In [ ]:
# ============================================================
# STEP 1 — CONFIGURATION, LOCKED LABELS, AND DATA INTEGRITY
# ============================================================
# Purpose:
# 1. Load only the final locked 2,227-household KMeans solution.
# 2. validate the official four-archetype distribution.
# 3. Audit monthly billing quality and duplicate household-months.
# 4. Quantify billing-match bias across household-months.
# 4. Quantify billing-match bias across archetypes.
# 5. Record input-file hashes for reproducibility.
#
# No model is trained in this step.
# ============================================================

from pathlib import Path
import hashlib
import json
import platform
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy.stats import chi2_contingency

# ------------------------------------------------------------
# 1. Reproducibility configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()

CLUSTER_FILE = PROJECT_ROOT / "clean_fm_clusters.pkl"
MONTHLY_FILE = PROJECT_ROOT / "monthly_consumption.csv"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "archetype_classifier_final_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ORDERED_CLUSTER_NAMES = [
    "Peak-Exposed",
    "Typical Household",
    "Heavy Continuous",
    "Daytime-Active"
]

EXPECTED_CLUSTER_COUNTS = {
    "Peak-Exposed": 145,
    "Typical Household": 1426,
    "Heavy Continuous": 515,
    "Daytime-Active": 141
}

EXPECTED_TOTAL_HOUSEHOLDS = 2227

FINAL_ARCHETYPE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]

KNOWN_CLUSTER_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_factor"
]

warnings.filterwarnings(
    "default"
)

# ------------------------------------------------------------
# 2. Utility functions
# ------------------------------------------------------------

def normalise_household_id(series):
    """Return consistently formatted household identifiers."""
    return (
        series
        .astype(str)
        .str.strip()
    )


def sha256_file(file_path, chunk_size=1024 * 1024):
    """Calculate a SHA-256 checksum without loading the whole file."""
    hash_object = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            hash_object.update(chunk)

    return hash_object.hexdigest()


def assert_required_columns(dataframe, required_columns, source_name):
    missing_columns = sorted(
        set(required_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{source_name} is missing required columns: "
            f"{missing_columns}"
        )


# ------------------------------------------------------------
# 3. Validate input-file availability
# ------------------------------------------------------------

missing_files = [
    str(file_path)
    for file_path in [CLUSTER_FILE, MONTHLY_FILE]
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required input files were not found:\n"
        + "\n".join(missing_files)
        + "\n\nPlace the files beside this notebook."
    )

input_hashes = {
    "clean_fm_clusters.pkl": sha256_file(
        CLUSTER_FILE
    ),
    "monthly_consumption.csv": sha256_file(
        MONTHLY_FILE
    )
}

with open(
    OUTPUT_DIR / "input_file_hashes.json",
    "w",
    encoding="utf-8"
) as hash_file:
    json.dump(
        input_hashes,
        hash_file,
        indent=4
    )

print("=" * 80)
print("INPUT FILES")
print("=" * 80)

for filename, checksum in input_hashes.items():
    print(f"{filename}: {checksum}")


# ------------------------------------------------------------
# 4. Load the locked KMeans output
# ------------------------------------------------------------

locked_raw = pd.read_pickle(
    CLUSTER_FILE
)

if not isinstance(locked_raw, pd.DataFrame):
    raise TypeError(
        "clean_fm_clusters.pkl must contain a pandas DataFrame."
    )

locked_raw = locked_raw.copy()

if "household_ID" in locked_raw.columns:
    locked = locked_raw.copy()
else:
    locked = locked_raw.reset_index()

    first_column = locked.columns[0]

    locked = locked.rename(
        columns={
            first_column: "household_ID"
        }
    )

assert_required_columns(
    locked,
    ["household_ID", "final_cluster"],
    "Locked clustering file"
)

locked["household_ID"] = normalise_household_id(
    locked["household_ID"]
)

# ------------------------------------------------------------
# 5. Strict household and label checks
# ------------------------------------------------------------

blank_household_ids = (
    locked["household_ID"]
    .isin(["", "nan", "None"])
    .sum()
)

if blank_household_ids > 0:
    raise ValueError(
        f"Found {blank_household_ids} blank household identifiers "
        "in the locked clustering file."
    )

duplicate_label_rows = (
    locked["household_ID"]
    .duplicated(keep=False)
)

if duplicate_label_rows.any():
    duplicate_ids = (
        locked.loc[
            duplicate_label_rows,
            "household_ID"
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "The locked clustering file must contain exactly one row "
        "per household. Duplicate household IDs found: "
        f"{duplicate_ids[:20]}"
    )

if locked["final_cluster"].isna().any():
    raise ValueError(
        "The locked clustering file contains missing final_cluster values."
    )

cluster_ids = sorted(
    locked["final_cluster"]
    .unique()
    .tolist(),
    key=lambda value: str(value)
)

if len(cluster_ids) != 4:
    raise ValueError(
        "Expected exactly four locked clusters, but found "
        f"{len(cluster_ids)}: {cluster_ids}"
    )


# ------------------------------------------------------------
# 6. Obtain the official archetype-name mapping
# ------------------------------------------------------------

def derive_archetype_mapping(cluster_dataframe):
    """
    Prefer trusted cluster_name values already saved by the main
    clustering pipeline.

    When cluster_name is unavailable, derive names from the locked
    cluster feature profiles and then validate the exact final counts.
    """

    if "cluster_name" in cluster_dataframe.columns:
        mapping_check = (
            cluster_dataframe[
                ["final_cluster", "cluster_name"]
            ]
            .dropna()
            .groupby("final_cluster")["cluster_name"]
            .nunique()
        )

        if (mapping_check > 1).any():
            raise ValueError(
                "At least one numeric cluster maps to multiple "
                "archetype names."
            )

        mapping = (
            cluster_dataframe[
                ["final_cluster", "cluster_name"]
            ]
            .dropna()
            .drop_duplicates()
            .set_index("final_cluster")["cluster_name"]
            .astype(str)
            .to_dict()
        )

        if (
            len(mapping) == 4
            and set(mapping.values())
            == set(ORDERED_CLUSTER_NAMES)
        ):
            return mapping, "trusted cluster_name column"

    missing_profile_columns = [
        column
        for column in KNOWN_CLUSTER_FEATURES
        if column not in cluster_dataframe.columns
    ]

    if missing_profile_columns:
        raise ValueError(
            "The locked file does not contain a valid cluster_name "
            "column, and the following profile columns required for "
            "safe fallback mapping are missing: "
            f"{missing_profile_columns}"
        )

    medians = (
        cluster_dataframe
        .groupby("final_cluster")[
            KNOWN_CLUSTER_FEATURES
        ]
        .median()
    )

    remaining_clusters = list(
        medians.index
    )

    mapping = {}

    heavy_cluster = (
        medians.loc[
            remaining_clusters,
            "load_factor"
        ]
        .idxmax()
    )

    mapping[heavy_cluster] = "Heavy Continuous"
    remaining_clusters.remove(heavy_cluster)

    peak_cluster = (
        medians.loc[
            remaining_clusters,
            "evening_peak_ratio"
        ]
        .idxmax()
    )

    mapping[peak_cluster] = "Peak-Exposed"
    remaining_clusters.remove(peak_cluster)

    daytime_cluster = (
        medians.loc[
            remaining_clusters,
            "night_baseload_ratio"
        ]
        .idxmin()
    )

    mapping[daytime_cluster] = "Daytime-Active"
    remaining_clusters.remove(daytime_cluster)

    if len(remaining_clusters) != 1:
        raise RuntimeError(
            "The remaining Typical Household cluster could not "
            "be uniquely identified."
        )

    mapping[
        remaining_clusters[0]
    ] = "Typical Household"

    return mapping, "locked feature-profile fallback"


cluster_name_map, mapping_source = (
    derive_archetype_mapping(
        locked
    )
)

locked["cluster_name"] = (
    locked["final_cluster"]
    .map(cluster_name_map)
)

if locked["cluster_name"].isna().any():
    raise ValueError(
        "Some locked cluster IDs could not be mapped to archetype names."
    )

if not set(
    locked["cluster_name"].unique()
).issubset(
    set(ORDERED_CLUSTER_NAMES)
):
    raise ValueError(
        "Unexpected archetype names were found after mapping."
    )

# ------------------------------------------------------------
# 7. Enforce the final 2,227-household endpoint
# ------------------------------------------------------------

locked_counts = (
    locked["cluster_name"]
    .value_counts()
    .reindex(
        ORDERED_CLUSTER_NAMES,
        fill_value=0
    )
    .astype(int)
)

expected_counts = pd.Series(
    EXPECTED_CLUSTER_COUNTS
).reindex(
    ORDERED_CLUSTER_NAMES
).astype(int)

if len(locked) != EXPECTED_TOTAL_HOUSEHOLDS:
    raise ValueError(
        "The classifier must use the final locked population of "
        f"{EXPECTED_TOTAL_HOUSEHOLDS:,} households. "
        f"The uploaded file contains {len(locked):,}."
    )

if not locked_counts.equals(
    expected_counts
):
    comparison = pd.DataFrame({
        "expected": expected_counts,
        "observed": locked_counts,
        "difference": (
            locked_counts
            - expected_counts
        )
    })

    raise ValueError(
        "The locked archetype distribution does not match the "
        "final accepted KMeans endpoint:\n"
        f"{comparison}"
    )

locked_labels = locked[
    [
        "household_ID",
        "final_cluster",
        "cluster_name"
    ]
].copy()

print("\n" + "=" * 80)
print("LOCKED CLUSTER LABEL AUDIT")
print("=" * 80)

print(
    "Mapping source:",
    mapping_source
)

print("\nNumeric cluster-to-name mapping:")

for cluster_id, cluster_name in sorted(
    cluster_name_map.items(),
    key=lambda item: str(item[0])
):
    print(
        f"{cluster_id!r} -> {cluster_name}"
    )

print("\nFinal locked distribution:")

display(
    locked_counts
    .rename("households")
    .to_frame()
)

print(
    f"\nPASS: exact final population confirmed: "
    f"{len(locked_labels):,} households."
)


# ------------------------------------------------------------
# 8. Load and audit monthly billing records
# ------------------------------------------------------------

monthly_raw = pd.read_csv(
    MONTHLY_FILE,
    low_memory=False
)

assert_required_columns(
    monthly_raw,
    [
        "household_ID",
        "month",
        "consumption"
    ],
    "Monthly billing file"
)

monthly = monthly_raw.copy()

monthly["household_ID"] = (
    normalise_household_id(
        monthly["household_ID"]
    )
)

monthly["month_original"] = (
    monthly["month"]
)

monthly["month"] = pd.to_datetime(
    monthly["month"],
    errors="coerce"
)

monthly["month"] = (
    monthly["month"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly["consumption"] = pd.to_numeric(
    monthly["consumption"]
    .astype(str)
    .str.replace(
        ",",
        "",
        regex=False
    ),
    errors="coerce"
)

invalid_household_ids = (
    monthly["household_ID"]
    .isin(["", "nan", "None"])
)

invalid_month_rows = (
    monthly["month"]
    .isna()
)

missing_consumption_rows = (
    monthly["consumption"]
    .isna()
)

negative_consumption_rows = (
    monthly["consumption"] < 0
)

duplicate_month_mask = (
    monthly
    .duplicated(
        subset=[
            "household_ID",
            "month"
        ],
        keep=False
    )
    & ~invalid_month_rows
    & ~invalid_household_ids
)

duplicate_household_month_rows = int(
    duplicate_month_mask.sum()
)

duplicate_household_month_pairs = int(
    monthly.loc[
        duplicate_month_mask,
        ["household_ID", "month"]
    ]
    .drop_duplicates()
    .shape[0]
)

monthly_audit = pd.DataFrame({
    "item": [
        "Raw rows",
        "Raw unique households",
        "Blank household-ID rows",
        "Invalid month rows",
        "Missing/non-numeric consumption rows",
        "Negative-consumption rows",
        "Duplicate household-month rows",
        "Duplicate household-month pairs",
        "Minimum valid month",
        "Maximum valid month"
    ],
    "value": [
        len(monthly_raw),
        monthly["household_ID"].nunique(),
        int(invalid_household_ids.sum()),
        int(invalid_month_rows.sum()),
        int(missing_consumption_rows.sum()),
        int(negative_consumption_rows.sum()),
        duplicate_household_month_rows,
        duplicate_household_month_pairs,
        monthly["month"].min(),
        monthly["month"].max()
    ]
})

print("\n" + "=" * 80)
print("MONTHLY BILLING DATA-QUALITY AUDIT")
print("=" * 80)

display(monthly_audit)

if invalid_household_ids.any():
    raise ValueError(
        "Blank household identifiers were found in monthly billing data."
    )

if invalid_month_rows.any():
    invalid_month_examples = (
        monthly.loc[
            invalid_month_rows,
            "month_original"
        ]
        .head(10)
        .tolist()
    )

    raise ValueError(
        "Invalid monthly dates were found. Examples: "
        f"{invalid_month_examples}"
    )

if negative_consumption_rows.any():
    negative_examples = (
        monthly.loc[
            negative_consumption_rows,
            [
                "household_ID",
                "month",
                "consumption"
            ]
        ]
        .head(10)
        .to_dict("records")
    )

    raise ValueError(
        "Negative monthly consumption values were found. "
        f"Examples: {negative_examples}"
    )

if duplicate_household_month_pairs > 0:
    duplicate_summary = (
        monthly.loc[
            duplicate_month_mask
        ]
        .groupby(
            [
                "household_ID",
                "month"
            ],
            as_index=False
        )
        .agg(
            row_count=(
                "consumption",
                "size"
            ),
            unique_consumption_values=(
                "consumption",
                "nunique"
            ),
            minimum_consumption=(
                "consumption",
                "min"
            ),
            maximum_consumption=(
                "consumption",
                "max"
            )
        )
    )

    duplicate_summary.to_csv(
        OUTPUT_DIR
        / "duplicate_household_month_audit.csv",
        index=False
    )

    raise ValueError(
        "Duplicate household-month records were found. "
        "They were not silently removed or aggregated. "
        "Review duplicate_household_month_audit.csv."
    )

print(
    "\nPASS: no duplicate household-month combinations were found."
)

# Missing consumption values are retained in the raw audit.
# They will be removed explicitly before feature engineering.

valid_monthly = monthly.loc[
    ~missing_consumption_rows,
    [
        "household_ID",
        "month",
        "consumption"
    ]
].copy()

valid_monthly = valid_monthly.sort_values(
    [
        "household_ID",
        "month"
    ]
).reset_index(drop=True)

coverage_by_household = (
    valid_monthly
    .groupby("household_ID")
    .agg(
        months_observed=(
            "month",
            "nunique"
        ),
        first_month=(
            "month",
            "min"
        ),
        last_month=(
            "month",
            "max"
        ),
        minimum_monthly_kwh=(
            "consumption",
            "min"
        ),
        maximum_monthly_kwh=(
            "consumption",
            "max"
        )
    )
    .reset_index()
)

coverage_summary = (
    coverage_by_household[
        "months_observed"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .rename("months_observed")
    .to_frame()
)

print("\nMonthly coverage distribution:")

display(coverage_summary)

print(
    "\nHouseholds with fewer than 3 valid months:",
    int(
        (
            coverage_by_household[
                "months_observed"
            ] < 3
        ).sum()
    )
)

print(
    "Households with fewer than 6 valid months:",
    int(
        (
            coverage_by_household[
                "months_observed"
            ] < 6
        ).sum()
    )
)


# ------------------------------------------------------------
# 9. Billing-match bias across locked archetypes
# ------------------------------------------------------------

billing_household_ids = set(
    valid_monthly["household_ID"]
    .unique()
)

match_audit = locked_labels.copy()

match_audit["has_valid_monthly_billing"] = (
    match_audit["household_ID"]
    .isin(
        billing_household_ids
    )
)

match_by_cluster = (
    match_audit
    .groupby(
        "cluster_name",
        observed=False
    )
    .agg(
        locked_households=(
            "household_ID",
            "size"
        ),
        matched_households=(
            "has_valid_monthly_billing",
            "sum"
        )
    )
    .reindex(
        ORDERED_CLUSTER_NAMES
    )
)

match_by_cluster["unmatched_households"] = (
    match_by_cluster[
        "locked_households"
    ]
    - match_by_cluster[
        "matched_households"
    ]
)

match_by_cluster["match_rate_percent"] = (
    match_by_cluster[
        "matched_households"
    ]
    / match_by_cluster[
        "locked_households"
    ]
    * 100
)

contingency_table = (
    pd.crosstab(
        match_audit["cluster_name"],
        match_audit[
            "has_valid_monthly_billing"
        ]
    )
    .reindex(
        index=ORDERED_CLUSTER_NAMES,
        columns=[False, True],
        fill_value=0
    )
)

chi_square, chi_p_value, degrees_freedom, expected = (
    chi2_contingency(
        contingency_table
    )
)

number_of_households = (
    contingency_table
    .to_numpy()
    .sum()
)

minimum_dimension = min(
    contingency_table.shape[0] - 1,
    contingency_table.shape[1] - 1
)

cramers_v = np.sqrt(
    chi_square
    / (
        number_of_households
        * minimum_dimension
    )
)

match_bias_statistics = pd.DataFrame({
    "statistic": [
        "Chi-square",
        "Degrees of freedom",
        "p-value",
        "Cramer's V",
        "Overall match rate percent"
    ],
    "value": [
        chi_square,
        degrees_freedom,
        chi_p_value,
        cramers_v,
        (
            match_audit[
                "has_valid_monthly_billing"
            ].mean()
            * 100
        )
    ]
})

print("\n" + "=" * 80)
print("BILLING-MATCH BIAS AUDIT")
print("=" * 80)

display(
    match_by_cluster.round(4)
)

display(
    match_bias_statistics
)

print(
    "\nMatched households:",
    int(
        match_audit[
            "has_valid_monthly_billing"
        ].sum()
    )
)

print(
    "Unmatched locked households:",
    int(
        (
            ~match_audit[
                "has_valid_monthly_billing"
            ]
        ).sum()
    )
)

if chi_p_value < 0.05:
    print(
        "\nInterpretation: billing availability differs significantly "
        "across archetypes. Classifier results therefore apply to the "
        "billing-matched subset and are not automatically representative "
        f"of all {len(locked_labels):,} clustered households."
    )
else:
    print(
        "\nInterpretation: no statistically significant cluster-dependent "
        "billing-match difference was detected."
    )


# ------------------------------------------------------------
# 10. Save Step 1 audit outputs
# ------------------------------------------------------------

locked_labels.to_csv(
    OUTPUT_DIR
    / "locked_archetype_labels_2227.csv",
    index=False
)

monthly_audit.to_csv(
    OUTPUT_DIR
    / "monthly_billing_data_quality_audit.csv",
    index=False
)

coverage_by_household.to_csv(
    OUTPUT_DIR
    / "monthly_billing_household_coverage.csv",
    index=False
)

coverage_summary.to_csv(
    OUTPUT_DIR
    / "monthly_billing_coverage_summary.csv"
)

match_audit.to_csv(
    OUTPUT_DIR
    / "billing_match_household_status.csv",
    index=False
)

match_by_cluster.to_csv(
    OUTPUT_DIR
    / "billing_match_rates_by_archetype.csv"
)

contingency_table.to_csv(
    OUTPUT_DIR
    / "billing_match_contingency_table.csv"
)

match_bias_statistics.to_csv(
    OUTPUT_DIR
    / "billing_match_bias_statistics.csv",
    index=False
)

environment_metadata = {
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
    "scikit_learn_version": sklearn.__version__,
    "random_state": RANDOM_STATE,
    "locked_households": int(
        len(locked_labels)
    ),
    "cluster_counts": {
        key: int(value)
        for key, value in (
            locked_counts.to_dict()
        ).items()
    },
    "input_hashes": input_hashes
}

with open(
    OUTPUT_DIR
    / "step1_environment_and_input_metadata.json",
    "w",
    encoding="utf-8"
) as metadata_file:
    json.dump(
        environment_metadata,
        metadata_file,
        indent=4
    )

print("\n" + "=" * 80)
print("STEP 1 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Locked labels:",
    len(locked_labels)
)

print(
    "Valid monthly billing rows:",
    len(valid_monthly)
)

print(
    "Valid monthly billing households:",
    valid_monthly[
        "household_ID"
    ].nunique()
)

print(
    "Billing-matched locked households:",
    int(
        match_audit[
            "has_valid_monthly_billing"
        ].sum()
    )
)

print(
    "\nSaved audit outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 2 — LEAKAGE-SAFE MONTHLY BILLING FEATURE ENGINEERING
# ============================================================
# Purpose:
# 1. Create one auditable billing-feature row per household.
# 2. Build two clearly separated datasets:
#    a) contemporaneous transfer using all available billing;
#    b) temporal sensitivity using billing before 2024-05-01.
# 3. Preserve a compact core feature set and an extended set.
# 4. Audit duplicates, coverage, missingness and leakage.
#
# No model is trained in this step.
# ============================================================

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Confirm that Step 1 was run
# ------------------------------------------------------------

REQUIRED_STEP1_OBJECTS = [
    "valid_monthly",
    "locked_labels",
    "ORDERED_CLUSTER_NAMES",
    "OUTPUT_DIR"
]

missing_step1_objects = [
    name
    for name in REQUIRED_STEP1_OBJECTS
    if name not in globals()
]

if missing_step1_objects:
    raise RuntimeError(
        "Run Step 1 first. Missing objects: "
        f"{missing_step1_objects}"
    )


GOLDEN_START = pd.Timestamp("2024-05-01")


# ------------------------------------------------------------
# 2. Define the feature sets
# ------------------------------------------------------------

CORE_FEATURES = [
    "billing_median_kwh",
    "billing_mean_kwh",
    "billing_std_kwh",
    "billing_iqr_kwh",
    "billing_cv",
    "billing_p10_kwh",
    "billing_p90_kwh",
    "billing_robust_range_kwh",
    "recent_3_month_median_kwh",
    "recent_6_month_median_kwh",
    "recent_vs_overall_log_ratio",
    "early_6_month_median_kwh",
    "late_6_month_median_kwh",
    "late_vs_early_log_ratio",
    "monthly_trend_slope_log1p",
    "median_abs_log_change",
    "near_zero_month_rate",
    "above_120_month_rate"
]

EXTENDED_ONLY_FEATURES = [
    "billing_min_kwh",
    "billing_max_kwh",
    "billing_range_kwh",
    "billing_log_std",
    "mean_abs_monthly_change_kwh",
    "maximum_monthly_increase_kwh",
    "maximum_monthly_decrease_kwh",
    "low_30_month_rate",
    "above_90_month_rate",
    "above_180_month_rate",
    "above_240_month_rate",
    "median_kwh_calendar_q1",
    "median_kwh_calendar_q2",
    "median_kwh_calendar_q3",
    "median_kwh_calendar_q4"
]

EXTENDED_FEATURES = (
    CORE_FEATURES
    + EXTENDED_ONLY_FEATURES
)

AUDIT_ONLY_COLUMNS = [
    "months_observed",
    "first_month",
    "last_month",
    "observation_span_months"
]


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def calculate_month_span(start_series, end_series):
    """
    Calculate the inclusive calendar-month span.
    """

    return (
        (end_series.dt.year - start_series.dt.year) * 12
        + (end_series.dt.month - start_series.dt.month)
        + 1
    )


def calculate_household_trend(group):
    """
    Estimate the linear trend in log1p monthly consumption per
    calendar month.

    Actual month positions are used, so missing calendar months
    are not treated as directly adjacent observations.
    """

    group = group.sort_values("month")

    if len(group) < 3:
        return np.nan

    calendar_month_index = (
        group["month"].dt.year.to_numpy() * 12
        + group["month"].dt.month.to_numpy()
    ).astype(float)

    calendar_month_index = (
        calendar_month_index
        - calendar_month_index.min()
    )

    if np.unique(calendar_month_index).size < 3:
        return np.nan

    log_consumption = np.log1p(
        group["consumption"].to_numpy(dtype=float)
    )

    return float(
        np.polyfit(
            calendar_month_index,
            log_consumption,
            1
        )[0]
    )


def build_household_billing_features(
    monthly_input,
    mode_name,
    cutoff=None
):
    """
    Create one billing-feature row per household.

    Parameters
    ----------
    monthly_input:
        Valid monthly billing rows produced by Step 1.

    mode_name:
        Name used in audits and errors.

    cutoff:
        When supplied, only months strictly before the cutoff
        are used.
    """

    df = monthly_input.copy()

    if cutoff is not None:
        df = df[
            df["month"] < cutoff
        ].copy()

    if df.empty:
        raise RuntimeError(
            f"No monthly rows remain for {mode_name}."
        )

    df = (
        df.sort_values(
            ["household_ID", "month"]
        )
        .reset_index(drop=True)
    )

    duplicate_pairs = int(
        df.duplicated(
            subset=[
                "household_ID",
                "month"
            ]
        ).sum()
    )

    if duplicate_pairs > 0:
        raise ValueError(
            f"{mode_name}: duplicate household-month "
            "records were detected."
        )

    df["log_consumption"] = np.log1p(
        df["consumption"]
    )

    grouped = df.groupby(
        "household_ID",
        sort=False
    )

    # --------------------------------------------------------
    # Distribution and coverage features
    # --------------------------------------------------------

    features = grouped.agg(
        billing_median_kwh=(
            "consumption",
            "median"
        ),
        billing_mean_kwh=(
            "consumption",
            "mean"
        ),
        billing_std_kwh=(
            "consumption",
            "std"
        ),
        billing_min_kwh=(
            "consumption",
            "min"
        ),
        billing_max_kwh=(
            "consumption",
            "max"
        ),
        billing_p10_kwh=(
            "consumption",
            lambda values: values.quantile(0.10)
        ),
        billing_p25_kwh=(
            "consumption",
            lambda values: values.quantile(0.25)
        ),
        billing_p75_kwh=(
            "consumption",
            lambda values: values.quantile(0.75)
        ),
        billing_p90_kwh=(
            "consumption",
            lambda values: values.quantile(0.90)
        ),
        billing_log_std=(
            "log_consumption",
            "std"
        ),
        months_observed=(
            "month",
            "nunique"
        ),
        first_month=(
            "month",
            "min"
        ),
        last_month=(
            "month",
            "max"
        )
    ).reset_index()

    features["billing_iqr_kwh"] = (
        features["billing_p75_kwh"]
        - features["billing_p25_kwh"]
    )

    features["billing_robust_range_kwh"] = (
        features["billing_p90_kwh"]
        - features["billing_p10_kwh"]
    )

    features["billing_range_kwh"] = (
        features["billing_max_kwh"]
        - features["billing_min_kwh"]
    )

    features["billing_cv"] = np.where(
        features["billing_mean_kwh"] > 0,
        (
            features["billing_std_kwh"]
            / features["billing_mean_kwh"]
        ),
        np.nan
    )

    features["observation_span_months"] = (
        calculate_month_span(
            features["first_month"],
            features["last_month"]
        )
    )

    # --------------------------------------------------------
    # Fixed target-independent consumption-rate features
    # --------------------------------------------------------

    rate_source = df.assign(
        near_zero_month=(
            df["consumption"] < 10
        ).astype(float),

        low_30_month=(
            df["consumption"] <= 30
        ).astype(float),

        above_90_month=(
            df["consumption"] > 90
        ).astype(float),

        above_120_month=(
            df["consumption"] > 120
        ).astype(float),

        above_180_month=(
            df["consumption"] > 180
        ).astype(float),

        above_240_month=(
            df["consumption"] > 240
        ).astype(float)
    )

    rate_features = (
        rate_source
        .groupby("household_ID")
        .agg(
            near_zero_month_rate=(
                "near_zero_month",
                "mean"
            ),
            low_30_month_rate=(
                "low_30_month",
                "mean"
            ),
            above_90_month_rate=(
                "above_90_month",
                "mean"
            ),
            above_120_month_rate=(
                "above_120_month",
                "mean"
            ),
            above_180_month_rate=(
                "above_180_month",
                "mean"
            ),
            above_240_month_rate=(
                "above_240_month",
                "mean"
            )
        )
        .reset_index()
    )

    features = features.merge(
        rate_features,
        on="household_ID",
        how="left",
        validate="one_to_one"
    )

    # --------------------------------------------------------
    # Early and recent billing windows
    # --------------------------------------------------------

    early_6 = (
        df.groupby(
            "household_ID",
            group_keys=False
        )
        .head(6)
        .groupby("household_ID")[
            "consumption"
        ]
        .median()
        .rename(
            "early_6_month_median_kwh"
        )
        .reset_index()
    )

    late_6 = (
        df.groupby(
            "household_ID",
            group_keys=False
        )
        .tail(6)
        .groupby("household_ID")[
            "consumption"
        ]
        .median()
        .rename(
            "late_6_month_median_kwh"
        )
        .reset_index()
    )

    recent_3 = (
        df.groupby(
            "household_ID",
            group_keys=False
        )
        .tail(3)
        .groupby("household_ID")[
            "consumption"
        ]
        .median()
        .rename(
            "recent_3_month_median_kwh"
        )
        .reset_index()
    )

    recent_6 = late_6.rename(
        columns={
            "late_6_month_median_kwh":
            "recent_6_month_median_kwh"
        }
    )

    for window_table in [
        early_6,
        late_6,
        recent_3,
        recent_6
    ]:
        features = features.merge(
            window_table,
            on="household_ID",
            how="left",
            validate="one_to_one"
        )

    # Log-ratio differences are safer than percentage changes
    # when consumption is close to zero.

    features["recent_vs_overall_log_ratio"] = (
        np.log1p(
            features[
                "recent_3_month_median_kwh"
            ]
        )
        - np.log1p(
            features["billing_median_kwh"]
        )
    )

    features["late_vs_early_log_ratio"] = (
        np.log1p(
            features[
                "late_6_month_median_kwh"
            ]
        )
        - np.log1p(
            features[
                "early_6_month_median_kwh"
            ]
        )
    )

    # --------------------------------------------------------
    # Month-to-month change features
    # --------------------------------------------------------

    # Retain differences only for genuinely consecutive calendar
    # months. This prevents a January-to-March change from being
    # treated as a one-month change when February is missing.
    df["calendar_month_number"] = (
        df["month"].dt.year * 12
        + df["month"].dt.month
    )

    df["month_gap"] = (
        df.groupby("household_ID")[
            "calendar_month_number"
        ]
        .diff()
    )

    raw_monthly_change = (
        df.groupby("household_ID")[
            "consumption"
        ]
        .diff()
    )

    raw_log_change = (
        df.groupby("household_ID")[
            "log_consumption"
        ]
        .diff()
    )

    df["monthly_change_kwh"] = (
        raw_monthly_change.where(
            df["month_gap"].eq(1)
        )
    )

    df["log_change"] = (
        raw_log_change.where(
            df["month_gap"].eq(1)
        )
    )

    dynamic_features = (
        df.groupby("household_ID")
        .agg(
            mean_abs_monthly_change_kwh=(
                "monthly_change_kwh",
                lambda values: (
                    values.abs().mean()
                )
            ),

            maximum_monthly_increase_kwh=(
                "monthly_change_kwh",
                lambda values: (
                    max(
                        float(
                            values.max(
                                skipna=True
                            )
                        ),
                        0.0
                    )
                    if values.notna().any()
                    else np.nan
                )
            ),

            maximum_monthly_decrease_kwh=(
                "monthly_change_kwh",
                lambda values: (
                    max(
                        -float(
                            values.min(
                                skipna=True
                            )
                        ),
                        0.0
                    )
                    if values.notna().any()
                    else np.nan
                )
            ),

            median_abs_log_change=(
                "log_change",
                lambda values: (
                    values.abs().median()
                )
            )
        )
        .reset_index()
    )

    features = features.merge(
        dynamic_features,
        on="household_ID",
        how="left",
        validate="one_to_one"
    )

    # --------------------------------------------------------
    # Calendar-quarter medians
    # --------------------------------------------------------

    df["calendar_quarter"] = (
        (df["month"].dt.month - 1) // 3
        + 1
    )

    quarter_features = (
        df.pivot_table(
            index="household_ID",
            columns="calendar_quarter",
            values="consumption",
            aggfunc="median"
        )
        .rename(
            columns={
                1: "median_kwh_calendar_q1",
                2: "median_kwh_calendar_q2",
                3: "median_kwh_calendar_q3",
                4: "median_kwh_calendar_q4"
            }
        )
        .reset_index()
    )

    quarter_columns = [
        "median_kwh_calendar_q1",
        "median_kwh_calendar_q2",
        "median_kwh_calendar_q3",
        "median_kwh_calendar_q4"
    ]

    for quarter_column in quarter_columns:
        if quarter_column not in quarter_features.columns:
            quarter_features[
                quarter_column
            ] = np.nan

    features = features.merge(
        quarter_features[
            ["household_ID"]
            + quarter_columns
        ],
        on="household_ID",
        how="left",
        validate="one_to_one"
    )

    # --------------------------------------------------------
    # Calendar-aware household trend
    # --------------------------------------------------------

    trend_records = []

    for household_id, household_group in (
        df.groupby("household_ID")
    ):
        trend_records.append({
            "household_ID": household_id,
            "monthly_trend_slope_log1p":
                calculate_household_trend(
                    household_group
                )
        })

    trend_features = pd.DataFrame(
        trend_records
    )

    features = features.merge(
        trend_features,
        on="household_ID",
        how="left",
        validate="one_to_one"
    )

    features = features.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if features[
        "household_ID"
    ].duplicated().any():
        raise RuntimeError(
            f"{mode_name}: duplicate household "
            "feature rows were produced."
        )

    expected_columns = (
        ["household_ID"]
        + CORE_FEATURES
        + EXTENDED_ONLY_FEATURES
        + AUDIT_ONLY_COLUMNS
    )

    missing_expected_columns = [
        column
        for column in expected_columns
        if column not in features.columns
    ]

    if missing_expected_columns:
        raise RuntimeError(
            f"{mode_name}: missing expected columns: "
            f"{missing_expected_columns}"
        )

    return (
        features[
            expected_columns
        ].copy(),
        df
    )


# ------------------------------------------------------------
# 4. Build the two feature datasets
# ------------------------------------------------------------

billing_features_all, monthly_rows_all = (
    build_household_billing_features(
        valid_monthly,
        mode_name=(
            "all_available_contemporaneous"
        ),
        cutoff=None
    )
)

billing_features_pre, monthly_rows_pre = (
    build_household_billing_features(
        valid_monthly,
        mode_name=(
            "pre_golden_sensitivity"
        ),
        cutoff=GOLDEN_START
    )
)


# ------------------------------------------------------------
# 5. Merge with the locked archetype labels
# ------------------------------------------------------------

def create_labelled_dataset(
    feature_table,
    dataset_name
):
    if feature_table[
        "household_ID"
    ].duplicated().any():
        raise ValueError(
            f"{dataset_name}: duplicate feature "
            "households detected."
        )

    labelled = feature_table.merge(
        locked_labels[
            [
                "household_ID",
                "final_cluster",
                "cluster_name"
            ]
        ],
        on="household_ID",
        how="inner",
        validate="one_to_one"
    )

    if labelled[
        "household_ID"
    ].duplicated().any():
        raise RuntimeError(
            f"{dataset_name}: duplicate labelled "
            "rows were produced."
        )

    labelled["cluster_name"] = (
        pd.Categorical(
            labelled["cluster_name"],
            categories=ORDERED_CLUSTER_NAMES,
            ordered=True
        )
    )

    return (
        labelled
        .sort_values("household_ID")
        .reset_index(drop=True)
    )


dataset_all = create_labelled_dataset(
    billing_features_all,
    "all_available_contemporaneous"
)

dataset_pre = create_labelled_dataset(
    billing_features_pre,
    "pre_golden_sensitivity"
)


# ------------------------------------------------------------
# 6. Leakage audit
# ------------------------------------------------------------

FORBIDDEN_PREDICTOR_COLUMNS = {
    "final_cluster",
    "cluster_name",
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
}

leakage_hits = sorted(
    FORBIDDEN_PREDICTOR_COLUMNS.intersection(
        set(EXTENDED_FEATURES)
    )
)

if leakage_hits:
    raise RuntimeError(
        "Direct target or smart-meter feature "
        f"leakage detected: {leakage_hits}"
    )


# ------------------------------------------------------------
# 7. Dataset audit helper
# ------------------------------------------------------------

def create_feature_audit(
    labelled,
    monthly_rows,
    dataset_name
):
    target_counts = (
        labelled["cluster_name"]
        .value_counts(sort=False)
        .reindex(
            ORDERED_CLUSTER_NAMES,
            fill_value=0
        )
        .astype(int)
    )

    coverage = labelled[
        "months_observed"
    ]

    audit = pd.DataFrame({
        "item": [
            "Dataset",
            "Monthly rows used",
            "Monthly households used",
            "Minimum month used",
            "Maximum month used",
            "Labelled households",
            "Core predictor count",
            "Extended predictor count",
            "Minimum months per labelled household",
            "Median months per labelled household",
            "Maximum months per labelled household",
            "Households with fewer than 6 months",
            "Households with fewer than 12 months",
            "Duplicate labelled households",
            "Completely missing core predictors",
            "Completely missing extended predictors",
            "Direct leakage columns"
        ],

        "value": [
            dataset_name,
            len(monthly_rows),
            monthly_rows[
                "household_ID"
            ].nunique(),
            monthly_rows["month"].min(),
            monthly_rows["month"].max(),
            len(labelled),
            len(CORE_FEATURES),
            len(EXTENDED_FEATURES),
            int(coverage.min()),
            float(coverage.median()),
            int(coverage.max()),
            int((coverage < 6).sum()),
            int((coverage < 12).sum()),
            int(
                labelled[
                    "household_ID"
                ].duplicated().sum()
            ),
            int(
                labelled[
                    CORE_FEATURES
                ].isna().all().sum()
            ),
            int(
                labelled[
                    EXTENDED_FEATURES
                ].isna().all().sum()
            ),
            len(leakage_hits)
        ]
    })

    missingness = pd.DataFrame({
        "feature": EXTENDED_FEATURES,

        "missing_count": [
            int(
                labelled[
                    feature
                ].isna().sum()
            )
            for feature in EXTENDED_FEATURES
        ],

        "missing_percent": [
            float(
                labelled[
                    feature
                ].isna().mean()
                * 100
            )
            for feature in EXTENDED_FEATURES
        ]
    }).sort_values(
        [
            "missing_percent",
            "feature"
        ],
        ascending=[
            False,
            True
        ]
    )

    return (
        audit,
        target_counts,
        missingness
    )


audit_all, counts_all, missingness_all = (
    create_feature_audit(
        dataset_all,
        monthly_rows_all,
        "all_available_contemporaneous"
    )
)

audit_pre, counts_pre, missingness_pre = (
    create_feature_audit(
        dataset_pre,
        monthly_rows_pre,
        "pre_golden_sensitivity"
    )
)


# ------------------------------------------------------------
# 8. Compare populations across the two feature modes
# ------------------------------------------------------------

all_available_ids = set(
    dataset_all["household_ID"]
)

pre_golden_ids = set(
    dataset_pre["household_ID"]
)

common_households = sorted(
    all_available_ids.intersection(
        pre_golden_ids
    )
)

comparison_population_audit = (
    pd.DataFrame({
        "item": [
            "All-available labelled households",
            "Pre-golden labelled households",
            "Common households",
            "All-available-only households",
            "Pre-golden-only households"
        ],

        "value": [
            len(dataset_all),
            len(dataset_pre),
            len(common_households),
            len(
                all_available_ids
                - set(common_households)
            ),
            len(
                pre_golden_ids
                - set(common_households)
            )
        ]
    })
)


# ------------------------------------------------------------
# 9. Display the results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 2 — FEATURE ENGINEERING AUDIT")
print("=" * 80)

print(
    "\nAll-available contemporaneous dataset:"
)
display(audit_all)

print(
    "\nAll-available target distribution:"
)
display(
    counts_all
    .rename("households")
    .to_frame()
)

print(
    "\nAll-available predictor missingness:"
)
display(
    missingness_all
    .head(20)
    .round(4)
)

print(
    "\nPre-golden sensitivity dataset:"
)
display(audit_pre)

print(
    "\nPre-golden target distribution:"
)
display(
    counts_pre
    .rename("households")
    .to_frame()
)

print(
    "\nPre-golden predictor missingness:"
)
display(
    missingness_pre
    .head(20)
    .round(4)
)

print(
    "\nPopulation overlap:"
)
display(
    comparison_population_audit
)

print("\nCore predictors:")
print(CORE_FEATURES)

print("\nExtended-only predictors:")
print(EXTENDED_ONLY_FEATURES)

print(
    "\nPASS: no target or direct smart-meter "
    "feature leakage was detected."
)


# ------------------------------------------------------------
# 10. Save the datasets and audits
# ------------------------------------------------------------

billing_features_all.to_csv(
    OUTPUT_DIR
    / "billing_features_all_available.csv",
    index=False
)

billing_features_pre.to_csv(
    OUTPUT_DIR
    / "billing_features_pre_golden.csv",
    index=False
)

dataset_all.to_csv(
    OUTPUT_DIR
    / "classifier_dataset_all_available.csv",
    index=False
)

dataset_pre.to_csv(
    OUTPUT_DIR
    / "classifier_dataset_pre_golden.csv",
    index=False
)

audit_all.to_csv(
    OUTPUT_DIR
    / "feature_audit_all_available.csv",
    index=False
)

audit_pre.to_csv(
    OUTPUT_DIR
    / "feature_audit_pre_golden.csv",
    index=False
)

missingness_all.to_csv(
    OUTPUT_DIR
    / "feature_missingness_all_available.csv",
    index=False
)

missingness_pre.to_csv(
    OUTPUT_DIR
    / "feature_missingness_pre_golden.csv",
    index=False
)

comparison_population_audit.to_csv(
    OUTPUT_DIR
    / "feature_mode_population_overlap.csv",
    index=False
)

feature_schema = {
    "core_features": CORE_FEATURES,

    "extended_only_features":
        EXTENDED_ONLY_FEATURES,

    "extended_features":
        EXTENDED_FEATURES,

    "audit_only_columns":
        AUDIT_ONLY_COLUMNS,

    "all_available_use_case": (
        "Contemporaneous monthly-billing "
        "archetype transfer; not future prediction."
    ),

    "pre_golden_use_case": (
        "Temporal sensitivity using billing "
        "records strictly before 2024-05-01."
    )
}

with open(
    OUTPUT_DIR
    / "billing_feature_schema.json",
    "w",
    encoding="utf-8"
) as schema_file:
    json.dump(
        feature_schema,
        schema_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 2 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "All-available labelled households:",
    len(dataset_all)
)

print(
    "Pre-golden labelled households:",
    len(dataset_pre)
)

print(
    "Common households:",
    len(common_households)
)

print(
    "Core predictors:",
    len(CORE_FEATURES)
)

print(
    "Extended predictors:",
    len(EXTENDED_FEATURES)
)

print(
    "\nSaved outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 3 — MODELLING READINESS, IMBALANCE, AND BASELINE AUDIT
# ============================================================
# Purpose:
# 1. Lock the target encoding and candidate feature sets.
# 2. Quantify class imbalance.
# 3. Establish majority and stratified-random baselines.
# 4. Audit constant, near-constant and highly correlated features.
# 5. Verify that 5x4 nested stratified cross-validation is
#    feasible for the smallest Daytime-Active class.
#
# No substantive classifier is selected in this step.
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)


# ------------------------------------------------------------
# 1. Confirm that Steps 1 and 2 were run
# ------------------------------------------------------------

REQUIRED_OBJECTS = [
    "dataset_all",
    "dataset_pre",
    "CORE_FEATURES",
    "EXTENDED_FEATURES",
    "ORDERED_CLUSTER_NAMES",
    "OUTPUT_DIR",
    "RANDOM_STATE"
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Steps 1 and 2 first. Missing objects: "
        f"{missing_objects}"
    )


# ------------------------------------------------------------
# 2. Lock the modelling configuration
# ------------------------------------------------------------

OUTER_FOLDS = 5
INNER_FOLDS = 4

PRIMARY_SELECTION_METRIC = "macro_f1"
SECONDARY_SELECTION_METRIC = "balanced_accuracy"

CLASS_TO_INDEX = {
    class_name: class_index
    for class_index, class_name
    in enumerate(ORDERED_CLUSTER_NAMES)
}

INDEX_TO_CLASS = {
    class_index: class_name
    for class_name, class_index
    in CLASS_TO_INDEX.items()
}

CANDIDATE_FEATURE_SETS = {
    "Core": list(CORE_FEATURES),
    "Extended": list(EXTENDED_FEATURES)
}

PRIMARY_IMBALANCE_APPROACHES = [
    "No class weighting",
    "Class weighting"
]

SENSITIVITY_IMBALANCE_APPROACHES = [
    "Random oversampling",
    "SMOTE"
]

# Random oversampling and SMOTE will be assessed later using
# training folds only. They are not applied in this audit.


# ------------------------------------------------------------
# 3. Prepare the primary contemporaneous dataset
# ------------------------------------------------------------

modelling_df = dataset_all.copy()

if modelling_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households were detected in dataset_all."
    )

if modelling_df["cluster_name"].isna().any():
    raise ValueError(
        "Missing archetype labels were detected."
    )

unexpected_classes = sorted(
    set(
        modelling_df["cluster_name"]
        .astype(str)
        .unique()
    )
    - set(ORDERED_CLUSTER_NAMES)
)

if unexpected_classes:
    raise ValueError(
        "Unexpected archetype labels were found: "
        f"{unexpected_classes}"
    )

y = (
    modelling_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
)

if y.isna().any():
    raise ValueError(
        "One or more archetype labels could not be encoded."
    )

y = y.astype(int)

X_core = modelling_df[
    CANDIDATE_FEATURE_SETS["Core"]
].copy()

X_extended = modelling_df[
    CANDIDATE_FEATURE_SETS["Extended"]
].copy()

for matrix_name, matrix in {
    "Core": X_core,
    "Extended": X_extended
}.items():

    for column in matrix.columns:
        matrix[column] = pd.to_numeric(
            matrix[column],
            errors="coerce"
        )

    matrix.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    if matrix.isna().all(axis=1).any():
        raise ValueError(
            f"{matrix_name}: at least one household has "
            "all predictor values missing."
        )

    if matrix.columns.duplicated().any():
        duplicate_columns = (
            matrix.columns[
                matrix.columns.duplicated()
            ]
            .tolist()
        )

        raise ValueError(
            f"{matrix_name}: duplicate feature names found: "
            f"{duplicate_columns}"
        )


# ------------------------------------------------------------
# 4. Class-imbalance audit
# ------------------------------------------------------------

class_counts = (
    modelling_df["cluster_name"]
    .astype(str)
    .value_counts()
    .reindex(
        ORDERED_CLUSTER_NAMES,
        fill_value=0
    )
    .astype(int)
)

class_percentages = (
    class_counts
    / class_counts.sum()
    * 100
)

largest_class_count = int(
    class_counts.max()
)

class_imbalance_audit = pd.DataFrame({
    "class_name": ORDERED_CLUSTER_NAMES,
    "class_index": [
        CLASS_TO_INDEX[name]
        for name in ORDERED_CLUSTER_NAMES
    ],
    "households": [
        int(class_counts[name])
        for name in ORDERED_CLUSTER_NAMES
    ],
    "percentage": [
        float(class_percentages[name])
        for name in ORDERED_CLUSTER_NAMES
    ],
    "largest_to_class_ratio": [
        (
            largest_class_count
            / class_counts[name]
            if class_counts[name] > 0
            else np.nan
        )
        for name in ORDERED_CLUSTER_NAMES
    ]
})

minimum_class_name = class_counts.idxmin()
minimum_class_count = int(
    class_counts.min()
)

print("=" * 80)
print("CLASS-IMBALANCE AUDIT")
print("=" * 80)

display(
    class_imbalance_audit.round(4)
)

print(
    "\nSmallest class:",
    minimum_class_name
)

print(
    "Smallest class count:",
    minimum_class_count
)

print(
    "Largest-to-smallest imbalance ratio:",
    round(
        largest_class_count
        / minimum_class_count,
        4
    )
)


# ------------------------------------------------------------
# 5. Feature-quality audit
# ------------------------------------------------------------

def audit_feature_set(
    feature_matrix,
    feature_set_name
):
    records = []

    for feature in feature_matrix.columns:
        values = feature_matrix[feature]

        value_frequencies = (
            values
            .value_counts(
                dropna=False,
                normalize=True
            )
        )

        maximum_frequency = (
            float(value_frequencies.iloc[0])
            if len(value_frequencies) > 0
            else np.nan
        )

        records.append({
            "feature_set": feature_set_name,
            "feature": feature,
            "non_missing_count": int(
                values.notna().sum()
            ),
            "missing_count": int(
                values.isna().sum()
            ),
            "missing_percent": float(
                values.isna().mean() * 100
            ),
            "unique_values": int(
                values.nunique(dropna=True)
            ),
            "standard_deviation": float(
                values.std(skipna=True)
            ),
            "maximum_single_value_rate": (
                maximum_frequency
            ),
            "constant_feature": bool(
                values.nunique(dropna=True) <= 1
            ),
            "near_constant_feature": bool(
                maximum_frequency >= 0.99
            )
        })

    return pd.DataFrame(records)


core_feature_audit = audit_feature_set(
    X_core,
    "Core"
)

extended_feature_audit = audit_feature_set(
    X_extended,
    "Extended"
)

feature_quality_audit = pd.concat(
    [
        core_feature_audit,
        extended_feature_audit
    ],
    ignore_index=True
)

constant_features = (
    feature_quality_audit.loc[
        feature_quality_audit[
            "constant_feature"
        ],
        [
            "feature_set",
            "feature"
        ]
    ]
    .drop_duplicates()
)

near_constant_features = (
    feature_quality_audit.loc[
        feature_quality_audit[
            "near_constant_feature"
        ],
        [
            "feature_set",
            "feature",
            "maximum_single_value_rate"
        ]
    ]
    .drop_duplicates()
)


# ------------------------------------------------------------
# 6. Feature-redundancy audit
# ------------------------------------------------------------

def create_high_correlation_table(
    feature_matrix,
    feature_set_name,
    threshold=0.95
):
    correlation_matrix = (
        feature_matrix
        .corr(method="spearman")
    )

    records = []

    columns = correlation_matrix.columns.tolist()

    for first_index in range(len(columns)):
        for second_index in range(
            first_index + 1,
            len(columns)
        ):
            first_feature = columns[first_index]
            second_feature = columns[second_index]

            correlation_value = (
                correlation_matrix.loc[
                    first_feature,
                    second_feature
                ]
            )

            if (
                pd.notna(correlation_value)
                and abs(correlation_value) >= threshold
            ):
                records.append({
                    "feature_set": feature_set_name,
                    "feature_1": first_feature,
                    "feature_2": second_feature,
                    "spearman_correlation": float(
                        correlation_value
                    ),
                    "absolute_correlation": float(
                        abs(correlation_value)
                    )
                })

    return (
        pd.DataFrame(records)
        .sort_values(
            "absolute_correlation",
            ascending=False
        )
        .reset_index(drop=True)
        if records
        else pd.DataFrame(
            columns=[
                "feature_set",
                "feature_1",
                "feature_2",
                "spearman_correlation",
                "absolute_correlation"
            ]
        )
    )


core_high_correlations = (
    create_high_correlation_table(
        X_core,
        "Core",
        threshold=0.95
    )
)

extended_high_correlations = (
    create_high_correlation_table(
        X_extended,
        "Extended",
        threshold=0.95
    )
)

high_correlation_pairs = pd.concat(
    [
        core_high_correlations,
        extended_high_correlations
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 7. Validate nested stratified CV feasibility
# ------------------------------------------------------------

outer_cv_audit = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_feasibility_records = []

for outer_fold, (
    outer_train_indices,
    outer_test_indices
) in enumerate(
    outer_cv_audit.split(X_core, y),
    start=1
):
    y_outer_train = y.iloc[
        outer_train_indices
    ]

    y_outer_test = y.iloc[
        outer_test_indices
    ]

    train_counts = (
        y_outer_train
        .value_counts()
        .reindex(
            range(len(ORDERED_CLUSTER_NAMES)),
            fill_value=0
        )
        .astype(int)
    )

    test_counts = (
        y_outer_test
        .value_counts()
        .reindex(
            range(len(ORDERED_CLUSTER_NAMES)),
            fill_value=0
        )
        .astype(int)
    )

    if (test_counts == 0).any():
        raise RuntimeError(
            f"Outer fold {outer_fold} does not contain "
            "all four archetypes."
        )

    inner_cv_audit = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_STATE + outer_fold
        )
    )

    minimum_inner_train_count = np.inf
    minimum_inner_validation_count = np.inf

    for (
        inner_train_indices,
        inner_validation_indices
    ) in inner_cv_audit.split(
        X_core.iloc[outer_train_indices],
        y_outer_train
    ):
        y_inner_train = (
            y_outer_train.iloc[
                inner_train_indices
            ]
        )

        y_inner_validation = (
            y_outer_train.iloc[
                inner_validation_indices
            ]
        )

        inner_train_counts = (
            y_inner_train
            .value_counts()
            .reindex(
                range(
                    len(ORDERED_CLUSTER_NAMES)
                ),
                fill_value=0
            )
        )

        inner_validation_counts = (
            y_inner_validation
            .value_counts()
            .reindex(
                range(
                    len(ORDERED_CLUSTER_NAMES)
                ),
                fill_value=0
            )
        )

        minimum_inner_train_count = min(
            minimum_inner_train_count,
            int(
                inner_train_counts.min()
            )
        )

        minimum_inner_validation_count = min(
            minimum_inner_validation_count,
            int(
                inner_validation_counts.min()
            )
        )

    cv_record = {
        "outer_fold": outer_fold,
        "outer_train_rows": len(
            outer_train_indices
        ),
        "outer_test_rows": len(
            outer_test_indices
        ),
        "minimum_outer_train_class_count": int(
            train_counts.min()
        ),
        "minimum_outer_test_class_count": int(
            test_counts.min()
        ),
        "minimum_inner_train_class_count": int(
            minimum_inner_train_count
        ),
        "minimum_inner_validation_class_count": int(
            minimum_inner_validation_count
        )
    }

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        safe_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        cv_record[
            f"outer_train_{safe_name}"
        ] = int(
            train_counts[class_index]
        )

        cv_record[
            f"outer_test_{safe_name}"
        ] = int(
            test_counts[class_index]
        )

    cv_feasibility_records.append(
        cv_record
    )


cv_feasibility_audit = pd.DataFrame(
    cv_feasibility_records
)

if (
    cv_feasibility_audit[
        "minimum_inner_validation_class_count"
    ].min() < 1
):
    raise RuntimeError(
        "The planned inner cross-validation creates "
        "a validation fold without all classes."
    )


# ------------------------------------------------------------
# 8. Establish leakage-safe dummy baselines
# ------------------------------------------------------------

baseline_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

baseline_models = {
    "Majority-class baseline": (
        DummyClassifier(
            strategy="most_frequent"
        )
    ),

    "Stratified-random baseline": (
        DummyClassifier(
            strategy="stratified",
            random_state=RANDOM_STATE
        )
    )
}

baseline_summary_records = []
baseline_class_records = []
baseline_prediction_records = []

for baseline_name, baseline_model in (
    baseline_models.items()
):
    baseline_predictions = cross_val_predict(
        baseline_model,
        X_core,
        y,
        cv=baseline_cv,
        method="predict",
        n_jobs=1
    )

    baseline_accuracy = accuracy_score(
        y,
        baseline_predictions
    )

    baseline_balanced_accuracy = (
        balanced_accuracy_score(
            y,
            baseline_predictions
        )
    )

    baseline_macro_f1 = f1_score(
        y,
        baseline_predictions,
        average="macro",
        zero_division=0
    )

    baseline_weighted_f1 = f1_score(
        y,
        baseline_predictions,
        average="weighted",
        zero_division=0
    )

    baseline_summary_records.append({
        "baseline": baseline_name,
        "accuracy": baseline_accuracy,
        "balanced_accuracy": (
            baseline_balanced_accuracy
        ),
        "macro_f1": baseline_macro_f1,
        "weighted_f1": baseline_weighted_f1
    })

    report = classification_report(
        y,
        baseline_predictions,
        labels=list(
            range(
                len(ORDERED_CLUSTER_NAMES)
            )
        ),
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in ORDERED_CLUSTER_NAMES:
        baseline_class_records.append({
            "baseline": baseline_name,
            "class_name": class_name,
            "precision": report[
                class_name
            ]["precision"],
            "recall": report[
                class_name
            ]["recall"],
            "f1_score": report[
                class_name
            ]["f1-score"],
            "support": int(
                report[
                    class_name
                ]["support"]
            )
        })

    for row_index in range(len(modelling_df)):
        baseline_prediction_records.append({
            "baseline": baseline_name,
            "household_ID": (
                modelling_df.iloc[
                    row_index
                ]["household_ID"]
            ),
            "true_class_index": int(
                y.iloc[row_index]
            ),
            "true_class_name": (
                INDEX_TO_CLASS[
                    int(y.iloc[row_index])
                ]
            ),
            "predicted_class_index": int(
                baseline_predictions[
                    row_index
                ]
            ),
            "predicted_class_name": (
                INDEX_TO_CLASS[
                    int(
                        baseline_predictions[
                            row_index
                        ]
                    )
                ]
            )
        })


baseline_summary = pd.DataFrame(
    baseline_summary_records
)

baseline_class_performance = pd.DataFrame(
    baseline_class_records
)

baseline_predictions_df = pd.DataFrame(
    baseline_prediction_records
)


# ------------------------------------------------------------
# 9. Display the audit
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FEATURE-QUALITY AUDIT")
print("=" * 80)

print("\nConstant predictors:")
display(constant_features)

print("\nNear-constant predictors:")
display(
    near_constant_features.round(4)
)

print(
    "\nHighly correlated feature pairs "
    "(absolute Spearman correlation >= 0.95):"
)

display(
    high_correlation_pairs.head(30).round(4)
)

print(
    "\nNumber of highly correlated pairs — Core:",
    len(core_high_correlations)
)

print(
    "Number of highly correlated pairs — Extended:",
    len(extended_high_correlations)
)


print("\n" + "=" * 80)
print("NESTED CV FEASIBILITY AUDIT")
print("=" * 80)

display(
    cv_feasibility_audit
)

print(
    "\nPASS: the planned 5-fold outer and "
    "4-fold inner stratified design contains "
    "all four classes in every evaluated fold."
)


print("\n" + "=" * 80)
print("DUMMY BASELINE PERFORMANCE")
print("=" * 80)

display(
    baseline_summary.round(4)
)

print("\nClass-specific baseline performance:")

display(
    baseline_class_performance.round(4)
)


# ------------------------------------------------------------
# 10. Save Step 3 outputs
# ------------------------------------------------------------

class_imbalance_audit.to_csv(
    OUTPUT_DIR
    / "class_imbalance_audit.csv",
    index=False
)

feature_quality_audit.to_csv(
    OUTPUT_DIR
    / "feature_quality_audit.csv",
    index=False
)

high_correlation_pairs.to_csv(
    OUTPUT_DIR
    / "high_correlation_feature_pairs.csv",
    index=False
)

cv_feasibility_audit.to_csv(
    OUTPUT_DIR
    / "nested_cv_feasibility_audit.csv",
    index=False
)

baseline_summary.to_csv(
    OUTPUT_DIR
    / "dummy_baseline_summary.csv",
    index=False
)

baseline_class_performance.to_csv(
    OUTPUT_DIR
    / "dummy_baseline_class_performance.csv",
    index=False
)

baseline_predictions_df.to_csv(
    OUTPUT_DIR
    / "dummy_baseline_predictions.csv",
    index=False
)

modelling_configuration = {
    "primary_dataset": (
        "all_available_contemporaneous"
    ),
    "primary_use_case": (
        "Contemporaneous transfer of locked "
        "smart-meter archetypes using monthly "
        "billing history; not future prediction."
    ),
    "class_order": ORDERED_CLUSTER_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "primary_selection_metric": (
        PRIMARY_SELECTION_METRIC
    ),
    "secondary_selection_metric": (
        SECONDARY_SELECTION_METRIC
    ),
    "candidate_feature_sets": (
        CANDIDATE_FEATURE_SETS
    ),
    "primary_imbalance_approaches": (
        PRIMARY_IMBALANCE_APPROACHES
    ),
    "sensitivity_imbalance_approaches": (
        SENSITIVITY_IMBALANCE_APPROACHES
    ),
    "ordinary_accuracy_is_primary": False,
    "top_two_accuracy_is_primary": False,
    "smote_is_primary": False,
    "random_state": RANDOM_STATE
}

with open(
    OUTPUT_DIR
    / "modelling_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        modelling_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Primary selection metric:",
    PRIMARY_SELECTION_METRIC
)

print(
    "Secondary selection metric:",
    SECONDARY_SELECTION_METRIC
)

print(
    "Outer folds:",
    OUTER_FOLDS
)

print(
    "Inner folds:",
    INNER_FOLDS
)

print(
    "Core predictors:",
    len(CORE_FEATURES)
)

print(
    "Extended predictors:",
    len(EXTENDED_FEATURES)
)

print(
    "\nSaved Step 3 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 4 — PRIMARY LEAKAGE-SAFE NESTED MODEL COMPARISON
# ============================================================
# Purpose:
# 1. Remove the exact duplicate six-month feature.
# 2. Compare Core and Extended monthly-billing feature sets.
# 3. Compare Logistic Regression, Random Forest and Extra Trees.
# 4. Compare unweighted and balanced class-weighted models.
# 5. Select configurations using inner-fold macro-F1.
# 6. Evaluate only on untouched outer folds.
# 7. Preserve one out-of-fold probability vector per household.
#
# Important:
# - Accuracy is supplementary only.
# - Macro-F1 is the primary model-selection metric.
# - Balanced accuracy is the first tie-breaker.
# - No oversampling is applied in this step.
# - Random oversampling and SMOTE are tested in Step 5.
# ============================================================

import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    top_k_accuracy_score
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    RobustScaler
)
from sklearn.metrics import make_scorer


# ------------------------------------------------------------
# 1. Confirm previous steps were completed
# ------------------------------------------------------------

REQUIRED_OBJECTS = [
    "dataset_all",
    "CORE_FEATURES",
    "EXTENDED_FEATURES",
    "ORDERED_CLUSTER_NAMES",
    "CLASS_TO_INDEX",
    "INDEX_TO_CLASS",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "baseline_summary"
]

missing_objects = [
    object_name
    for object_name in REQUIRED_OBJECTS
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Steps 1–3 first. Missing objects: "
        f"{missing_objects}"
    )


# ------------------------------------------------------------
# 2. Remove the deterministically duplicated feature
# ------------------------------------------------------------

DUPLICATED_FEATURE_TO_REMOVE = (
    "late_6_month_median_kwh"
)

DUPLICATED_FEATURE_TO_RETAIN = (
    "recent_6_month_median_kwh"
)

duplicate_values_match = np.allclose(
    dataset_all[
        DUPLICATED_FEATURE_TO_REMOVE
    ].to_numpy(dtype=float),
    dataset_all[
        DUPLICATED_FEATURE_TO_RETAIN
    ].to_numpy(dtype=float),
    equal_nan=True
)

if not duplicate_values_match:
    raise RuntimeError(
        "The two six-month variables are no longer identical. "
        "Review their definitions before removing either feature."
    )

MODEL_CORE_FEATURES = [
    feature
    for feature in CORE_FEATURES
    if feature != DUPLICATED_FEATURE_TO_REMOVE
]

MODEL_EXTENDED_FEATURES = [
    feature
    for feature in EXTENDED_FEATURES
    if feature != DUPLICATED_FEATURE_TO_REMOVE
]

MODEL_FEATURE_SETS = {
    "Core reduced": MODEL_CORE_FEATURES,
    "Extended reduced": MODEL_EXTENDED_FEATURES
}

if len(MODEL_CORE_FEATURES) != 17:
    raise RuntimeError(
        "Expected 17 corrected core predictors, but found "
        f"{len(MODEL_CORE_FEATURES)}."
    )

if len(MODEL_EXTENDED_FEATURES) != 32:
    raise RuntimeError(
        "Expected 32 corrected extended predictors, but found "
        f"{len(MODEL_EXTENDED_FEATURES)}."
    )

print("=" * 80)
print("CORRECTED MODELLING FEATURE SETS")
print("=" * 80)

print(
    "Removed exact duplicate:",
    DUPLICATED_FEATURE_TO_REMOVE
)

print(
    "Retained equivalent feature:",
    DUPLICATED_FEATURE_TO_RETAIN
)

print(
    "Corrected core predictors:",
    len(MODEL_CORE_FEATURES)
)

print(
    "Corrected extended predictors:",
    len(MODEL_EXTENDED_FEATURES)
)


# ------------------------------------------------------------
# 3. Prepare the target and predictor source
# ------------------------------------------------------------

nested_df = (
    dataset_all
    .copy()
    .reset_index(drop=True)
)

if nested_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households were detected before modelling."
    )

y_nested = (
    nested_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
    .astype(int)
)

all_model_features = sorted(
    set(MODEL_EXTENDED_FEATURES)
)

X_nested_source = (
    nested_df[
        all_model_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

if X_nested_source.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all predictors missing."
    )

FIXED_CLASS_INDICES = list(
    range(len(ORDERED_CLUSTER_NAMES))
)


# ------------------------------------------------------------
# 4. Scoring definitions
# ------------------------------------------------------------

SCORING = {
    "macro_f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    ),

    "balanced_accuracy": (
        "balanced_accuracy"
    ),

    "weighted_f1": make_scorer(
        f1_score,
        average="weighted",
        zero_division=0
    ),

    "accuracy": "accuracy"
}


# ------------------------------------------------------------
# 5. Candidate-pipeline builder
# ------------------------------------------------------------

def build_candidate_pipeline(
    model_family,
    class_weight
):
    """
    Construct model-specific preprocessing.

    Logistic Regression:
    - median imputation
    - RobustScaler
    - L2-regularised Logistic Regression

    Random Forest and Extra Trees:
    - median imputation
    - no scaling, because tree split ordering is invariant to
      monotonic feature rescaling
    """

    if model_family == "Logistic Regression":

        estimator = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            max_iter=5000,
            class_weight=class_weight,
            random_state=RANDOM_STATE
        )

        pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                RobustScaler()
            ),
            (
                "model",
                estimator
            )
        ])

        parameter_grid = {
            "model__C": [
                0.001,
                0.01,
                0.1,
                1.0,
                10.0
            ]
        }

    elif model_family == "Random Forest":

        estimator = RandomForestClassifier(
            n_estimators=400,
            bootstrap=True,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=1
        )

        pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                estimator
            )
        ])

        # Curated regularised configurations. A list of dictionaries
        # avoids an unnecessarily large Cartesian grid while testing
        # stronger depth, leaf-size and split-size regularisation.
        parameter_grid = [
            # Strongly regularised candidate.
            {
                "model__max_depth": [3],
                "model__min_samples_leaf": [12],
                "model__min_samples_split": [24],
                "model__max_features": ["sqrt"],
                "model__max_samples": [0.70],
                "model__ccp_alpha": [0.001]
            },
            # Conservative shallow candidate.
            {
                "model__max_depth": [4],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.001]
            },
            # Moderate feature subsampling.
            {
                "model__max_depth": [4],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": [0.30],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.0]
            },
            # Balanced middle-complexity candidate.
            {
                "model__max_depth": [5],
                "model__min_samples_leaf": [8],
                "model__min_samples_split": [16],
                "model__max_features": [0.30],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.0]
            },
            # Moderate pruning candidate.
            {
                "model__max_depth": [5],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": [0.50],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.001]
            },
            # Retain the previous best-performing configuration.
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [5],
                "model__min_samples_split": [10],
                "model__max_features": [0.50],
                "model__max_samples": [None],
                "model__ccp_alpha": [0.0]
            },
            # Stronger leaf and bootstrap regularisation at depth 6.
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.001]
            },
            # One moderately deeper candidate, still regularised.
            {
                "model__max_depth": [8],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": [0.30],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.001]
            }
        ]

    elif model_family == "Extra Trees":

        estimator = ExtraTreesClassifier(
            n_estimators=400,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=1
        )

        pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                estimator
            )
        ])

        # Curated regularised configurations. A list of dictionaries
        # avoids an unnecessarily large Cartesian grid while testing
        # stronger depth, leaf-size and split-size regularisation.
        parameter_grid = [
            {
                "model__max_depth": [3],
                "model__min_samples_leaf": [12],
                "model__min_samples_split": [24],
                "model__max_features": ["sqrt"],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [4],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [4],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": [0.30],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [5],
                "model__min_samples_leaf": [8],
                "model__min_samples_split": [16],
                "model__max_features": [0.30],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [5],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": [0.50],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [5],
                "model__min_samples_split": [10],
                "model__max_features": [0.50],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [8],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": [0.30],
                "model__ccp_alpha": [0.001]
            }
        ]

    else:
        raise ValueError(
            f"Unknown model family: {model_family}"
        )

    return pipeline, parameter_grid


# ------------------------------------------------------------
# 5A. Balanced inner-CV selection rule
# ------------------------------------------------------------

INNER_MACRO_F1_PRACTICAL_MARGIN = 0.01
INNER_BALANCED_ACCURACY_PRACTICAL_MARGIN = 0.01


def _max_features_complexity(value):
    """Map max_features to an approximate lower-is-simpler rank."""

    if value is None:
        return 1.0

    if isinstance(value, str):
        if value == "sqrt":
            return 0.25
        if value == "log2":
            return 0.20
        return 1.0

    return float(value)


def _regularisation_rank(parameters, model_family):
    """
    Deterministic complexity rank used only after validation
    performance is practically equivalent.
    """

    if model_family == "Logistic Regression":
        return (
            float(parameters.get("model__C", 1.0)),
        )

    depth = parameters.get("model__max_depth")
    depth_rank = 999 if depth is None else int(depth)

    leaf_size = int(
        parameters.get("model__min_samples_leaf", 1)
    )

    split_size = int(
        parameters.get("model__min_samples_split", 2)
    )

    feature_rank = _max_features_complexity(
        parameters.get("model__max_features")
    )

    sample_fraction = parameters.get(
        "model__max_samples",
        None
    )

    sample_rank = (
        1.0
        if sample_fraction is None
        else float(sample_fraction)
    )

    pruning_strength = float(
        parameters.get("model__ccp_alpha", 0.0)
    )

    return (
        depth_rank,
        -leaf_size,
        -split_size,
        feature_rank,
        sample_rank,
        -pruning_strength
    )


def select_balanced_cv_candidate(
    search_results,
    model_family
):
    """
    Select a defensible candidate from inner/repeated CV results.

    Rule:
    1. Retain candidates within 0.01 macro-F1 of the best.
    2. Within that set, retain candidates within 0.01 balanced
       accuracy of the best remaining candidate.
    3. Prefer the smallest train-validation macro-F1 gap.
    4. Use stronger regularisation only as a deterministic final
       tie-breaker.

    This controls overfitting without accepting a clearly weaker
    validation model merely because its training score is lower.
    """

    results = search_results.copy()

    required_columns = {
        "params",
        "mean_test_macro_f1",
        "mean_test_balanced_accuracy",
        "mean_train_macro_f1"
    }

    missing_columns = sorted(
        required_columns - set(results.columns)
    )

    if missing_columns:
        raise ValueError(
            "CV results are missing required columns: "
            f"{missing_columns}"
        )

    results["train_validation_macro_f1_gap"] = (
        results["mean_train_macro_f1"]
        - results["mean_test_macro_f1"]
    )

    best_macro_f1 = float(
        results["mean_test_macro_f1"].max()
    )

    shortlist = results[
        results["mean_test_macro_f1"]
        >= (
            best_macro_f1
            - INNER_MACRO_F1_PRACTICAL_MARGIN
        )
    ].copy()

    best_balanced_accuracy = float(
        shortlist[
            "mean_test_balanced_accuracy"
        ].max()
    )

    shortlist = shortlist[
        shortlist["mean_test_balanced_accuracy"]
        >= (
            best_balanced_accuracy
            - INNER_BALANCED_ACCURACY_PRACTICAL_MARGIN
        )
    ].copy()

    shortlist["regularisation_rank"] = (
        shortlist["params"].map(
            lambda parameters: _regularisation_rank(
                parameters,
                model_family
            )
        )
    )

    shortlist = (
        shortlist
        .sort_values(
            [
                "train_validation_macro_f1_gap",
                "mean_test_macro_f1",
                "mean_test_balanced_accuracy",
                "regularisation_rank"
            ],
            ascending=[
                True,
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

    selected = shortlist.iloc[0].copy()
    selected["selection_shortlist_size"] = int(
        len(shortlist)
    )

    return selected


# ------------------------------------------------------------
# 6. Define primary candidate strategies
# ------------------------------------------------------------

PRIMARY_CANDIDATE_SPECIFICATIONS = []

for model_family in [
    "Logistic Regression",
    "Random Forest",
    "Extra Trees"
]:
    for weighting_name, class_weight in [
        (
            "No class weighting",
            None
        ),
        (
            "Balanced class weighting",
            "balanced"
        )
    ]:
        PRIMARY_CANDIDATE_SPECIFICATIONS.append({
            "model_family": model_family,
            "imbalance_strategy": weighting_name,
            "class_weight": class_weight
        })

print("\nCandidate model strategies:")

display(
    pd.DataFrame(
        PRIMARY_CANDIDATE_SPECIFICATIONS
    )
)


# ------------------------------------------------------------
# 7. Metric utilities
# ------------------------------------------------------------

def align_class_probabilities(
    raw_probabilities,
    fitted_classes
):
    """
    Align estimator probability columns to the fixed class order
    0, 1, 2, 3.
    """

    aligned_probabilities = np.zeros(
        (
            raw_probabilities.shape[0],
            len(FIXED_CLASS_INDICES)
        ),
        dtype=float
    )

    for source_column, class_index in enumerate(
        fitted_classes
    ):
        aligned_probabilities[
            :,
            int(class_index)
        ] = raw_probabilities[
            :,
            source_column
        ]

    row_sums = aligned_probabilities.sum(
        axis=1
    )

    if not np.allclose(
        row_sums,
        1.0,
        atol=1e-8
    ):
        raise RuntimeError(
            "Aligned probability rows do not sum to one."
        )

    return aligned_probabilities


def multiclass_brier_score(
    y_true,
    probabilities
):
    """
    Multiclass Brier score using the summed squared error
    across all four probability columns.
    """

    one_hot_targets = np.eye(
        len(FIXED_CLASS_INDICES)
    )[np.asarray(y_true, dtype=int)]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot_targets
                ) ** 2,
                axis=1
            )
        )
    )


def calculate_multiclass_metrics(
    y_true,
    predicted_classes,
    probabilities
):
    return {
        "accuracy": accuracy_score(
            y_true,
            predicted_classes
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predicted_classes
            )
        ),

        "macro_f1": f1_score(
            y_true,
            predicted_classes,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            predicted_classes,
            average="weighted",
            zero_division=0
        ),

        "top_2_accuracy": (
            top_k_accuracy_score(
                y_true,
                probabilities,
                k=2,
                labels=FIXED_CLASS_INDICES
            )
        ),

        "multiclass_log_loss": log_loss(
            y_true,
            probabilities,
            labels=FIXED_CLASS_INDICES
        ),

        "multiclass_brier_sum": (
            multiclass_brier_score(
                y_true,
                probabilities
            )
        )
    }


def serialise_parameters(parameter_dictionary):
    """
    Convert NumPy scalar values to ordinary Python values before
    writing JSON text.
    """

    serialisable = {}

    for key, value in parameter_dictionary.items():

        if isinstance(
            value,
            (
                np.integer,
                np.floating,
                np.bool_
            )
        ):
            value = value.item()

        serialisable[key] = value

    return json.dumps(
        serialisable,
        sort_keys=True
    )


# ------------------------------------------------------------
# 8. Nested cross-validation
# ------------------------------------------------------------

outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_candidate_records = []
outer_selection_records = []
outer_metric_records = []
outer_class_metric_records = []
outer_confusion_records = []
oof_prediction_records = []

nested_start_time = time.time()

for outer_fold, (
    outer_train_indices,
    outer_test_indices
) in enumerate(
    outer_cv.split(
        X_nested_source,
        y_nested
    ),
    start=1
):
    print("\n" + "=" * 80)
    print(
        f"PRIMARY NESTED CV — OUTER FOLD "
        f"{outer_fold}/{OUTER_FOLDS}"
    )
    print("=" * 80)

    y_outer_train = (
        y_nested.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_test = (
        y_nested.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_train_source = (
        X_nested_source.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_source = (
        X_nested_source.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_metadata = (
        nested_df.iloc[
            outer_test_indices
        ][
            [
                "household_ID",
                "cluster_name"
            ]
        ]
        .reset_index(drop=True)
    )

    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_STATE + outer_fold
        )
    )

    fold_best_candidates = []

    for feature_set_name, feature_columns in (
        MODEL_FEATURE_SETS.items()
    ):
        X_outer_train = (
            outer_train_source[
                feature_columns
            ]
        )

        for candidate_specification in (
            PRIMARY_CANDIDATE_SPECIFICATIONS
        ):
            model_family = (
                candidate_specification[
                    "model_family"
                ]
            )

            imbalance_strategy = (
                candidate_specification[
                    "imbalance_strategy"
                ]
            )

            class_weight = (
                candidate_specification[
                    "class_weight"
                ]
            )

            pipeline, parameter_grid = (
                build_candidate_pipeline(
                    model_family,
                    class_weight
                )
            )

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=parameter_grid,
                scoring=SCORING,
                refit=False,
                cv=inner_cv,
                n_jobs=-1,
                pre_dispatch="2*n_jobs",
                return_train_score=True,
                error_score="raise"
            )

            candidate_start_time = time.time()

            search.fit(
                X_outer_train,
                y_outer_train
            )

            candidate_runtime_seconds = (
                time.time()
                - candidate_start_time
            )

            candidate_results = pd.DataFrame(
                search.cv_results_
            )

            candidate_results = (
                candidate_results
                .sort_values(
                    [
                        "mean_test_macro_f1",
                        "mean_test_balanced_accuracy",
                        "mean_test_weighted_f1",
                        "mean_test_accuracy"
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                        False
                    ]
                )
                .reset_index(drop=True)
            )

            best_result = select_balanced_cv_candidate(
                candidate_results,
                model_family
            )

            best_parameters = (
                best_result["params"]
            )

            candidate_record = {
                "outer_fold": outer_fold,
                "feature_set": feature_set_name,
                "feature_count": len(
                    feature_columns
                ),
                "model_family": model_family,
                "imbalance_strategy": (
                    imbalance_strategy
                ),
                "class_weight": (
                    str(class_weight)
                ),
                "selected_parameters": (
                    serialise_parameters(
                        best_parameters
                    )
                ),
                "mean_inner_macro_f1": float(
                    best_result[
                        "mean_test_macro_f1"
                    ]
                ),
                "std_inner_macro_f1": float(
                    best_result[
                        "std_test_macro_f1"
                    ]
                ),
                "mean_inner_balanced_accuracy": float(
                    best_result[
                        "mean_test_balanced_accuracy"
                    ]
                ),
                "mean_inner_weighted_f1": float(
                    best_result[
                        "mean_test_weighted_f1"
                    ]
                ),
                "mean_inner_accuracy": float(
                    best_result[
                        "mean_test_accuracy"
                    ]
                ),
                "mean_inner_train_macro_f1": float(
                    best_result[
                        "mean_train_macro_f1"
                    ]
                ),
                "inner_macro_f1_gap": float(
                    best_result[
                        "mean_train_macro_f1"
                    ]
                    - best_result[
                        "mean_test_macro_f1"
                    ]
                ),
                "search_runtime_seconds": float(
                    candidate_runtime_seconds
                )
            }

            inner_candidate_records.append(
                candidate_record
            )

            fold_best_candidates.append({
                **candidate_record,
                "feature_columns": list(
                    feature_columns
                ),
                "pipeline_template": pipeline,
                "best_parameters_dict": dict(
                    best_parameters
                )
            })

            print(
                f"{feature_set_name} | "
                f"{model_family} | "
                f"{imbalance_strategy}: "
                f"inner macro-F1="
                f"{candidate_record['mean_inner_macro_f1']:.4f}, "
                f"balanced accuracy="
                f"{candidate_record['mean_inner_balanced_accuracy']:.4f}"
            )

    # --------------------------------------------------------
    # Select complete workflow using inner validation only
    # --------------------------------------------------------

    fold_best_candidates = sorted(
        fold_best_candidates,
        key=lambda record: (
            record["mean_inner_macro_f1"],
            record[
                "mean_inner_balanced_accuracy"
            ],
            record["mean_inner_weighted_f1"],
            record["mean_inner_accuracy"]
        ),
        reverse=True
    )

    selected_candidate = (
        fold_best_candidates[0]
    )

    selected_pipeline = clone(
        selected_candidate[
            "pipeline_template"
        ]
    )

    selected_pipeline.set_params(
        **selected_candidate[
            "best_parameters_dict"
        ]
    )

    selected_features = (
        selected_candidate[
            "feature_columns"
        ]
    )

    selected_pipeline.fit(
        outer_train_source[
            selected_features
        ],
        y_outer_train
    )

    outer_predicted_classes = (
        selected_pipeline.predict(
            outer_test_source[
                selected_features
            ]
        )
    )

    raw_outer_probabilities = (
        selected_pipeline.predict_proba(
            outer_test_source[
                selected_features
            ]
        )
    )

    outer_probabilities = (
        align_class_probabilities(
            raw_outer_probabilities,
            selected_pipeline.classes_
        )
    )

    outer_metrics = (
        calculate_multiclass_metrics(
            y_outer_test,
            outer_predicted_classes,
            outer_probabilities
        )
    )

    outer_selection_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_candidate[
                "feature_set"
            ]
        ),
        "selected_feature_count": len(
            selected_features
        ),
        "selected_model_family": (
            selected_candidate[
                "model_family"
            ]
        ),
        "selected_imbalance_strategy": (
            selected_candidate[
                "imbalance_strategy"
            ]
        ),
        "selected_class_weight": (
            selected_candidate[
                "class_weight"
            ]
        ),
        "selected_parameters": (
            selected_candidate[
                "selected_parameters"
            ]
        ),
        "inner_macro_f1": (
            selected_candidate[
                "mean_inner_macro_f1"
            ]
        ),
        "inner_balanced_accuracy": (
            selected_candidate[
                "mean_inner_balanced_accuracy"
            ]
        )
    })

    outer_metric_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_candidate[
                "feature_set"
            ]
        ),
        "selected_model_family": (
            selected_candidate[
                "model_family"
            ]
        ),
        "selected_imbalance_strategy": (
            selected_candidate[
                "imbalance_strategy"
            ]
        ),
        **outer_metrics
    })

    class_report = classification_report(
        y_outer_test,
        outer_predicted_classes,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in ORDERED_CLUSTER_NAMES:
        outer_class_metric_records.append({
            "outer_fold": outer_fold,
            "class_name": class_name,
            "precision": class_report[
                class_name
            ]["precision"],
            "recall": class_report[
                class_name
            ]["recall"],
            "f1_score": class_report[
                class_name
            ]["f1-score"],
            "support": int(
                class_report[
                    class_name
                ]["support"]
            )
        })

    fold_confusion_matrix = confusion_matrix(
        y_outer_test,
        outer_predicted_classes,
        labels=FIXED_CLASS_INDICES
    )

    for true_index, true_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        for predicted_index, predicted_name in enumerate(
            ORDERED_CLUSTER_NAMES
        ):
            outer_confusion_records.append({
                "outer_fold": outer_fold,
                "true_class": true_name,
                "predicted_class": predicted_name,
                "count": int(
                    fold_confusion_matrix[
                        true_index,
                        predicted_index
                    ]
                )
            })

    sorted_probability_indices = np.argsort(
        outer_probabilities,
        axis=1
    )[:, ::-1]

    for row_position in range(
        len(outer_test_indices)
    ):
        predicted_index = int(
            outer_predicted_classes[
                row_position
            ]
        )

        highest_index = int(
            sorted_probability_indices[
                row_position,
                0
            ]
        )

        second_index = int(
            sorted_probability_indices[
                row_position,
                1
            ]
        )

        probability_margin = float(
            outer_probabilities[
                row_position,
                highest_index
            ]
            - outer_probabilities[
                row_position,
                second_index
            ]
        )

        prediction_record = {
            "household_ID": (
                outer_test_metadata.iloc[
                    row_position
                ]["household_ID"]
            ),
            "outer_fold": outer_fold,
            "true_class_index": int(
                y_outer_test.iloc[
                    row_position
                ]
            ),
            "true_class_name": (
                outer_test_metadata.iloc[
                    row_position
                ]["cluster_name"]
            ),
            "predicted_class_index": (
                predicted_index
            ),
            "predicted_class_name": (
                INDEX_TO_CLASS[
                    predicted_index
                ]
            ),
            "highest_probability": float(
                outer_probabilities[
                    row_position,
                    highest_index
                ]
            ),
            "second_highest_probability": float(
                outer_probabilities[
                    row_position,
                    second_index
                ]
            ),
            "probability_margin": (
                probability_margin
            ),
            "top_1_class_name": (
                INDEX_TO_CLASS[
                    highest_index
                ]
            ),
            "top_2_class_name": (
                INDEX_TO_CLASS[
                    second_index
                ]
            )
        }

        for class_index, class_name in enumerate(
            ORDERED_CLUSTER_NAMES
        ):
            safe_class_name = (
                class_name
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
                .replace("/", "_")
            )

            prediction_record[
                f"probability_{safe_class_name}"
            ] = float(
                outer_probabilities[
                    row_position,
                    class_index
                ]
            )

        oof_prediction_records.append(
            prediction_record
        )

    print("\nSelected complete workflow:")

    print(
        "Feature set:",
        selected_candidate[
            "feature_set"
        ]
    )

    print(
        "Model:",
        selected_candidate[
            "model_family"
        ]
    )

    print(
        "Imbalance strategy:",
        selected_candidate[
            "imbalance_strategy"
        ]
    )

    print(
        "Outer macro-F1:",
        round(
            outer_metrics["macro_f1"],
            4
        )
    )

    print(
        "Outer balanced accuracy:",
        round(
            outer_metrics[
                "balanced_accuracy"
            ],
            4
        )
    )


# ------------------------------------------------------------
# 9. Compile nested-CV outputs
# ------------------------------------------------------------

inner_candidate_results_df = pd.DataFrame(
    inner_candidate_records
)

outer_selection_df = pd.DataFrame(
    outer_selection_records
)

outer_metrics_df = pd.DataFrame(
    outer_metric_records
)

outer_class_metrics_df = pd.DataFrame(
    outer_class_metric_records
)

outer_confusion_df = pd.DataFrame(
    outer_confusion_records
)

oof_predictions_df = pd.DataFrame(
    oof_prediction_records
)

nested_runtime_seconds = (
    time.time()
    - nested_start_time
)

if len(oof_predictions_df) != len(nested_df):
    raise RuntimeError(
        "The number of out-of-fold predictions does not "
        "match the modelling population."
    )

if oof_predictions_df[
    "household_ID"
].duplicated().any():
    raise RuntimeError(
        "Duplicate household out-of-fold predictions were created."
    )


# ------------------------------------------------------------
# 10. Pooled out-of-fold performance
# ------------------------------------------------------------

pooled_true_classes = (
    oof_predictions_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

pooled_predicted_classes = (
    oof_predictions_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

probability_columns = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_class_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    probability_columns.append(
        f"probability_{safe_class_name}"
    )

pooled_probabilities = (
    oof_predictions_df[
        probability_columns
    ]
    .to_numpy(dtype=float)
)

pooled_metrics = (
    calculate_multiclass_metrics(
        pooled_true_classes,
        pooled_predicted_classes,
        pooled_probabilities
    )
)

pooled_performance_df = pd.DataFrame([
    {
        "evaluation": (
            "Primary nested out-of-fold"
        ),
        **pooled_metrics
    }
])

pooled_classification_report = (
    classification_report(
        pooled_true_classes,
        pooled_predicted_classes,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )
)

pooled_class_metrics_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    pooled_class_metrics_records.append({
        "class_name": class_name,
        "precision": (
            pooled_classification_report[
                class_name
            ]["precision"]
        ),
        "recall": (
            pooled_classification_report[
                class_name
            ]["recall"]
        ),
        "f1_score": (
            pooled_classification_report[
                class_name
            ]["f1-score"]
        ),
        "support": int(
            pooled_classification_report[
                class_name
            ]["support"]
        )
    })

pooled_class_metrics_df = pd.DataFrame(
    pooled_class_metrics_records
)

pooled_confusion_matrix = confusion_matrix(
    pooled_true_classes,
    pooled_predicted_classes,
    labels=FIXED_CLASS_INDICES
)

pooled_confusion_df = pd.DataFrame(
    pooled_confusion_matrix,
    index=ORDERED_CLUSTER_NAMES,
    columns=ORDERED_CLUSTER_NAMES
)

outer_metric_summary = (
    outer_metrics_df[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "top_2_accuracy",
            "multiclass_log_loss",
            "multiclass_brier_sum"
        ]
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    )
    .T
)

selected_workflow_counts = (
    outer_selection_df
    .groupby(
        [
            "selected_feature_set",
            "selected_model_family",
            "selected_imbalance_strategy"
        ]
    )
    .size()
    .reset_index(
        name="outer_folds_selected"
    )
    .sort_values(
        "outer_folds_selected",
        ascending=False
    )
)


# ------------------------------------------------------------
# 11. Compare against dummy baselines
# ------------------------------------------------------------

baseline_comparison = pd.concat(
    [
        baseline_summary[
            [
                "baseline",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1"
            ]
        ].rename(
            columns={
                "baseline": "evaluation"
            }
        ),

        pooled_performance_df[
            [
                "evaluation",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1"
            ]
        ]
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 12. Display final Step 4 results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PRIMARY NESTED-CV RESULTS")
print("=" * 80)

print("\nWorkflow selected in each outer fold:")

display(
    outer_selection_df
)

print("\nWorkflow-selection frequency:")

display(
    selected_workflow_counts
)

print("\nOuter-fold metric summary:")

display(
    outer_metric_summary.round(4)
)

print("\nPooled untouched out-of-fold performance:")

display(
    pooled_performance_df.round(4)
)

print("\nPooled class-specific performance:")

display(
    pooled_class_metrics_df.round(4)
)

print("\nPooled confusion matrix:")

display(
    pooled_confusion_df
)

print("\nComparison with dummy baselines:")

display(
    baseline_comparison.round(4)
)

print(
    "\nTotal nested-CV runtime in minutes:",
    round(
        nested_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 13. Save Step 4 outputs
# ------------------------------------------------------------

inner_candidate_results_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_inner_candidates.csv",
    index=False
)

outer_selection_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_outer_selections.csv",
    index=False
)

outer_metrics_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_outer_metrics.csv",
    index=False
)

outer_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_outer_class_metrics.csv",
    index=False
)

outer_confusion_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_outer_confusion_records.csv",
    index=False
)

oof_predictions_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_oof_predictions.csv",
    index=False
)

pooled_performance_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_pooled_performance.csv",
    index=False
)

pooled_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_pooled_class_metrics.csv",
    index=False
)

pooled_confusion_df.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_pooled_confusion_matrix.csv"
)

outer_metric_summary.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_outer_metric_summary.csv"
)

selected_workflow_counts.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_workflow_selection_counts.csv",
    index=False
)

baseline_comparison.to_csv(
    OUTPUT_DIR
    / "primary_nested_cv_baseline_comparison.csv",
    index=False
)

corrected_feature_schema = {
    "removed_exact_duplicate": (
        DUPLICATED_FEATURE_TO_REMOVE
    ),
    "retained_equivalent_feature": (
        DUPLICATED_FEATURE_TO_RETAIN
    ),
    "core_model_features": (
        MODEL_CORE_FEATURES
    ),
    "extended_model_features": (
        MODEL_EXTENDED_FEATURES
    ),
    "primary_selection_metric": (
        "macro_f1"
    ),
    "first_tie_breaker": (
        "balanced_accuracy"
    ),
    "second_tie_breaker": (
        "weighted_f1"
    ),
    "ordinary_accuracy_primary": False,
    "top_two_accuracy_primary": False,
    "oversampling_applied": False
}

with open(
    OUTPUT_DIR
    / "corrected_modelling_feature_schema.json",
    "w",
    encoding="utf-8"
) as schema_file:
    json.dump(
        corrected_feature_schema,
        schema_file,
        indent=4
    )

print("\n" + "=" * 80)
print("STEP 4 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "OOF households:",
    len(oof_predictions_df)
)

print(
    "Pooled macro-F1:",
    round(
        pooled_metrics["macro_f1"],
        4
    )
)

print(
    "Pooled balanced accuracy:",
    round(
        pooled_metrics[
            "balanced_accuracy"
        ],
        4
    )
)

print(
    "Pooled top-2 accuracy "
    "(supplementary only):",
    round(
        pooled_metrics[
            "top_2_accuracy"
        ],
        4
    )
)

print(
    "\nSaved Step 4 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 5 — IMBALANCE-TREATMENT SENSITIVITY
# ============================================================
# Purpose:
# 1. Compare balanced class weighting with training-only
#    random oversampling and SMOTE.
# 2. Prevent leakage by applying every sampler inside the
#    cross-validation pipeline.
# 3. Select complete workflows using inner-fold macro-F1.
# 4. Evaluate selected workflows on untouched outer folds.
#
# Strategies:
# - Balanced class weighting
# - Moderate random oversampling
# - Full random oversampling
# - Moderate SMOTE
#
# Important:
# - Validation and test folds remain untouched.
# - Macro-F1 is the primary selection metric.
# - Balanced accuracy is the first tie-breaker.
# - Minimum class recall is an additional minority-class
#   safeguard.
# ============================================================

import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    recall_score
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import RobustScaler

try:
    import imblearn

    from imblearn.over_sampling import (
        RandomOverSampler,
        SMOTE
    )

    from imblearn.pipeline import (
        Pipeline as ImbalancedPipeline
    )

except ImportError as error:
    raise ImportError(
        "Step 5 requires imbalanced-learn. Install it using:\n"
        "pip install imbalanced-learn\n"
        "Then restart the kernel and rerun Steps 1–5."
    ) from error


# ------------------------------------------------------------
# 1. Confirm Step 4 objects
# ------------------------------------------------------------

REQUIRED_STEP4_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "y_nested",
    "MODEL_FEATURE_SETS",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "CLASS_TO_INDEX",
    "INDEX_TO_CLASS",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "align_class_probabilities",
    "calculate_multiclass_metrics",
    "serialise_parameters",
    "pooled_performance_df",
    "pooled_class_metrics_df",
    "oof_predictions_df"
]

missing_step4_objects = [
    name
    for name in REQUIRED_STEP4_OBJECTS
    if name not in globals()
]

if missing_step4_objects:
    raise RuntimeError(
        "Run Steps 1–4 first. Missing objects: "
        f"{missing_step4_objects}"
    )


# ------------------------------------------------------------
# 2. Define cautious oversampling targets
# ------------------------------------------------------------

def moderate_sampling_strategy(y_values):
    """
    Moderately increase minority classes without forcing every
    class to equal the majority class.

    For each minority class, the target count is limited to:
    - at most 50% of the majority-class count; and
    - at most four times its original count.

    This prevents the small Daytime-Active class from
    being synthetically expanded to the full Typical Household
    population in one operation.
    """

    class_counts_local = (
        pd.Series(y_values)
        .value_counts()
        .sort_index()
    )

    majority_count = int(
        class_counts_local.max()
    )

    half_majority = int(
        np.ceil(0.50 * majority_count)
    )

    strategy = {}

    for class_index, class_count in (
        class_counts_local.items()
    ):
        class_count = int(class_count)

        if class_count >= majority_count:
            continue

        target_count = min(
            half_majority,
            class_count * 4
        )

        if target_count > class_count:
            strategy[int(class_index)] = int(
                target_count
            )

    return strategy


# ------------------------------------------------------------
# 3. Additional minority-sensitive scorer
# ------------------------------------------------------------

def minimum_class_recall(
    y_true,
    y_predicted
):
    recalls = recall_score(
        y_true,
        y_predicted,
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    return float(
        np.min(recalls)
    )


STEP5_SCORING = {
    "macro_f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    ),

    "balanced_accuracy": (
        "balanced_accuracy"
    ),

    "minimum_class_recall": make_scorer(
        minimum_class_recall
    ),

    "weighted_f1": make_scorer(
        f1_score,
        average="weighted",
        zero_division=0
    ),

    "accuracy": "accuracy"
}


# ------------------------------------------------------------
# 4. Build model-and-sampler pipelines
# ------------------------------------------------------------

def build_step5_pipeline(
    model_family,
    imbalance_strategy
):
    """
    Create a leakage-safe model pipeline.

    Class weighting:
        Uses class_weight='balanced' without resampling.

    Oversampling:
        Uses class_weight=None and applies the sampler after
        imputation, inside each training fold.
    """

    if imbalance_strategy == "Balanced class weighting":
        class_weight = "balanced"
        sampler = None

    elif imbalance_strategy == "Moderate random oversampling":
        class_weight = None

        sampler = RandomOverSampler(
            sampling_strategy=moderate_sampling_strategy,
            random_state=RANDOM_STATE
        )

    elif imbalance_strategy == "Full random oversampling":
        class_weight = None

        sampler = RandomOverSampler(
            sampling_strategy="not majority",
            random_state=RANDOM_STATE
        )

    elif imbalance_strategy == "Moderate SMOTE":
        class_weight = None

        sampler = SMOTE(
            sampling_strategy=moderate_sampling_strategy,
            k_neighbors=3,
            random_state=RANDOM_STATE
        )

    else:
        raise ValueError(
            f"Unknown imbalance strategy: "
            f"{imbalance_strategy}"
        )

    if model_family == "Logistic Regression":

        model = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            max_iter=5000,
            class_weight=class_weight,
            random_state=RANDOM_STATE
        )

        parameter_grid = {
            "model__C": [
                0.001,
                0.01,
                0.1,
                1.0,
                10.0
            ]
        }

        if sampler is None:
            pipeline = SklearnPipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "scaler",
                    RobustScaler()
                ),
                (
                    "model",
                    model
                )
            ])

        else:
            pipeline = ImbalancedPipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "scaler",
                    RobustScaler()
                ),
                (
                    "sampler",
                    sampler
                ),
                (
                    "model",
                    model
                )
            ])

    elif model_family == "Random Forest":

        model = RandomForestClassifier(
            n_estimators=400,
            bootstrap=True,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=1
        )

        parameter_grid = [
            {
                "model__max_depth": [3],
                "model__min_samples_leaf": [12],
                "model__min_samples_split": [24],
                "model__max_features": ["sqrt"],
                "model__max_samples": [0.70],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [4],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [5],
                "model__min_samples_leaf": [8],
                "model__min_samples_split": [16],
                "model__max_features": [0.30],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [5],
                "model__min_samples_split": [10],
                "model__max_features": [0.50],
                "model__max_samples": [None],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__max_samples": [0.80],
                "model__ccp_alpha": [0.001]
            }
        ]

        if sampler is None:
            pipeline = SklearnPipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "model",
                    model
                )
            ])

        else:
            pipeline = ImbalancedPipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "sampler",
                    sampler
                ),
                (
                    "model",
                    model
                )
            ])

    elif model_family == "Extra Trees":

        model = ExtraTreesClassifier(
            n_estimators=400,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=1
        )

        parameter_grid = [
            {
                "model__max_depth": [3],
                "model__min_samples_leaf": [12],
                "model__min_samples_split": [24],
                "model__max_features": ["sqrt"],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [4],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__ccp_alpha": [0.001]
            },
            {
                "model__max_depth": [5],
                "model__min_samples_leaf": [8],
                "model__min_samples_split": [16],
                "model__max_features": [0.30],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [5],
                "model__min_samples_split": [10],
                "model__max_features": [0.50],
                "model__ccp_alpha": [0.0]
            },
            {
                "model__max_depth": [6],
                "model__min_samples_leaf": [10],
                "model__min_samples_split": [20],
                "model__max_features": ["sqrt"],
                "model__ccp_alpha": [0.001]
            }
        ]

        if sampler is None:
            pipeline = SklearnPipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "model",
                    model
                )
            ])

        else:
            pipeline = ImbalancedPipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "sampler",
                    sampler
                ),
                (
                    "model",
                    model
                )
            ])

    else:
        raise ValueError(
            f"Unknown model family: "
            f"{model_family}"
        )

    return pipeline, parameter_grid


# ------------------------------------------------------------
# 5. Candidate workflow definitions
# ------------------------------------------------------------

STEP5_MODEL_FAMILIES = [
    "Logistic Regression",
    "Random Forest",
    "Extra Trees"
]

STEP5_IMBALANCE_STRATEGIES = [
    "Balanced class weighting",
    "Moderate random oversampling",
    "Full random oversampling",
    "Moderate SMOTE"
]

step5_candidate_table = pd.DataFrame([
    {
        "model_family": model_family,
        "imbalance_strategy": imbalance_strategy
    }
    for model_family in STEP5_MODEL_FAMILIES
    for imbalance_strategy
    in STEP5_IMBALANCE_STRATEGIES
])

print("=" * 80)
print("STEP 5 CANDIDATE WORKFLOWS")
print("=" * 80)

display(step5_candidate_table)


# ------------------------------------------------------------
# 6. Nested cross-validation
# ------------------------------------------------------------

step5_outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

step5_inner_candidate_records = []
step5_outer_selection_records = []
step5_outer_metric_records = []
step5_outer_class_records = []
step5_oof_prediction_records = []

step5_start_time = time.time()

for outer_fold, (
    outer_train_indices,
    outer_test_indices
) in enumerate(
    step5_outer_cv.split(
        X_nested_source,
        y_nested
    ),
    start=1
):
    print("\n" + "=" * 80)

    print(
        f"IMBALANCE NESTED CV — OUTER FOLD "
        f"{outer_fold}/{OUTER_FOLDS}"
    )

    print("=" * 80)

    outer_train_source = (
        X_nested_source.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_source = (
        X_nested_source.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_train = (
        y_nested.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_test = (
        y_nested.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_metadata = (
        nested_df.iloc[
            outer_test_indices
        ][
            [
                "household_ID",
                "cluster_name"
            ]
        ]
        .reset_index(drop=True)
    )

    step5_inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_STATE + outer_fold
        )
    )

    fold_candidates = []

    for feature_set_name, feature_columns in (
        MODEL_FEATURE_SETS.items()
    ):
        X_outer_train = (
            outer_train_source[
                feature_columns
            ]
        )

        for model_family in (
            STEP5_MODEL_FAMILIES
        ):
            for imbalance_strategy in (
                STEP5_IMBALANCE_STRATEGIES
            ):
                pipeline, parameter_grid = (
                    build_step5_pipeline(
                        model_family,
                        imbalance_strategy
                    )
                )

                search = GridSearchCV(
                    estimator=pipeline,
                    param_grid=parameter_grid,
                    scoring=STEP5_SCORING,
                    refit=False,
                    cv=step5_inner_cv,
                    n_jobs=1,
                    return_train_score=True,
                    error_score="raise"
                )

                search_start = time.time()

                search.fit(
                    X_outer_train,
                    y_outer_train
                )

                search_runtime = (
                    time.time()
                    - search_start
                )

                results = pd.DataFrame(
                    search.cv_results_
                )

                results = (
                    results
                    .sort_values(
                        [
                            "mean_test_macro_f1",
                            "mean_test_balanced_accuracy",
                            "mean_test_minimum_class_recall",
                            "mean_test_weighted_f1",
                            "mean_test_accuracy"
                        ],
                        ascending=[
                            False,
                            False,
                            False,
                            False,
                            False
                        ]
                    )
                    .reset_index(drop=True)
                )

                best_result = select_balanced_cv_candidate(
                results,
                model_family
            )
                best_parameters = dict(
                    best_result["params"]
                )

                candidate_record = {
                    "outer_fold": outer_fold,
                    "feature_set": (
                        feature_set_name
                    ),
                    "feature_count": len(
                        feature_columns
                    ),
                    "model_family": (
                        model_family
                    ),
                    "imbalance_strategy": (
                        imbalance_strategy
                    ),
                    "selected_parameters": (
                        serialise_parameters(
                            best_parameters
                        )
                    ),
                    "mean_inner_macro_f1": float(
                        best_result[
                            "mean_test_macro_f1"
                        ]
                    ),
                    "mean_inner_balanced_accuracy": float(
                        best_result[
                            "mean_test_balanced_accuracy"
                        ]
                    ),
                    "mean_inner_minimum_class_recall": float(
                        best_result[
                            "mean_test_minimum_class_recall"
                        ]
                    ),
                    "mean_inner_weighted_f1": float(
                        best_result[
                            "mean_test_weighted_f1"
                        ]
                    ),
                    "mean_inner_accuracy": float(
                        best_result[
                            "mean_test_accuracy"
                        ]
                    ),
                    "mean_inner_train_macro_f1": float(
                        best_result[
                            "mean_train_macro_f1"
                        ]
                    ),
                    "inner_macro_f1_gap": float(
                        best_result[
                            "mean_train_macro_f1"
                        ]
                        - best_result[
                            "mean_test_macro_f1"
                        ]
                    ),
                    "search_runtime_seconds": float(
                        search_runtime
                    )
                }

                step5_inner_candidate_records.append(
                    candidate_record
                )

                fold_candidates.append({
                    **candidate_record,
                    "feature_columns": list(
                        feature_columns
                    ),
                    "pipeline_template": pipeline,
                    "best_parameters_dict": (
                        best_parameters
                    )
                })

                print(
                    f"{feature_set_name} | "
                    f"{model_family} | "
                    f"{imbalance_strategy}: "
                    f"macro-F1="
                    f"{candidate_record['mean_inner_macro_f1']:.4f}, "
                    f"balanced accuracy="
                    f"{candidate_record['mean_inner_balanced_accuracy']:.4f}, "
                    f"minimum recall="
                    f"{candidate_record['mean_inner_minimum_class_recall']:.4f}"
                )

    # --------------------------------------------------------
    # Select complete workflow using training data only
    # --------------------------------------------------------

    fold_candidates = sorted(
        fold_candidates,
        key=lambda record: (
            record["mean_inner_macro_f1"],
            record[
                "mean_inner_balanced_accuracy"
            ],
            record[
                "mean_inner_minimum_class_recall"
            ],
            record["mean_inner_weighted_f1"],
            record["mean_inner_accuracy"]
        ),
        reverse=True
    )

    selected_candidate = fold_candidates[0]

    selected_pipeline = clone(
        selected_candidate[
            "pipeline_template"
        ]
    )

    selected_pipeline.set_params(
        **selected_candidate[
            "best_parameters_dict"
        ]
    )

    selected_features = (
        selected_candidate[
            "feature_columns"
        ]
    )

    selected_pipeline.fit(
        outer_train_source[
            selected_features
        ],
        y_outer_train
    )

    outer_predictions = (
        selected_pipeline.predict(
            outer_test_source[
                selected_features
            ]
        )
    )

    raw_probabilities = (
        selected_pipeline.predict_proba(
            outer_test_source[
                selected_features
            ]
        )
    )

    outer_probabilities = (
        align_class_probabilities(
            raw_probabilities,
            selected_pipeline.classes_
        )
    )

    outer_metrics = (
        calculate_multiclass_metrics(
            y_outer_test,
            outer_predictions,
            outer_probabilities
        )
    )

    class_report = classification_report(
        y_outer_test,
        outer_predictions,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )

    outer_class_recalls = recall_score(
        y_outer_test,
        outer_predictions,
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    outer_minimum_class_recall = float(
        np.min(outer_class_recalls)
    )

    step5_outer_selection_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_candidate[
                "feature_set"
            ]
        ),
        "selected_model_family": (
            selected_candidate[
                "model_family"
            ]
        ),
        "selected_imbalance_strategy": (
            selected_candidate[
                "imbalance_strategy"
            ]
        ),
        "selected_parameters": (
            selected_candidate[
                "selected_parameters"
            ]
        ),
        "inner_macro_f1": (
            selected_candidate[
                "mean_inner_macro_f1"
            ]
        ),
        "inner_balanced_accuracy": (
            selected_candidate[
                "mean_inner_balanced_accuracy"
            ]
        ),
        "inner_minimum_class_recall": (
            selected_candidate[
                "mean_inner_minimum_class_recall"
            ]
        )
    })

    step5_outer_metric_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_candidate[
                "feature_set"
            ]
        ),
        "selected_model_family": (
            selected_candidate[
                "model_family"
            ]
        ),
        "selected_imbalance_strategy": (
            selected_candidate[
                "imbalance_strategy"
            ]
        ),
        "minimum_class_recall": (
            outer_minimum_class_recall
        ),
        **outer_metrics
    })

    for class_name in ORDERED_CLUSTER_NAMES:
        step5_outer_class_records.append({
            "outer_fold": outer_fold,
            "class_name": class_name,
            "precision": (
                class_report[
                    class_name
                ]["precision"]
            ),
            "recall": (
                class_report[
                    class_name
                ]["recall"]
            ),
            "f1_score": (
                class_report[
                    class_name
                ]["f1-score"]
            ),
            "support": int(
                class_report[
                    class_name
                ]["support"]
            )
        })

    probability_order = np.argsort(
        outer_probabilities,
        axis=1
    )[:, ::-1]

    for row_position in range(
        len(outer_test_indices)
    ):
        predicted_index = int(
            outer_predictions[
                row_position
            ]
        )

        first_index = int(
            probability_order[
                row_position,
                0
            ]
        )

        second_index = int(
            probability_order[
                row_position,
                1
            ]
        )

        record = {
            "household_ID": (
                outer_test_metadata.iloc[
                    row_position
                ]["household_ID"]
            ),
            "outer_fold": outer_fold,
            "true_class_index": int(
                y_outer_test.iloc[
                    row_position
                ]
            ),
            "true_class_name": (
                outer_test_metadata.iloc[
                    row_position
                ]["cluster_name"]
            ),
            "predicted_class_index": (
                predicted_index
            ),
            "predicted_class_name": (
                INDEX_TO_CLASS[
                    predicted_index
                ]
            ),
            "selected_feature_set": (
                selected_candidate[
                    "feature_set"
                ]
            ),
            "selected_model_family": (
                selected_candidate[
                    "model_family"
                ]
            ),
            "selected_imbalance_strategy": (
                selected_candidate[
                    "imbalance_strategy"
                ]
            ),
            "highest_probability": float(
                outer_probabilities[
                    row_position,
                    first_index
                ]
            ),
            "second_highest_probability": float(
                outer_probabilities[
                    row_position,
                    second_index
                ]
            ),
            "probability_margin": float(
                outer_probabilities[
                    row_position,
                    first_index
                ]
                - outer_probabilities[
                    row_position,
                    second_index
                ]
            )
        }

        for class_index, class_name in enumerate(
            ORDERED_CLUSTER_NAMES
        ):
            safe_class_name = (
                class_name
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
                .replace("/", "_")
            )

            record[
                f"probability_{safe_class_name}"
            ] = float(
                outer_probabilities[
                    row_position,
                    class_index
                ]
            )

        step5_oof_prediction_records.append(
            record
        )

    print("\nSelected imbalance workflow:")

    print(
        "Feature set:",
        selected_candidate[
            "feature_set"
        ]
    )

    print(
        "Model:",
        selected_candidate[
            "model_family"
        ]
    )

    print(
        "Strategy:",
        selected_candidate[
            "imbalance_strategy"
        ]
    )

    print(
        "Outer macro-F1:",
        round(
            outer_metrics["macro_f1"],
            4
        )
    )

    print(
        "Outer balanced accuracy:",
        round(
            outer_metrics[
                "balanced_accuracy"
            ],
            4
        )
    )

    print(
        "Outer minimum class recall:",
        round(
            outer_minimum_class_recall,
            4
        )
    )


# ------------------------------------------------------------
# 7. Compile outputs
# ------------------------------------------------------------

step5_inner_candidates_df = pd.DataFrame(
    step5_inner_candidate_records
)

step5_outer_selections_df = pd.DataFrame(
    step5_outer_selection_records
)

step5_outer_metrics_df = pd.DataFrame(
    step5_outer_metric_records
)

step5_outer_class_metrics_df = pd.DataFrame(
    step5_outer_class_records
)

step5_oof_predictions_df = pd.DataFrame(
    step5_oof_prediction_records
)

step5_runtime_seconds = (
    time.time() - step5_start_time
)

if len(step5_oof_predictions_df) != len(
    nested_df
):
    raise RuntimeError(
        "Step 5 did not create exactly one OOF "
        "prediction per household."
    )

if step5_oof_predictions_df[
    "household_ID"
].duplicated().any():
    raise RuntimeError(
        "Duplicate Step 5 OOF household predictions "
        "were detected."
    )


# ------------------------------------------------------------
# 8. Pooled OOF performance
# ------------------------------------------------------------

step5_true = (
    step5_oof_predictions_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

step5_predicted = (
    step5_oof_predictions_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

step5_probability_columns = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_class_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    step5_probability_columns.append(
        f"probability_{safe_class_name}"
    )

step5_probabilities = (
    step5_oof_predictions_df[
        step5_probability_columns
    ]
    .to_numpy(dtype=float)
)

step5_pooled_metrics = (
    calculate_multiclass_metrics(
        step5_true,
        step5_predicted,
        step5_probabilities
    )
)

step5_pooled_metrics[
    "minimum_class_recall"
] = float(
    np.min(
        recall_score(
            step5_true,
            step5_predicted,
            labels=FIXED_CLASS_INDICES,
            average=None,
            zero_division=0
        )
    )
)

step5_pooled_performance_df = pd.DataFrame([
    {
        "evaluation": (
            "Nested imbalance-treatment workflow"
        ),
        **step5_pooled_metrics
    }
])

step5_classification_report = (
    classification_report(
        step5_true,
        step5_predicted,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )
)

step5_pooled_class_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    step5_pooled_class_records.append({
        "class_name": class_name,
        "precision": (
            step5_classification_report[
                class_name
            ]["precision"]
        ),
        "recall": (
            step5_classification_report[
                class_name
            ]["recall"]
        ),
        "f1_score": (
            step5_classification_report[
                class_name
            ]["f1-score"]
        ),
        "support": int(
            step5_classification_report[
                class_name
            ]["support"]
        )
    })

step5_pooled_class_metrics_df = pd.DataFrame(
    step5_pooled_class_records
)

step5_pooled_confusion_matrix = (
    confusion_matrix(
        step5_true,
        step5_predicted,
        labels=FIXED_CLASS_INDICES
    )
)

step5_pooled_confusion_df = pd.DataFrame(
    step5_pooled_confusion_matrix,
    index=ORDERED_CLUSTER_NAMES,
    columns=ORDERED_CLUSTER_NAMES
)

step5_selection_counts = (
    step5_outer_selections_df
    .groupby(
        [
            "selected_feature_set",
            "selected_model_family",
            "selected_imbalance_strategy"
        ]
    )
    .size()
    .reset_index(
        name="outer_folds_selected"
    )
    .sort_values(
        "outer_folds_selected",
        ascending=False
    )
)


# ------------------------------------------------------------
# 9. Compare Step 4 and Step 5
# ------------------------------------------------------------

step4_summary_for_comparison = (
    pooled_performance_df.copy()
)

step4_summary_for_comparison[
    "minimum_class_recall"
] = float(
    pooled_class_metrics_df[
        "recall"
    ].min()
)

imbalance_workflow_comparison = pd.concat(
    [
        step4_summary_for_comparison[
            [
                "evaluation",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
                "top_2_accuracy",
                "multiclass_log_loss",
                "multiclass_brier_sum",
                "minimum_class_recall"
            ]
        ],

        step5_pooled_performance_df[
            [
                "evaluation",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
                "top_2_accuracy",
                "multiclass_log_loss",
                "multiclass_brier_sum",
                "minimum_class_recall"
            ]
        ]
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 9A. Interpret Step 5 without post-hoc promotion
# ------------------------------------------------------------

step4_comparison_row = (
    imbalance_workflow_comparison.iloc[0]
)

step5_comparison_row = (
    imbalance_workflow_comparison.iloc[1]
)

step5_metric_differences = {
    "macro_f1_difference_step5_minus_step4": float(
        step5_comparison_row["macro_f1"]
        - step4_comparison_row["macro_f1"]
    ),
    "balanced_accuracy_difference_step5_minus_step4": float(
        step5_comparison_row["balanced_accuracy"]
        - step4_comparison_row["balanced_accuracy"]
    ),
    "minimum_class_recall_difference_step5_minus_step4": float(
        step5_comparison_row["minimum_class_recall"]
        - step4_comparison_row["minimum_class_recall"]
    )
}

STEP5_PROMOTED_TO_PRIMARY = False

STEP5_ROLE = (
    "Secondary imbalance-treatment sensitivity analysis"
)

STEP5_NON_PROMOTION_REASON = (
    "Step 5 was introduced as a secondary sensitivity analysis "
    "after the primary Step 4 workflow had been defined. Its "
    "outer-fold results were therefore not used to replace the "
    "prespecified primary estimate. The analysis may show a small "
    "numerical change in macro-F1 or balanced accuracy, but it did "
    "not provide a stable, clearly superior minority-class solution "
    "across folds. Promoting it after inspecting the same outer-fold "
    "results would make those folds part of post-hoc model selection."
)

step5_secondary_interpretation_df = pd.DataFrame([
    {
        "analysis_role": STEP5_ROLE,
        **step5_metric_differences,
        "promoted_to_primary": STEP5_PROMOTED_TO_PRIMARY,
        "interpretation": STEP5_NON_PROMOTION_REASON
    }
])


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 5 — IMBALANCE-TREATMENT RESULTS")
print("=" * 80)

print("\nSelected workflow by outer fold:")

display(
    step5_outer_selections_df
)

print("\nWorkflow-selection frequency:")

display(
    step5_selection_counts
)

print("\nPooled imbalance-treatment performance:")

display(
    step5_pooled_performance_df.round(4)
)

print("\nPooled class-specific performance:")

display(
    step5_pooled_class_metrics_df.round(4)
)

print("\nPooled confusion matrix:")

display(
    step5_pooled_confusion_df
)

print("\nStep 4 versus Step 5:")

display(
    imbalance_workflow_comparison.round(4)
)

print("\nStep 5 interpretation:")

display(
    step5_secondary_interpretation_df.round(4)
)

print(
    "\nStep 5 runtime in minutes:",
    round(
        step5_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 11. Save Step 5 outputs
# ------------------------------------------------------------

step5_inner_candidates_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_inner_candidates.csv",
    index=False
)

step5_outer_selections_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_outer_selections.csv",
    index=False
)

step5_outer_metrics_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_outer_metrics.csv",
    index=False
)

step5_outer_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_outer_class_metrics.csv",
    index=False
)

step5_oof_predictions_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_oof_predictions.csv",
    index=False
)

step5_pooled_performance_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_pooled_performance.csv",
    index=False
)

step5_pooled_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_pooled_class_metrics.csv",
    index=False
)

step5_pooled_confusion_df.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_pooled_confusion_matrix.csv"
)

step5_selection_counts.to_csv(
    OUTPUT_DIR
    / "imbalance_nested_cv_selection_counts.csv",
    index=False
)

imbalance_workflow_comparison.to_csv(
    OUTPUT_DIR
    / "step4_vs_step5_imbalance_comparison.csv",
    index=False
)

step5_secondary_interpretation_df.to_csv(
    OUTPUT_DIR
    / "step5_secondary_sensitivity_interpretation.csv",
    index=False
)

step5_configuration = {
    "strategies_tested": (
        STEP5_IMBALANCE_STRATEGIES
    ),
    "smote_k_neighbors": 3,
    "moderate_sampling_rule": (
        "Each minority class increased to the smaller "
        "of 50 percent of the majority count or four "
        "times its original training count."
    ),
    "full_random_oversampling": (
        "All minority training classes increased to "
        "the majority training-class count."
    ),
    "samplers_applied_before_cv": False,
    "samplers_applied_inside_training_pipeline": True,
    "primary_selection_metric": "macro_f1",
    "first_tie_breaker": "balanced_accuracy",
    "minority_guardrail": "minimum_class_recall",
    "validation_distribution_modified": False,
    "test_distribution_modified": False,
    "analysis_role": STEP5_ROLE,
    "promoted_to_primary": STEP5_PROMOTED_TO_PRIMARY,
    "non_promotion_reason": STEP5_NON_PROMOTION_REASON,
    "metric_differences_step5_minus_step4": (
        step5_metric_differences
    ),
    "imbalanced_learn_version": (
        imblearn.__version__
    )
}

with open(
    OUTPUT_DIR
    / "imbalance_treatment_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        step5_configuration,
        configuration_file,
        indent=4
    )

print("\n" + "=" * 80)
print("STEP 5 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Pooled macro-F1:",
    round(
        step5_pooled_metrics[
            "macro_f1"
        ],
        4
    )
)

print(
    "Pooled balanced accuracy:",
    round(
        step5_pooled_metrics[
            "balanced_accuracy"
        ],
        4
    )
)

print(
    "Minimum class recall:",
    round(
        step5_pooled_metrics[
            "minimum_class_recall"
        ],
        4
    )
)

print(
    "\nSaved Step 5 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# 95% confidence intervals using the primary Step 4 untouched out-of-fold predictions.

# ============================================================
# STEP 6 — BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================
# Purpose:
# 1. Quantify uncertainty in the primary Step 4 nested-CV
#    out-of-fold results.
# 2. Calculate 95% confidence intervals for overall metrics.
# 3. Calculate class-specific precision, recall and F1 intervals.
# 4. Preserve the original class distribution through a
#    stratified household-level bootstrap.
#
# Important:
# - The primary source is Step 4, not Step 5.
# - Step 5 remains a secondary sensitivity analysis and was not
#   promoted post hoc using the same outer-fold results.
# - Each household has exactly one untouched OOF prediction.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    top_k_accuracy_score
)


# ------------------------------------------------------------
# 1. Confirm required Step 4 objects
# ------------------------------------------------------------

REQUIRED_STEP6_OBJECTS = [
    "oof_predictions_df",
    "dataset_all",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "probability_columns",
    "multiclass_brier_score",
    "OUTPUT_DIR",
    "RANDOM_STATE"
]

missing_step6_objects = [
    object_name
    for object_name in REQUIRED_STEP6_OBJECTS
    if object_name not in globals()
]

if missing_step6_objects:
    raise RuntimeError(
        "Run Steps 1–4 before Step 6. Missing objects: "
        f"{missing_step6_objects}"
    )


# ------------------------------------------------------------
# 2. Validate the primary OOF prediction table
# ------------------------------------------------------------

bootstrap_source = (
    oof_predictions_df
    .copy()
    .reset_index(drop=True)
)

expected_oof_households = len(dataset_all)

if len(bootstrap_source) != expected_oof_households:
    raise ValueError(
        f"Expected {expected_oof_households:,} primary OOF "
        f"predictions, but found {len(bootstrap_source):,}."
    )

if bootstrap_source["household_ID"].duplicated().any():
    raise ValueError(
        "Each household must have exactly one primary OOF prediction."
    )

required_prediction_columns = [
    "household_ID",
    "true_class_index",
    "predicted_class_index"
] + probability_columns

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in bootstrap_source.columns
]

if missing_prediction_columns:
    raise ValueError(
        "Primary OOF prediction columns are missing: "
        f"{missing_prediction_columns}"
    )

y_true_primary = (
    bootstrap_source["true_class_index"]
    .to_numpy(dtype=int)
)

y_predicted_primary = (
    bootstrap_source["predicted_class_index"]
    .to_numpy(dtype=int)
)

probabilities_primary = (
    bootstrap_source[probability_columns]
    .to_numpy(dtype=float)
)

if not np.allclose(
    probabilities_primary.sum(axis=1),
    1.0,
    atol=1e-8
):
    raise ValueError(
        "Primary probability rows do not sum to one."
    )


# ------------------------------------------------------------
# 3. Metric calculation helper
# ------------------------------------------------------------

def calculate_bootstrap_metrics(
    y_true_values,
    y_predicted_values,
    probability_values
):
    overall_metrics = {
        "accuracy": accuracy_score(
            y_true_values,
            y_predicted_values
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true_values,
                y_predicted_values
            )
        ),

        "macro_f1": f1_score(
            y_true_values,
            y_predicted_values,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true_values,
            y_predicted_values,
            average="weighted",
            zero_division=0
        ),

        "top_2_accuracy": top_k_accuracy_score(
            y_true_values,
            probability_values,
            k=2,
            labels=FIXED_CLASS_INDICES
        ),

        "multiclass_log_loss": log_loss(
            y_true_values,
            probability_values,
            labels=FIXED_CLASS_INDICES
        ),

        "multiclass_brier_sum": (
            multiclass_brier_score(
                y_true_values,
                probability_values
            )
        )
    }

    class_precision = precision_score(
        y_true_values,
        y_predicted_values,
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    class_recall = recall_score(
        y_true_values,
        y_predicted_values,
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    class_f1 = f1_score(
        y_true_values,
        y_predicted_values,
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    class_metrics = {}

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        safe_class_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        class_metrics[
            f"{safe_class_name}_precision"
        ] = class_precision[class_index]

        class_metrics[
            f"{safe_class_name}_recall"
        ] = class_recall[class_index]

        class_metrics[
            f"{safe_class_name}_f1"
        ] = class_f1[class_index]

    return {
        **overall_metrics,
        **class_metrics
    }


# ------------------------------------------------------------
# 4. Point estimates
# ------------------------------------------------------------

primary_point_metrics = (
    calculate_bootstrap_metrics(
        y_true_primary,
        y_predicted_primary,
        probabilities_primary
    )
)

primary_point_estimates_df = pd.DataFrame([
    {
        "evaluation": (
            "Primary Step 4 nested out-of-fold"
        ),
        **primary_point_metrics
    }
])


# ------------------------------------------------------------
# 5. Stratified household-level bootstrap
# ------------------------------------------------------------

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = RANDOM_STATE

random_generator = np.random.default_rng(
    BOOTSTRAP_SEED
)

class_indices = {
    class_index: np.where(
        y_true_primary == class_index
    )[0]
    for class_index in FIXED_CLASS_INDICES
}

for class_index in FIXED_CLASS_INDICES:
    if len(class_indices[class_index]) == 0:
        raise RuntimeError(
            f"Class {class_index} contains no households."
        )

bootstrap_metric_records = []

for bootstrap_iteration in range(
    1,
    BOOTSTRAP_ITERATIONS + 1
):
    sampled_indices_by_class = []

    for class_index in FIXED_CLASS_INDICES:
        original_indices = (
            class_indices[class_index]
        )

        sampled_indices = (
            random_generator.choice(
                original_indices,
                size=len(original_indices),
                replace=True
            )
        )

        sampled_indices_by_class.append(
            sampled_indices
        )

    bootstrap_indices = np.concatenate(
        sampled_indices_by_class
    )

    random_generator.shuffle(
        bootstrap_indices
    )

    bootstrap_metrics = (
        calculate_bootstrap_metrics(
            y_true_primary[
                bootstrap_indices
            ],
            y_predicted_primary[
                bootstrap_indices
            ],
            probabilities_primary[
                bootstrap_indices
            ]
        )
    )

    bootstrap_metric_records.append({
        "bootstrap_iteration": (
            bootstrap_iteration
        ),
        **bootstrap_metrics
    })


bootstrap_distributions_df = pd.DataFrame(
    bootstrap_metric_records
)


# ------------------------------------------------------------
# 6. Create confidence-interval tables
# ------------------------------------------------------------

overall_metric_names = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "top_2_accuracy",
    "multiclass_log_loss",
    "multiclass_brier_sum"
]

overall_ci_records = []

for metric_name in overall_metric_names:
    overall_ci_records.append({
        "metric": metric_name,

        "point_estimate": (
            primary_point_metrics[
                metric_name
            ]
        ),

        "bootstrap_mean": (
            bootstrap_distributions_df[
                metric_name
            ].mean()
        ),

        "bootstrap_standard_error": (
            bootstrap_distributions_df[
                metric_name
            ].std(ddof=1)
        ),

        "ci_2.5_percent": (
            bootstrap_distributions_df[
                metric_name
            ].quantile(0.025)
        ),

        "ci_97.5_percent": (
            bootstrap_distributions_df[
                metric_name
            ].quantile(0.975)
        )
    })

overall_bootstrap_ci_df = pd.DataFrame(
    overall_ci_records
)


class_ci_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_class_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    for metric_type in [
        "precision",
        "recall",
        "f1"
    ]:
        metric_column = (
            f"{safe_class_name}_"
            f"{metric_type}"
        )

        class_ci_records.append({
            "class_name": class_name,
            "metric": metric_type,

            "point_estimate": (
                primary_point_metrics[
                    metric_column
                ]
            ),

            "bootstrap_mean": (
                bootstrap_distributions_df[
                    metric_column
                ].mean()
            ),

            "bootstrap_standard_error": (
                bootstrap_distributions_df[
                    metric_column
                ].std(ddof=1)
            ),

            "ci_2.5_percent": (
                bootstrap_distributions_df[
                    metric_column
                ].quantile(0.025)
            ),

            "ci_97.5_percent": (
                bootstrap_distributions_df[
                    metric_column
                ].quantile(0.975)
            )
        })

class_bootstrap_ci_df = pd.DataFrame(
    class_ci_records
)


# ------------------------------------------------------------
# 7. Minority-class uncertainty summary
# ------------------------------------------------------------

minority_uncertainty_summary = (
    class_bootstrap_ci_df[
        class_bootstrap_ci_df[
            "class_name"
        ].isin(
            [
                "Peak-Exposed",
                "Daytime-Active"
            ]
        )
    ]
    .copy()
)

minority_uncertainty_summary[
    "confidence_interval_width"
] = (
    minority_uncertainty_summary[
        "ci_97.5_percent"
    ]
    - minority_uncertainty_summary[
        "ci_2.5_percent"
    ]
)


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 6 — BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 80)

print("\nOverall primary-model confidence intervals:")

display(
    overall_bootstrap_ci_df.round(4)
)

print("\nClass-specific confidence intervals:")

display(
    class_bootstrap_ci_df.round(4)
)

print("\nMinority-class uncertainty:")

display(
    minority_uncertainty_summary.round(4)
)

print(
    "\nBootstrap iterations:",
    BOOTSTRAP_ITERATIONS
)

print(
    "Bootstrap design: stratified household-level "
    "resampling with the original class counts retained."
)


# ------------------------------------------------------------
# 9. Save Step 6 outputs
# ------------------------------------------------------------

primary_point_estimates_df.to_csv(
    OUTPUT_DIR
    / "primary_oof_point_estimates.csv",
    index=False
)

overall_bootstrap_ci_df.to_csv(
    OUTPUT_DIR
    / "primary_oof_overall_bootstrap_ci.csv",
    index=False
)

class_bootstrap_ci_df.to_csv(
    OUTPUT_DIR
    / "primary_oof_class_bootstrap_ci.csv",
    index=False
)

minority_uncertainty_summary.to_csv(
    OUTPUT_DIR
    / "primary_oof_minority_uncertainty.csv",
    index=False
)

bootstrap_distributions_df.to_csv(
    OUTPUT_DIR
    / "primary_oof_bootstrap_distributions.csv",
    index=False
)

bootstrap_configuration = {
    "evaluation_source": (
        "Step 4 primary nested out-of-fold predictions"
    ),
    "bootstrap_iterations": (
        BOOTSTRAP_ITERATIONS
    ),
    "bootstrap_seed": (
        BOOTSTRAP_SEED
    ),
    "resampling_unit": "household",
    "stratified_by_true_archetype": True,
    "original_class_counts_retained": True,
    "confidence_interval_method": (
        "Percentile bootstrap"
    ),
    "confidence_level": 0.95,
    "step5_promoted_to_primary": False,
    "step5_role": (
        "Secondary imbalance-treatment sensitivity analysis"
    ),
    "reason_step5_not_promoted": (
        "Step 5 was retained as a secondary sensitivity analysis. "
        "It was not promoted after inspecting the same outer-fold "
        "results because that would convert the primary evaluation "
        "folds into a post-hoc selection stage. Any small numerical "
        "improvement is therefore reported separately rather than "
        "attributed to the prespecified Step 4 procedure."
    )
}

with open(
    OUTPUT_DIR
    / "bootstrap_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        bootstrap_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Primary macro-F1:",
    round(
        primary_point_metrics[
            "macro_f1"
        ],
        4
    )
)

macro_f1_row = (
    overall_bootstrap_ci_df[
        overall_bootstrap_ci_df[
            "metric"
        ] == "macro_f1"
    ].iloc[0]
)

print(
    "Macro-F1 95% CI:",
    (
        round(
            macro_f1_row[
                "ci_2.5_percent"
            ],
            4
        ),
        round(
            macro_f1_row[
                "ci_97.5_percent"
            ],
            4
        )
    )
)

print(
    "\nSaved Step 6 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 7 — LEAKAGE-SAFE PROBABILITY CALIBRATION
# ============================================================
# Purpose:
# 1. Assess whether the Step 4 selected workflows produce
#    reliable multiclass probabilities.
# 2. Compare uncalibrated, sigmoid-calibrated and isotonic-
#    calibrated probabilities.
# 3. Select calibration using only outer-training data.
# 4. Evaluate the selected calibration method on untouched
#    outer-test households.
#
# Primary calibration criterion:
# - multiclass log loss
#
# Tie-breakers:
# - multiclass Brier score
# - top-label expected calibration error
#
# Isotonic calibration is treated cautiously because the
# Daytime-Active class is small in the billing-matched sample.
# ============================================================

import inspect
import json
import time

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    top_k_accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP7_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "y_nested",
    "MODEL_FEATURE_SETS",
    "outer_selection_df",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "INDEX_TO_CLASS",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "build_candidate_pipeline",
    "align_class_probabilities",
    "calculate_multiclass_metrics",
    "multiclass_brier_score",
    "oof_predictions_df",
    "pooled_performance_df"
]

missing_step7_objects = [
    object_name
    for object_name in REQUIRED_STEP7_OBJECTS
    if object_name not in globals()
]

if missing_step7_objects:
    raise RuntimeError(
        "Run Steps 1–4 before Step 7. Missing objects: "
        f"{missing_step7_objects}"
    )


# ------------------------------------------------------------
# 2. Calibration configuration
# ------------------------------------------------------------

CALIBRATION_METHODS = [
    "Uncalibrated",
    "Sigmoid",
    "Isotonic"
]

CALIBRATION_INTERNAL_FOLDS = 3
ECE_BINS = 10

daytime_calibration_count = int(
    (
        nested_df["cluster_name"]
        .astype(str)
        == "Daytime-Active"
    ).sum()
)


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def create_calibrated_classifier(
    estimator,
    method,
    cv
):
    """
    Create CalibratedClassifierCV while supporting both newer
    and older scikit-learn parameter names.
    """

    constructor_parameters = (
        inspect.signature(
            CalibratedClassifierCV
        ).parameters
    )

    keyword_arguments = {
        "method": method,
        "cv": cv,
        "ensemble": True
    }

    if "estimator" in constructor_parameters:
        keyword_arguments["estimator"] = estimator
    else:
        keyword_arguments["base_estimator"] = estimator

    return CalibratedClassifierCV(
        **keyword_arguments
    )


def build_calibration_candidate(
    base_pipeline,
    calibration_method
):
    if calibration_method == "Uncalibrated":
        return clone(base_pipeline)

    if calibration_method == "Sigmoid":
        return create_calibrated_classifier(
            estimator=clone(base_pipeline),
            method="sigmoid",
            cv=CALIBRATION_INTERNAL_FOLDS
        )

    if calibration_method == "Isotonic":
        return create_calibrated_classifier(
            estimator=clone(base_pipeline),
            method="isotonic",
            cv=CALIBRATION_INTERNAL_FOLDS
        )

    raise ValueError(
        f"Unknown calibration method: "
        f"{calibration_method}"
    )


def top_label_expected_calibration_error(
    y_true,
    probabilities,
    number_of_bins=10
):
    """
    Calculate top-label expected calibration error.

    This compares each prediction's maximum probability with
    whether its highest-probability class was correct.
    """

    predicted_classes = np.argmax(
        probabilities,
        axis=1
    )

    confidences = np.max(
        probabilities,
        axis=1
    )

    correctness = (
        predicted_classes
        == np.asarray(y_true)
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1
    )

    expected_calibration_error = 0.0

    for bin_index in range(number_of_bins):
        lower_bound = bin_edges[bin_index]
        upper_bound = bin_edges[bin_index + 1]

        if bin_index == number_of_bins - 1:
            in_bin = (
                (confidences >= lower_bound)
                & (confidences <= upper_bound)
            )
        else:
            in_bin = (
                (confidences >= lower_bound)
                & (confidences < upper_bound)
            )

        bin_count = int(in_bin.sum())

        if bin_count == 0:
            continue

        bin_accuracy = float(
            correctness[in_bin].mean()
        )

        bin_confidence = float(
            confidences[in_bin].mean()
        )

        bin_weight = (
            bin_count
            / len(confidences)
        )

        expected_calibration_error += (
            bin_weight
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(
        expected_calibration_error
    )


def calculate_calibration_metrics(
    y_true,
    probabilities
):
    predicted_classes = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            predicted_classes
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predicted_classes
            )
        ),

        "macro_f1": f1_score(
            y_true,
            predicted_classes,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            predicted_classes,
            average="weighted",
            zero_division=0
        ),

        "top_2_accuracy": (
            top_k_accuracy_score(
                y_true,
                probabilities,
                k=2,
                labels=FIXED_CLASS_INDICES
            )
        ),

        "multiclass_log_loss": log_loss(
            y_true,
            probabilities,
            labels=FIXED_CLASS_INDICES
        ),

        "multiclass_brier_sum": (
            multiclass_brier_score(
                y_true,
                probabilities
            )
        ),

        "top_label_ece": (
            top_label_expected_calibration_error(
                y_true,
                probabilities,
                number_of_bins=ECE_BINS
            )
        )
    }


def parse_saved_parameters(
    parameter_text
):
    if isinstance(
        parameter_text,
        dict
    ):
        return parameter_text

    return json.loads(
        parameter_text
    )


# ------------------------------------------------------------
# 4. Recreate the exact Step 4 outer folds
# ------------------------------------------------------------

calibration_outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

calibration_selection_records = []
calibration_outer_metric_records = []
calibration_outer_class_records = []
calibrated_oof_records = []

calibration_start_time = time.time()

for outer_fold, (
    outer_train_indices,
    outer_test_indices
) in enumerate(
    calibration_outer_cv.split(
        X_nested_source,
        y_nested
    ),
    start=1
):
    print("\n" + "=" * 80)
    print(
        f"CALIBRATION — OUTER FOLD "
        f"{outer_fold}/{OUTER_FOLDS}"
    )
    print("=" * 80)

    selected_row = (
        outer_selection_df.loc[
            outer_selection_df[
                "outer_fold"
            ] == outer_fold
        ]
    )

    if len(selected_row) != 1:
        raise RuntimeError(
            f"Expected one Step 4 selection for "
            f"outer fold {outer_fold}, but found "
            f"{len(selected_row)}."
        )

    selected_row = selected_row.iloc[0]

    selected_feature_set = (
        selected_row[
            "selected_feature_set"
        ]
    )

    selected_model_family = (
        selected_row[
            "selected_model_family"
        ]
    )

    selected_imbalance_strategy = (
        selected_row[
            "selected_imbalance_strategy"
        ]
    )

    if (
        selected_imbalance_strategy
        != "Balanced class weighting"
    ):
        raise RuntimeError(
            "Step 7 expects the primary Step 4 "
            "class-weighted workflow."
        )

    selected_features = (
        MODEL_FEATURE_SETS[
            selected_feature_set
        ]
    )

    selected_parameters = (
        parse_saved_parameters(
            selected_row[
                "selected_parameters"
            ]
        )
    )

    base_pipeline, _ = (
        build_candidate_pipeline(
            model_family=selected_model_family,
            class_weight="balanced"
        )
    )

    base_pipeline.set_params(
        **selected_parameters
    )

    X_outer_train = (
        X_nested_source.iloc[
            outer_train_indices
        ][selected_features]
        .reset_index(drop=True)
    )

    X_outer_test = (
        X_nested_source.iloc[
            outer_test_indices
        ][selected_features]
        .reset_index(drop=True)
    )

    y_outer_train = (
        y_nested.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_test = (
        y_nested.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_metadata = (
        nested_df.iloc[
            outer_test_indices
        ][
            [
                "household_ID",
                "cluster_name"
            ]
        ]
        .reset_index(drop=True)
    )

    calibration_selection_cv = (
        StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=(
                RANDOM_STATE
                + 100
                + outer_fold
            )
        )
    )

    fold_calibration_candidates = []

    for calibration_method in (
        CALIBRATION_METHODS
    ):
        candidate_estimator = (
            build_calibration_candidate(
                base_pipeline=base_pipeline,
                calibration_method=(
                    calibration_method
                )
            )
        )

        candidate_start_time = time.time()

        inner_probabilities = (
            cross_val_predict(
                estimator=candidate_estimator,
                X=X_outer_train,
                y=y_outer_train,
                cv=calibration_selection_cv,
                method="predict_proba",
                n_jobs=1
            )
        )

        candidate_runtime = (
            time.time()
            - candidate_start_time
        )

        if (
            inner_probabilities.shape[1]
            != len(FIXED_CLASS_INDICES)
        ):
            raise RuntimeError(
                "Unexpected probability-column count "
                "during calibration selection."
            )

        if not np.allclose(
            inner_probabilities.sum(axis=1),
            1.0,
            atol=1e-8
        ):
            raise RuntimeError(
                "Calibration-selection probabilities "
                "do not sum to one."
            )

        inner_metrics = (
            calculate_calibration_metrics(
                y_true=y_outer_train,
                probabilities=inner_probabilities
            )
        )

        candidate_record = {
            "outer_fold": outer_fold,
            "selected_feature_set": (
                selected_feature_set
            ),
            "selected_model_family": (
                selected_model_family
            ),
            "calibration_method": (
                calibration_method
            ),
            "selection_log_loss": (
                inner_metrics[
                    "multiclass_log_loss"
                ]
            ),
            "selection_brier_sum": (
                inner_metrics[
                    "multiclass_brier_sum"
                ]
            ),
            "selection_top_label_ece": (
                inner_metrics[
                    "top_label_ece"
                ]
            ),
            "selection_macro_f1": (
                inner_metrics[
                    "macro_f1"
                ]
            ),
            "selection_balanced_accuracy": (
                inner_metrics[
                    "balanced_accuracy"
                ]
            ),
            "runtime_seconds": (
                candidate_runtime
            )
        }

        calibration_selection_records.append(
            candidate_record
        )

        fold_calibration_candidates.append(
            candidate_record
        )

        print(
            f"{calibration_method}: "
            f"log loss="
            f"{inner_metrics['multiclass_log_loss']:.4f}, "
            f"Brier="
            f"{inner_metrics['multiclass_brier_sum']:.4f}, "
            f"ECE="
            f"{inner_metrics['top_label_ece']:.4f}, "
            f"macro-F1="
            f"{inner_metrics['macro_f1']:.4f}"
        )

    # --------------------------------------------------------
    # Select calibration method using outer-training data only
    # --------------------------------------------------------

    fold_calibration_candidates = sorted(
        fold_calibration_candidates,
        key=lambda record: (
            record[
                "selection_log_loss"
            ],
            record[
                "selection_brier_sum"
            ],
            record[
                "selection_top_label_ece"
            ],
            -record[
                "selection_macro_f1"
            ]
        )
    )

    selected_calibration = (
        fold_calibration_candidates[0]
    )

    selected_method = (
        selected_calibration[
            "calibration_method"
        ]
    )

    final_outer_estimator = (
        build_calibration_candidate(
            base_pipeline=base_pipeline,
            calibration_method=(
                selected_method
            )
        )
    )

    final_outer_estimator.fit(
        X_outer_train,
        y_outer_train
    )

    raw_outer_probabilities = (
        final_outer_estimator.predict_proba(
            X_outer_test
        )
    )

    outer_probabilities = (
        align_class_probabilities(
            raw_probabilities=(
                raw_outer_probabilities
            ),
            fitted_classes=(
                final_outer_estimator.classes_
            )
        )
    )

    outer_predictions = np.argmax(
        outer_probabilities,
        axis=1
    )

    outer_metrics = (
        calculate_calibration_metrics(
            y_true=y_outer_test,
            probabilities=outer_probabilities
        )
    )

    calibration_outer_metric_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_feature_set
        ),
        "selected_model_family": (
            selected_model_family
        ),
        "selected_calibration_method": (
            selected_method
        ),
        **outer_metrics
    })

    outer_report = classification_report(
        y_outer_test,
        outer_predictions,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in ORDERED_CLUSTER_NAMES:
        calibration_outer_class_records.append({
            "outer_fold": outer_fold,
            "calibration_method": (
                selected_method
            ),
            "class_name": class_name,
            "precision": (
                outer_report[
                    class_name
                ]["precision"]
            ),
            "recall": (
                outer_report[
                    class_name
                ]["recall"]
            ),
            "f1_score": (
                outer_report[
                    class_name
                ]["f1-score"]
            ),
            "support": int(
                outer_report[
                    class_name
                ]["support"]
            )
        })

    probability_rankings = np.argsort(
        outer_probabilities,
        axis=1
    )[:, ::-1]

    for row_position in range(
        len(outer_test_indices)
    ):
        first_class_index = int(
            probability_rankings[
                row_position,
                0
            ]
        )

        second_class_index = int(
            probability_rankings[
                row_position,
                1
            ]
        )

        prediction_record = {
            "household_ID": (
                outer_test_metadata.iloc[
                    row_position
                ]["household_ID"]
            ),
            "outer_fold": outer_fold,
            "true_class_index": int(
                y_outer_test.iloc[
                    row_position
                ]
            ),
            "true_class_name": (
                outer_test_metadata.iloc[
                    row_position
                ]["cluster_name"]
            ),
            "predicted_class_index": int(
                outer_predictions[
                    row_position
                ]
            ),
            "predicted_class_name": (
                INDEX_TO_CLASS[
                    int(
                        outer_predictions[
                            row_position
                        ]
                    )
                ]
            ),
            "selected_feature_set": (
                selected_feature_set
            ),
            "selected_model_family": (
                selected_model_family
            ),
            "selected_calibration_method": (
                selected_method
            ),
            "highest_probability": float(
                outer_probabilities[
                    row_position,
                    first_class_index
                ]
            ),
            "second_highest_probability": float(
                outer_probabilities[
                    row_position,
                    second_class_index
                ]
            ),
            "probability_margin": float(
                outer_probabilities[
                    row_position,
                    first_class_index
                ]
                - outer_probabilities[
                    row_position,
                    second_class_index
                ]
            )
        }

        for class_index, class_name in enumerate(
            ORDERED_CLUSTER_NAMES
        ):
            safe_class_name = (
                class_name
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
                .replace("/", "_")
            )

            prediction_record[
                f"probability_{safe_class_name}"
            ] = float(
                outer_probabilities[
                    row_position,
                    class_index
                ]
            )

        calibrated_oof_records.append(
            prediction_record
        )

    print("\nSelected calibration method:")
    print(selected_method)

    print(
        "Outer log loss:",
        round(
            outer_metrics[
                "multiclass_log_loss"
            ],
            4
        )
    )

    print(
        "Outer Brier score:",
        round(
            outer_metrics[
                "multiclass_brier_sum"
            ],
            4
        )
    )

    print(
        "Outer ECE:",
        round(
            outer_metrics[
                "top_label_ece"
            ],
            4
        )
    )


# ------------------------------------------------------------
# 5. Compile calibration outputs
# ------------------------------------------------------------

calibration_selection_df = pd.DataFrame(
    calibration_selection_records
)

calibration_outer_metrics_df = pd.DataFrame(
    calibration_outer_metric_records
)

calibration_outer_class_df = pd.DataFrame(
    calibration_outer_class_records
)

calibrated_oof_predictions_df = pd.DataFrame(
    calibrated_oof_records
)

calibration_runtime_seconds = (
    time.time()
    - calibration_start_time
)

if len(
    calibrated_oof_predictions_df
) != len(nested_df):
    raise RuntimeError(
        "Calibration did not create one OOF "
        "prediction per household."
    )

if calibrated_oof_predictions_df[
    "household_ID"
].duplicated().any():
    raise RuntimeError(
        "Duplicate calibrated OOF household "
        "predictions were generated."
    )


# ------------------------------------------------------------
# 6. Pooled calibrated performance
# ------------------------------------------------------------

calibrated_true = (
    calibrated_oof_predictions_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

calibrated_predicted = (
    calibrated_oof_predictions_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

calibrated_probability_columns = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_class_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    calibrated_probability_columns.append(
        f"probability_{safe_class_name}"
    )

calibrated_probabilities = (
    calibrated_oof_predictions_df[
        calibrated_probability_columns
    ]
    .to_numpy(dtype=float)
)

calibrated_pooled_metrics = (
    calculate_calibration_metrics(
        y_true=calibrated_true,
        probabilities=calibrated_probabilities
    )
)

calibrated_pooled_performance_df = pd.DataFrame([
    {
        "evaluation": (
            "Nested calibration-selected OOF"
        ),
        **calibrated_pooled_metrics
    }
])

calibrated_report = classification_report(
    calibrated_true,
    calibrated_predicted,
    labels=FIXED_CLASS_INDICES,
    target_names=ORDERED_CLUSTER_NAMES,
    output_dict=True,
    zero_division=0
)

calibrated_class_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    calibrated_class_records.append({
        "class_name": class_name,
        "precision": (
            calibrated_report[
                class_name
            ]["precision"]
        ),
        "recall": (
            calibrated_report[
                class_name
            ]["recall"]
        ),
        "f1_score": (
            calibrated_report[
                class_name
            ]["f1-score"]
        ),
        "support": int(
            calibrated_report[
                class_name
            ]["support"]
        )
    })

calibrated_class_metrics_df = pd.DataFrame(
    calibrated_class_records
)

calibrated_confusion_df = pd.DataFrame(
    confusion_matrix(
        calibrated_true,
        calibrated_predicted,
        labels=FIXED_CLASS_INDICES
    ),
    index=ORDERED_CLUSTER_NAMES,
    columns=ORDERED_CLUSTER_NAMES
)

calibration_selection_counts = (
    calibration_outer_metrics_df[
        "selected_calibration_method"
    ]
    .value_counts()
    .rename_axis(
        "calibration_method"
    )
    .reset_index(
        name="outer_folds_selected"
    )
)


# ------------------------------------------------------------
# 7. Compare calibrated and uncalibrated results
# ------------------------------------------------------------

uncalibrated_ece = (
    top_label_expected_calibration_error(
        y_true=(
            oof_predictions_df[
                "true_class_index"
            ].to_numpy(dtype=int)
        ),
        probabilities=(
            oof_predictions_df[
                probability_columns
            ].to_numpy(dtype=float)
        ),
        number_of_bins=ECE_BINS
    )
)

uncalibrated_comparison_row = {
    "evaluation": (
        "Primary Step 4 uncalibrated OOF"
    ),
    "accuracy": float(
        pooled_performance_df.iloc[0][
            "accuracy"
        ]
    ),
    "balanced_accuracy": float(
        pooled_performance_df.iloc[0][
            "balanced_accuracy"
        ]
    ),
    "macro_f1": float(
        pooled_performance_df.iloc[0][
            "macro_f1"
        ]
    ),
    "weighted_f1": float(
        pooled_performance_df.iloc[0][
            "weighted_f1"
        ]
    ),
    "top_2_accuracy": float(
        pooled_performance_df.iloc[0][
            "top_2_accuracy"
        ]
    ),
    "multiclass_log_loss": float(
        pooled_performance_df.iloc[0][
            "multiclass_log_loss"
        ]
    ),
    "multiclass_brier_sum": float(
        pooled_performance_df.iloc[0][
            "multiclass_brier_sum"
        ]
    ),
    "top_label_ece": float(
        uncalibrated_ece
    )
}

calibration_comparison_df = pd.DataFrame([
    uncalibrated_comparison_row,
    {
        "evaluation": (
            "Nested calibration-selected OOF"
        ),
        **calibrated_pooled_metrics
    }
])

calibration_difference_df = pd.DataFrame([
    {
        "metric": metric_name,
        "calibrated_minus_uncalibrated": (
            calibrated_pooled_metrics[
                metric_name
            ]
            - uncalibrated_comparison_row[
                metric_name
            ]
        )
    }
    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "weighted_f1",
        "top_2_accuracy",
        "multiclass_log_loss",
        "multiclass_brier_sum",
        "top_label_ece"
    ]
])


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 7 — CALIBRATION RESULTS")
print("=" * 80)

print("\nCalibration method selected by fold:")

display(
    calibration_outer_metrics_df[
        [
            "outer_fold",
            "selected_feature_set",
            "selected_model_family",
            "selected_calibration_method",
            "multiclass_log_loss",
            "multiclass_brier_sum",
            "top_label_ece",
            "macro_f1",
            "balanced_accuracy"
        ]
    ].round(4)
)

print("\nCalibration-selection frequency:")

display(
    calibration_selection_counts
)

print("\nPooled calibrated performance:")

display(
    calibrated_pooled_performance_df.round(4)
)

print("\nCalibrated class-specific performance:")

display(
    calibrated_class_metrics_df.round(4)
)

print("\nCalibrated confusion matrix:")

display(
    calibrated_confusion_df
)

print("\nCalibrated versus uncalibrated:")

display(
    calibration_comparison_df.round(4)
)

print("\nMetric differences:")

display(
    calibration_difference_df.round(4)
)

print(
    "\nCalibration runtime in minutes:",
    round(
        calibration_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 9. Save Step 7 outputs
# ------------------------------------------------------------

calibration_selection_df.to_csv(
    OUTPUT_DIR
    / "calibration_inner_selection_results.csv",
    index=False
)

calibration_outer_metrics_df.to_csv(
    OUTPUT_DIR
    / "calibration_outer_fold_metrics.csv",
    index=False
)

calibration_outer_class_df.to_csv(
    OUTPUT_DIR
    / "calibration_outer_class_metrics.csv",
    index=False
)

calibrated_oof_predictions_df.to_csv(
    OUTPUT_DIR
    / "calibrated_nested_oof_predictions.csv",
    index=False
)

calibrated_pooled_performance_df.to_csv(
    OUTPUT_DIR
    / "calibrated_nested_pooled_performance.csv",
    index=False
)

calibrated_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "calibrated_nested_class_metrics.csv",
    index=False
)

calibrated_confusion_df.to_csv(
    OUTPUT_DIR
    / "calibrated_nested_confusion_matrix.csv"
)

calibration_selection_counts.to_csv(
    OUTPUT_DIR
    / "calibration_selection_counts.csv",
    index=False
)

calibration_comparison_df.to_csv(
    OUTPUT_DIR
    / "calibrated_vs_uncalibrated_comparison.csv",
    index=False
)

calibration_difference_df.to_csv(
    OUTPUT_DIR
    / "calibration_metric_differences.csv",
    index=False
)

calibration_configuration = {
    "methods_compared": (
        CALIBRATION_METHODS
    ),
    "selection_metric": (
        "multiclass_log_loss"
    ),
    "first_tie_breaker": (
        "multiclass_brier_sum"
    ),
    "second_tie_breaker": (
        "top_label_expected_calibration_error"
    ),
    "selection_performed_inside_outer_training": True,
    "outer_test_used_for_selection": False,
    "calibration_internal_folds": (
        CALIBRATION_INTERNAL_FOLDS
    ),
    "expected_calibration_error_bins": (
        ECE_BINS
    ),
    "isotonic_interpretation": (
        f"Sensitivity method interpreted cautiously because "
        f"Daytime-Active has {daytime_calibration_count} matched households."
    )
}

with open(
    OUTPUT_DIR
    / "calibration_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        calibration_configuration,
        configuration_file,
        indent=4
    )

print("\n" + "=" * 80)
print("STEP 7 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Uncalibrated log loss:",
    round(
        uncalibrated_comparison_row[
            "multiclass_log_loss"
        ],
        4
    )
)

print(
    "Calibration-selected log loss:",
    round(
        calibrated_pooled_metrics[
            "multiclass_log_loss"
        ],
        4
    )
)

print(
    "Uncalibrated Brier score:",
    round(
        uncalibrated_comparison_row[
            "multiclass_brier_sum"
        ],
        4
    )
)

print(
    "Calibration-selected Brier score:",
    round(
        calibrated_pooled_metrics[
            "multiclass_brier_sum"
        ],
        4
    )
)

print(
    "\nSaved Step 7 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 8 — ARGMAX-PRESERVING TEMPERATURE CALIBRATION
# ============================================================
# Purpose:
# 1. Improve the probability quality of the primary Step 4
#    classifier without changing its class predictions.
# 2. Estimate a scalar temperature using only each outer
#    training partition.
# 3. Evaluate temperature-adjusted probabilities on untouched
#    outer-test households.
#
# Method:
# p_calibrated(c) ∝ p_original(c) ** (1 / temperature)
#
# Because the same positive monotonic transformation is applied
# to every class probability, the highest-probability class is
# preserved. Therefore, macro-F1, balanced accuracy and the
# confusion matrix should remain identical to Step 4.
# ============================================================

import json
import time

import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    top_k_accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP8_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "y_nested",
    "MODEL_FEATURE_SETS",
    "outer_selection_df",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "INDEX_TO_CLASS",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "build_candidate_pipeline",
    "align_class_probabilities",
    "multiclass_brier_score",
    "top_label_expected_calibration_error",
    "oof_predictions_df",
    "pooled_performance_df",
    "calibrated_pooled_performance_df"
]

missing_step8_objects = [
    object_name
    for object_name in REQUIRED_STEP8_OBJECTS
    if object_name not in globals()
]

if missing_step8_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 7 before Step 8. "
        f"Missing objects: {missing_step8_objects}"
    )


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

TEMPERATURE_LOWER_BOUND = 0.05
TEMPERATURE_UPPER_BOUND = 20.0
TEMPERATURE_TOLERANCE = 1e-5
PROBABILITY_EPSILON = 1e-12
TEMPERATURE_ECE_BINS = 10


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def parse_temperature_parameters(parameter_text):
    if isinstance(parameter_text, dict):
        return parameter_text

    return json.loads(parameter_text)


def apply_temperature(probabilities, temperature):
    """
    Apply scalar temperature transformation to multiclass
    probabilities while preserving class ordering.

    Temperature > 1 softens probabilities.
    Temperature < 1 sharpens probabilities.
    """

    if temperature <= 0:
        raise ValueError(
            "Temperature must be strictly positive."
        )

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    clipped_probabilities = np.clip(
        probabilities,
        PROBABILITY_EPSILON,
        1.0
    )

    log_probabilities = np.log(
        clipped_probabilities
    )

    scaled_log_probabilities = (
        log_probabilities
        / float(temperature)
    )

    scaled_log_probabilities = (
        scaled_log_probabilities
        - scaled_log_probabilities.max(
            axis=1,
            keepdims=True
        )
    )

    exponentiated = np.exp(
        scaled_log_probabilities
    )

    calibrated_probabilities = (
        exponentiated
        / exponentiated.sum(
            axis=1,
            keepdims=True
        )
    )

    if not np.allclose(
        calibrated_probabilities.sum(axis=1),
        1.0,
        atol=1e-8
    ):
        raise RuntimeError(
            "Temperature-adjusted probabilities do not sum to one."
        )

    return calibrated_probabilities


def fit_temperature(
    y_true,
    uncalibrated_probabilities
):
    """
    Select one positive temperature by minimizing multiclass
    log loss on out-of-fold predictions from the outer-training
    partition.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    uncalibrated_probabilities = np.asarray(
        uncalibrated_probabilities,
        dtype=float
    )

    def objective(temperature):
        adjusted_probabilities = apply_temperature(
            uncalibrated_probabilities,
            temperature
        )

        return log_loss(
            y_true,
            adjusted_probabilities,
            labels=FIXED_CLASS_INDICES
        )

    optimisation_result = minimize_scalar(
        objective,
        bounds=(
            TEMPERATURE_LOWER_BOUND,
            TEMPERATURE_UPPER_BOUND
        ),
        method="bounded",
        options={
            "xatol": TEMPERATURE_TOLERANCE
        }
    )

    if not optimisation_result.success:
        raise RuntimeError(
            "Temperature optimisation failed: "
            f"{optimisation_result.message}"
        )

    selected_temperature = float(
        optimisation_result.x
    )

    return (
        selected_temperature,
        float(optimisation_result.fun)
    )


def calculate_temperature_metrics(
    y_true,
    probabilities
):
    predicted_classes = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            predicted_classes
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predicted_classes
            )
        ),

        "macro_f1": f1_score(
            y_true,
            predicted_classes,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            predicted_classes,
            average="weighted",
            zero_division=0
        ),

        "top_2_accuracy": (
            top_k_accuracy_score(
                y_true,
                probabilities,
                k=2,
                labels=FIXED_CLASS_INDICES
            )
        ),

        "multiclass_log_loss": log_loss(
            y_true,
            probabilities,
            labels=FIXED_CLASS_INDICES
        ),

        "multiclass_brier_sum": (
            multiclass_brier_score(
                y_true,
                probabilities
            )
        ),

        "top_label_ece": (
            top_label_expected_calibration_error(
                y_true,
                probabilities,
                number_of_bins=(
                    TEMPERATURE_ECE_BINS
                )
            )
        )
    }


# ------------------------------------------------------------
# 4. Recreate the exact Step 4 outer folds
# ------------------------------------------------------------

temperature_outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

temperature_selection_records = []
temperature_outer_metric_records = []
temperature_oof_records = []

temperature_start_time = time.time()

for outer_fold, (
    outer_train_indices,
    outer_test_indices
) in enumerate(
    temperature_outer_cv.split(
        X_nested_source,
        y_nested
    ),
    start=1
):
    print("\n" + "=" * 80)

    print(
        f"TEMPERATURE CALIBRATION — OUTER FOLD "
        f"{outer_fold}/{OUTER_FOLDS}"
    )

    print("=" * 80)

    selected_rows = outer_selection_df.loc[
        outer_selection_df[
            "outer_fold"
        ] == outer_fold
    ]

    if len(selected_rows) != 1:
        raise RuntimeError(
            f"Expected exactly one Step 4 selection "
            f"for outer fold {outer_fold}."
        )

    selected_row = selected_rows.iloc[0]

    selected_feature_set = (
        selected_row[
            "selected_feature_set"
        ]
    )

    selected_model_family = (
        selected_row[
            "selected_model_family"
        ]
    )

    selected_imbalance_strategy = (
        selected_row[
            "selected_imbalance_strategy"
        ]
    )

    if (
        selected_imbalance_strategy
        != "Balanced class weighting"
    ):
        raise RuntimeError(
            "Step 8 requires the primary balanced "
            "class-weighted Step 4 workflow."
        )

    selected_features = MODEL_FEATURE_SETS[
        selected_feature_set
    ]

    selected_parameters = (
        parse_temperature_parameters(
            selected_row[
                "selected_parameters"
            ]
        )
    )

    base_pipeline, _ = (
        build_candidate_pipeline(
            model_family=selected_model_family,
            class_weight="balanced"
        )
    )

    base_pipeline.set_params(
        **selected_parameters
    )

    X_outer_train = (
        X_nested_source.iloc[
            outer_train_indices
        ][selected_features]
        .reset_index(drop=True)
    )

    X_outer_test = (
        X_nested_source.iloc[
            outer_test_indices
        ][selected_features]
        .reset_index(drop=True)
    )

    y_outer_train = (
        y_nested.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_test = (
        y_nested.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_metadata = (
        nested_df.iloc[
            outer_test_indices
        ][
            [
                "household_ID",
                "cluster_name"
            ]
        ]
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Fit temperature using outer-training OOF probabilities
    # --------------------------------------------------------

    temperature_inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_STATE
            + 300
            + outer_fold
        )
    )

    inner_uncalibrated_probabilities = (
        cross_val_predict(
            estimator=clone(base_pipeline),
            X=X_outer_train,
            y=y_outer_train,
            cv=temperature_inner_cv,
            method="predict_proba",
            n_jobs=1
        )
    )

    if (
        inner_uncalibrated_probabilities.shape[1]
        != len(FIXED_CLASS_INDICES)
    ):
        raise RuntimeError(
            "Unexpected inner probability-column count."
        )

    selected_temperature, selection_log_loss = (
        fit_temperature(
            y_true=y_outer_train,
            uncalibrated_probabilities=(
                inner_uncalibrated_probabilities
            )
        )
    )

    inner_temperature_probabilities = (
        apply_temperature(
            inner_uncalibrated_probabilities,
            selected_temperature
        )
    )

    inner_uncalibrated_log_loss = log_loss(
        y_outer_train,
        inner_uncalibrated_probabilities,
        labels=FIXED_CLASS_INDICES
    )

    inner_uncalibrated_brier = (
        multiclass_brier_score(
            y_outer_train,
            inner_uncalibrated_probabilities
        )
    )

    inner_temperature_brier = (
        multiclass_brier_score(
            y_outer_train,
            inner_temperature_probabilities
        )
    )

    # --------------------------------------------------------
    # Fit base classifier on the full outer-training partition
    # --------------------------------------------------------

    fitted_outer_pipeline = clone(
        base_pipeline
    )

    fitted_outer_pipeline.fit(
        X_outer_train,
        y_outer_train
    )

    raw_outer_probabilities = (
        fitted_outer_pipeline.predict_proba(
            X_outer_test
        )
    )

    uncalibrated_outer_probabilities = (
        align_class_probabilities(
            raw_probabilities=(
                raw_outer_probabilities
            ),
            fitted_classes=(
                fitted_outer_pipeline.classes_
            )
        )
    )

    temperature_outer_probabilities = (
        apply_temperature(
            uncalibrated_outer_probabilities,
            selected_temperature
        )
    )

    uncalibrated_outer_predictions = np.argmax(
        uncalibrated_outer_probabilities,
        axis=1
    )

    temperature_outer_predictions = np.argmax(
        temperature_outer_probabilities,
        axis=1
    )

    if not np.array_equal(
        uncalibrated_outer_predictions,
        temperature_outer_predictions
    ):
        raise RuntimeError(
            "Temperature transformation changed at least "
            "one class prediction. This should not occur."
        )

    uncalibrated_outer_metrics = (
        calculate_temperature_metrics(
            y_true=y_outer_test,
            probabilities=(
                uncalibrated_outer_probabilities
            )
        )
    )

    temperature_outer_metrics = (
        calculate_temperature_metrics(
            y_true=y_outer_test,
            probabilities=(
                temperature_outer_probabilities
            )
        )
    )

    temperature_selection_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_feature_set
        ),
        "selected_model_family": (
            selected_model_family
        ),
        "selected_temperature": (
            selected_temperature
        ),
        "inner_uncalibrated_log_loss": (
            inner_uncalibrated_log_loss
        ),
        "inner_temperature_log_loss": (
            selection_log_loss
        ),
        "inner_log_loss_improvement": (
            inner_uncalibrated_log_loss
            - selection_log_loss
        ),
        "inner_uncalibrated_brier": (
            inner_uncalibrated_brier
        ),
        "inner_temperature_brier": (
            inner_temperature_brier
        )
    })

    temperature_outer_metric_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_feature_set
        ),
        "selected_model_family": (
            selected_model_family
        ),
        "selected_temperature": (
            selected_temperature
        ),

        "uncalibrated_log_loss": (
            uncalibrated_outer_metrics[
                "multiclass_log_loss"
            ]
        ),
        "temperature_log_loss": (
            temperature_outer_metrics[
                "multiclass_log_loss"
            ]
        ),

        "uncalibrated_brier_sum": (
            uncalibrated_outer_metrics[
                "multiclass_brier_sum"
            ]
        ),
        "temperature_brier_sum": (
            temperature_outer_metrics[
                "multiclass_brier_sum"
            ]
        ),

        "uncalibrated_top_label_ece": (
            uncalibrated_outer_metrics[
                "top_label_ece"
            ]
        ),
        "temperature_top_label_ece": (
            temperature_outer_metrics[
                "top_label_ece"
            ]
        ),

        "macro_f1": (
            temperature_outer_metrics[
                "macro_f1"
            ]
        ),

        "balanced_accuracy": (
            temperature_outer_metrics[
                "balanced_accuracy"
            ]
        )
    })

    probability_ranking = np.argsort(
        temperature_outer_probabilities,
        axis=1
    )[:, ::-1]

    for row_position in range(
        len(outer_test_indices)
    ):
        first_class_index = int(
            probability_ranking[
                row_position,
                0
            ]
        )

        second_class_index = int(
            probability_ranking[
                row_position,
                1
            ]
        )

        prediction_record = {
            "household_ID": (
                outer_test_metadata.iloc[
                    row_position
                ]["household_ID"]
            ),
            "outer_fold": outer_fold,
            "true_class_index": int(
                y_outer_test.iloc[
                    row_position
                ]
            ),
            "true_class_name": (
                outer_test_metadata.iloc[
                    row_position
                ]["cluster_name"]
            ),
            "predicted_class_index": int(
                temperature_outer_predictions[
                    row_position
                ]
            ),
            "predicted_class_name": (
                INDEX_TO_CLASS[
                    int(
                        temperature_outer_predictions[
                            row_position
                        ]
                    )
                ]
            ),
            "selected_feature_set": (
                selected_feature_set
            ),
            "selected_model_family": (
                selected_model_family
            ),
            "selected_temperature": (
                selected_temperature
            ),
            "highest_probability": float(
                temperature_outer_probabilities[
                    row_position,
                    first_class_index
                ]
            ),
            "second_highest_probability": float(
                temperature_outer_probabilities[
                    row_position,
                    second_class_index
                ]
            ),
            "probability_margin": float(
                temperature_outer_probabilities[
                    row_position,
                    first_class_index
                ]
                - temperature_outer_probabilities[
                    row_position,
                    second_class_index
                ]
            )
        }

        for class_index, class_name in enumerate(
            ORDERED_CLUSTER_NAMES
        ):
            safe_class_name = (
                class_name
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
                .replace("/", "_")
            )

            prediction_record[
                f"probability_{safe_class_name}"
            ] = float(
                temperature_outer_probabilities[
                    row_position,
                    class_index
                ]
            )

        temperature_oof_records.append(
            prediction_record
        )

    print(
        "Selected temperature:",
        round(
            selected_temperature,
            4
        )
    )

    print(
        "Outer uncalibrated log loss:",
        round(
            uncalibrated_outer_metrics[
                "multiclass_log_loss"
            ],
            4
        )
    )

    print(
        "Outer temperature log loss:",
        round(
            temperature_outer_metrics[
                "multiclass_log_loss"
            ],
            4
        )
    )

    print(
        "Class predictions changed:",
        False
    )


# ------------------------------------------------------------
# 5. Compile OOF outputs
# ------------------------------------------------------------

temperature_selection_df = pd.DataFrame(
    temperature_selection_records
)

temperature_outer_metrics_df = pd.DataFrame(
    temperature_outer_metric_records
)

temperature_oof_predictions_df = pd.DataFrame(
    temperature_oof_records
)

temperature_runtime_seconds = (
    time.time()
    - temperature_start_time
)

if len(temperature_oof_predictions_df) != len(
    nested_df
):
    raise RuntimeError(
        "Temperature calibration did not produce one "
        "OOF prediction per household."
    )

if temperature_oof_predictions_df[
    "household_ID"
].duplicated().any():
    raise RuntimeError(
        "Duplicate temperature-calibrated OOF predictions "
        "were created."
    )


# ------------------------------------------------------------
# 6. Confirm exact agreement with primary Step 4 decisions
# ------------------------------------------------------------

step4_decisions = (
    oof_predictions_df[
        [
            "household_ID",
            "predicted_class_index"
        ]
    ]
    .rename(
        columns={
            "predicted_class_index":
            "step4_predicted_class_index"
        }
    )
)

temperature_decision_check = (
    temperature_oof_predictions_df[
        [
            "household_ID",
            "predicted_class_index"
        ]
    ]
    .rename(
        columns={
            "predicted_class_index":
            "temperature_predicted_class_index"
        }
    )
    .merge(
        step4_decisions,
        on="household_ID",
        how="inner",
        validate="one_to_one"
    )
)

temperature_decision_check[
    "same_prediction"
] = (
    temperature_decision_check[
        "temperature_predicted_class_index"
    ]
    == temperature_decision_check[
        "step4_predicted_class_index"
    ]
)

if not temperature_decision_check[
    "same_prediction"
].all():
    raise RuntimeError(
        "Temperature OOF predictions do not exactly match "
        "the primary Step 4 class decisions."
    )


# ------------------------------------------------------------
# 7. Pooled temperature-calibrated performance
# ------------------------------------------------------------

temperature_true = (
    temperature_oof_predictions_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

temperature_predicted = (
    temperature_oof_predictions_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

temperature_probability_columns = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_class_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    temperature_probability_columns.append(
        f"probability_{safe_class_name}"
    )

temperature_probabilities = (
    temperature_oof_predictions_df[
        temperature_probability_columns
    ]
    .to_numpy(dtype=float)
)

temperature_pooled_metrics = (
    calculate_temperature_metrics(
        y_true=temperature_true,
        probabilities=temperature_probabilities
    )
)

temperature_pooled_performance_df = pd.DataFrame([
    {
        "evaluation": (
            "Argmax-preserving temperature-calibrated OOF"
        ),
        **temperature_pooled_metrics
    }
])

temperature_report = classification_report(
    temperature_true,
    temperature_predicted,
    labels=FIXED_CLASS_INDICES,
    target_names=ORDERED_CLUSTER_NAMES,
    output_dict=True,
    zero_division=0
)

temperature_class_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    temperature_class_records.append({
        "class_name": class_name,
        "precision": (
            temperature_report[
                class_name
            ]["precision"]
        ),
        "recall": (
            temperature_report[
                class_name
            ]["recall"]
        ),
        "f1_score": (
            temperature_report[
                class_name
            ]["f1-score"]
        ),
        "support": int(
            temperature_report[
                class_name
            ]["support"]
        )
    })

temperature_class_metrics_df = pd.DataFrame(
    temperature_class_records
)

temperature_confusion_df = pd.DataFrame(
    confusion_matrix(
        temperature_true,
        temperature_predicted,
        labels=FIXED_CLASS_INDICES
    ),
    index=ORDERED_CLUSTER_NAMES,
    columns=ORDERED_CLUSTER_NAMES
)


# ------------------------------------------------------------
# 8. Compare all three probability approaches
# ------------------------------------------------------------

step4_uncalibrated_row = {
    "evaluation": (
        "Primary Step 4 uncalibrated OOF"
    ),
    "accuracy": float(
        pooled_performance_df.iloc[0][
            "accuracy"
        ]
    ),
    "balanced_accuracy": float(
        pooled_performance_df.iloc[0][
            "balanced_accuracy"
        ]
    ),
    "macro_f1": float(
        pooled_performance_df.iloc[0][
            "macro_f1"
        ]
    ),
    "weighted_f1": float(
        pooled_performance_df.iloc[0][
            "weighted_f1"
        ]
    ),
    "top_2_accuracy": float(
        pooled_performance_df.iloc[0][
            "top_2_accuracy"
        ]
    ),
    "multiclass_log_loss": float(
        pooled_performance_df.iloc[0][
            "multiclass_log_loss"
        ]
    ),
    "multiclass_brier_sum": float(
        pooled_performance_df.iloc[0][
            "multiclass_brier_sum"
        ]
    ),
    "top_label_ece": float(
        top_label_expected_calibration_error(
            oof_predictions_df[
                "true_class_index"
            ].to_numpy(dtype=int),
            oof_predictions_df[
                probability_columns
            ].to_numpy(dtype=float),
            number_of_bins=(
                TEMPERATURE_ECE_BINS
            )
        )
    )
}

sigmoid_row = (
    calibrated_pooled_performance_df
    .iloc[0]
    .to_dict()
)

probability_method_comparison_df = pd.DataFrame([
    step4_uncalibrated_row,
    {
        "evaluation": (
            "Sigmoid calibration-selected OOF"
        ),
        **{
            key: sigmoid_row[key]
            for key in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
                "top_2_accuracy",
                "multiclass_log_loss",
                "multiclass_brier_sum",
                "top_label_ece"
            ]
        }
    },
    {
        "evaluation": (
            "Argmax-preserving temperature-calibrated OOF"
        ),
        **temperature_pooled_metrics
    }
])


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 8 — TEMPERATURE CALIBRATION RESULTS")
print("=" * 80)

print("\nSelected temperature by outer fold:")

display(
    temperature_selection_df.round(4)
)

print("\nOuter-fold probability metrics:")

display(
    temperature_outer_metrics_df.round(4)
)

print("\nPooled temperature-calibrated performance:")

display(
    temperature_pooled_performance_df.round(4)
)

print("\nClass-specific performance:")

display(
    temperature_class_metrics_df.round(4)
)

print("\nConfusion matrix:")

display(
    temperature_confusion_df
)

print("\nProbability-method comparison:")

display(
    probability_method_comparison_df.round(4)
)

print(
    "\nExact Step 4 decision agreement:",
    float(
        temperature_decision_check[
            "same_prediction"
        ].mean()
    )
)

print(
    "Temperature-calibration runtime in minutes:",
    round(
        temperature_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

temperature_selection_df.to_csv(
    OUTPUT_DIR
    / "temperature_calibration_selection.csv",
    index=False
)

temperature_outer_metrics_df.to_csv(
    OUTPUT_DIR
    / "temperature_calibration_outer_metrics.csv",
    index=False
)

temperature_oof_predictions_df.to_csv(
    OUTPUT_DIR
    / "temperature_calibrated_oof_predictions.csv",
    index=False
)

temperature_pooled_performance_df.to_csv(
    OUTPUT_DIR
    / "temperature_calibrated_pooled_performance.csv",
    index=False
)

temperature_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "temperature_calibrated_class_metrics.csv",
    index=False
)

temperature_confusion_df.to_csv(
    OUTPUT_DIR
    / "temperature_calibrated_confusion_matrix.csv"
)

probability_method_comparison_df.to_csv(
    OUTPUT_DIR
    / "probability_calibration_method_comparison.csv",
    index=False
)

temperature_decision_check.to_csv(
    OUTPUT_DIR
    / "temperature_step4_decision_agreement.csv",
    index=False
)

temperature_configuration = {
    "method": (
        "Scalar argmax-preserving probability "
        "temperature transformation"
    ),
    "selection_objective": (
        "Multiclass log loss"
    ),
    "temperature_lower_bound": (
        TEMPERATURE_LOWER_BOUND
    ),
    "temperature_upper_bound": (
        TEMPERATURE_UPPER_BOUND
    ),
    "temperature_selected_inside_outer_training": True,
    "outer_test_used_for_selection": False,
    "class_predictions_preserved": True,
    "sigmoid_used_for_final_class_predictions": False,
    "sigmoid_status": (
        "Supplementary probability-calibration sensitivity only"
    ),
    "primary_classification_source": (
        "Step 4 uncalibrated balanced class-weighted workflow"
    )
}

with open(
    OUTPUT_DIR
    / "temperature_calibration_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        temperature_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 8 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Primary macro-F1 retained:",
    round(
        temperature_pooled_metrics[
            "macro_f1"
        ],
        4
    )
)

print(
    "Primary balanced accuracy retained:",
    round(
        temperature_pooled_metrics[
            "balanced_accuracy"
        ],
        4
    )
)

print(
    "Temperature log loss:",
    round(
        temperature_pooled_metrics[
            "multiclass_log_loss"
        ],
        4
    )
)

print(
    "Temperature Brier score:",
    round(
        temperature_pooled_metrics[
            "multiclass_brier_sum"
        ],
        4
    )
)

print(
    "Exact decision agreement:",
    round(
        temperature_decision_check[
            "same_prediction"
        ].mean(),
        4
    )
)

print(
    "\nSaved Step 8 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 9 — REPEATED-CV FINAL FIXED-WORKFLOW SELECTION
# ============================================================
# Purpose:
# 1. Select one stable feature-set, model-family and
#    hyperparameter configuration for final full-data fitting.
# 2. Compare only balanced class-weighted workflows because
#    Steps 4 and 5 established this as the preferred imbalance
#    treatment.
# 3. Use repeated stratified CV to reduce dependence on one
#    particular fold allocation.
#
# Important:
# - This step selects the fixed research-prototype configuration.
# - It does not replace the unbiased Step 4 nested OOF estimate.
# - Oversampling and sigmoid calibration are excluded.
# - No final model is fitted or saved yet.
# ============================================================

import json
import re
import time

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold
)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP9_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "y_nested",
    "MODEL_FEATURE_SETS",
    "ORDERED_CLUSTER_NAMES",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "SCORING",
    "build_candidate_pipeline",
    "serialise_parameters",
    "pooled_performance_df"
]

missing_step9_objects = [
    object_name
    for object_name in REQUIRED_STEP9_OBJECTS
    if object_name not in globals()
]

if missing_step9_objects:
    raise RuntimeError(
        "Run Steps 1–4 before Step 9. Missing objects: "
        f"{missing_step9_objects}"
    )


# ------------------------------------------------------------
# 2. Repeated-CV configuration
# ------------------------------------------------------------

FINAL_SELECTION_FOLDS = 5
FINAL_SELECTION_REPEATS = 5
FINAL_SELECTION_TOTAL_SPLITS = (
    FINAL_SELECTION_FOLDS
    * FINAL_SELECTION_REPEATS
)

FINAL_SELECTION_MODEL_FAMILIES = [
    "Logistic Regression",
    "Random Forest",
    "Extra Trees"
]

final_selection_cv = RepeatedStratifiedKFold(
    n_splits=FINAL_SELECTION_FOLDS,
    n_repeats=FINAL_SELECTION_REPEATS,
    random_state=RANDOM_STATE
)


# ------------------------------------------------------------
# 3. Run repeated-CV search for each fixed workflow
# ------------------------------------------------------------

workflow_summary_records = []
workflow_split_records = []
workflow_parameter_lookup = {}
workflow_pipeline_lookup = {}
workflow_feature_lookup = {}

selection_start_time = time.time()

for feature_set_name, feature_columns in (
    MODEL_FEATURE_SETS.items()
):
    X_workflow = X_nested_source[
        feature_columns
    ].copy()

    for model_family in (
        FINAL_SELECTION_MODEL_FAMILIES
    ):
        workflow_id = (
            f"{feature_set_name} | "
            f"{model_family} | "
            "Balanced class weighting"
        )

        print("\n" + "=" * 80)
        print("FINAL FIXED-WORKFLOW SEARCH")
        print(workflow_id)
        print("=" * 80)

        pipeline, parameter_grid = (
            build_candidate_pipeline(
                model_family=model_family,
                class_weight="balanced"
            )
        )

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=parameter_grid,
            scoring=SCORING,
            refit=False,
            cv=final_selection_cv,
            n_jobs=-1,
            pre_dispatch="2*n_jobs",
            return_train_score=True,
            error_score="raise"
        )

        workflow_start_time = time.time()

        search.fit(
            X_workflow,
            y_nested
        )

        workflow_runtime = (
            time.time()
            - workflow_start_time
        )

        search_results = pd.DataFrame(
            search.cv_results_
        )

        search_results = (
            search_results
            .sort_values(
                [
                    "mean_test_macro_f1",
                    "mean_test_balanced_accuracy",
                    "mean_test_weighted_f1",
                    "mean_test_accuracy"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    False
                ]
            )
            .reset_index(drop=True)
        )

        best_result = select_balanced_cv_candidate(
            search_results,
            model_family
        )
        best_parameters = dict(
            best_result["params"]
        )

        workflow_parameter_lookup[
            workflow_id
        ] = best_parameters

        workflow_pipeline_lookup[
            workflow_id
        ] = pipeline

        workflow_feature_lookup[
            workflow_id
        ] = list(feature_columns)

        summary_record = {
            "workflow_id": workflow_id,
            "feature_set": feature_set_name,
            "feature_count": len(feature_columns),
            "model_family": model_family,
            "imbalance_strategy": (
                "Balanced class weighting"
            ),
            "selected_parameters": (
                serialise_parameters(
                    best_parameters
                )
            ),
            "mean_macro_f1": float(
                best_result[
                    "mean_test_macro_f1"
                ]
            ),
            "std_macro_f1": float(
                best_result[
                    "std_test_macro_f1"
                ]
            ),
            "naive_descriptive_standard_error_macro_f1": float(
                best_result[
                    "std_test_macro_f1"
                ]
                / np.sqrt(
                    FINAL_SELECTION_TOTAL_SPLITS
                )
            ),
            "mean_balanced_accuracy": float(
                best_result[
                    "mean_test_balanced_accuracy"
                ]
            ),
            "std_balanced_accuracy": float(
                best_result[
                    "std_test_balanced_accuracy"
                ]
            ),
            "mean_weighted_f1": float(
                best_result[
                    "mean_test_weighted_f1"
                ]
            ),
            "mean_accuracy": float(
                best_result[
                    "mean_test_accuracy"
                ]
            ),
            "mean_train_macro_f1": float(
                best_result[
                    "mean_train_macro_f1"
                ]
            ),
            "train_validation_macro_f1_gap": float(
                best_result[
                    "mean_train_macro_f1"
                ]
                - best_result[
                    "mean_test_macro_f1"
                ]
            ),
            "runtime_seconds": float(
                workflow_runtime
            )
        }

        workflow_summary_records.append(
            summary_record
        )

        # ----------------------------------------------------
        # Save split-specific scores for the chosen parameter
        # configuration within this workflow
        # ----------------------------------------------------

        split_macro_columns = sorted(
            [
                column
                for column in search_results.columns
                if re.fullmatch(
                    r"split\d+_test_macro_f1",
                    column
                )
            ],
            key=lambda column: int(
                re.search(
                    r"split(\d+)",
                    column
                ).group(1)
            )
        )

        for macro_column in split_macro_columns:
            split_index = int(
                re.search(
                    r"split(\d+)",
                    macro_column
                ).group(1)
            )

            balanced_column = (
                f"split{split_index}_"
                "test_balanced_accuracy"
            )

            weighted_column = (
                f"split{split_index}_"
                "test_weighted_f1"
            )

            accuracy_column = (
                f"split{split_index}_"
                "test_accuracy"
            )

            workflow_split_records.append({
                "workflow_id": workflow_id,
                "feature_set": feature_set_name,
                "model_family": model_family,
                "split_index": split_index,
                "repeat_index": (
                    split_index
                    // FINAL_SELECTION_FOLDS
                ) + 1,
                "fold_within_repeat": (
                    split_index
                    % FINAL_SELECTION_FOLDS
                ) + 1,
                "macro_f1": float(
                    best_result[
                        macro_column
                    ]
                ),
                "balanced_accuracy": float(
                    best_result[
                        balanced_column
                    ]
                ),
                "weighted_f1": float(
                    best_result[
                        weighted_column
                    ]
                ),
                "accuracy": float(
                    best_result[
                        accuracy_column
                    ]
                )
            })

        print(
            "Selected parameters:",
            best_parameters
        )

        print(
            "Repeated-CV macro-F1:",
            round(
                summary_record[
                    "mean_macro_f1"
                ],
                4
            ),
            "±",
            round(
                summary_record[
                    "std_macro_f1"
                ],
                4
            )
        )

        print(
            "Repeated-CV balanced accuracy:",
            round(
                summary_record[
                    "mean_balanced_accuracy"
                ],
                4
            )
        )


# ------------------------------------------------------------
# 4. Rank fixed workflows
# ------------------------------------------------------------

fixed_workflow_summary_df = pd.DataFrame(
    workflow_summary_records
)

fixed_workflow_summary_df = (
    fixed_workflow_summary_df
    .sort_values(
        [
            "mean_macro_f1",
            "mean_balanced_accuracy",
            "mean_weighted_f1",
            "mean_accuracy"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

fixed_workflow_summary_df[
    "selection_rank"
] = np.arange(
    1,
    len(fixed_workflow_summary_df) + 1
)

fixed_workflow_split_df = pd.DataFrame(
    workflow_split_records
)


# ------------------------------------------------------------
# 5. Diagnostic split-level winning frequency
# ------------------------------------------------------------

split_winners = (
    fixed_workflow_split_df
    .sort_values(
        [
            "split_index",
            "macro_f1",
            "balanced_accuracy",
            "weighted_f1",
            "accuracy"
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False
        ]
    )
    .groupby(
        "split_index",
        as_index=False
    )
    .head(1)
)

split_win_counts = (
    split_winners[
        "workflow_id"
    ]
    .value_counts()
    .rename_axis(
        "workflow_id"
    )
    .reset_index(
        name="split_wins"
    )
)

fixed_workflow_summary_df = (
    fixed_workflow_summary_df
    .merge(
        split_win_counts,
        on="workflow_id",
        how="left"
    )
)

fixed_workflow_summary_df[
    "split_wins"
] = (
    fixed_workflow_summary_df[
        "split_wins"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 6. Lock the final fixed workflow
# ------------------------------------------------------------

selected_fixed_row = (
    fixed_workflow_summary_df
    .sort_values(
        [
            "mean_macro_f1",
            "mean_balanced_accuracy",
            "mean_weighted_f1",
            "mean_accuracy"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .iloc[0]
)

FINAL_WORKFLOW_ID = (
    selected_fixed_row[
        "workflow_id"
    ]
)

FINAL_WORKFLOW_FEATURE_SET = (
    selected_fixed_row[
        "feature_set"
    ]
)

FINAL_WORKFLOW_MODEL_FAMILY = (
    selected_fixed_row[
        "model_family"
    ]
)

FINAL_WORKFLOW_FEATURES = (
    workflow_feature_lookup[
        FINAL_WORKFLOW_ID
    ]
)

FINAL_WORKFLOW_PARAMS = (
    workflow_parameter_lookup[
        FINAL_WORKFLOW_ID
    ]
)

final_workflow_pipeline_template = clone(
    workflow_pipeline_lookup[
        FINAL_WORKFLOW_ID
    ]
)

final_workflow_pipeline_template.set_params(
    **FINAL_WORKFLOW_PARAMS
)


# ------------------------------------------------------------
# 7. Compare selection-stage and evaluation-stage results
# ------------------------------------------------------------

selection_vs_evaluation_df = pd.DataFrame([
    {
        "result_type": (
            "Primary approximately unbiased internal estimate"
        ),
        "source": (
            "Step 4 nested out-of-fold selection procedure"
        ),
        "macro_f1": float(
            pooled_performance_df.iloc[0][
                "macro_f1"
            ]
        ),
        "balanced_accuracy": float(
            pooled_performance_df.iloc[0][
                "balanced_accuracy"
            ]
        ),
        "accuracy": float(
            pooled_performance_df.iloc[0][
                "accuracy"
            ]
        ),
        "interpretation": (
            "Use for thesis reporting as the estimate of the adaptive nested development procedure."
        )
    },
    {
        "result_type": (
            "Final fixed-workflow selection estimate"
        ),
        "source": (
            "Step 9 repeated stratified cross-validation"
        ),
        "macro_f1": float(
            selected_fixed_row[
                "mean_macro_f1"
            ]
        ),
        "balanced_accuracy": float(
            selected_fixed_row[
                "mean_balanced_accuracy"
            ]
        ),
        "accuracy": float(
            selected_fixed_row[
                "mean_accuracy"
            ]
        ),
        "interpretation": (
            "Use only to choose the final deployable "
            "configuration; do not replace Step 4 results."
        )
    }
])


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

selection_runtime_seconds = (
    time.time()
    - selection_start_time
)

print("\n" + "=" * 80)
print("STEP 9 — FIXED-WORKFLOW SELECTION RESULTS")
print("=" * 80)

print("\nRanked fixed workflows:")

display(
    fixed_workflow_summary_df[
        [
            "selection_rank",
            "workflow_id",
            "feature_count",
            "selected_parameters",
            "mean_macro_f1",
            "std_macro_f1",
            "naive_descriptive_standard_error_macro_f1",
            "mean_balanced_accuracy",
            "std_balanced_accuracy",
            "mean_weighted_f1",
            "mean_accuracy",
            "mean_train_macro_f1",
            "train_validation_macro_f1_gap",
            "split_wins"
        ]
    ].round(4)
)

print("\nSplit-level winning frequency:")

display(
    split_win_counts
)

print("\nSelected final fixed workflow:")

selected_fixed_display = pd.DataFrame([
    {
        "workflow_id": (
            FINAL_WORKFLOW_ID
        ),
        "feature_set": (
            FINAL_WORKFLOW_FEATURE_SET
        ),
        "feature_count": len(
            FINAL_WORKFLOW_FEATURES
        ),
        "model_family": (
            FINAL_WORKFLOW_MODEL_FAMILY
        ),
        "imbalance_strategy": (
            "Balanced class weighting"
        ),
        "selected_parameters": (
            serialise_parameters(
                FINAL_WORKFLOW_PARAMS
            )
        ),
        "repeated_cv_macro_f1": float(
            selected_fixed_row[
                "mean_macro_f1"
            ]
        ),
        "repeated_cv_balanced_accuracy": float(
            selected_fixed_row[
                "mean_balanced_accuracy"
            ]
        )
    }
])

display(
    selected_fixed_display
)

print("\nPerformance reporting distinction:")

display(
    selection_vs_evaluation_df.round(4)
)

print(
    "\nStep 9 runtime in minutes:",
    round(
        selection_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

fixed_workflow_summary_df.to_csv(
    OUTPUT_DIR
    / "final_fixed_workflow_repeated_cv_summary.csv",
    index=False
)

fixed_workflow_split_df.to_csv(
    OUTPUT_DIR
    / "final_fixed_workflow_split_scores.csv",
    index=False
)

split_winners.to_csv(
    OUTPUT_DIR
    / "final_fixed_workflow_split_winners.csv",
    index=False
)

split_win_counts.to_csv(
    OUTPUT_DIR
    / "final_fixed_workflow_split_win_counts.csv",
    index=False
)

selected_fixed_display.to_csv(
    OUTPUT_DIR
    / "selected_final_fixed_workflow.csv",
    index=False
)

selection_vs_evaluation_df.to_csv(
    OUTPUT_DIR
    / "selection_vs_primary_evaluation_results.csv",
    index=False
)

final_workflow_configuration = {
    "workflow_id": (
        FINAL_WORKFLOW_ID
    ),
    "feature_set": (
        FINAL_WORKFLOW_FEATURE_SET
    ),
    "features": (
        FINAL_WORKFLOW_FEATURES
    ),
    "feature_count": len(
        FINAL_WORKFLOW_FEATURES
    ),
    "model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "imbalance_strategy": (
        "Balanced class weighting"
    ),
    "selected_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "selection_folds": (
        FINAL_SELECTION_FOLDS
    ),
    "selection_repeats": (
        FINAL_SELECTION_REPEATS
    ),
    "total_selection_splits": (
        FINAL_SELECTION_TOTAL_SPLITS
    ),
    "selection_metric": (
        "macro_f1"
    ),
    "first_tie_breaker": (
        "balanced_accuracy"
    ),
    "oversampling_used": False,
    "sigmoid_calibration_used_for_labels": False,
    "primary_performance_source": (
        "Step 4 nested out-of-fold results"
    ),
    "repeated_cv_role": (
        "Final fixed-workflow selection only; secondary post-selection estimate"
    )
}

with open(
    OUTPUT_DIR
    / "selected_final_fixed_workflow.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        final_workflow_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 9 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Selected feature set:",
    FINAL_WORKFLOW_FEATURE_SET
)

print(
    "Selected model family:",
    FINAL_WORKFLOW_MODEL_FAMILY
)

print(
    "Selected parameters:",
    FINAL_WORKFLOW_PARAMS
)

print(
    "Selected predictor count:",
    len(FINAL_WORKFLOW_FEATURES)
)

print(
    "\nPrimary thesis macro-F1 remains:",
    round(
        float(
            pooled_performance_df.iloc[0][
                "macro_f1"
            ]
        ),
        4
    )
)

print(
    "Primary thesis balanced accuracy remains:",
    round(
        float(
            pooled_performance_df.iloc[0][
                "balanced_accuracy"
            ]
        ),
        4
    )
)

print(
    "\nSaved Step 9 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 9A — PRACTICAL-TIE AND WORKFLOW-STABILITY DECISION
# ============================================================
# Purpose:
# 1. Compare the two strongest fixed workflows using identical
#    repeated-CV splits.
# 2. Avoid selecting a final model based on a negligible
#    macro-F1 difference.
# 3. Prefer the more stable and better-balanced workflow when
#    predictive performance is practically equivalent.
#
# Important:
# - This is a deployable-model selection rule.
# - It does not replace the Step 4 nested OOF evaluation.
# - No formal independence assumption is made for the 25
#   overlapping repeated-CV fold scores.
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.base import clone


# ------------------------------------------------------------
# 1. Confirm Step 9 objects
# ------------------------------------------------------------

REQUIRED_STEP9A_OBJECTS = [
    "fixed_workflow_summary_df",
    "fixed_workflow_split_df",
    "workflow_feature_lookup",
    "workflow_parameter_lookup",
    "workflow_pipeline_lookup",
    "OUTPUT_DIR",
    "serialise_parameters",
    "pooled_performance_df"
]

missing_step9a_objects = [
    object_name
    for object_name in REQUIRED_STEP9A_OBJECTS
    if object_name not in globals()
]

if missing_step9a_objects:
    raise RuntimeError(
        "Run Step 9 first. Missing objects: "
        f"{missing_step9a_objects}"
    )


# ------------------------------------------------------------
# 2. Predeclare the practical-equivalence rule
# ------------------------------------------------------------

PRACTICAL_MACRO_F1_MARGIN = 0.01
PRACTICAL_BALANCED_ACCURACY_MARGIN = 0.01
PRACTICAL_TRAIN_VALIDATION_GAP_MARGIN = 0.01

TOP_FEATURE_SET = "Extended reduced"

TOP_MODEL_FAMILIES = [
    "Random Forest",
    "Extra Trees"
]


# ------------------------------------------------------------
# 3. Extract the two leading extended workflows
# ------------------------------------------------------------

top_workflows = (
    fixed_workflow_summary_df.loc[
        (
            fixed_workflow_summary_df[
                "feature_set"
            ] == TOP_FEATURE_SET
        )
        & (
            fixed_workflow_summary_df[
                "model_family"
            ].isin(TOP_MODEL_FAMILIES)
        )
    ]
    .copy()
)

if len(top_workflows) != 2:
    raise RuntimeError(
        "Expected exactly two leading extended tree workflows, "
        f"but found {len(top_workflows)}."
    )

top_workflows = (
    top_workflows
    .sort_values(
        "model_family"
    )
    .reset_index(drop=True)
)

random_forest_row = (
    top_workflows.loc[
        top_workflows[
            "model_family"
        ] == "Random Forest"
    ]
)

extra_trees_row = (
    top_workflows.loc[
        top_workflows[
            "model_family"
        ] == "Extra Trees"
    ]
)

if (
    len(random_forest_row) != 1
    or len(extra_trees_row) != 1
):
    raise RuntimeError(
        "Could not uniquely identify the Random Forest and "
        "Extra Trees workflows."
    )

random_forest_row = random_forest_row.iloc[0]
extra_trees_row = extra_trees_row.iloc[0]

random_forest_id = (
    random_forest_row["workflow_id"]
)

extra_trees_id = (
    extra_trees_row["workflow_id"]
)


# ------------------------------------------------------------
# 4. Paired comparison across identical CV splits
# ------------------------------------------------------------

paired_split_source = (
    fixed_workflow_split_df.loc[
        fixed_workflow_split_df[
            "workflow_id"
        ].isin(
            [
                random_forest_id,
                extra_trees_id
            ]
        )
    ]
    .copy()
)

paired_macro_f1 = (
    paired_split_source
    .pivot(
        index=[
            "repeat_index",
            "fold_within_repeat",
            "split_index"
        ],
        columns="workflow_id",
        values="macro_f1"
    )
    .reset_index()
)

paired_balanced_accuracy = (
    paired_split_source
    .pivot(
        index=[
            "repeat_index",
            "fold_within_repeat",
            "split_index"
        ],
        columns="workflow_id",
        values="balanced_accuracy"
    )
    .reset_index()
)

if (
    len(paired_macro_f1) != 25
    or len(paired_balanced_accuracy) != 25
):
    raise RuntimeError(
        "Expected 25 paired repeated-CV split results."
    )

paired_split_comparison = (
    paired_macro_f1[
        [
            "repeat_index",
            "fold_within_repeat",
            "split_index",
            random_forest_id,
            extra_trees_id
        ]
    ]
    .rename(
        columns={
            random_forest_id:
                "random_forest_macro_f1",

            extra_trees_id:
                "extra_trees_macro_f1"
        }
    )
    .merge(
        paired_balanced_accuracy[
            [
                "repeat_index",
                "fold_within_repeat",
                "split_index",
                random_forest_id,
                extra_trees_id
            ]
        ].rename(
            columns={
                random_forest_id:
                    "random_forest_balanced_accuracy",

                extra_trees_id:
                    "extra_trees_balanced_accuracy"
            }
        ),
        on=[
            "repeat_index",
            "fold_within_repeat",
            "split_index"
        ],
        how="inner",
        validate="one_to_one"
    )
)

paired_split_comparison[
    "macro_f1_difference_rf_minus_et"
] = (
    paired_split_comparison[
        "random_forest_macro_f1"
    ]
    - paired_split_comparison[
        "extra_trees_macro_f1"
    ]
)

paired_split_comparison[
    "balanced_accuracy_difference_rf_minus_et"
] = (
    paired_split_comparison[
        "random_forest_balanced_accuracy"
    ]
    - paired_split_comparison[
        "extra_trees_balanced_accuracy"
    ]
)

paired_split_comparison[
    "macro_f1_winner"
] = np.select(
    [
        paired_split_comparison[
            "macro_f1_difference_rf_minus_et"
        ] > 0,

        paired_split_comparison[
            "macro_f1_difference_rf_minus_et"
        ] < 0
    ],
    [
        "Random Forest",
        "Extra Trees"
    ],
    default="Tie"
)


# ------------------------------------------------------------
# 5. Repeat-level comparison
# ------------------------------------------------------------

repeat_level_comparison = (
    paired_split_comparison
    .groupby(
        "repeat_index",
        as_index=False
    )
    .agg(
        random_forest_macro_f1=(
            "random_forest_macro_f1",
            "mean"
        ),
        extra_trees_macro_f1=(
            "extra_trees_macro_f1",
            "mean"
        ),
        random_forest_balanced_accuracy=(
            "random_forest_balanced_accuracy",
            "mean"
        ),
        extra_trees_balanced_accuracy=(
            "extra_trees_balanced_accuracy",
            "mean"
        )
    )
)

repeat_level_comparison[
    "macro_f1_difference_rf_minus_et"
] = (
    repeat_level_comparison[
        "random_forest_macro_f1"
    ]
    - repeat_level_comparison[
        "extra_trees_macro_f1"
    ]
)

repeat_level_comparison[
    "balanced_accuracy_difference_rf_minus_et"
] = (
    repeat_level_comparison[
        "random_forest_balanced_accuracy"
    ]
    - repeat_level_comparison[
        "extra_trees_balanced_accuracy"
    ]
)

repeat_level_comparison[
    "macro_f1_winner"
] = np.select(
    [
        repeat_level_comparison[
            "macro_f1_difference_rf_minus_et"
        ] > 0,

        repeat_level_comparison[
            "macro_f1_difference_rf_minus_et"
        ] < 0
    ],
    [
        "Random Forest",
        "Extra Trees"
    ],
    default="Tie"
)


# ------------------------------------------------------------
# 6. Apply the practical-equivalence rule
# ------------------------------------------------------------

extra_trees_macro_f1 = float(
    extra_trees_row["mean_macro_f1"]
)

random_forest_macro_f1 = float(
    random_forest_row["mean_macro_f1"]
)

extra_trees_balanced_accuracy = float(
    extra_trees_row["mean_balanced_accuracy"]
)

random_forest_balanced_accuracy = float(
    random_forest_row["mean_balanced_accuracy"]
)

absolute_macro_f1_difference = abs(
    extra_trees_macro_f1
    - random_forest_macro_f1
)

absolute_balanced_accuracy_difference = abs(
    extra_trees_balanced_accuracy
    - random_forest_balanced_accuracy
)

macro_f1_practically_equivalent = bool(
    absolute_macro_f1_difference
    < PRACTICAL_MACRO_F1_MARGIN
)

balanced_accuracy_practically_equivalent = bool(
    absolute_balanced_accuracy_difference
    < PRACTICAL_BALANCED_ACCURACY_MARGIN
)

decision_comparison = pd.DataFrame([
    {
        "criterion": "Mean macro-F1",
        "Random Forest": float(
            random_forest_row[
                "mean_macro_f1"
            ]
        ),
        "Extra Trees": float(
            extra_trees_row[
                "mean_macro_f1"
            ]
        ),
        "preferred_workflow": (
            "Random Forest"
            if random_forest_row[
                "mean_macro_f1"
            ] > extra_trees_row[
                "mean_macro_f1"
            ]
            else "Extra Trees"
        )
    },
    {
        "criterion": "Mean balanced accuracy",
        "Random Forest": float(
            random_forest_row[
                "mean_balanced_accuracy"
            ]
        ),
        "Extra Trees": float(
            extra_trees_row[
                "mean_balanced_accuracy"
            ]
        ),
        "preferred_workflow": (
            "Random Forest"
            if random_forest_row[
                "mean_balanced_accuracy"
            ] > extra_trees_row[
                "mean_balanced_accuracy"
            ]
            else "Extra Trees"
        )
    },
    {
        "criterion": "Train-validation macro-F1 gap",
        "Random Forest": float(
            random_forest_row[
                "train_validation_macro_f1_gap"
            ]
        ),
        "Extra Trees": float(
            extra_trees_row[
                "train_validation_macro_f1_gap"
            ]
        ),
        "preferred_workflow": (
            "Random Forest"
            if random_forest_row[
                "train_validation_macro_f1_gap"
            ] < extra_trees_row[
                "train_validation_macro_f1_gap"
            ]
            else "Extra Trees"
        )
    },
    {
        "criterion": "Split wins",
        "Random Forest": int(
            random_forest_row[
                "split_wins"
            ]
        ),
        "Extra Trees": int(
            extra_trees_row[
                "split_wins"
            ]
        ),
        "preferred_workflow": (
            "Random Forest"
            if random_forest_row[
                "split_wins"
            ] > extra_trees_row[
                "split_wins"
            ]
            else "Extra Trees"
        )
    }
])


if not macro_f1_practically_equivalent:
    selected_model_family = (
        "Random Forest"
        if random_forest_macro_f1
        > extra_trees_macro_f1
        else "Extra Trees"
    )

    decision_reason = (
        "The mean macro-F1 difference exceeded the documented "
        "practical-equivalence margin, so the workflow with the "
        "higher mean macro-F1 was retained."
    )

elif not balanced_accuracy_practically_equivalent:
    selected_model_family = (
        "Random Forest"
        if random_forest_balanced_accuracy
        > extra_trees_balanced_accuracy
        else "Extra Trees"
    )

    decision_reason = (
        "The workflows were practically equivalent in macro-F1, "
        "but their balanced-accuracy difference exceeded the "
        "documented practical-equivalence margin. The workflow "
        "with higher balanced accuracy was retained."
    )

else:
    random_forest_gap = float(
        random_forest_row[
            "train_validation_macro_f1_gap"
        ]
    )

    extra_trees_gap = float(
        extra_trees_row[
            "train_validation_macro_f1_gap"
        ]
    )

    absolute_train_validation_gap_difference = abs(
        random_forest_gap
        - extra_trees_gap
    )

    if (
        absolute_train_validation_gap_difference
        >= PRACTICAL_TRAIN_VALIDATION_GAP_MARGIN
    ):
        selected_model_family = (
            "Random Forest"
            if random_forest_gap
            < extra_trees_gap
            else "Extra Trees"
        )

        decision_reason = (
            "The workflows were practically equivalent in both "
            "macro-F1 and balanced accuracy. The workflow with "
            "the smaller train-validation macro-F1 gap was "
            "retained as the more stable option."
        )

    elif (
        int(random_forest_row["split_wins"])
        != int(extra_trees_row["split_wins"])
    ):
        selected_model_family = (
            "Random Forest"
            if int(random_forest_row["split_wins"])
            > int(extra_trees_row["split_wins"])
            else "Extra Trees"
        )

        decision_reason = (
            "The performance and train-validation gaps were tied "
            "within the declared rules, so split-level winning "
            "frequency was used as the final stability criterion."
        )

    else:
        selected_model_family = (
            "Random Forest"
            if random_forest_macro_f1
            >= extra_trees_macro_f1
            else "Extra Trees"
        )

        decision_reason = (
            "All declared criteria were tied. The workflow with "
            "the marginally higher mean macro-F1 was retained as "
            "a deterministic final tie-breaker."
        )


# ------------------------------------------------------------
# 7. Lock the corrected final workflow
# ------------------------------------------------------------

if selected_model_family == "Random Forest":
    selected_stability_row = (
        random_forest_row
    )
else:
    selected_stability_row = (
        extra_trees_row
    )

FINAL_WORKFLOW_ID = (
    selected_stability_row[
        "workflow_id"
    ]
)

FINAL_WORKFLOW_FEATURE_SET = (
    selected_stability_row[
        "feature_set"
    ]
)

FINAL_WORKFLOW_MODEL_FAMILY = (
    selected_stability_row[
        "model_family"
    ]
)

FINAL_WORKFLOW_FEATURES = list(
    workflow_feature_lookup[
        FINAL_WORKFLOW_ID
    ]
)

FINAL_WORKFLOW_PARAMS = dict(
    workflow_parameter_lookup[
        FINAL_WORKFLOW_ID
    ]
)

final_workflow_pipeline_template = clone(
    workflow_pipeline_lookup[
        FINAL_WORKFLOW_ID
    ]
)

final_workflow_pipeline_template.set_params(
    **FINAL_WORKFLOW_PARAMS
)


# ------------------------------------------------------------
# 8. Final decision summary
# ------------------------------------------------------------

workflow_stability_decision_df = pd.DataFrame([
    {
        "practical_macro_f1_margin": (
            PRACTICAL_MACRO_F1_MARGIN
        ),
        "practical_balanced_accuracy_margin": (
            PRACTICAL_BALANCED_ACCURACY_MARGIN
        ),
        "practical_train_validation_gap_margin": (
            PRACTICAL_TRAIN_VALIDATION_GAP_MARGIN
        ),
        "extra_trees_mean_macro_f1": (
            extra_trees_macro_f1
        ),
        "random_forest_mean_macro_f1": (
            random_forest_macro_f1
        ),
        "absolute_macro_f1_difference": (
            absolute_macro_f1_difference
        ),
        "absolute_balanced_accuracy_difference": (
            absolute_balanced_accuracy_difference
        ),
        "macro_f1_practically_equivalent": (
            macro_f1_practically_equivalent
        ),
        "balanced_accuracy_practically_equivalent": (
            balanced_accuracy_practically_equivalent
        ),
        "selected_feature_set": (
            FINAL_WORKFLOW_FEATURE_SET
        ),
        "selected_model_family": (
            FINAL_WORKFLOW_MODEL_FAMILY
        ),
        "selected_parameters": (
            serialise_parameters(
                FINAL_WORKFLOW_PARAMS
            )
        ),
        "selected_predictor_count": len(
            FINAL_WORKFLOW_FEATURES
        ),
        "decision_reason": (
            decision_reason
        ),
        "primary_performance_source": (
            "Step 4 nested out-of-fold results"
        )
    }
])


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 9A — PRACTICAL-TIE AND STABILITY RESULTS")
print("=" * 80)

print("\nDirect workflow comparison:")

display(
    top_workflows[
        [
            "workflow_id",
            "mean_macro_f1",
            "std_macro_f1",
            "mean_balanced_accuracy",
            "std_balanced_accuracy",
            "mean_train_macro_f1",
            "train_validation_macro_f1_gap",
            "split_wins"
        ]
    ].round(4)
)

print("\nDecision criteria:")

display(
    decision_comparison.round(4)
)

print("\nPaired split-level summary:")

paired_split_summary = pd.DataFrame([
    {
        "metric": (
            "Mean RF minus Extra Trees macro-F1"
        ),
        "value": (
            paired_split_comparison[
                "macro_f1_difference_rf_minus_et"
            ].mean()
        )
    },
    {
        "metric": (
            "Median RF minus Extra Trees macro-F1"
        ),
        "value": (
            paired_split_comparison[
                "macro_f1_difference_rf_minus_et"
            ].median()
        )
    },
    {
        "metric": (
            "Random Forest split wins"
        ),
        "value": int(
            (
                paired_split_comparison[
                    "macro_f1_winner"
                ] == "Random Forest"
            ).sum()
        )
    },
    {
        "metric": (
            "Extra Trees split wins"
        ),
        "value": int(
            (
                paired_split_comparison[
                    "macro_f1_winner"
                ] == "Extra Trees"
            ).sum()
        )
    }
])

display(
    paired_split_summary.round(4)
)

print("\nRepeat-level comparison:")

display(
    repeat_level_comparison.round(4)
)

print("\nFinal fixed-workflow decision:")

display(
    workflow_stability_decision_df
)

print(
    "\nSelected model family:",
    FINAL_WORKFLOW_MODEL_FAMILY
)

print(
    "Selected feature set:",
    FINAL_WORKFLOW_FEATURE_SET
)

print(
    "Selected parameters:",
    FINAL_WORKFLOW_PARAMS
)

print(
    "\nPrimary thesis macro-F1 remains:",
    round(
        float(
            pooled_performance_df.iloc[0][
                "macro_f1"
            ]
        ),
        4
    )
)

print(
    "Primary thesis balanced accuracy remains:",
    round(
        float(
            pooled_performance_df.iloc[0][
                "balanced_accuracy"
            ]
        ),
        4
    )
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

top_workflows.to_csv(
    OUTPUT_DIR
    / "top_workflow_near_tie_comparison.csv",
    index=False
)

paired_split_comparison.to_csv(
    OUTPUT_DIR
    / "top_workflow_paired_split_comparison.csv",
    index=False
)

repeat_level_comparison.to_csv(
    OUTPUT_DIR
    / "top_workflow_repeat_level_comparison.csv",
    index=False
)

decision_comparison.to_csv(
    OUTPUT_DIR
    / "top_workflow_decision_criteria.csv",
    index=False
)

workflow_stability_decision_df.to_csv(
    OUTPUT_DIR
    / "final_workflow_stability_decision.csv",
    index=False
)

final_stability_configuration = {
    "practical_macro_f1_margin": (
        PRACTICAL_MACRO_F1_MARGIN
    ),
    "practical_balanced_accuracy_margin": (
        PRACTICAL_BALANCED_ACCURACY_MARGIN
    ),
    "practical_train_validation_gap_margin": (
        PRACTICAL_TRAIN_VALIDATION_GAP_MARGIN
    ),
    "practical_margin_interpretation": (
        "Differences below 0.01 in macro-F1 and balanced "
        "accuracy were treated as practically negligible for "
        "fixed-workflow selection."
    ),
    "selection_hierarchy": [
        "Higher macro-F1 when difference is at least 0.01",
        "Higher balanced accuracy when macro-F1 is equivalent "
        "and balanced-accuracy difference is at least 0.01",
        "Lower train-validation macro-F1 gap when both metrics "
        "are practically equivalent and the gap difference is "
        "at least 0.01",
        "More split-level wins",
        "Marginally higher mean macro-F1 as deterministic final tie-breaker"
    ],
    "selected_workflow_id": (
        FINAL_WORKFLOW_ID
    ),
    "selected_feature_set": (
        FINAL_WORKFLOW_FEATURE_SET
    ),
    "selected_model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "selected_features": (
        FINAL_WORKFLOW_FEATURES
    ),
    "selected_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "repeated_cv_split_scores_independent": False,
    "repeated_cv_standard_error_used_for_formal_inference": False,
    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation"
    )
}

with open(
    OUTPUT_DIR
    / "final_workflow_stability_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        final_stability_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 9A COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Locked deployable workflow:",
    FINAL_WORKFLOW_ID
)

print(
    "\nSaved Step 9A outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# GROUPED FEATURE ABLATION — FINAL FIXED WORKFLOW
# ============================================================
# Purpose:
# 1. Quantify how much each logical billing-feature family
#    contributes to archetype-transfer performance.
# 2. Remove ONE feature family at a time while keeping:
#       - the same households,
#       - the same CV splits,
#       - the same fitted workflow specification,
#       - the same hyperparameters.
# 3. Compare every ablation directly with the full 32-feature
#    fixed workflow on identical test folds.
#
# IMPORTANT:
# - This is a SECONDARY INTERPRETABILITY / SENSITIVITY analysis.
# - It does NOT replace the primary Step 4 nested OOF result.
# - Hyperparameters are NOT retuned after removing features,
#   because the question is:
#       "How much does the locked model rely on this information?"
# - Results must NOT be used post-hoc to claim a new unbiased
#   primary model.
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score
)


# ------------------------------------------------------------
# 1. Check required objects
# ------------------------------------------------------------

REQUIRED_ABLATION_OBJECTS = [
    "dataset_all",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "CLASS_TO_INDEX",
    "RANDOM_STATE",
    "OUTPUT_DIR"
]

missing_objects = [
    name
    for name in REQUIRED_ABLATION_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the notebook through the final-workflow "
        f"selection step first. Missing: {missing_objects}"
    )


# ------------------------------------------------------------
# 2. Prepare one row per household
# ------------------------------------------------------------

ablation_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

if ablation_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households detected before feature ablation."
    )

X_ablation = (
    ablation_df[FINAL_WORKFLOW_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

y_ablation = (
    ablation_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
)

if y_ablation.isna().any():
    unknown_labels = sorted(
        ablation_df.loc[
            y_ablation.isna(),
            "cluster_name"
        ]
        .astype(str)
        .unique()
    )

    raise ValueError(
        f"Unmapped archetype labels detected: {unknown_labels}"
    )

y_ablation = y_ablation.astype(int)

if X_ablation.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all final predictors missing."
    )


# ------------------------------------------------------------
# 3. Define the FOUR prespecified feature families
# ------------------------------------------------------------

FEATURE_GROUPS = {

    "Distribution / dispersion": [
        "billing_median_kwh",
        "billing_mean_kwh",
        "billing_std_kwh",
        "billing_iqr_kwh",
        "billing_cv",
        "billing_p10_kwh",
        "billing_p90_kwh",
        "billing_robust_range_kwh",
        "billing_min_kwh",
        "billing_max_kwh",
        "billing_range_kwh",
        "billing_log_std",
    ],

    "Temporal / history / change": [
        "recent_3_month_median_kwh",
        "recent_6_month_median_kwh",
        "recent_vs_overall_log_ratio",
        "early_6_month_median_kwh",
        "late_vs_early_log_ratio",
        "monthly_trend_slope_log1p",
        "median_abs_log_change",
        "mean_abs_monthly_change_kwh",
        "maximum_monthly_increase_kwh",
        "maximum_monthly_decrease_kwh",
    ],

    "Consumption-rate features": [
        "near_zero_month_rate",
        "low_30_month_rate",
        "above_90_month_rate",
        "above_120_month_rate",
        "above_180_month_rate",
        "above_240_month_rate",
    ],

    "Calendar-quarter medians": [
        "median_kwh_calendar_q1",
        "median_kwh_calendar_q2",
        "median_kwh_calendar_q3",
        "median_kwh_calendar_q4",
    ],
}


# ------------------------------------------------------------
# 4. Audit the feature-group definition
# ------------------------------------------------------------

grouped_features = [
    feature
    for group_features in FEATURE_GROUPS.values()
    for feature in group_features
]

# No duplicated feature across groups
if len(grouped_features) != len(set(grouped_features)):
    duplicates = (
        pd.Series(grouped_features)
        .value_counts()
    )

    duplicates = duplicates[
        duplicates > 1
    ].index.tolist()

    raise RuntimeError(
        "A feature appears in more than one ablation group: "
        f"{duplicates}"
    )

# All grouped features must belong to the locked final model
unexpected_features = sorted(
    set(grouped_features)
    - set(FINAL_WORKFLOW_FEATURES)
)

if unexpected_features:
    raise RuntimeError(
        "Ablation groups contain predictors not present in the "
        f"final locked workflow: {unexpected_features}"
    )

# Every locked final feature must be represented exactly once
missing_group_assignments = sorted(
    set(FINAL_WORKFLOW_FEATURES)
    - set(grouped_features)
)

if missing_group_assignments:
    raise RuntimeError(
        "Some final predictors were not assigned to an ablation "
        f"group: {missing_group_assignments}"
    )

if len(grouped_features) != len(FINAL_WORKFLOW_FEATURES):
    raise RuntimeError(
        "Grouped feature count does not match final predictor count."
    )

print("=" * 80)
print("GROUPED FEATURE ABLATION")
print("=" * 80)

print(
    "Locked model:",
    FINAL_WORKFLOW_MODEL_FAMILY
)

print(
    "Full predictor count:",
    len(FINAL_WORKFLOW_FEATURES)
)

print("\nFeature-family sizes:")

for group_name, features in FEATURE_GROUPS.items():
    print(
        f"  {group_name}: {len(features)}"
    )


# ------------------------------------------------------------
# 5. Create IDENTICAL repeated-CV splits
# ------------------------------------------------------------
# 5 folds x 5 repeats = 25 matched train/test comparisons.
#
# We precompute the splits ONCE so the full model and every
# ablation are evaluated on exactly the same households.

ABLATION_CV_FOLDS = 5
ABLATION_CV_REPEATS = 5

ablation_cv = RepeatedStratifiedKFold(
    n_splits=ABLATION_CV_FOLDS,
    n_repeats=ABLATION_CV_REPEATS,
    random_state=RANDOM_STATE
)

cv_splits = list(
    ablation_cv.split(
        X_ablation,
        y_ablation
    )
)

print(
    "\nMatched CV test splits:",
    len(cv_splits)
)


# ------------------------------------------------------------
# 6. Define full model + leave-one-GROUP-out variants
# ------------------------------------------------------------

ABLATION_VARIANTS = {
    "Full model (32 predictors)": list(
        FINAL_WORKFLOW_FEATURES
    )
}

for group_name, removed_features in FEATURE_GROUPS.items():

    retained_features = [
        feature
        for feature in FINAL_WORKFLOW_FEATURES
        if feature not in removed_features
    ]

    if len(retained_features) == 0:
        raise RuntimeError(
            f"Ablating '{group_name}' leaves no predictors."
        )

    ABLATION_VARIANTS[
        f"Remove: {group_name}"
    ] = retained_features


# ------------------------------------------------------------
# 7. Evaluate every variant on IDENTICAL splits
# ------------------------------------------------------------

ablation_records = []

for split_number, (train_idx, test_idx) in enumerate(
    cv_splits,
    start=1
):

    X_train_full = X_ablation.iloc[train_idx]
    X_test_full = X_ablation.iloc[test_idx]

    y_train = y_ablation.iloc[train_idx]
    y_test = y_ablation.iloc[test_idx]

    repeat_number = (
        (split_number - 1)
        // ABLATION_CV_FOLDS
        + 1
    )

    fold_number = (
        (split_number - 1)
        % ABLATION_CV_FOLDS
        + 1
    )

    for variant_name, retained_features in (
        ABLATION_VARIANTS.items()
    ):

        # Clone the EXACT locked workflow.
        # This preserves:
        # - imputation,
        # - class weighting,
        # - Random Forest specification,
        # - final hyperparameters.
        model = clone(
            final_workflow_pipeline_template
        )

        X_train = X_train_full[
            retained_features
        ]

        X_test = X_test_full[
            retained_features
        ]

        model.fit(
            X_train,
            y_train
        )

        train_predictions = model.predict(
            X_train
        )

        test_predictions = model.predict(
            X_test
        )

        train_macro_f1 = f1_score(
            y_train,
            train_predictions,
            average="macro",
            zero_division=0
        )

        test_macro_f1 = f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        )

        train_balanced_accuracy = (
            balanced_accuracy_score(
                y_train,
                train_predictions
            )
        )

        test_balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                test_predictions
            )
        )

        ablation_records.append({

            "split": split_number,
            "repeat": repeat_number,
            "fold": fold_number,

            "variant": variant_name,

            "retained_feature_count": len(
                retained_features
            ),

            "removed_feature_count": (
                len(FINAL_WORKFLOW_FEATURES)
                - len(retained_features)
            ),

            "train_macro_f1": (
                train_macro_f1
            ),

            "test_macro_f1": (
                test_macro_f1
            ),

            "macro_f1_train_test_gap": (
                train_macro_f1
                - test_macro_f1
            ),

            "train_balanced_accuracy": (
                train_balanced_accuracy
            ),

            "test_balanced_accuracy": (
                test_balanced_accuracy
            ),

            "balanced_accuracy_train_test_gap": (
                train_balanced_accuracy
                - test_balanced_accuracy
            )
        })


ablation_raw_df = pd.DataFrame(
    ablation_records
)


# ------------------------------------------------------------
# 8. Pair every ablation with its SAME-SPLIT full-model result
# ------------------------------------------------------------

full_split_scores = (
    ablation_raw_df[
        ablation_raw_df["variant"]
        == "Full model (32 predictors)"
    ][
        [
            "split",
            "test_macro_f1",
            "test_balanced_accuracy"
        ]
    ]
    .rename(
        columns={
            "test_macro_f1":
                "full_test_macro_f1",

            "test_balanced_accuracy":
                "full_test_balanced_accuracy"
        }
    )
)

ablation_paired_df = (
    ablation_raw_df
    .merge(
        full_split_scores,
        on="split",
        how="left",
        validate="many_to_one"
    )
)

# Positive value = performance was LOST after ablation.
ablation_paired_df[
    "macro_f1_loss_vs_full"
] = (
    ablation_paired_df[
        "full_test_macro_f1"
    ]
    - ablation_paired_df[
        "test_macro_f1"
    ]
)

ablation_paired_df[
    "balanced_accuracy_loss_vs_full"
] = (
    ablation_paired_df[
        "full_test_balanced_accuracy"
    ]
    - ablation_paired_df[
        "test_balanced_accuracy"
    ]
)


# ------------------------------------------------------------
# 9. Create summary table
# ------------------------------------------------------------

summary_records = []

for variant_name in ABLATION_VARIANTS:

    subset = (
        ablation_paired_df[
            ablation_paired_df["variant"]
            == variant_name
        ]
        .copy()
    )

    summary_records.append({

        "variant": variant_name,

        "retained_features": int(
            subset[
                "retained_feature_count"
            ].iloc[0]
        ),

        "removed_features": int(
            subset[
                "removed_feature_count"
            ].iloc[0]
        ),

        "mean_test_macro_f1": (
            subset[
                "test_macro_f1"
            ].mean()
        ),

        "std_test_macro_f1": (
            subset[
                "test_macro_f1"
            ].std(ddof=1)
        ),

        "mean_macro_f1_loss_vs_full": (
            subset[
                "macro_f1_loss_vs_full"
            ].mean()
        ),

        "median_macro_f1_loss_vs_full": (
            subset[
                "macro_f1_loss_vs_full"
            ].median()
        ),

        "splits_ablation_worse_macro_f1": int(
            (
                subset[
                    "macro_f1_loss_vs_full"
                ] > 0
            ).sum()
        ),

        "mean_test_balanced_accuracy": (
            subset[
                "test_balanced_accuracy"
            ].mean()
        ),

        "std_test_balanced_accuracy": (
            subset[
                "test_balanced_accuracy"
            ].std(ddof=1)
        ),

        "mean_balanced_accuracy_loss_vs_full": (
            subset[
                "balanced_accuracy_loss_vs_full"
            ].mean()
        ),

        "splits_ablation_worse_balanced_accuracy": int(
            (
                subset[
                    "balanced_accuracy_loss_vs_full"
                ] > 0
            ).sum()
        ),

        "mean_train_test_macro_f1_gap": (
            subset[
                "macro_f1_train_test_gap"
            ].mean()
        )
    })


ablation_summary_df = pd.DataFrame(
    summary_records
)


# Full model should be shown first.
# Ablated groups are then ranked by average macro-F1 loss.
full_row = (
    ablation_summary_df[
        ablation_summary_df["variant"]
        == "Full model (32 predictors)"
    ]
)

ablated_rows = (
    ablation_summary_df[
        ablation_summary_df["variant"]
        != "Full model (32 predictors)"
    ]
    .sort_values(
        "mean_macro_f1_loss_vs_full",
        ascending=False
    )
)

ablation_summary_df = pd.concat(
    [
        full_row,
        ablated_rows
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 10. Add a simple interpretation category
# ------------------------------------------------------------
# These labels are DESCRIPTIVE only.
# The raw metric differences should remain the main result.

def describe_ablation_effect(delta):

    if pd.isna(delta):
        return "Not available"

    if delta >= 0.03:
        return "Clear performance loss"

    if delta >= 0.01:
        return "Small performance loss"

    if delta > -0.01:
        return "Little practical change"

    return "Performance improved after removal"


ablation_summary_df[
    "macro_f1_interpretation"
] = (
    ablation_summary_df[
        "mean_macro_f1_loss_vs_full"
    ]
    .apply(describe_ablation_effect)
)

# Full model is the reference, not an ablation.
ablation_summary_df.loc[
    ablation_summary_df["variant"]
    == "Full model (32 predictors)",
    "macro_f1_interpretation"
] = "Reference"


# ------------------------------------------------------------
# 11. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GROUPED FEATURE ABLATION RESULTS")
print("=" * 80)

display(
    ablation_summary_df[
        [
            "variant",
            "retained_features",
            "mean_test_macro_f1",
            "mean_macro_f1_loss_vs_full",
            "mean_test_balanced_accuracy",
            "mean_balanced_accuracy_loss_vs_full",
            "splits_ablation_worse_macro_f1",
            "mean_train_test_macro_f1_gap",
            "macro_f1_interpretation"
        ]
    ]
    .round(4)
)


# ------------------------------------------------------------
# 12. Save outputs
# ------------------------------------------------------------

ablation_raw_path = (
    OUTPUT_DIR
    / "grouped_feature_ablation_raw_scores.csv"
)

ablation_paired_path = (
    OUTPUT_DIR
    / "grouped_feature_ablation_paired_scores.csv"
)

ablation_summary_path = (
    OUTPUT_DIR
    / "grouped_feature_ablation_summary.csv"
)

ablation_raw_df.to_csv(
    ablation_raw_path,
    index=False
)

ablation_paired_df.to_csv(
    ablation_paired_path,
    index=False
)

ablation_summary_df.to_csv(
    ablation_summary_path,
    index=False
)


ablation_configuration = {

    "analysis_role": (
        "Secondary grouped feature-ablation sensitivity analysis"
    ),

    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation remains primary"
    ),

    "model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),

    "model_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),

    "full_feature_count": (
        len(FINAL_WORKFLOW_FEATURES)
    ),

    "feature_groups": (
        FEATURE_GROUPS
    ),

    "cv_design": {
        "method": "RepeatedStratifiedKFold",
        "folds": ABLATION_CV_FOLDS,
        "repeats": ABLATION_CV_REPEATS,
        "total_splits": len(cv_splits),
        "same_splits_for_every_variant": True,
        "random_state": RANDOM_STATE
    },

    "hyperparameters_retuned_after_ablation": False,

    "reason_no_retuning": (
        "The purpose is to measure dependence of the locked "
        "final workflow on each prespecified information family, "
        "not to select a new model."
    ),

    "primary_ablation_metric": "macro_f1",

    "supporting_ablation_metric": (
        "balanced_accuracy"
    ),

    "interpretation": (
        "Positive full-minus-ablated differences indicate "
        "performance loss after removal of the feature family."
    ),

    "limitations": [
        (
            "Repeated-CV folds overlap, so fold-level scores are "
            "descriptive paired sensitivity results rather than "
            "independent observations for formal hypothesis tests."
        ),
        (
            "Ablation does not establish causality or unique "
            "feature importance because predictors within and "
            "between groups may remain correlated."
        ),
        (
            "Results must not replace the prespecified primary "
            "nested-CV performance estimate."
        )
    ]
}


with open(
    OUTPUT_DIR
    / "grouped_feature_ablation_configuration.json",
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        ablation_configuration,
        handle,
        indent=4
    )


print("\nSaved:")
print(ablation_raw_path)
print(ablation_paired_path)
print(ablation_summary_path)

In [ ]:
# ============================================================
# STEP 10 — PRE-GOLDEN TEMPORAL SENSITIVITY
# ============================================================
# Purpose:
# 1. Evaluate the locked final fixed workflow using:
#    a) all available monthly billing history;
#    b) billing history strictly before 2024-05-01.
# 2. Use the same households, labels and CV splits.
# 3. Quantify paired performance differences using a
#    stratified household-level bootstrap.s
#
# Important:
# - This is a secondary temporal-sensitivity analysis.
# - It does not replace the Step 4 nested OOF estimate.
# - The pre-golden analysis assesses whether earlier billing
#   histories can transfer later smart-meter archetypes.
# - It is not a causal or future-default prediction analysis.
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import StratifiedKFold


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP10_OBJECTS = [
    "dataset_all",
    "dataset_pre",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "ORDERED_CLUSTER_NAMES",
    "CLASS_TO_INDEX",
    "INDEX_TO_CLASS",
    "FIXED_CLASS_INDICES",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "align_class_probabilities",
    "calculate_multiclass_metrics"
]

missing_step10_objects = [
    object_name
    for object_name in REQUIRED_STEP10_OBJECTS
    if object_name not in globals()
]

if missing_step10_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 9A first. Missing objects: "
        f"{missing_step10_objects}"
    )


# ------------------------------------------------------------
# 2. Align the two datasets strictly
# ------------------------------------------------------------

temporal_all_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

temporal_pre_df = (
    dataset_pre
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

expected_temporal_households = len(dataset_all)

if len(temporal_all_df) != expected_temporal_households:
    raise ValueError(
        f"Expected {expected_temporal_households:,} all-available "
        f"households, but found {len(temporal_all_df):,}."
    )

if len(temporal_pre_df) != expected_temporal_households:
    raise ValueError(
        f"Expected {expected_temporal_households:,} pre-golden "
        f"households, but found {len(temporal_pre_df):,}."
    )

if not temporal_all_df["household_ID"].equals(
    temporal_pre_df["household_ID"]
):
    raise ValueError(
        "The contemporaneous and pre-golden household orders "
        "do not match."
    )

if not (
    temporal_all_df["cluster_name"].astype(str).to_numpy()
    == temporal_pre_df["cluster_name"].astype(str).to_numpy()
).all():
    raise ValueError(
        "The archetype labels differ between temporal datasets."
    )

missing_all_features = sorted(
    set(FINAL_WORKFLOW_FEATURES)
    - set(temporal_all_df.columns)
)

missing_pre_features = sorted(
    set(FINAL_WORKFLOW_FEATURES)
    - set(temporal_pre_df.columns)
)

if missing_all_features:
    raise ValueError(
        "All-available data are missing final predictors: "
        f"{missing_all_features}"
    )

if missing_pre_features:
    raise ValueError(
        "Pre-golden data are missing final predictors: "
        f"{missing_pre_features}"
    )


# ------------------------------------------------------------
# 3. Prepare identical target and predictor matrices
# ------------------------------------------------------------

temporal_y = (
    temporal_all_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
    .astype(int)
)

temporal_X_all = (
    temporal_all_df[FINAL_WORKFLOW_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

temporal_X_pre = (
    temporal_pre_df[FINAL_WORKFLOW_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

if temporal_X_all.isna().all(axis=1).any():
    raise ValueError(
        "At least one all-available household has all final "
        "predictors missing."
    )

if temporal_X_pre.isna().all(axis=1).any():
    raise ValueError(
        "At least one pre-golden household has all final "
        "predictors missing."
    )


# ------------------------------------------------------------
# 4. Fixed-workflow OOF evaluation helper
# ------------------------------------------------------------

def evaluate_fixed_workflow_oof(
    feature_matrix,
    target,
    household_ids,
    evaluation_name
):
    """
    Evaluate the already locked workflow using fixed five-fold
    stratified OOF predictions.

    No workflow or hyperparameter selection occurs here.
    """

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    number_of_rows = len(feature_matrix)
    number_of_classes = len(FIXED_CLASS_INDICES)

    oof_predictions = np.full(
        number_of_rows,
        -1,
        dtype=int
    )

    oof_probabilities = np.full(
        (number_of_rows, number_of_classes),
        np.nan,
        dtype=float
    )

    fold_assignments = np.full(
        number_of_rows,
        -1,
        dtype=int
    )

    fold_metric_records = []

    for fold_number, (
        train_indices,
        test_indices
    ) in enumerate(
        cv.split(feature_matrix, target),
        start=1
    ):
        estimator = clone(
            final_workflow_pipeline_template
        )

        estimator.fit(
            feature_matrix.iloc[train_indices],
            target.iloc[train_indices]
        )

        fold_predictions = estimator.predict(
            feature_matrix.iloc[test_indices]
        )

        raw_fold_probabilities = estimator.predict_proba(
            feature_matrix.iloc[test_indices]
        )

        fold_probabilities = align_class_probabilities(
            raw_probabilities=raw_fold_probabilities,
            fitted_classes=estimator.classes_
        )

        oof_predictions[test_indices] = (
            fold_predictions
        )

        oof_probabilities[test_indices, :] = (
            fold_probabilities
        )

        fold_assignments[test_indices] = (
            fold_number
        )

        fold_metrics = calculate_multiclass_metrics(
            target.iloc[test_indices],
            fold_predictions,
            fold_probabilities
        )

        fold_metric_records.append({
            "evaluation": evaluation_name,
            "fold": fold_number,
            "train_rows": len(train_indices),
            "test_rows": len(test_indices),
            **fold_metrics
        })

    if (oof_predictions < 0).any():
        raise RuntimeError(
            f"{evaluation_name}: missing OOF class predictions."
        )

    if np.isnan(oof_probabilities).any():
        raise RuntimeError(
            f"{evaluation_name}: missing OOF probabilities."
        )

    if (fold_assignments < 1).any():
        raise RuntimeError(
            f"{evaluation_name}: missing fold assignments."
        )

    pooled_metrics = calculate_multiclass_metrics(
        target,
        oof_predictions,
        oof_probabilities
    )

    class_report = classification_report(
        target,
        oof_predictions,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )

    class_records = []

    for class_name in ORDERED_CLUSTER_NAMES:
        class_records.append({
            "evaluation": evaluation_name,
            "class_name": class_name,
            "precision": class_report[
                class_name
            ]["precision"],
            "recall": class_report[
                class_name
            ]["recall"],
            "f1_score": class_report[
                class_name
            ]["f1-score"],
            "support": int(
                class_report[
                    class_name
                ]["support"]
            )
        })

    confusion_df = pd.DataFrame(
        confusion_matrix(
            target,
            oof_predictions,
            labels=FIXED_CLASS_INDICES
        ),
        index=ORDERED_CLUSTER_NAMES,
        columns=ORDERED_CLUSTER_NAMES
    )

    probability_column_names = []

    for class_name in ORDERED_CLUSTER_NAMES:
        safe_class_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        probability_column_names.append(
            f"probability_{safe_class_name}"
        )

    prediction_df = pd.DataFrame({
        "household_ID": household_ids,
        "evaluation": evaluation_name,
        "fold": fold_assignments,
        "true_class_index": target.to_numpy(dtype=int),
        "true_class_name": [
            INDEX_TO_CLASS[int(value)]
            for value in target
        ],
        "predicted_class_index": oof_predictions,
        "predicted_class_name": [
            INDEX_TO_CLASS[int(value)]
            for value in oof_predictions
        ]
    })

    for column_index, probability_column in enumerate(
        probability_column_names
    ):
        prediction_df[probability_column] = (
            oof_probabilities[:, column_index]
        )

    return {
        "predictions": prediction_df,
        "probability_columns": probability_column_names,
        "pooled_metrics": pooled_metrics,
        "class_metrics": pd.DataFrame(class_records),
        "confusion_matrix": confusion_df,
        "fold_metrics": pd.DataFrame(
            fold_metric_records
        )
    }


# ------------------------------------------------------------
# 5. Evaluate both observation windows
# ------------------------------------------------------------

all_available_fixed_results = (
    evaluate_fixed_workflow_oof(
        feature_matrix=temporal_X_all,
        target=temporal_y,
        household_ids=(
            temporal_all_df["household_ID"]
        ),
        evaluation_name=(
            f"All-available fixed {FINAL_WORKFLOW_MODEL_FAMILY}"
        )
    )
)

pre_golden_fixed_results = (
    evaluate_fixed_workflow_oof(
        feature_matrix=temporal_X_pre,
        target=temporal_y,
        household_ids=(
            temporal_pre_df["household_ID"]
        ),
        evaluation_name=(
            f"Pre-golden fixed {FINAL_WORKFLOW_MODEL_FAMILY}"
        )
    )
)


# ------------------------------------------------------------
# 6. Overall comparison
# ------------------------------------------------------------

temporal_overall_comparison_df = pd.DataFrame([
    {
        "evaluation": (
            f"All-available fixed {FINAL_WORKFLOW_MODEL_FAMILY}"
        ),
        **all_available_fixed_results[
            "pooled_metrics"
        ]
    },
    {
        "evaluation": (
            f"Pre-golden fixed {FINAL_WORKFLOW_MODEL_FAMILY}"
        ),
        **pre_golden_fixed_results[
            "pooled_metrics"
        ]
    }
])

temporal_metric_names = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "top_2_accuracy",
    "multiclass_log_loss",
    "multiclass_brier_sum"
]

temporal_point_difference_df = pd.DataFrame([
    {
        "metric": metric_name,
        "pre_golden_minus_all_available": (
            pre_golden_fixed_results[
                "pooled_metrics"
            ][metric_name]
            - all_available_fixed_results[
                "pooled_metrics"
            ][metric_name]
        )
    }
    for metric_name in temporal_metric_names
])


# ------------------------------------------------------------
# 7. Class-specific comparison
# ------------------------------------------------------------

temporal_class_comparison_df = (
    all_available_fixed_results[
        "class_metrics"
    ]
    .drop(columns=["evaluation"])
    .rename(
        columns={
            "precision":
                "all_available_precision",
            "recall":
                "all_available_recall",
            "f1_score":
                "all_available_f1"
        }
    )
    .merge(
        pre_golden_fixed_results[
            "class_metrics"
        ]
        .drop(columns=["evaluation", "support"])
        .rename(
            columns={
                "precision":
                    "pre_golden_precision",
                "recall":
                    "pre_golden_recall",
                "f1_score":
                    "pre_golden_f1"
            }
        ),
        on="class_name",
        how="inner",
        validate="one_to_one"
    )
)

temporal_class_comparison_df[
    "precision_difference_pre_minus_all"
] = (
    temporal_class_comparison_df[
        "pre_golden_precision"
    ]
    - temporal_class_comparison_df[
        "all_available_precision"
    ]
)

temporal_class_comparison_df[
    "recall_difference_pre_minus_all"
] = (
    temporal_class_comparison_df[
        "pre_golden_recall"
    ]
    - temporal_class_comparison_df[
        "all_available_recall"
    ]
)

temporal_class_comparison_df[
    "f1_difference_pre_minus_all"
] = (
    temporal_class_comparison_df[
        "pre_golden_f1"
    ]
    - temporal_class_comparison_df[
        "all_available_f1"
    ]
)


# ------------------------------------------------------------
# 8. Paired stratified bootstrap
# ------------------------------------------------------------

TEMPORAL_BOOTSTRAP_ITERATIONS = 2000
TEMPORAL_BOOTSTRAP_SEED = (
    RANDOM_STATE + 1000
)

bootstrap_generator = np.random.default_rng(
    TEMPORAL_BOOTSTRAP_SEED
)

all_prediction_df = (
    all_available_fixed_results[
        "predictions"
    ]
)

pre_prediction_df = (
    pre_golden_fixed_results[
        "predictions"
    ]
)

if not all_prediction_df["household_ID"].equals(
    pre_prediction_df["household_ID"]
):
    raise RuntimeError(
        "OOF prediction household order differs between "
        "temporal analyses."
    )

all_true = (
    all_prediction_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

pre_true = (
    pre_prediction_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

if not np.array_equal(all_true, pre_true):
    raise RuntimeError(
        "Temporal analyses do not use identical targets."
    )

all_predicted = (
    all_prediction_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

pre_predicted = (
    pre_prediction_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

all_probabilities = (
    all_prediction_df[
        all_available_fixed_results[
            "probability_columns"
        ]
    ]
    .to_numpy(dtype=float)
)

pre_probabilities = (
    pre_prediction_df[
        pre_golden_fixed_results[
            "probability_columns"
        ]
    ]
    .to_numpy(dtype=float)
)

bootstrap_class_indices = {
    class_index: np.where(
        all_true == class_index
    )[0]
    for class_index in FIXED_CLASS_INDICES
}

temporal_bootstrap_records = []

for bootstrap_iteration in range(
    1,
    TEMPORAL_BOOTSTRAP_ITERATIONS + 1
):
    sampled_indices = np.concatenate([
        bootstrap_generator.choice(
            bootstrap_class_indices[
                class_index
            ],
            size=len(
                bootstrap_class_indices[
                    class_index
                ]
            ),
            replace=True
        )
        for class_index in FIXED_CLASS_INDICES
    ])

    bootstrap_generator.shuffle(
        sampled_indices
    )

    all_bootstrap_metrics = (
        calculate_multiclass_metrics(
            all_true[sampled_indices],
            all_predicted[sampled_indices],
            all_probabilities[sampled_indices]
        )
    )

    pre_bootstrap_metrics = (
        calculate_multiclass_metrics(
            pre_true[sampled_indices],
            pre_predicted[sampled_indices],
            pre_probabilities[sampled_indices]
        )
    )

    bootstrap_record = {
        "bootstrap_iteration": (
            bootstrap_iteration
        )
    }

    for metric_name in temporal_metric_names:
        bootstrap_record[
            f"{metric_name}_difference_pre_minus_all"
        ] = (
            pre_bootstrap_metrics[metric_name]
            - all_bootstrap_metrics[metric_name]
        )

    all_class_recall = recall_score(
        all_true[sampled_indices],
        all_predicted[sampled_indices],
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    pre_class_recall = recall_score(
        pre_true[sampled_indices],
        pre_predicted[sampled_indices],
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    all_class_f1 = f1_score(
        all_true[sampled_indices],
        all_predicted[sampled_indices],
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    pre_class_f1 = f1_score(
        pre_true[sampled_indices],
        pre_predicted[sampled_indices],
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        safe_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        bootstrap_record[
            f"{safe_name}_recall_difference_pre_minus_all"
        ] = (
            pre_class_recall[class_index]
            - all_class_recall[class_index]
        )

        bootstrap_record[
            f"{safe_name}_f1_difference_pre_minus_all"
        ] = (
            pre_class_f1[class_index]
            - all_class_f1[class_index]
        )

    temporal_bootstrap_records.append(
        bootstrap_record
    )


temporal_bootstrap_distributions_df = (
    pd.DataFrame(
        temporal_bootstrap_records
    )
)


# ------------------------------------------------------------
# 9. Bootstrap confidence intervals
# ------------------------------------------------------------

temporal_difference_ci_records = []

for metric_name in temporal_metric_names:
    column_name = (
        f"{metric_name}_"
        "difference_pre_minus_all"
    )

    point_difference = float(
        temporal_point_difference_df.loc[
            temporal_point_difference_df[
                "metric"
            ] == metric_name,
            "pre_golden_minus_all_available"
        ].iloc[0]
    )

    temporal_difference_ci_records.append({
        "metric": metric_name,
        "point_difference_pre_minus_all": (
            point_difference
        ),
        "bootstrap_mean_difference": (
            temporal_bootstrap_distributions_df[
                column_name
            ].mean()
        ),
        "ci_2.5_percent": (
            temporal_bootstrap_distributions_df[
                column_name
            ].quantile(0.025)
        ),
        "ci_97.5_percent": (
            temporal_bootstrap_distributions_df[
                column_name
            ].quantile(0.975)
        )
    })

temporal_difference_ci_df = pd.DataFrame(
    temporal_difference_ci_records
)


temporal_class_difference_ci_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    point_class_row = (
        temporal_class_comparison_df.loc[
            temporal_class_comparison_df[
                "class_name"
            ] == class_name
        ].iloc[0]
    )

    for metric_name in ["recall", "f1"]:
        column_name = (
            f"{safe_name}_{metric_name}_"
            "difference_pre_minus_all"
        )

        point_column = (
            f"{metric_name}_"
            "difference_pre_minus_all"
        )

        temporal_class_difference_ci_records.append({
            "class_name": class_name,
            "metric": metric_name,
            "point_difference_pre_minus_all": (
                point_class_row[
                    point_column
                ]
            ),
            "bootstrap_mean_difference": (
                temporal_bootstrap_distributions_df[
                    column_name
                ].mean()
            ),
            "ci_2.5_percent": (
                temporal_bootstrap_distributions_df[
                    column_name
                ].quantile(0.025)
            ),
            "ci_97.5_percent": (
                temporal_bootstrap_distributions_df[
                    column_name
                ].quantile(0.975)
            )
        })

temporal_class_difference_ci_df = (
    pd.DataFrame(
        temporal_class_difference_ci_records
    )
)


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 10 — PRE-GOLDEN TEMPORAL SENSITIVITY")
print("=" * 80)

print("\nOverall fixed-workflow comparison:")

display(
    temporal_overall_comparison_df.round(4)
)

print("\nPoint differences: pre-golden minus all-available:")

display(
    temporal_point_difference_df.round(4)
)

print("\nClass-specific comparison:")

display(
    temporal_class_comparison_df.round(4)
)

print("\nPaired bootstrap confidence intervals:")

display(
    temporal_difference_ci_df.round(4)
)

print("\nClass-specific paired confidence intervals:")

display(
    temporal_class_difference_ci_df.round(4)
)

print("\nAll-available confusion matrix:")

display(
    all_available_fixed_results[
        "confusion_matrix"
    ]
)

print("\nPre-golden confusion matrix:")

display(
    pre_golden_fixed_results[
        "confusion_matrix"
    ]
)

print(
    "\nBootstrap iterations:",
    TEMPORAL_BOOTSTRAP_ITERATIONS
)

print(
    "Population agreement:",
    float(
        all_prediction_df[
            "household_ID"
        ].equals(
            pre_prediction_df[
                "household_ID"
            ]
        )
    )
)


# ------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------

temporal_overall_comparison_df.to_csv(
    OUTPUT_DIR
    / "temporal_sensitivity_overall_comparison.csv",
    index=False
)

temporal_point_difference_df.to_csv(
    OUTPUT_DIR
    / "temporal_sensitivity_point_differences.csv",
    index=False
)

temporal_class_comparison_df.to_csv(
    OUTPUT_DIR
    / "temporal_sensitivity_class_comparison.csv",
    index=False
)

temporal_difference_ci_df.to_csv(
    OUTPUT_DIR
    / "temporal_sensitivity_overall_difference_ci.csv",
    index=False
)

temporal_class_difference_ci_df.to_csv(
    OUTPUT_DIR
    / "temporal_sensitivity_class_difference_ci.csv",
    index=False
)

temporal_bootstrap_distributions_df.to_csv(
    OUTPUT_DIR
    / "temporal_sensitivity_bootstrap_distributions.csv",
    index=False
)

all_available_fixed_results[
    "predictions"
].to_csv(
    OUTPUT_DIR
    / "fixed_rf_all_available_oof_predictions.csv",
    index=False
)

pre_golden_fixed_results[
    "predictions"
].to_csv(
    OUTPUT_DIR
    / "fixed_rf_pre_golden_oof_predictions.csv",
    index=False
)

all_available_fixed_results[
    "fold_metrics"
].to_csv(
    OUTPUT_DIR
    / "fixed_rf_all_available_fold_metrics.csv",
    index=False
)

pre_golden_fixed_results[
    "fold_metrics"
].to_csv(
    OUTPUT_DIR
    / "fixed_rf_pre_golden_fold_metrics.csv",
    index=False
)

all_available_fixed_results[
    "confusion_matrix"
].to_csv(
    OUTPUT_DIR
    / "fixed_rf_all_available_confusion_matrix.csv"
)

pre_golden_fixed_results[
    "confusion_matrix"
].to_csv(
    OUTPUT_DIR
    / "fixed_rf_pre_golden_confusion_matrix.csv"
)

temporal_sensitivity_configuration = {
    "population": len(temporal_all_df),
    "population_identical_across_windows": True,
    "target_identical_across_windows": True,
    "feature_set": FINAL_WORKFLOW_FEATURES,
    "model_family": FINAL_WORKFLOW_MODEL_FAMILY,
    "model_parameters": FINAL_WORKFLOW_PARAMS,
    "imbalance_strategy": (
        "Balanced class weighting"
    ),
    "all_available_period": (
        "2022-10 through 2024-10"
    ),
    "pre_golden_period": (
        "2022-10 through 2024-04"
    ),
    "golden_window_start": "2024-05-01",
    "cv_folds": 5,
    "same_cv_splits_used": True,
    "bootstrap_iterations": (
        TEMPORAL_BOOTSTRAP_ITERATIONS
    ),
    "bootstrap_unit": "household",
    "bootstrap_stratified_by_archetype": True,
    "analysis_role": (
        "Secondary temporal-sensitivity analysis"
    ),
    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation"
    )
}

with open(
    OUTPUT_DIR
    / "temporal_sensitivity_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        temporal_sensitivity_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 10 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "All-available macro-F1:",
    round(
        all_available_fixed_results[
            "pooled_metrics"
        ]["macro_f1"],
        4
    )
)

print(
    "Pre-golden macro-F1:",
    round(
        pre_golden_fixed_results[
            "pooled_metrics"
        ]["macro_f1"],
        4
    )
)

print(
    "Macro-F1 difference:",
    round(
        pre_golden_fixed_results[
            "pooled_metrics"
        ]["macro_f1"]
        - all_available_fixed_results[
            "pooled_metrics"
        ]["macro_f1"],
        4
    )
)

print(
    "\nSaved Step 10 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 11 — FINAL WORKFLOW LEARNING CURVES
# ============================================================
# Purpose:
# 1. Diagnose overfitting and sample-size limitations in the
#    locked final fixed workflow.
# 2. Estimate training and validation learning curves for:
#    - macro-F1
#    - balanced accuracy
# 3. Use stratified training subsets so that every subset
#    preserves all four archetypes.
#
# Important:
# - The model configuration is already locked.
# - This step does not select another model.
# - Validation step does not select another model.
# - Validation scores are secondary diagnostic results.
# - Step 4 remains the primary performance estimate.
# ============================================================

import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    train_test_split
)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP11_OBJECTS = [
    "dataset_all",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "CLASS_TO_INDEX",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "pooled_performance_df"
]

missing_step11_objects = [
    object_name
    for object_name in REQUIRED_STEP11_OBJECTS
    if object_name not in globals()
]

if missing_step11_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 9A first. Missing objects: "
        f"{missing_step11_objects}"
    )


# ------------------------------------------------------------
# 2. Prepare the locked modelling dataset
# ------------------------------------------------------------

learning_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

if learning_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households were detected before "
        "learning-curve evaluation."
    )

X_learning = (
    learning_df[FINAL_WORKFLOW_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

y_learning = (
    learning_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
    .astype(int)
)

if X_learning.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all final predictors missing."
    )

if sorted(y_learning.unique()) != FIXED_CLASS_INDICES:
    raise ValueError(
        "The learning-curve target does not contain the "
        "expected four classes."
    )


# ------------------------------------------------------------
# 3. Learning-curve configuration
# ------------------------------------------------------------

LEARNING_CURVE_FOLDS = 5
LEARNING_CURVE_REPEATS = 3

LEARNING_CURVE_FRACTIONS = [
    0.40,
    0.55,
    0.70,
    0.85,
    1.00
]

learning_curve_cv = RepeatedStratifiedKFold(
    n_splits=LEARNING_CURVE_FOLDS,
    n_repeats=LEARNING_CURVE_REPEATS,
    random_state=RANDOM_STATE
)

learning_curve_records = []

learning_curve_start_time = time.time()


# ------------------------------------------------------------
# 4. Custom stratified learning-curve evaluation
# ------------------------------------------------------------

for split_number, (
    full_train_indices,
    validation_indices
) in enumerate(
    learning_curve_cv.split(
        X_learning,
        y_learning
    ),
    start=1
):
    repeat_number = (
        (split_number - 1)
        // LEARNING_CURVE_FOLDS
        + 1
    )

    fold_number = (
        (split_number - 1)
        % LEARNING_CURVE_FOLDS
        + 1
    )

    full_train_indices = np.asarray(
        full_train_indices
    )

    validation_indices = np.asarray(
        validation_indices
    )

    y_full_train = y_learning.iloc[
        full_train_indices
    ]

    for fraction_index, training_fraction in enumerate(
        LEARNING_CURVE_FRACTIONS,
        start=1
    ):
        if training_fraction < 1.0:
            sampled_train_indices, _ = train_test_split(
                full_train_indices,
                train_size=training_fraction,
                stratify=y_full_train,
                random_state=(
                    RANDOM_STATE
                    + split_number * 100
                    + fraction_index
                )
            )

            sampled_train_indices = np.asarray(
                sampled_train_indices
            )

        else:
            sampled_train_indices = (
                full_train_indices.copy()
            )

        X_train_subset = X_learning.iloc[
            sampled_train_indices
        ]

        y_train_subset = y_learning.iloc[
            sampled_train_indices
        ]

        X_validation = X_learning.iloc[
            validation_indices
        ]

        y_validation = y_learning.iloc[
            validation_indices
        ]

        subset_class_counts = (
            y_train_subset
            .value_counts()
            .reindex(
                FIXED_CLASS_INDICES,
                fill_value=0
            )
            .astype(int)
        )

        if (subset_class_counts == 0).any():
            raise RuntimeError(
                "A learning-curve training subset does not "
                "contain all four archetypes."
            )

        estimator = clone(
            final_workflow_pipeline_template
        )

        estimator.fit(
            X_train_subset,
            y_train_subset
        )

        train_predictions = estimator.predict(
            X_train_subset
        )

        validation_predictions = estimator.predict(
            X_validation
        )

        train_macro_f1 = f1_score(
            y_train_subset,
            train_predictions,
            average="macro",
            zero_division=0
        )

        validation_macro_f1 = f1_score(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0
        )

        train_balanced_accuracy = (
            balanced_accuracy_score(
                y_train_subset,
                train_predictions
            )
        )

        validation_balanced_accuracy = (
            balanced_accuracy_score(
                y_validation,
                validation_predictions
            )
        )

        learning_curve_records.append({
            "split_number": split_number,
            "repeat_number": repeat_number,
            "fold_number": fold_number,
            "training_fraction": (
                training_fraction
            ),
            "training_households": len(
                sampled_train_indices
            ),
            "validation_households": len(
                validation_indices
            ),
            "minimum_training_class_count": int(
                subset_class_counts.min()
            ),
            "train_macro_f1": (
                train_macro_f1
            ),
            "validation_macro_f1": (
                validation_macro_f1
            ),
            "macro_f1_gap": (
                train_macro_f1
                - validation_macro_f1
            ),
            "train_balanced_accuracy": (
                train_balanced_accuracy
            ),
            "validation_balanced_accuracy": (
                validation_balanced_accuracy
            ),
            "balanced_accuracy_gap": (
                train_balanced_accuracy
                - validation_balanced_accuracy
            )
        })


# ------------------------------------------------------------
# 5. Summarise the learning curves
# ------------------------------------------------------------

learning_curve_raw_df = pd.DataFrame(
    learning_curve_records
)

learning_curve_summary_df = (
    learning_curve_raw_df
    .groupby(
        "training_fraction",
        as_index=False
    )
    .agg(
        mean_training_households=(
            "training_households",
            "mean"
        ),
        minimum_training_class_count=(
            "minimum_training_class_count",
            "min"
        ),
        train_macro_f1_mean=(
            "train_macro_f1",
            "mean"
        ),
        train_macro_f1_std=(
            "train_macro_f1",
            "std"
        ),
        validation_macro_f1_mean=(
            "validation_macro_f1",
            "mean"
        ),
        validation_macro_f1_std=(
            "validation_macro_f1",
            "std"
        ),
        macro_f1_gap_mean=(
            "macro_f1_gap",
            "mean"
        ),
        train_balanced_accuracy_mean=(
            "train_balanced_accuracy",
            "mean"
        ),
        train_balanced_accuracy_std=(
            "train_balanced_accuracy",
            "std"
        ),
        validation_balanced_accuracy_mean=(
            "validation_balanced_accuracy",
            "mean"
        ),
        validation_balanced_accuracy_std=(
            "validation_balanced_accuracy",
            "std"
        ),
        balanced_accuracy_gap_mean=(
            "balanced_accuracy_gap",
            "mean"
        )
    )
)

learning_curve_summary_df[
    "mean_training_households"
] = (
    learning_curve_summary_df[
        "mean_training_households"
    ]
    .round()
    .astype(int)
)

learning_curve_runtime_seconds = (
    time.time()
    - learning_curve_start_time
)


# ------------------------------------------------------------
# 6. Plateau diagnostics
# ------------------------------------------------------------

last_learning_row = (
    learning_curve_summary_df
    .iloc[-1]
)

previous_learning_row = (
    learning_curve_summary_df
    .iloc[-2]
)

learning_curve_plateau_df = pd.DataFrame([
    {
        "diagnostic": (
            "Validation macro-F1 change from "
            "85% to 100% training data"
        ),
        "value": (
            last_learning_row[
                "validation_macro_f1_mean"
            ]
            - previous_learning_row[
                "validation_macro_f1_mean"
            ]
        )
    },
    {
        "diagnostic": (
            "Validation balanced-accuracy change "
            "from 85% to 100% training data"
        ),
        "value": (
            last_learning_row[
                "validation_balanced_accuracy_mean"
            ]
            - previous_learning_row[
                "validation_balanced_accuracy_mean"
            ]
        )
    },
    {
        "diagnostic": (
            "Final train-validation macro-F1 gap"
        ),
        "value": (
            last_learning_row[
                "macro_f1_gap_mean"
            ]
        )
    },
    {
        "diagnostic": (
            "Final train-validation balanced-accuracy gap"
        ),
        "value": (
            last_learning_row[
                "balanced_accuracy_gap_mean"
            ]
        )
    }
])


# ------------------------------------------------------------
# 7. Create figures
# ------------------------------------------------------------

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

training_sizes = (
    learning_curve_summary_df[
        "mean_training_households"
    ]
    .to_numpy()
)


# Macro-F1 learning curve

plt.figure(figsize=(8, 5))

plt.plot(
    training_sizes,
    learning_curve_summary_df[
        "train_macro_f1_mean"
    ],
    marker="o",
    label="Training macro-F1"
)

plt.plot(
    training_sizes,
    learning_curve_summary_df[
        "validation_macro_f1_mean"
    ],
    marker="o",
    label="Validation macro-F1"
)

plt.fill_between(
    training_sizes,
    (
        learning_curve_summary_df[
            "train_macro_f1_mean"
        ]
        - learning_curve_summary_df[
            "train_macro_f1_std"
        ]
    ),
    (
        learning_curve_summary_df[
            "train_macro_f1_mean"
        ]
        + learning_curve_summary_df[
            "train_macro_f1_std"
        ]
    ),
    alpha=0.2
)

plt.fill_between(
    training_sizes,
    (
        learning_curve_summary_df[
            "validation_macro_f1_mean"
        ]
        - learning_curve_summary_df[
            "validation_macro_f1_std"
        ]
    ),
    (
        learning_curve_summary_df[
            "validation_macro_f1_mean"
        ]
        + learning_curve_summary_df[
            "validation_macro_f1_std"
        ]
    ),
    alpha=0.2
)

plt.xlabel("Training households")
plt.ylabel("Macro-F1")
plt.title(
    "Learning Curve of the Final Billing-Based "
    "Archetype-Transfer Model"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "final_model_macro_f1_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Balanced-accuracy learning curve

plt.figure(figsize=(8, 5))

plt.plot(
    training_sizes,
    learning_curve_summary_df[
        "train_balanced_accuracy_mean"
    ],
    marker="o",
    label="Training balanced accuracy"
)

plt.plot(
    training_sizes,
    learning_curve_summary_df[
        "validation_balanced_accuracy_mean"
    ],
    marker="o",
    label="Validation balanced accuracy"
)

plt.fill_between(
    training_sizes,
    (
        learning_curve_summary_df[
            "train_balanced_accuracy_mean"
        ]
        - learning_curve_summary_df[
            "train_balanced_accuracy_std"
        ]
    ),
    (
        learning_curve_summary_df[
            "train_balanced_accuracy_mean"
        ]
        + learning_curve_summary_df[
            "train_balanced_accuracy_std"
        ]
    ),
    alpha=0.2
)

plt.fill_between(
    training_sizes,
    (
        learning_curve_summary_df[
            "validation_balanced_accuracy_mean"
        ]
        - learning_curve_summary_df[
            "validation_balanced_accuracy_std"
        ]
    ),
    (
        learning_curve_summary_df[
            "validation_balanced_accuracy_mean"
        ]
        + learning_curve_summary_df[
            "validation_balanced_accuracy_std"
        ]
    ),
    alpha=0.2
)

plt.xlabel("Training households")
plt.ylabel("Balanced accuracy")
plt.title(
    "Balanced-Accuracy Learning Curve of the Final "
    "Archetype-Transfer Model"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.ylim(0, 1)
plt.savefig(
    FIGURE_DIR
    / "final_model_balanced_accuracy_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 11 — FINAL WORKFLOW LEARNING CURVES")
print("=" * 80)

print("\nLearning-curve summary:")

display(
    learning_curve_summary_df.round(4)
)

print("\nPlateau and gap diagnostics:")

display(
    learning_curve_plateau_df.round(4)
)

print(
    "\nRepeated CV splits:",
    (
        LEARNING_CURVE_FOLDS
        * LEARNING_CURVE_REPEATS
    )
)

print(
    "Total fitted learning-curve models:",
    len(learning_curve_raw_df)
)

print(
    "Runtime in minutes:",
    round(
        learning_curve_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

learning_curve_raw_df.to_csv(
    OUTPUT_DIR
    / "final_model_learning_curve_raw_scores.csv",
    index=False
)

learning_curve_summary_df.to_csv(
    OUTPUT_DIR
    / "final_model_learning_curve_summary.csv",
    index=False
)

learning_curve_plateau_df.to_csv(
    OUTPUT_DIR
    / "final_model_learning_curve_plateau_diagnostics.csv",
    index=False
)

learning_curve_configuration = {
    "model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "model_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "feature_count": len(
        FINAL_WORKFLOW_FEATURES
    ),
    "feature_set": (
        FINAL_WORKFLOW_FEATURES
    ),
    "cv_folds": (
        LEARNING_CURVE_FOLDS
    ),
    "cv_repeats": (
        LEARNING_CURVE_REPEATS
    ),
    "training_fractions": (
        LEARNING_CURVE_FRACTIONS
    ),
    "training_subsets_stratified": True,
    "primary_diagnostic_metrics": [
        "macro_f1",
        "balanced_accuracy"
    ],
    "ordinary_accuracy_used": False,
    "role": (
        "Secondary model-capacity and sample-size diagnostic"
    ),
    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation"
    )
}

with open(
    OUTPUT_DIR
    / "final_model_learning_curve_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        learning_curve_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 11 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Final validation macro-F1:",
    round(
        last_learning_row[
            "validation_macro_f1_mean"
        ],
        4
    )
)

print(
    "Final macro-F1 train-validation gap:",
    round(
        last_learning_row[
            "macro_f1_gap_mean"
        ],
        4
    )
)

print(
    "Final validation balanced accuracy:",
    round(
        last_learning_row[
            "validation_balanced_accuracy_mean"
        ],
        4
    )
)

print(
    "\nSaved Step 11 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 12 — CROSS-VALIDATED PERMUTATION IMPORTANCE
# ============================================================
# Purpose:
# 1. Estimate feature importance on unseen validation households.
# 2. Use the locked final fixed workflow.
# 3. Calculate importance for both macro-F1 and balanced accuracy.
# 4. Assess importance stability across five validation folds.
#
# Important:
# - Permutation is performed on validation folds only.
# - These are predictive contributions, not causal effects.
# - Correlated predictors may share or mask importance.
# - This step does not change the locked final model.
# ============================================================

import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    make_scorer
)
from sklearn.model_selection import StratifiedKFold


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP12_OBJECTS = [
    "dataset_all",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "CLASS_TO_INDEX",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "RANDOM_STATE",
    "OUTPUT_DIR"
]

missing_step12_objects = [
    object_name
    for object_name in REQUIRED_STEP12_OBJECTS
    if object_name not in globals()
]

if missing_step12_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 9A first. Missing objects: "
        f"{missing_step12_objects}"
    )


# ------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------

importance_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

if importance_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households were detected before "
        "permutation-importance analysis."
    )

X_importance = (
    importance_df[FINAL_WORKFLOW_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

y_importance = (
    importance_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
    .astype(int)
)

if X_importance.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all final predictors missing."
    )


# ------------------------------------------------------------
# 3. Configuration
# ------------------------------------------------------------

PERMUTATION_FOLDS = 5
PERMUTATION_REPEATS = 20

permutation_cv = StratifiedKFold(
    n_splits=PERMUTATION_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

macro_f1_scorer = make_scorer(
    f1_score,
    average="macro",
    zero_division=0
)

balanced_accuracy_scorer = make_scorer(
    balanced_accuracy_score
)

permutation_fold_records = []
permutation_repeat_records = []
permutation_baseline_records = []

permutation_start_time = time.time()


# ------------------------------------------------------------
# 4. Validation-fold permutation importance
# ------------------------------------------------------------

for fold_number, (
    train_indices,
    validation_indices
) in enumerate(
    permutation_cv.split(
        X_importance,
        y_importance
    ),
    start=1
):
    print("\n" + "=" * 80)
    print(
        f"PERMUTATION IMPORTANCE — FOLD "
        f"{fold_number}/{PERMUTATION_FOLDS}"
    )
    print("=" * 80)

    X_train = X_importance.iloc[
        train_indices
    ]

    y_train = y_importance.iloc[
        train_indices
    ]

    X_validation = X_importance.iloc[
        validation_indices
    ]

    y_validation = y_importance.iloc[
        validation_indices
    ]

    estimator = clone(
        final_workflow_pipeline_template
    )

    estimator.fit(
        X_train,
        y_train
    )

    validation_predictions = estimator.predict(
        X_validation
    )

    baseline_macro_f1 = f1_score(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0
    )

    baseline_balanced_accuracy = (
        balanced_accuracy_score(
            y_validation,
            validation_predictions
        )
    )

    permutation_baseline_records.append({
        "fold": fold_number,
        "train_households": len(train_indices),
        "validation_households": len(validation_indices),
        "baseline_macro_f1": baseline_macro_f1,
        "baseline_balanced_accuracy": (
            baseline_balanced_accuracy
        )
    })

    macro_result = permutation_importance(
        estimator=estimator,
        X=X_validation,
        y=y_validation,
        scoring=macro_f1_scorer,
        n_repeats=PERMUTATION_REPEATS,
        random_state=(
            RANDOM_STATE
            + fold_number
        ),
        n_jobs=-1
    )

    balanced_result = permutation_importance(
        estimator=estimator,
        X=X_validation,
        y=y_validation,
        scoring=balanced_accuracy_scorer,
        n_repeats=PERMUTATION_REPEATS,
        random_state=(
            RANDOM_STATE
            + 100
            + fold_number
        ),
        n_jobs=-1
    )

    for feature_index, feature_name in enumerate(
        FINAL_WORKFLOW_FEATURES
    ):
        macro_repeat_values = (
            macro_result.importances[
                feature_index
            ]
        )

        balanced_repeat_values = (
            balanced_result.importances[
                feature_index
            ]
        )

        permutation_fold_records.append({
            "fold": fold_number,
            "feature": feature_name,

            "macro_f1_importance_mean": float(
                macro_repeat_values.mean()
            ),
            "macro_f1_importance_std": float(
                macro_repeat_values.std(
                    ddof=1
                )
            ),

            "balanced_accuracy_importance_mean": float(
                balanced_repeat_values.mean()
            ),
            "balanced_accuracy_importance_std": float(
                balanced_repeat_values.std(
                    ddof=1
                )
            )
        })

        for repeat_number in range(
            PERMUTATION_REPEATS
        ):
            permutation_repeat_records.append({
                "fold": fold_number,
                "repeat": repeat_number + 1,
                "feature": feature_name,

                "macro_f1_importance": float(
                    macro_repeat_values[
                        repeat_number
                    ]
                ),

                "balanced_accuracy_importance": float(
                    balanced_repeat_values[
                        repeat_number
                    ]
                )
            })

    fold_top_features = (
        pd.DataFrame(
            permutation_fold_records
        )
        .loc[
            lambda frame:
            frame["fold"] == fold_number
        ]
        .sort_values(
            "macro_f1_importance_mean",
            ascending=False
        )
        .head(5)
    )

    print(
        "Baseline macro-F1:",
        round(
            baseline_macro_f1,
            4
        )
    )

    print(
        "Baseline balanced accuracy:",
        round(
            baseline_balanced_accuracy,
            4
        )
    )

    print("\nTop five fold-specific features:")

    display(
        fold_top_features[
            [
                "feature",
                "macro_f1_importance_mean",
                "balanced_accuracy_importance_mean"
            ]
        ].round(4)
    )


# ------------------------------------------------------------
# 5. Compile fold-level and repeat-level results
# ------------------------------------------------------------

permutation_fold_df = pd.DataFrame(
    permutation_fold_records
)

permutation_repeat_df = pd.DataFrame(
    permutation_repeat_records
)

permutation_baseline_df = pd.DataFrame(
    permutation_baseline_records
)


# ------------------------------------------------------------
# 6. Aggregate importance across validation folds
# ------------------------------------------------------------

permutation_summary_df = (
    permutation_fold_df
    .groupby(
        "feature",
        as_index=False
    )
    .agg(
        macro_f1_importance_mean=(
            "macro_f1_importance_mean",
            "mean"
        ),
        macro_f1_importance_std_across_folds=(
            "macro_f1_importance_mean",
            "std"
        ),
        macro_f1_importance_median=(
            "macro_f1_importance_mean",
            "median"
        ),
        macro_f1_importance_min=(
            "macro_f1_importance_mean",
            "min"
        ),
        macro_f1_importance_max=(
            "macro_f1_importance_mean",
            "max"
        ),

        balanced_accuracy_importance_mean=(
            "balanced_accuracy_importance_mean",
            "mean"
        ),
        balanced_accuracy_importance_std_across_folds=(
            "balanced_accuracy_importance_mean",
            "std"
        ),
        balanced_accuracy_importance_median=(
            "balanced_accuracy_importance_mean",
            "median"
        ),
        balanced_accuracy_importance_min=(
            "balanced_accuracy_importance_mean",
            "min"
        ),
        balanced_accuracy_importance_max=(
            "balanced_accuracy_importance_mean",
            "max"
        )
    )
)

positive_fold_counts = (
    permutation_fold_df
    .assign(
        positive_macro_f1=(
            permutation_fold_df[
                "macro_f1_importance_mean"
            ] > 0
        ),
        positive_balanced_accuracy=(
            permutation_fold_df[
                "balanced_accuracy_importance_mean"
            ] > 0
        )
    )
    .groupby(
        "feature",
        as_index=False
    )
    .agg(
        positive_macro_f1_folds=(
            "positive_macro_f1",
            "sum"
        ),
        positive_balanced_accuracy_folds=(
            "positive_balanced_accuracy",
            "sum"
        )
    )
)

permutation_summary_df = (
    permutation_summary_df
    .merge(
        positive_fold_counts,
        on="feature",
        how="left",
        validate="one_to_one"
    )
)

permutation_summary_df = (
    permutation_summary_df
    .sort_values(
        [
            "macro_f1_importance_mean",
            "balanced_accuracy_importance_mean"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

permutation_summary_df[
    "macro_f1_rank"
] = np.arange(
    1,
    len(permutation_summary_df) + 1
)

permutation_summary_df[
    "stable_positive_macro_f1"
] = (
    permutation_summary_df[
        "positive_macro_f1_folds"
    ] >= 4
)

permutation_summary_df[
    "stable_positive_balanced_accuracy"
] = (
    permutation_summary_df[
        "positive_balanced_accuracy_folds"
    ] >= 4
)


# ------------------------------------------------------------
# 7. Create interpretation groups
# ------------------------------------------------------------

def classify_importance_stability(row):
    if (
        row["positive_macro_f1_folds"] >= 4
        and row[
            "macro_f1_importance_mean"
        ] > 0
    ):
        return "Consistently positive"

    if (
        row["positive_macro_f1_folds"] >= 2
        and row[
            "macro_f1_importance_mean"
        ] > 0
    ):
        return "Mixed positive"

    return "Unstable or negligible"


permutation_summary_df[
    "importance_stability"
] = permutation_summary_df.apply(
    classify_importance_stability,
    axis=1
)


# ------------------------------------------------------------
# 8. Create figures
# ------------------------------------------------------------

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TOP_FEATURES_TO_PLOT = 15

top_macro_features = (
    permutation_summary_df
    .head(TOP_FEATURES_TO_PLOT)
    .sort_values(
        "macro_f1_importance_mean",
        ascending=True
    )
)

plt.figure(figsize=(9, 7))

plt.barh(
    top_macro_features["feature"],
    top_macro_features[
        "macro_f1_importance_mean"
    ],
    xerr=top_macro_features[
        "macro_f1_importance_std_across_folds"
    ]
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel(
    "Decrease in validation macro-F1 after permutation"
)

plt.ylabel("Monthly billing feature")

plt.title(
    "Cross-Validated Permutation Importance "
    "of the Final Archetype-Transfer Model"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "final_model_cv_permutation_importance_macro_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


top_balanced_features = (
    permutation_summary_df
    .sort_values(
        "balanced_accuracy_importance_mean",
        ascending=False
    )
    .head(TOP_FEATURES_TO_PLOT)
    .sort_values(
        "balanced_accuracy_importance_mean",
        ascending=True
    )
)

plt.figure(figsize=(9, 7))

plt.barh(
    top_balanced_features["feature"],
    top_balanced_features[
        "balanced_accuracy_importance_mean"
    ],
    xerr=top_balanced_features[
        "balanced_accuracy_importance_std_across_folds"
    ]
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel(
    "Decrease in validation balanced accuracy "
    "after permutation"
)

plt.ylabel("Monthly billing feature")

plt.title(
    "Balanced-Accuracy Permutation Importance "
    "of the Final Archetype-Transfer Model"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "final_model_cv_permutation_importance_balanced_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

permutation_runtime_seconds = (
    time.time()
    - permutation_start_time
)

print("\n" + "=" * 80)
print("STEP 12 — CROSS-VALIDATED PERMUTATION IMPORTANCE")
print("=" * 80)

print("\nValidation-fold baselines:")

display(
    permutation_baseline_df.round(4)
)

print("\nRanked feature importance:")

display(
    permutation_summary_df[
        [
            "macro_f1_rank",
            "feature",
            "macro_f1_importance_mean",
            "macro_f1_importance_std_across_folds",
            "positive_macro_f1_folds",
            "balanced_accuracy_importance_mean",
            "balanced_accuracy_importance_std_across_folds",
            "positive_balanced_accuracy_folds",
            "importance_stability"
        ]
    ].round(4)
)

print("\nConsistently positive features:")

display(
    permutation_summary_df.loc[
        permutation_summary_df[
            "importance_stability"
        ] == "Consistently positive",
        [
            "feature",
            "macro_f1_importance_mean",
            "positive_macro_f1_folds",
            "balanced_accuracy_importance_mean",
            "positive_balanced_accuracy_folds"
        ]
    ].round(4)
)

print(
    "\nPermutation folds:",
    PERMUTATION_FOLDS
)

print(
    "Repeats per feature per fold:",
    PERMUTATION_REPEATS
)

print(
    "Runtime in minutes:",
    round(
        permutation_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

permutation_baseline_df.to_csv(
    OUTPUT_DIR
    / "final_model_permutation_validation_baselines.csv",
    index=False
)

permutation_fold_df.to_csv(
    OUTPUT_DIR
    / "final_model_permutation_importance_by_fold.csv",
    index=False
)

permutation_repeat_df.to_csv(
    OUTPUT_DIR
    / "final_model_permutation_importance_raw_repeats.csv",
    index=False
)

permutation_summary_df.to_csv(
    OUTPUT_DIR
    / "final_model_permutation_importance_summary.csv",
    index=False
)

permutation_configuration = {
    "model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "model_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "feature_count": len(
        FINAL_WORKFLOW_FEATURES
    ),
    "cv_folds": (
        PERMUTATION_FOLDS
    ),
    "permutation_repeats_per_fold": (
        PERMUTATION_REPEATS
    ),
    "importance_data": (
        "Untouched validation folds only"
    ),
    "importance_metrics": [
        "macro_f1",
        "balanced_accuracy"
    ],
    "stable_positive_definition": (
        "Positive mean importance in at least "
        "four of five validation folds"
    ),
    "causal_interpretation_allowed": False,
    "correlated_feature_warning": (
        "Correlated predictors may divide or mask "
        "permutation importance."
    ),
    "model_selection_performed": False
}

with open(
    OUTPUT_DIR
    / "final_model_permutation_importance_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        permutation_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 12 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Features with consistently positive "
    "macro-F1 importance:",
    int(
        (
            permutation_summary_df[
                "importance_stability"
            ] == "Consistently positive"
        ).sum()
    )
)

print(
    "\nSaved Step 12 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 13 — THREE-CLASS SCARCITY SENSITIVITY
# ============================================================
# Purpose:
# 1. Examine whether severe minority-class scarcity is a major
#    reason for weak four-class archetype transfer.
# 2. Merge Peak-Exposed and Daytime-Active only for a
#    supplementary supervised-learning sensitivity analysis.
# 3. Repeat leakage-safe nested model and feature-set selection.
# 4. Generate bootstrap confidence intervals.
#
# Important:
# - The K=4 unsupervised solution remains unchanged.
# - The merged class combines behaviourally different temporal
#   profiles and must not be interpreted as a new archetype.
# - Three-class performance is not directly comparable with
#   four-class performance because the task is easier.
# ============================================================

import json
import time

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    top_k_accuracy_score
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP13_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "MODEL_FEATURE_SETS",
    "PRIMARY_CANDIDATE_SPECIFICATIONS",
    "build_candidate_pipeline",
    "SCORING",
    "serialise_parameters",
    "ORDERED_CLUSTER_NAMES",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "pooled_performance_df",
    "counts_all"
]

missing_step13_objects = [
    object_name
    for object_name in REQUIRED_STEP13_OBJECTS
    if object_name not in globals()
]

if missing_step13_objects:
    raise RuntimeError(
        "Run Steps 1–4 before Step 13. Missing objects: "
        f"{missing_step13_objects}"
    )


# ------------------------------------------------------------
# 2. Define the supplementary three-class target
# ------------------------------------------------------------

THREE_CLASS_NAMES = [
    "Peak/Daytime Temporal Archetypes",
    "Typical Household",
    "Heavy Continuous"
]

THREE_CLASS_TO_INDEX = {
    class_name: class_index
    for class_index, class_name
    in enumerate(THREE_CLASS_NAMES)
}

THREE_INDEX_TO_CLASS = {
    class_index: class_name
    for class_name, class_index
    in THREE_CLASS_TO_INDEX.items()
}

THREE_FIXED_CLASS_INDICES = list(
    range(len(THREE_CLASS_NAMES))
)

FOUR_TO_THREE_CLASS_MAP = {
    "Peak-Exposed":
        "Peak/Daytime Temporal Archetypes",

    "Daytime-Active":
        "Peak/Daytime Temporal Archetypes",

    "Typical Household":
        "Typical Household",

    "Heavy Continuous":
        "Heavy Continuous"
}

three_class_df = (
    nested_df
    .copy()
    .reset_index(drop=True)
)

three_class_df["three_class_name"] = (
    three_class_df["cluster_name"]
    .astype(str)
    .map(FOUR_TO_THREE_CLASS_MAP)
)

if three_class_df[
    "three_class_name"
].isna().any():
    raise ValueError(
        "At least one four-class label could not be mapped "
        "to the three-class sensitivity target."
    )

y_three = (
    three_class_df["three_class_name"]
    .map(THREE_CLASS_TO_INDEX)
    .astype(int)
)

three_class_counts = (
    three_class_df["three_class_name"]
    .value_counts()
    .reindex(
        THREE_CLASS_NAMES,
        fill_value=0
    )
    .astype(int)
)

expected_three_class_counts = pd.Series({
    "Peak/Daytime Temporal Archetypes": int(
        counts_all.get("Peak-Exposed", 0)
        + counts_all.get("Daytime-Active", 0)
    ),
    "Typical Household": int(
        counts_all.get("Typical Household", 0)
    ),
    "Heavy Continuous": int(
        counts_all.get("Heavy Continuous", 0)
    )
}).reindex(THREE_CLASS_NAMES)

if not three_class_counts.equals(
    expected_three_class_counts
):
    raise ValueError(
        "Unexpected three-class distribution:\n"
        f"{pd.DataFrame({'expected': expected_three_class_counts, 'observed': three_class_counts})}"
    )


# ------------------------------------------------------------
# 3. Three-class probability and metric helpers
# ------------------------------------------------------------

def align_three_class_probabilities(
    raw_probabilities,
    fitted_classes
):
    aligned = np.zeros(
        (
            raw_probabilities.shape[0],
            len(THREE_FIXED_CLASS_INDICES)
        ),
        dtype=float
    )

    for source_column, class_index in enumerate(
        fitted_classes
    ):
        aligned[:, int(class_index)] = (
            raw_probabilities[:, source_column]
        )

    if not np.allclose(
        aligned.sum(axis=1),
        1.0,
        atol=1e-8
    ):
        raise RuntimeError(
            "Three-class probabilities do not sum to one."
        )

    return aligned


def three_class_brier_score(
    y_true,
    probabilities
):
    one_hot_targets = np.eye(
        len(THREE_FIXED_CLASS_INDICES)
    )[np.asarray(y_true, dtype=int)]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot_targets
                ) ** 2,
                axis=1
            )
        )
    )


def calculate_three_class_metrics(
    y_true,
    predicted_classes,
    probabilities
):
    return {
        "accuracy": accuracy_score(
            y_true,
            predicted_classes
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predicted_classes
            )
        ),

        "macro_f1": f1_score(
            y_true,
            predicted_classes,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            predicted_classes,
            average="weighted",
            zero_division=0
        ),

        "top_2_accuracy": (
            top_k_accuracy_score(
                y_true,
                probabilities,
                k=2,
                labels=THREE_FIXED_CLASS_INDICES
            )
        ),

        "multiclass_log_loss": log_loss(
            y_true,
            probabilities,
            labels=THREE_FIXED_CLASS_INDICES
        ),

        "multiclass_brier_sum": (
            three_class_brier_score(
                y_true,
                probabilities
            )
        )
    }


# ------------------------------------------------------------
# 4. Nested CV configuration
# ------------------------------------------------------------

THREE_OUTER_FOLDS = 5
THREE_INNER_FOLDS = 4

three_outer_cv = StratifiedKFold(
    n_splits=THREE_OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

three_inner_candidate_records = []
three_outer_selection_records = []
three_outer_metric_records = []
three_oof_records = []

three_class_start_time = time.time()


# ------------------------------------------------------------
# 5. Leakage-safe nested model selection
# ------------------------------------------------------------

for outer_fold, (
    outer_train_indices,
    outer_test_indices
) in enumerate(
    three_outer_cv.split(
        X_nested_source,
        y_three
    ),
    start=1
):
    print("\n" + "=" * 80)

    print(
        f"THREE-CLASS NESTED CV — OUTER FOLD "
        f"{outer_fold}/{THREE_OUTER_FOLDS}"
    )

    print("=" * 80)

    outer_train_source = (
        X_nested_source.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_source = (
        X_nested_source.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_train = (
        y_three.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    y_outer_test = (
        y_three.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_metadata = (
        three_class_df.iloc[
            outer_test_indices
        ][
            [
                "household_ID",
                "three_class_name"
            ]
        ]
        .reset_index(drop=True)
    )

    three_inner_cv = StratifiedKFold(
        n_splits=THREE_INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_STATE + outer_fold
        )
    )

    fold_candidates = []

    for feature_set_name, feature_columns in (
        MODEL_FEATURE_SETS.items()
    ):
        X_outer_train = (
            outer_train_source[
                feature_columns
            ]
        )

        for candidate_specification in (
            PRIMARY_CANDIDATE_SPECIFICATIONS
        ):
            model_family = (
                candidate_specification[
                    "model_family"
                ]
            )

            imbalance_strategy = (
                candidate_specification[
                    "imbalance_strategy"
                ]
            )

            class_weight = (
                candidate_specification[
                    "class_weight"
                ]
            )

            pipeline, parameter_grid = (
                build_candidate_pipeline(
                    model_family=model_family,
                    class_weight=class_weight
                )
            )

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=parameter_grid,
                scoring=SCORING,
                refit=False,
                cv=three_inner_cv,
                n_jobs=-1,
                pre_dispatch="2*n_jobs",
                return_train_score=True,
                error_score="raise"
            )

            search.fit(
                X_outer_train,
                y_outer_train
            )

            results = pd.DataFrame(
                search.cv_results_
            )

            results = (
                results
                .sort_values(
                    [
                        "mean_test_macro_f1",
                        "mean_test_balanced_accuracy",
                        "mean_test_weighted_f1",
                        "mean_test_accuracy"
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                        False
                    ]
                )
                .reset_index(drop=True)
            )

            best_result = select_balanced_cv_candidate(
                results,
                model_family
            )
            best_parameters = dict(
                best_result["params"]
            )

            candidate_record = {
                "outer_fold": outer_fold,
                "feature_set": feature_set_name,
                "feature_count": len(feature_columns),
                "model_family": model_family,
                "imbalance_strategy": (
                    imbalance_strategy
                ),
                "selected_parameters": (
                    serialise_parameters(
                        best_parameters
                    )
                ),
                "inner_macro_f1": float(
                    best_result[
                        "mean_test_macro_f1"
                    ]
                ),
                "inner_balanced_accuracy": float(
                    best_result[
                        "mean_test_balanced_accuracy"
                    ]
                ),
                "inner_weighted_f1": float(
                    best_result[
                        "mean_test_weighted_f1"
                    ]
                ),
                "inner_accuracy": float(
                    best_result[
                        "mean_test_accuracy"
                    ]
                ),
                "inner_train_macro_f1": float(
                    best_result[
                        "mean_train_macro_f1"
                    ]
                )
            }

            candidate_record[
                "inner_macro_f1_gap"
            ] = (
                candidate_record[
                    "inner_train_macro_f1"
                ]
                - candidate_record[
                    "inner_macro_f1"
                ]
            )

            three_inner_candidate_records.append(
                candidate_record
            )

            fold_candidates.append({
                **candidate_record,
                "feature_columns": list(
                    feature_columns
                ),
                "pipeline_template": pipeline,
                "parameter_dictionary": (
                    best_parameters
                )
            })

    fold_candidates = sorted(
        fold_candidates,
        key=lambda record: (
            record["inner_macro_f1"],
            record["inner_balanced_accuracy"],
            record["inner_weighted_f1"],
            record["inner_accuracy"]
        ),
        reverse=True
    )

    selected_candidate = fold_candidates[0]

    selected_pipeline = clone(
        selected_candidate[
            "pipeline_template"
        ]
    )

    selected_pipeline.set_params(
        **selected_candidate[
            "parameter_dictionary"
        ]
    )

    selected_features = (
        selected_candidate[
            "feature_columns"
        ]
    )

    selected_pipeline.fit(
        outer_train_source[
            selected_features
        ],
        y_outer_train
    )

    outer_predictions = (
        selected_pipeline.predict(
            outer_test_source[
                selected_features
            ]
        )
    )

    raw_outer_probabilities = (
        selected_pipeline.predict_proba(
            outer_test_source[
                selected_features
            ]
        )
    )

    outer_probabilities = (
        align_three_class_probabilities(
            raw_outer_probabilities,
            selected_pipeline.classes_
        )
    )

    outer_metrics = (
        calculate_three_class_metrics(
            y_outer_test,
            outer_predictions,
            outer_probabilities
        )
    )

    three_outer_selection_records.append({
        "outer_fold": outer_fold,
        "selected_feature_set": (
            selected_candidate[
                "feature_set"
            ]
        ),
        "selected_model_family": (
            selected_candidate[
                "model_family"
            ]
        ),
        "selected_imbalance_strategy": (
            selected_candidate[
                "imbalance_strategy"
            ]
        ),
        "selected_parameters": (
            selected_candidate[
                "selected_parameters"
            ]
        ),
        "inner_macro_f1": (
            selected_candidate[
                "inner_macro_f1"
            ]
        ),
        "inner_balanced_accuracy": (
            selected_candidate[
                "inner_balanced_accuracy"
            ]
        )
    })

    three_outer_metric_records.append({
        "outer_fold": outer_fold,
        **outer_metrics
    })

    for row_position in range(
        len(outer_test_indices)
    ):
        prediction_record = {
            "household_ID": (
                outer_test_metadata.iloc[
                    row_position
                ]["household_ID"]
            ),
            "outer_fold": outer_fold,
            "true_class_index": int(
                y_outer_test.iloc[
                    row_position
                ]
            ),
            "true_class_name": (
                outer_test_metadata.iloc[
                    row_position
                ][
                    "three_class_name"
                ]
            ),
            "predicted_class_index": int(
                outer_predictions[
                    row_position
                ]
            ),
            "predicted_class_name": (
                THREE_INDEX_TO_CLASS[
                    int(
                        outer_predictions[
                            row_position
                        ]
                    )
                ]
            )
        }

        for class_index, class_name in enumerate(
            THREE_CLASS_NAMES
        ):
            safe_name = (
                class_name
                .lower()
                .replace("-", "_")
                .replace("/", "_")
                .replace(" ", "_")
            )

            prediction_record[
                f"probability_{safe_name}"
            ] = float(
                outer_probabilities[
                    row_position,
                    class_index
                ]
            )

        three_oof_records.append(
            prediction_record
        )

    print(
        "Selected workflow:",
        selected_candidate[
            "feature_set"
        ],
        "|",
        selected_candidate[
            "model_family"
        ],
        "|",
        selected_candidate[
            "imbalance_strategy"
        ]
    )

    print(
        "Outer macro-F1:",
        round(
            outer_metrics[
                "macro_f1"
            ],
            4
        )
    )


# ------------------------------------------------------------
# 6. Compile pooled OOF results
# ------------------------------------------------------------

three_inner_candidates_df = pd.DataFrame(
    three_inner_candidate_records
)

three_outer_selections_df = pd.DataFrame(
    three_outer_selection_records
)

three_outer_metrics_df = pd.DataFrame(
    three_outer_metric_records
)

three_oof_predictions_df = pd.DataFrame(
    three_oof_records
)

if len(three_oof_predictions_df) != len(
    three_class_df
):
    raise RuntimeError(
        "Three-class sensitivity did not create exactly "
        "one OOF prediction per household."
    )

if three_oof_predictions_df[
    "household_ID"
].duplicated().any():
    raise RuntimeError(
        "Duplicate three-class OOF predictions were created."
    )

three_true = (
    three_oof_predictions_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

three_predicted = (
    three_oof_predictions_df[
        "predicted_class_index"
    ]
    .to_numpy(dtype=int)
)

three_probability_columns = []

for class_name in THREE_CLASS_NAMES:
    safe_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    three_probability_columns.append(
        f"probability_{safe_name}"
    )

three_probabilities = (
    three_oof_predictions_df[
        three_probability_columns
    ]
    .to_numpy(dtype=float)
)

three_pooled_metrics = (
    calculate_three_class_metrics(
        three_true,
        three_predicted,
        three_probabilities
    )
)

three_pooled_performance_df = pd.DataFrame([
    {
        "evaluation": (
            "Supplementary three-class nested OOF"
        ),
        **three_pooled_metrics
    }
])

three_report = classification_report(
    three_true,
    three_predicted,
    labels=THREE_FIXED_CLASS_INDICES,
    target_names=THREE_CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

three_class_metric_records = []

for class_name in THREE_CLASS_NAMES:
    three_class_metric_records.append({
        "class_name": class_name,
        "precision": (
            three_report[
                class_name
            ]["precision"]
        ),
        "recall": (
            three_report[
                class_name
            ]["recall"]
        ),
        "f1_score": (
            three_report[
                class_name
            ]["f1-score"]
        ),
        "support": int(
            three_report[
                class_name
            ]["support"]
        )
    })

three_class_metrics_df = pd.DataFrame(
    three_class_metric_records
)

three_confusion_df = pd.DataFrame(
    confusion_matrix(
        three_true,
        three_predicted,
        labels=THREE_FIXED_CLASS_INDICES
    ),
    index=THREE_CLASS_NAMES,
    columns=THREE_CLASS_NAMES
)

three_selection_counts = (
    three_outer_selections_df
    .groupby(
        [
            "selected_feature_set",
            "selected_model_family",
            "selected_imbalance_strategy"
        ]
    )
    .size()
    .reset_index(
        name="outer_folds_selected"
    )
    .sort_values(
        "outer_folds_selected",
        ascending=False
    )
)


# ------------------------------------------------------------
# 7. Stratified bootstrap confidence intervals
# ------------------------------------------------------------

THREE_BOOTSTRAP_ITERATIONS = 2000

three_bootstrap_generator = (
    np.random.default_rng(
        RANDOM_STATE + 1300
    )
)

three_class_row_indices = {
    class_index: np.where(
        three_true == class_index
    )[0]
    for class_index
    in THREE_FIXED_CLASS_INDICES
}

three_bootstrap_records = []

for bootstrap_iteration in range(
    1,
    THREE_BOOTSTRAP_ITERATIONS + 1
):
    sampled_indices = np.concatenate([
        three_bootstrap_generator.choice(
            three_class_row_indices[
                class_index
            ],
            size=len(
                three_class_row_indices[
                    class_index
                ]
            ),
            replace=True
        )
        for class_index
        in THREE_FIXED_CLASS_INDICES
    ])

    three_bootstrap_generator.shuffle(
        sampled_indices
    )

    sampled_metrics = (
        calculate_three_class_metrics(
            three_true[sampled_indices],
            three_predicted[sampled_indices],
            three_probabilities[
                sampled_indices
            ]
        )
    )

    sampled_precision = precision_score(
        three_true[sampled_indices],
        three_predicted[sampled_indices],
        labels=THREE_FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    sampled_recall = recall_score(
        three_true[sampled_indices],
        three_predicted[sampled_indices],
        labels=THREE_FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    sampled_f1 = f1_score(
        three_true[sampled_indices],
        three_predicted[sampled_indices],
        labels=THREE_FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    record = {
        "bootstrap_iteration":
            bootstrap_iteration,
        **sampled_metrics
    }

    for class_index, class_name in enumerate(
        THREE_CLASS_NAMES
    ):
        safe_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace("/", "_")
            .replace(" ", "_")
        )

        record[
            f"{safe_name}_precision"
        ] = sampled_precision[class_index]

        record[
            f"{safe_name}_recall"
        ] = sampled_recall[class_index]

        record[
            f"{safe_name}_f1"
        ] = sampled_f1[class_index]

    three_bootstrap_records.append(
        record
    )

three_bootstrap_df = pd.DataFrame(
    three_bootstrap_records
)

three_overall_ci_records = []

for metric_name in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "top_2_accuracy",
    "multiclass_log_loss",
    "multiclass_brier_sum"
]:
    three_overall_ci_records.append({
        "metric": metric_name,
        "point_estimate": (
            three_pooled_metrics[
                metric_name
            ]
        ),
        "ci_2.5_percent": (
            three_bootstrap_df[
                metric_name
            ].quantile(0.025)
        ),
        "ci_97.5_percent": (
            three_bootstrap_df[
                metric_name
            ].quantile(0.975)
        )
    })

three_overall_ci_df = pd.DataFrame(
    three_overall_ci_records
)

three_class_ci_records = []

for class_name in THREE_CLASS_NAMES:
    safe_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    point_row = (
        three_class_metrics_df.loc[
            three_class_metrics_df[
                "class_name"
            ] == class_name
        ].iloc[0]
    )

    for metric_name in [
        "precision",
        "recall",
        "f1"
    ]:
        point_column = (
            "f1_score"
            if metric_name == "f1"
            else metric_name
        )

        bootstrap_column = (
            f"{safe_name}_{metric_name}"
        )

        three_class_ci_records.append({
            "class_name": class_name,
            "metric": metric_name,
            "point_estimate": (
                point_row[
                    point_column
                ]
            ),
            "ci_2.5_percent": (
                three_bootstrap_df[
                    bootstrap_column
                ].quantile(0.025)
            ),
            "ci_97.5_percent": (
                three_bootstrap_df[
                    bootstrap_column
                ].quantile(0.975)
            )
        })

three_class_ci_df = pd.DataFrame(
    three_class_ci_records
)


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

three_class_runtime_seconds = (
    time.time()
    - three_class_start_time
)

print("\n" + "=" * 80)
print("STEP 13 — THREE-CLASS SENSITIVITY RESULTS")
print("=" * 80)

print("\nThree-class target distribution:")

display(
    three_class_counts
    .rename("households")
    .to_frame()
)

print("\nSelected workflow in each outer fold:")

display(
    three_outer_selections_df
)

print("\nWorkflow-selection frequency:")

display(
    three_selection_counts
)

print("\nPooled three-class performance:")

display(
    three_pooled_performance_df.round(4)
)

print("\nThree-class performance intervals:")

display(
    three_overall_ci_df.round(4)
)

print("\nClass-specific performance:")

display(
    three_class_metrics_df.round(4)
)

print("\nClass-specific intervals:")

display(
    three_class_ci_df.round(4)
)

print("\nThree-class confusion matrix:")

display(
    three_confusion_df
)

print(
    "\nRuntime in minutes:",
    round(
        three_class_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

three_inner_candidates_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_inner_candidates.csv",
    index=False
)

three_outer_selections_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_outer_selections.csv",
    index=False
)

three_outer_metrics_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_outer_metrics.csv",
    index=False
)

three_oof_predictions_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_oof_predictions.csv",
    index=False
)

three_pooled_performance_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_pooled_performance.csv",
    index=False
)

three_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_class_metrics.csv",
    index=False
)

three_confusion_df.to_csv(
    OUTPUT_DIR
    / "three_class_nested_confusion_matrix.csv"
)

three_overall_ci_df.to_csv(
    OUTPUT_DIR
    / "three_class_overall_bootstrap_ci.csv",
    index=False
)

three_class_ci_df.to_csv(
    OUTPUT_DIR
    / "three_class_class_bootstrap_ci.csv",
    index=False
)

three_selection_counts.to_csv(
    OUTPUT_DIR
    / "three_class_workflow_selection_counts.csv",
    index=False
)

three_class_configuration = {
    "analysis_role": (
        "Supplementary class-scarcity sensitivity"
    ),
    "original_clustering_solution_changed": False,
    "primary_clustering_k": 4,
    "merged_classes": [
        "Peak-Exposed",
        "Daytime-Active"
    ],
    "merged_class_name": (
        "Peak/Daytime Temporal Archetypes"
    ),
    "merged_classes_behaviourally_equivalent": False,
    "reason_for_merge": (
        "To examine the effect of severe supervised-class "
        "scarcity, not to redefine the archetypes."
    ),
    "outer_folds": THREE_OUTER_FOLDS,
    "inner_folds": THREE_INNER_FOLDS,
    "bootstrap_iterations": (
        THREE_BOOTSTRAP_ITERATIONS
    ),
    "primary_metric": "macro_f1",
    "direct_numeric_comparison_with_four_class_task": (
        "Must be interpreted cautiously because the "
        "classification difficulty differs."
    )
}

with open(
    OUTPUT_DIR
    / "three_class_sensitivity_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        three_class_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 13 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Three-class macro-F1:",
    round(
        three_pooled_metrics[
            "macro_f1"
        ],
        4
    )
)

print(
    "Three-class balanced accuracy:",
    round(
        three_pooled_metrics[
            "balanced_accuracy"
        ],
        4
    )
)

print(
    "\nThe primary K=4 archetype solution remains unchanged."
)

print(
    "Saved Step 13 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 14 — MULTICLASS ROC AND PRECISION–RECALL AUDIT
# ============================================================
# Purpose:
# 1. Evaluate one-vs-rest discrimination using the untouched
#    Step 4 nested out-of-fold probabilities.
# 2. Report per-class ROC-AUC and average precision.
# 3. Report macro and weighted probability metrics.
# 4. Calculate stratified household-level bootstrap intervals.
# 5. Compare the primary nested procedure with the selected
#    fixed workflow as a secondary check.
#
# Important:
# - Average precision is used as the precision–recall summary.
# - Step 4 remains the primary approximately unbiased internal
#   evaluation source for the adaptive selection procedure.
# - The selected fixed-workflow result is supplementary because its
#   configuration was selected using the complete dataset.
# ============================================================

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve
)
from sklearn.preprocessing import label_binarize


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP14_OBJECTS = [
    "oof_predictions_df",
    "probability_columns",
    "all_available_fixed_results",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "OUTPUT_DIR",
    "RANDOM_STATE"
]

missing_step14_objects = [
    object_name
    for object_name in REQUIRED_STEP14_OBJECTS
    if object_name not in globals()
]

if missing_step14_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 10 first. Missing objects: "
        f"{missing_step14_objects}"
    )


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def validate_probability_dataset(
    true_classes,
    probabilities,
    evaluation_name
):
    true_classes = np.asarray(
        true_classes,
        dtype=int
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    if probabilities.ndim != 2:
        raise ValueError(
            f"{evaluation_name}: probability input must be "
            "a two-dimensional matrix."
        )

    if probabilities.shape[0] != len(
        true_classes
    ):
        raise ValueError(
            f"{evaluation_name}: target and probability row "
            "counts differ."
        )

    if probabilities.shape[1] != len(
        FIXED_CLASS_INDICES
    ):
        raise ValueError(
            f"{evaluation_name}: expected "
            f"{len(FIXED_CLASS_INDICES)} probability columns."
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise ValueError(
            f"{evaluation_name}: non-finite probabilities found."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-8
    ):
        raise ValueError(
            f"{evaluation_name}: probabilities do not sum to one."
        )

    observed_classes = sorted(
        np.unique(true_classes).tolist()
    )

    if observed_classes != FIXED_CLASS_INDICES:
        raise ValueError(
            f"{evaluation_name}: expected classes "
            f"{FIXED_CLASS_INDICES}, found {observed_classes}."
        )

    return true_classes, probabilities


def calculate_discrimination_metrics(
    true_classes,
    probabilities,
    evaluation_name
):
    true_classes, probabilities = (
        validate_probability_dataset(
            true_classes,
            probabilities,
            evaluation_name
        )
    )

    binary_targets = label_binarize(
        true_classes,
        classes=FIXED_CLASS_INDICES
    )

    overall_metrics = {
        "evaluation": evaluation_name,

        "macro_roc_auc_ovr": roc_auc_score(
            binary_targets,
            probabilities,
            average="macro"
        ),

        "weighted_roc_auc_ovr": roc_auc_score(
            binary_targets,
            probabilities,
            average="weighted"
        ),

        "macro_average_precision": (
            average_precision_score(
                binary_targets,
                probabilities,
                average="macro"
            )
        ),

        "weighted_average_precision": (
            average_precision_score(
                binary_targets,
                probabilities,
                average="weighted"
            )
        ),

        "micro_average_precision": (
            average_precision_score(
                binary_targets,
                probabilities,
                average="micro"
            )
        )
    }

    class_records = []

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        binary_class_target = (
            binary_targets[:, class_index]
        )

        class_prevalence = float(
            binary_class_target.mean()
        )

        class_records.append({
            "evaluation": evaluation_name,
            "class_name": class_name,
            "support": int(
                binary_class_target.sum()
            ),
            "prevalence": class_prevalence,
            "no_skill_average_precision": (
                class_prevalence
            ),
            "roc_auc_ovr": roc_auc_score(
                binary_class_target,
                probabilities[:, class_index]
            ),
            "average_precision": (
                average_precision_score(
                    binary_class_target,
                    probabilities[:, class_index]
                )
            )
        })

    return (
        pd.DataFrame([overall_metrics]),
        pd.DataFrame(class_records),
        binary_targets
    )


# ------------------------------------------------------------
# 3. Primary Step 4 nested OOF probabilities
# ------------------------------------------------------------

primary_true = (
    oof_predictions_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

primary_probabilities = (
    oof_predictions_df[
        probability_columns
    ]
    .to_numpy(dtype=float)
)

(
    primary_discrimination_overall_df,
    primary_discrimination_class_df,
    primary_binary_targets
) = calculate_discrimination_metrics(
    true_classes=primary_true,
    probabilities=primary_probabilities,
    evaluation_name=(
        "Primary Step 4 nested out-of-fold"
    )
)


# ------------------------------------------------------------
# 4. Selected fixed-workflow OOF probabilities
# ------------------------------------------------------------

fixed_prediction_df = (
    all_available_fixed_results[
        "predictions"
    ]
    .copy()
    .reset_index(drop=True)
)

fixed_probability_columns = (
    all_available_fixed_results[
        "probability_columns"
    ]
)

fixed_true = (
    fixed_prediction_df[
        "true_class_index"
    ]
    .to_numpy(dtype=int)
)

fixed_probabilities = (
    fixed_prediction_df[
        fixed_probability_columns
    ]
    .to_numpy(dtype=float)
)

(
    fixed_discrimination_overall_df,
    fixed_discrimination_class_df,
    fixed_binary_targets
) = calculate_discrimination_metrics(
    true_classes=fixed_true,
    probabilities=fixed_probabilities,
    evaluation_name=(
        f"Locked fixed {FINAL_WORKFLOW_MODEL_FAMILY} out-of-fold"
    )
)


# ------------------------------------------------------------
# 5. Combined point-estimate comparison
# ------------------------------------------------------------

discrimination_overall_comparison_df = (
    pd.concat(
        [
            primary_discrimination_overall_df,
            fixed_discrimination_overall_df
        ],
        ignore_index=True
    )
)

discrimination_class_comparison_df = (
    pd.concat(
        [
            primary_discrimination_class_df,
            fixed_discrimination_class_df
        ],
        ignore_index=True
    )
)


# ------------------------------------------------------------
# 6. Stratified bootstrap for the primary Step 4 results
# ------------------------------------------------------------

DISCRIMINATION_BOOTSTRAP_ITERATIONS = 2000
DISCRIMINATION_BOOTSTRAP_SEED = (
    RANDOM_STATE + 1400
)

bootstrap_generator = np.random.default_rng(
    DISCRIMINATION_BOOTSTRAP_SEED
)

class_row_indices = {
    class_index: np.where(
        primary_true == class_index
    )[0]
    for class_index in FIXED_CLASS_INDICES
}

bootstrap_records = []

for bootstrap_iteration in range(
    1,
    DISCRIMINATION_BOOTSTRAP_ITERATIONS + 1
):
    sampled_indices = np.concatenate([
        bootstrap_generator.choice(
            class_row_indices[class_index],
            size=len(
                class_row_indices[class_index]
            ),
            replace=True
        )
        for class_index in FIXED_CLASS_INDICES
    ])

    bootstrap_generator.shuffle(
        sampled_indices
    )

    sampled_true = primary_true[
        sampled_indices
    ]

    sampled_probabilities = primary_probabilities[
        sampled_indices
    ]

    sampled_binary_targets = label_binarize(
        sampled_true,
        classes=FIXED_CLASS_INDICES
    )

    record = {
        "bootstrap_iteration": (
            bootstrap_iteration
        ),

        "macro_roc_auc_ovr": roc_auc_score(
            sampled_binary_targets,
            sampled_probabilities,
            average="macro"
        ),

        "weighted_roc_auc_ovr": roc_auc_score(
            sampled_binary_targets,
            sampled_probabilities,
            average="weighted"
        ),

        "macro_average_precision": (
            average_precision_score(
                sampled_binary_targets,
                sampled_probabilities,
                average="macro"
            )
        ),

        "weighted_average_precision": (
            average_precision_score(
                sampled_binary_targets,
                sampled_probabilities,
                average="weighted"
            )
        ),

        "micro_average_precision": (
            average_precision_score(
                sampled_binary_targets,
                sampled_probabilities,
                average="micro"
            )
        )
    }

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        safe_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        binary_target = (
            sampled_binary_targets[
                :,
                class_index
            ]
        )

        record[
            f"{safe_name}_roc_auc"
        ] = roc_auc_score(
            binary_target,
            sampled_probabilities[
                :,
                class_index
            ]
        )

        record[
            f"{safe_name}_average_precision"
        ] = average_precision_score(
            binary_target,
            sampled_probabilities[
                :,
                class_index
            ]
        )

    bootstrap_records.append(record)


discrimination_bootstrap_df = pd.DataFrame(
    bootstrap_records
)


# ------------------------------------------------------------
# 7. Overall confidence intervals
# ------------------------------------------------------------

primary_overall_point = (
    primary_discrimination_overall_df
    .iloc[0]
)

overall_metric_names = [
    "macro_roc_auc_ovr",
    "weighted_roc_auc_ovr",
    "macro_average_precision",
    "weighted_average_precision",
    "micro_average_precision"
]

discrimination_overall_ci_records = []

for metric_name in overall_metric_names:
    discrimination_overall_ci_records.append({
        "metric": metric_name,
        "point_estimate": float(
            primary_overall_point[
                metric_name
            ]
        ),
        "bootstrap_mean": float(
            discrimination_bootstrap_df[
                metric_name
            ].mean()
        ),
        "ci_2.5_percent": float(
            discrimination_bootstrap_df[
                metric_name
            ].quantile(0.025)
        ),
        "ci_97.5_percent": float(
            discrimination_bootstrap_df[
                metric_name
            ].quantile(0.975)
        )
    })

discrimination_overall_ci_df = pd.DataFrame(
    discrimination_overall_ci_records
)


# ------------------------------------------------------------
# 8. Class-specific confidence intervals
# ------------------------------------------------------------

discrimination_class_ci_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    point_row = (
        primary_discrimination_class_df.loc[
            primary_discrimination_class_df[
                "class_name"
            ] == class_name
        ]
        .iloc[0]
    )

    for metric_name, bootstrap_column in [
        (
            "roc_auc_ovr",
            f"{safe_name}_roc_auc"
        ),
        (
            "average_precision",
            f"{safe_name}_average_precision"
        )
    ]:
        discrimination_class_ci_records.append({
            "class_name": class_name,
            "metric": metric_name,
            "support": int(
                point_row["support"]
            ),
            "prevalence": float(
                point_row["prevalence"]
            ),
            "no_skill_average_precision": float(
                point_row[
                    "no_skill_average_precision"
                ]
            ),
            "point_estimate": float(
                point_row[metric_name]
            ),
            "bootstrap_mean": float(
                discrimination_bootstrap_df[
                    bootstrap_column
                ].mean()
            ),
            "ci_2.5_percent": float(
                discrimination_bootstrap_df[
                    bootstrap_column
                ].quantile(0.025)
            ),
            "ci_97.5_percent": float(
                discrimination_bootstrap_df[
                    bootstrap_column
                ].quantile(0.975)
            )
        })

discrimination_class_ci_df = pd.DataFrame(
    discrimination_class_ci_records
)


# ------------------------------------------------------------
# 9. Create ROC and precision–recall figures
# ------------------------------------------------------------

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ROC curves

plt.figure(figsize=(8, 6))

for class_index, class_name in enumerate(
    ORDERED_CLUSTER_NAMES
):
    false_positive_rate, true_positive_rate, _ = (
        roc_curve(
            primary_binary_targets[
                :,
                class_index
            ],
            primary_probabilities[
                :,
                class_index
            ]
        )
    )

    class_auc = float(
        primary_discrimination_class_df.loc[
            primary_discrimination_class_df[
                "class_name"
            ] == class_name,
            "roc_auc_ovr"
        ].iloc[0]
    )

    plt.plot(
        false_positive_rate,
        true_positive_rate,
        label=(
            f"{class_name} "
            f"(AUC={class_auc:.3f})"
        )
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="No-skill reference"
)

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate")
plt.title(
    "One-vs-Rest ROC Curves for the "
    "Billing-Based Archetype-Transfer Model"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "primary_nested_oof_multiclass_roc_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Precision–recall curves

plt.figure(figsize=(8, 6))

for class_index, class_name in enumerate(
    ORDERED_CLUSTER_NAMES
):
    precision_values, recall_values, _ = (
        precision_recall_curve(
            primary_binary_targets[
                :,
                class_index
            ],
            primary_probabilities[
                :,
                class_index
            ]
        )
    )

    class_ap = float(
        primary_discrimination_class_df.loc[
            primary_discrimination_class_df[
                "class_name"
            ] == class_name,
            "average_precision"
        ].iloc[0]
    )

    plt.plot(
        recall_values,
        precision_values,
        label=(
            f"{class_name} "
            f"(AP={class_ap:.3f})"
        )
    )

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "One-vs-Rest Precision–Recall Curves for the "
    "Billing-Based Archetype-Transfer Model"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "primary_nested_oof_multiclass_precision_recall_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 14 — MULTICLASS DISCRIMINATION RESULTS")
print("=" * 80)

print("\nOverall discrimination comparison:")

display(
    discrimination_overall_comparison_df.round(4)
)

print("\nPrimary class-specific discrimination:")

display(
    primary_discrimination_class_df.round(4)
)

print("\nOverall primary confidence intervals:")

display(
    discrimination_overall_ci_df.round(4)
)

print("\nClass-specific confidence intervals:")

display(
    discrimination_class_ci_df.round(4)
)

print(
    "\nBootstrap iterations:",
    DISCRIMINATION_BOOTSTRAP_ITERATIONS
)

print(
    "Primary evaluation source:",
    "Step 4 nested out-of-fold probabilities"
)


# ------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------

discrimination_overall_comparison_df.to_csv(
    OUTPUT_DIR
    / "multiclass_discrimination_overall_comparison.csv",
    index=False
)

discrimination_class_comparison_df.to_csv(
    OUTPUT_DIR
    / "multiclass_discrimination_class_comparison.csv",
    index=False
)

primary_discrimination_class_df.to_csv(
    OUTPUT_DIR
    / "primary_multiclass_class_discrimination.csv",
    index=False
)

discrimination_overall_ci_df.to_csv(
    OUTPUT_DIR
    / "primary_multiclass_discrimination_overall_ci.csv",
    index=False
)

discrimination_class_ci_df.to_csv(
    OUTPUT_DIR
    / "primary_multiclass_discrimination_class_ci.csv",
    index=False
)

discrimination_bootstrap_df.to_csv(
    OUTPUT_DIR
    / "primary_multiclass_discrimination_bootstrap.csv",
    index=False
)

discrimination_configuration = {
    "primary_evaluation_source": (
        "Step 4 nested out-of-fold probabilities"
    ),
    "secondary_evaluation_source": (
        f"Locked fixed {FINAL_WORKFLOW_MODEL_FAMILY} out-of-fold probabilities"
    ),
    "multiclass_strategy": "One-vs-rest",
    "precision_recall_summary": (
        "Average precision"
    ),
    "bootstrap_iterations": (
        DISCRIMINATION_BOOTSTRAP_ITERATIONS
    ),
    "bootstrap_unit": "household",
    "bootstrap_stratified_by_true_class": True,
    "ordinary_accuracy_primary": False,
    "average_precision_interpretation": (
        "Must be interpreted relative to each class prevalence."
    )
}

with open(
    OUTPUT_DIR
    / "multiclass_discrimination_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        discrimination_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 14 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Primary macro ROC-AUC:",
    round(
        float(
            primary_overall_point[
                "macro_roc_auc_ovr"
            ]
        ),
        4
    )
)

print(
    "Primary macro average precision:",
    round(
        float(
            primary_overall_point[
                "macro_average_precision"
            ]
        ),
        4
    )
)

print(
    "\nSaved Step 14 outputs to:",
    OUTPUT_DIR
) 

In [ ]:
# ============================================================
# STEP 15 — CONDITIONAL LABEL-PERMUTATION CHECK
# ============================================================
# Purpose:
# 1. Check whether the already selected fixed workflow performs
#    better than the same workflow trained on permuted labels.
# 2. Use the complete leakage-safe pipeline in every CV fold.
# 3. Evaluate macro-F1 and balanced accuracy under the null.
#
# Important:
# - This tests predictive information beyond random labels.
# - It does not establish causality or external validity.
# - The fixed model configuration is not changed.
# - Step 4 remains the primary performance estimate.
# - This check is conditional on the already selected workflow.
# - It does not repeat the complete model-selection process under
#   every permutation and is therefore exploratory, not a formal
#   significance test of the full development pipeline.
# ============================================================

import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    f1_score,
    make_scorer
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP15_OBJECTS = [
    "dataset_all",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "CLASS_TO_INDEX",
    "FIXED_CLASS_INDICES",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "pooled_performance_df"
]

missing_step15_objects = [
    object_name
    for object_name in REQUIRED_STEP15_OBJECTS
    if object_name not in globals()
]

if missing_step15_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 9A first. Missing objects: "
        f"{missing_step15_objects}"
    )


# ------------------------------------------------------------
# 2. Prepare the locked modelling data
# ------------------------------------------------------------

permutation_test_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

if permutation_test_df[
    "household_ID"
].duplicated().any():
    raise ValueError(
        "Duplicate households were detected before "
        "the permutation test."
    )

X_permutation_test = (
    permutation_test_df[
        FINAL_WORKFLOW_FEATURES
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

y_permutation_test = (
    permutation_test_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
    .astype(int)
)

if X_permutation_test.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all final predictors missing."
    )

if (
    sorted(
        y_permutation_test.unique().tolist()
    )
    != FIXED_CLASS_INDICES
):
    raise ValueError(
        "The target does not contain the expected four classes."
    )


# ------------------------------------------------------------
# 3. Permutation-test configuration
# ------------------------------------------------------------

PERMUTATION_TEST_ITERATIONS = 200
PERMUTATION_TEST_FOLDS = 5
PERMUTATION_TEST_SEED = (
    RANDOM_STATE + 1500
)

permutation_cv = StratifiedKFold(
    n_splits=PERMUTATION_TEST_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

permutation_scoring = {
    "macro_f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    ),
    "balanced_accuracy": (
        "balanced_accuracy"
    ),
    "accuracy": "accuracy"
}


# ------------------------------------------------------------
# 4. Calculate the observed fixed-workflow CV scores
# ------------------------------------------------------------

observed_cv_results = cross_validate(
    estimator=clone(
        final_workflow_pipeline_template
    ),
    X=X_permutation_test,
    y=y_permutation_test,
    cv=permutation_cv,
    scoring=permutation_scoring,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)

observed_macro_f1 = float(
    observed_cv_results[
        "test_macro_f1"
    ].mean()
)

observed_balanced_accuracy = float(
    observed_cv_results[
        "test_balanced_accuracy"
    ].mean()
)

observed_accuracy = float(
    observed_cv_results[
        "test_accuracy"
    ].mean()
)

observed_fold_scores_df = pd.DataFrame({
    "fold": np.arange(
        1,
        PERMUTATION_TEST_FOLDS + 1
    ),
    "macro_f1": (
        observed_cv_results[
            "test_macro_f1"
        ]
    ),
    "balanced_accuracy": (
        observed_cv_results[
            "test_balanced_accuracy"
        ]
    ),
    "accuracy": (
        observed_cv_results[
            "test_accuracy"
        ]
    )
})


# ------------------------------------------------------------
# 5. Generate the random-label null distribution
# ------------------------------------------------------------

random_generator = np.random.default_rng(
    PERMUTATION_TEST_SEED
)

true_target_array = (
    y_permutation_test
    .to_numpy(dtype=int)
)

permutation_records = []

permutation_start_time = time.time()

for permutation_number in range(
    1,
    PERMUTATION_TEST_ITERATIONS + 1
):
    permuted_target = (
        random_generator.permutation(
            true_target_array
        )
    )

    null_cv_results = cross_validate(
        estimator=clone(
            final_workflow_pipeline_template
        ),
        X=X_permutation_test,
        y=permuted_target,
        cv=permutation_cv,
        scoring=permutation_scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    permutation_records.append({
        "permutation": permutation_number,
        "mean_macro_f1": float(
            null_cv_results[
                "test_macro_f1"
            ].mean()
        ),
        "mean_balanced_accuracy": float(
            null_cv_results[
                "test_balanced_accuracy"
            ].mean()
        ),
        "mean_accuracy": float(
            null_cv_results[
                "test_accuracy"
            ].mean()
        )
    })

    if (
        permutation_number % 25 == 0
        or permutation_number
        == PERMUTATION_TEST_ITERATIONS
    ):
        print(
            f"Completed {permutation_number}/"
            f"{PERMUTATION_TEST_ITERATIONS} permutations."
        )


permutation_null_df = pd.DataFrame(
    permutation_records
)

permutation_runtime_seconds = (
    time.time()
    - permutation_start_time
)


# ------------------------------------------------------------
# 6. Calculate Monte Carlo p-values
# ------------------------------------------------------------

macro_f1_exceedances = int(
    (
        permutation_null_df[
            "mean_macro_f1"
        ] >= observed_macro_f1
    ).sum()
)

balanced_accuracy_exceedances = int(
    (
        permutation_null_df[
            "mean_balanced_accuracy"
        ] >= observed_balanced_accuracy
    ).sum()
)

accuracy_exceedances = int(
    (
        permutation_null_df[
            "mean_accuracy"
        ] >= observed_accuracy
    ).sum()
)

macro_f1_p_value = (
    macro_f1_exceedances + 1
) / (
    PERMUTATION_TEST_ITERATIONS + 1
)

balanced_accuracy_p_value = (
    balanced_accuracy_exceedances + 1
) / (
    PERMUTATION_TEST_ITERATIONS + 1
)

accuracy_p_value = (
    accuracy_exceedances + 1
) / (
    PERMUTATION_TEST_ITERATIONS + 1
)


# ------------------------------------------------------------
# 7. Compile the conditional permutation summary
# ------------------------------------------------------------

permutation_test_summary_df = pd.DataFrame([
    {
        "metric": "macro_f1",
        "observed_score": (
            observed_macro_f1
        ),
        "null_mean": (
            permutation_null_df[
                "mean_macro_f1"
            ].mean()
        ),
        "null_standard_deviation": (
            permutation_null_df[
                "mean_macro_f1"
            ].std(ddof=1)
        ),
        "null_2.5_percent": (
            permutation_null_df[
                "mean_macro_f1"
            ].quantile(0.025)
        ),
        "null_97.5_percent": (
            permutation_null_df[
                "mean_macro_f1"
            ].quantile(0.975)
        ),
        "null_scores_at_or_above_observed": (
            macro_f1_exceedances
        ),
        "monte_carlo_p_value": (
            macro_f1_p_value
        )
    },
    {
        "metric": "balanced_accuracy",
        "observed_score": (
            observed_balanced_accuracy
        ),
        "null_mean": (
            permutation_null_df[
                "mean_balanced_accuracy"
            ].mean()
        ),
        "null_standard_deviation": (
            permutation_null_df[
                "mean_balanced_accuracy"
            ].std(ddof=1)
        ),
        "null_2.5_percent": (
            permutation_null_df[
                "mean_balanced_accuracy"
            ].quantile(0.025)
        ),
        "null_97.5_percent": (
            permutation_null_df[
                "mean_balanced_accuracy"
            ].quantile(0.975)
        ),
        "null_scores_at_or_above_observed": (
            balanced_accuracy_exceedances
        ),
        "monte_carlo_p_value": (
            balanced_accuracy_p_value
        )
    },
    {
        "metric": "accuracy",
        "observed_score": observed_accuracy,
        "null_mean": (
            permutation_null_df[
                "mean_accuracy"
            ].mean()
        ),
        "null_standard_deviation": (
            permutation_null_df[
                "mean_accuracy"
            ].std(ddof=1)
        ),
        "null_2.5_percent": (
            permutation_null_df[
                "mean_accuracy"
            ].quantile(0.025)
        ),
        "null_97.5_percent": (
            permutation_null_df[
                "mean_accuracy"
            ].quantile(0.975)
        ),
        "null_scores_at_or_above_observed": (
            accuracy_exceedances
        ),
        "monte_carlo_p_value": (
            accuracy_p_value
        )
    }
])


# ------------------------------------------------------------
# 8. Create null-distribution figures
# ------------------------------------------------------------

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


plt.figure(figsize=(8, 5))

plt.hist(
    permutation_null_df[
        "mean_macro_f1"
    ],
    bins=25
)

plt.axvline(
    observed_macro_f1,
    linestyle="--",
    linewidth=2,
    label=(
        f"Observed macro-F1 = "
        f"{observed_macro_f1:.3f}"
    )
)

plt.xlabel(
    "Cross-validated macro-F1 under permuted labels"
)

plt.ylabel("Frequency")

plt.title(
    "Label-Permutation Null Distribution "
    "for Macro-F1"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "final_model_macro_f1_label_permutation_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


plt.figure(figsize=(8, 5))

plt.hist(
    permutation_null_df[
        "mean_balanced_accuracy"
    ],
    bins=25
)

plt.axvline(
    observed_balanced_accuracy,
    linestyle="--",
    linewidth=2,
    label=(
        "Observed balanced accuracy = "
        f"{observed_balanced_accuracy:.3f}"
    )
)

plt.xlabel(
    "Cross-validated balanced accuracy "
    "under permuted labels"
)

plt.ylabel("Frequency")

plt.title(
    "Label-Permutation Null Distribution "
    "for Balanced Accuracy"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "final_model_balanced_accuracy_label_permutation_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 15 — CONDITIONAL LABEL-PERMUTATION RESULTS")
print("=" * 80)

print("\nObserved fold scores:")

display(
    observed_fold_scores_df.round(4)
)

print("\nPermutation-test summary:")

display(
    permutation_test_summary_df.round(6)
)

print(
    "\nPermutations:",
    PERMUTATION_TEST_ITERATIONS
)

print(
    "Interpretation: exploratory and conditional on the "
    "already selected fixed workflow; the complete selection "
    "procedure was not repeated for each permuted target."
)

print(
    "Minimum attainable Monte Carlo p-value:",
    round(
        1 / (
            PERMUTATION_TEST_ITERATIONS + 1
        ),
        6
    )
)

print(
    "Runtime in minutes:",
    round(
        permutation_runtime_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

observed_fold_scores_df.to_csv(
    OUTPUT_DIR
    / "conditional_label_permutation_observed_fold_scores.csv",
    index=False
)

permutation_null_df.to_csv(
    OUTPUT_DIR
    / "conditional_label_permutation_null_distribution.csv",
    index=False
)

permutation_test_summary_df.to_csv(
    OUTPUT_DIR
    / "conditional_label_permutation_check_summary.csv",
    index=False
)

permutation_test_configuration = {
    "model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "model_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "feature_count": len(
        FINAL_WORKFLOW_FEATURES
    ),
    "cross_validation_folds": (
        PERMUTATION_TEST_FOLDS
    ),
    "permutations": (
        PERMUTATION_TEST_ITERATIONS
    ),
    "random_seed": (
        PERMUTATION_TEST_SEED
    ),
    "primary_test_metric": "macro_f1",
    "secondary_test_metric": (
        "balanced_accuracy"
    ),
    "p_value_method": (
        "(null scores >= observed + 1) / "
        "(permutations + 1)"
    ),
    "full_pipeline_refitted_for_each_fold": True,
    "labels_permuted_before_cross_validation": True,
    "conditional_on_selected_workflow": True,
    "full_model_selection_repeated_per_permutation": False,
    "formal_significance_test_of_entire_pipeline": False,
    "minimum_attainable_p_value": (
        1.0 / (PERMUTATION_TEST_ITERATIONS + 1)
    ),
    "causal_interpretation_allowed": False,
    "external_validity_established": False,
    "analysis_role": (
        "Exploratory conditional label-permutation check"
    )
}

with open(
    OUTPUT_DIR
    / "conditional_label_permutation_check_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        permutation_test_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 15 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Observed macro-F1:",
    round(
        observed_macro_f1,
        4
    )
)

print(
    "Macro-F1 permutation p-value:",
    round(
        macro_f1_p_value,
        6
    )
)

print(
    "Observed balanced accuracy:",
    round(
        observed_balanced_accuracy,
        4
    )
)

print(
    "Balanced-accuracy permutation p-value:",
    round(
        balanced_accuracy_p_value,
        6
    )
)

print(
    "\nSaved Step 15 outputs to:",
    OUTPUT_DIR
) 

In [ ]:
# ============================================================
# STEP 16 — CORRELATED-FEATURE REDUCTION SENSITIVITY
# ============================================================
# Purpose:
# 1. Identify connected groups of highly correlated predictors.
# 2. Retain one interpretable representative from each group.
# 3. Compare the full-feature fixed workflow with the
#    correlation-reduced version using identical CV folds.
# 4. Calculate paired household-level bootstrap intervals.
#
# Important:
# - Feature grouping uses predictors only, without the target.
# - This is a sensitivity analysis, not a replacement for the
#   primary Step 4 nested evaluation.
# - Correlation reduction does not imply that removed features
#   are scientifically irrelevant.
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.model_selection import StratifiedKFold


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

REQUIRED_STEP16_OBJECTS = [
    "dataset_all",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "CLASS_TO_INDEX",
    "ORDERED_CLUSTER_NAMES",
    "FIXED_CLASS_INDICES",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "align_class_probabilities",
    "calculate_multiclass_metrics"
]

missing_step16_objects = [
    object_name
    for object_name in REQUIRED_STEP16_OBJECTS
    if object_name not in globals()
]

if missing_step16_objects:
    raise RuntimeError(
        "Run Steps 1–4 and Step 9A first. Missing objects: "
        f"{missing_step16_objects}"
    )


# ------------------------------------------------------------
# 2. Prepare modelling data
# ------------------------------------------------------------

redundancy_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

if redundancy_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households were detected."
    )

X_full_features = (
    redundancy_df[FINAL_WORKFLOW_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

y_redundancy = (
    redundancy_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
    .astype(int)
)

if X_full_features.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all predictors missing."
    )


# ------------------------------------------------------------
# 3. Correlation-reduction configuration
# ------------------------------------------------------------

CORRELATION_REDUCTION_THRESHOLD = 0.95

# Representatives are preferred according to robustness,
# interpretability and temporal relevance. The target and
# model performance are not used in this ordering.

FEATURE_REPRESENTATIVE_PRIORITY = [
    "billing_median_kwh",
    "billing_iqr_kwh",
    "recent_6_month_median_kwh",
    "billing_cv",
    "billing_p10_kwh",
    "billing_p90_kwh",
    "billing_robust_range_kwh",
    "billing_min_kwh",
    "maximum_monthly_increase_kwh",
    "recent_vs_overall_log_ratio",
    "early_6_month_median_kwh",
    "late_vs_early_log_ratio",
    "monthly_trend_slope_log1p",
    "median_abs_log_change",
    "near_zero_month_rate",
    "above_120_month_rate",
    "billing_mean_kwh",
    "billing_std_kwh",
    "recent_3_month_median_kwh",
    "billing_max_kwh",
    "billing_range_kwh",
    "billing_log_std",
    "mean_abs_monthly_change_kwh",
    "maximum_monthly_decrease_kwh",
    "low_30_month_rate",
    "above_90_month_rate",
    "above_180_month_rate",
    "above_240_month_rate",
    "median_kwh_calendar_q1",
    "median_kwh_calendar_q2",
    "median_kwh_calendar_q3",
    "median_kwh_calendar_q4"
]

priority_rank = {
    feature: rank
    for rank, feature in enumerate(
        FEATURE_REPRESENTATIVE_PRIORITY
    )
}

original_feature_rank = {
    feature: rank
    for rank, feature in enumerate(
        FINAL_WORKFLOW_FEATURES
    )
}


# ------------------------------------------------------------
# 4. Build a high-correlation feature graph
# ------------------------------------------------------------

spearman_correlation = (
    X_full_features
    .corr(method="spearman")
)

feature_adjacency = {
    feature: set()
    for feature in FINAL_WORKFLOW_FEATURES
}

high_correlation_records = []

for first_index, first_feature in enumerate(
    FINAL_WORKFLOW_FEATURES
):
    for second_index in range(
        first_index + 1,
        len(FINAL_WORKFLOW_FEATURES)
    ):
        second_feature = (
            FINAL_WORKFLOW_FEATURES[
                second_index
            ]
        )

        correlation_value = (
            spearman_correlation.loc[
                first_feature,
                second_feature
            ]
        )

        if (
            pd.notna(correlation_value)
            and abs(correlation_value)
            >= CORRELATION_REDUCTION_THRESHOLD
        ):
            feature_adjacency[
                first_feature
            ].add(second_feature)

            feature_adjacency[
                second_feature
            ].add(first_feature)

            high_correlation_records.append({
                "feature_1": first_feature,
                "feature_2": second_feature,
                "spearman_correlation": float(
                    correlation_value
                ),
                "absolute_correlation": float(
                    abs(correlation_value)
                )
            })


# ------------------------------------------------------------
# 5. Identify connected correlation groups
# ------------------------------------------------------------

visited_features = set()
correlation_components = []

for starting_feature in FINAL_WORKFLOW_FEATURES:
    if starting_feature in visited_features:
        continue

    stack = [starting_feature]
    component = []

    while stack:
        current_feature = stack.pop()

        if current_feature in visited_features:
            continue

        visited_features.add(
            current_feature
        )

        component.append(
            current_feature
        )

        stack.extend(
            sorted(
                feature_adjacency[
                    current_feature
                ]
                - visited_features
            )
        )

    correlation_components.append(
        sorted(
            component,
            key=lambda feature:
                original_feature_rank[feature]
        )
    )


def representative_sort_key(feature):
    return (
        priority_rank.get(
            feature,
            len(
                FEATURE_REPRESENTATIVE_PRIORITY
            ) + original_feature_rank[feature]
        ),
        original_feature_rank[feature]
    )


component_records = []
features_to_remove = set()

for component_number, component in enumerate(
    correlation_components,
    start=1
):
    representative = min(
        component,
        key=representative_sort_key
    )

    for feature in component:
        retained = (
            feature == representative
        )

        component_records.append({
            "component": component_number,
            "component_size": len(component),
            "feature": feature,
            "representative": representative,
            "retained": retained,
            "reason": (
                "Highest-priority representative"
                if retained
                else (
                    "Removed only for correlation-"
                    "reduction sensitivity"
                )
            )
        })

        if not retained:
            features_to_remove.add(feature)


correlation_component_df = pd.DataFrame(
    component_records
)

high_correlation_pairs_step16_df = pd.DataFrame(
    high_correlation_records
)

REDUCED_CORRELATION_FEATURES = [
    feature
    for feature in FINAL_WORKFLOW_FEATURES
    if feature not in features_to_remove
]

if len(REDUCED_CORRELATION_FEATURES) >= len(
    FINAL_WORKFLOW_FEATURES
):
    raise RuntimeError(
        "Correlation reduction did not remove any features."
    )

if len(REDUCED_CORRELATION_FEATURES) == 0:
    raise RuntimeError(
        "Correlation reduction removed every feature."
    )


# ------------------------------------------------------------
# 6. Fixed-workflow OOF evaluation helper
# ------------------------------------------------------------

def evaluate_redundancy_workflow(
    feature_columns,
    evaluation_name
):
    feature_matrix = (
        redundancy_df[feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    number_of_rows = len(feature_matrix)
    number_of_classes = len(
        FIXED_CLASS_INDICES
    )

    oof_predictions = np.full(
        number_of_rows,
        -1,
        dtype=int
    )

    oof_probabilities = np.full(
        (
            number_of_rows,
            number_of_classes
        ),
        np.nan,
        dtype=float
    )

    fold_assignments = np.full(
        number_of_rows,
        -1,
        dtype=int
    )

    fold_records = []

    for fold_number, (
        train_indices,
        test_indices
    ) in enumerate(
        cv.split(
            feature_matrix,
            y_redundancy
        ),
        start=1
    ):
        estimator = clone(
            final_workflow_pipeline_template
        )

        estimator.fit(
            feature_matrix.iloc[
                train_indices
            ],
            y_redundancy.iloc[
                train_indices
            ]
        )

        predictions = estimator.predict(
            feature_matrix.iloc[
                test_indices
            ]
        )

        raw_probabilities = (
            estimator.predict_proba(
                feature_matrix.iloc[
                    test_indices
                ]
            )
        )

        probabilities = (
            align_class_probabilities(
                raw_probabilities,
                estimator.classes_
            )
        )

        oof_predictions[
            test_indices
        ] = predictions

        oof_probabilities[
            test_indices,
            :
        ] = probabilities

        fold_assignments[
            test_indices
        ] = fold_number

        fold_metrics = (
            calculate_multiclass_metrics(
                y_redundancy.iloc[
                    test_indices
                ],
                predictions,
                probabilities
            )
        )

        fold_records.append({
            "evaluation": evaluation_name,
            "fold": fold_number,
            "feature_count": len(
                feature_columns
            ),
            **fold_metrics
        })

    if (oof_predictions < 0).any():
        raise RuntimeError(
            f"{evaluation_name}: missing predictions."
        )

    if np.isnan(
        oof_probabilities
    ).any():
        raise RuntimeError(
            f"{evaluation_name}: missing probabilities."
        )

    pooled_metrics = (
        calculate_multiclass_metrics(
            y_redundancy,
            oof_predictions,
            oof_probabilities
        )
    )

    classification_results = (
        classification_report(
            y_redundancy,
            oof_predictions,
            labels=FIXED_CLASS_INDICES,
            target_names=(
                ORDERED_CLUSTER_NAMES
            ),
            output_dict=True,
            zero_division=0
        )
    )

    class_records = []

    for class_name in ORDERED_CLUSTER_NAMES:
        class_records.append({
            "evaluation": evaluation_name,
            "class_name": class_name,
            "precision": (
                classification_results[
                    class_name
                ]["precision"]
            ),
            "recall": (
                classification_results[
                    class_name
                ]["recall"]
            ),
            "f1_score": (
                classification_results[
                    class_name
                ]["f1-score"]
            ),
            "support": int(
                classification_results[
                    class_name
                ]["support"]
            )
        })

    confusion_df = pd.DataFrame(
        confusion_matrix(
            y_redundancy,
            oof_predictions,
            labels=FIXED_CLASS_INDICES
        ),
        index=ORDERED_CLUSTER_NAMES,
        columns=ORDERED_CLUSTER_NAMES
    )

    probability_column_names = [
        (
            "probability_"
            + class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )
        for class_name in ORDERED_CLUSTER_NAMES
    ]

    prediction_df = pd.DataFrame({
        "household_ID": (
            redundancy_df[
                "household_ID"
            ]
        ),
        "evaluation": evaluation_name,
        "fold": fold_assignments,
        "true_class_index": (
            y_redundancy.to_numpy(
                dtype=int
            )
        ),
        "predicted_class_index": (
            oof_predictions
        )
    })

    for column_index, probability_column in enumerate(
        probability_column_names
    ):
        prediction_df[
            probability_column
        ] = oof_probabilities[
            :,
            column_index
        ]

    return {
        "feature_columns": list(
            feature_columns
        ),
        "predictions": prediction_df,
        "probability_columns": (
            probability_column_names
        ),
        "pooled_metrics": pooled_metrics,
        "fold_metrics": pd.DataFrame(
            fold_records
        ),
        "class_metrics": pd.DataFrame(
            class_records
        ),
        "confusion_matrix": confusion_df
    }


# ------------------------------------------------------------
# 7. Evaluate full and reduced feature sets
# ------------------------------------------------------------

full_feature_results = (
    evaluate_redundancy_workflow(
        FINAL_WORKFLOW_FEATURES,
        f"Fixed {FINAL_WORKFLOW_MODEL_FAMILY} — full feature set"
    )
)

reduced_feature_results = (
    evaluate_redundancy_workflow(
        REDUCED_CORRELATION_FEATURES,
        (
            f"Fixed {FINAL_WORKFLOW_MODEL_FAMILY} — "
            "correlation-reduced features"
        )
    )
)


# ------------------------------------------------------------
# 8. Overall and class-specific comparisons
# ------------------------------------------------------------

redundancy_overall_comparison_df = pd.DataFrame([
    {
        "evaluation": (
            f"Fixed {FINAL_WORKFLOW_MODEL_FAMILY} — full feature set"
        ),
        "feature_count": len(
            FINAL_WORKFLOW_FEATURES
        ),
        **full_feature_results[
            "pooled_metrics"
        ]
    },
    {
        "evaluation": (
            f"Fixed {FINAL_WORKFLOW_MODEL_FAMILY} — "
            "correlation-reduced features"
        ),
        "feature_count": len(
            REDUCED_CORRELATION_FEATURES
        ),
        **reduced_feature_results[
            "pooled_metrics"
        ]
    }
])

redundancy_class_comparison_df = (
    full_feature_results[
        "class_metrics"
    ]
    .drop(columns=["evaluation"])
    .rename(
        columns={
            "precision": "full_precision",
            "recall": "full_recall",
            "f1_score": "full_f1"
        }
    )
    .merge(
        reduced_feature_results[
            "class_metrics"
        ]
        .drop(
            columns=[
                "evaluation",
                "support"
            ]
        )
        .rename(
            columns={
                "precision":
                    "reduced_precision",
                "recall":
                    "reduced_recall",
                "f1_score":
                    "reduced_f1"
            }
        ),
        on="class_name",
        how="inner",
        validate="one_to_one"
    )
)

redundancy_class_comparison_df[
    "f1_difference_reduced_minus_full"
] = (
    redundancy_class_comparison_df[
        "reduced_f1"
    ]
    - redundancy_class_comparison_df[
        "full_f1"
    ]
)


# ------------------------------------------------------------
# 9. Paired stratified bootstrap
# ------------------------------------------------------------

REDUNDANCY_BOOTSTRAP_ITERATIONS = 2000
REDUNDANCY_BOOTSTRAP_SEED = (
    RANDOM_STATE + 1600
)

bootstrap_generator = np.random.default_rng(
    REDUNDANCY_BOOTSTRAP_SEED
)

true_classes = (
    full_feature_results[
        "predictions"
    ]["true_class_index"]
    .to_numpy(dtype=int)
)

full_predictions = (
    full_feature_results[
        "predictions"
    ]["predicted_class_index"]
    .to_numpy(dtype=int)
)

reduced_predictions = (
    reduced_feature_results[
        "predictions"
    ]["predicted_class_index"]
    .to_numpy(dtype=int)
)

full_probabilities = (
    full_feature_results[
        "predictions"
    ][
        full_feature_results[
            "probability_columns"
        ]
    ]
    .to_numpy(dtype=float)
)

reduced_probabilities = (
    reduced_feature_results[
        "predictions"
    ][
        reduced_feature_results[
            "probability_columns"
        ]
    ]
    .to_numpy(dtype=float)
)

bootstrap_class_indices = {
    class_index: np.where(
        true_classes == class_index
    )[0]
    for class_index in FIXED_CLASS_INDICES
}

comparison_metric_names = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "top_2_accuracy",
    "multiclass_log_loss",
    "multiclass_brier_sum"
]

redundancy_bootstrap_records = []

for bootstrap_iteration in range(
    1,
    REDUNDANCY_BOOTSTRAP_ITERATIONS + 1
):
    sampled_indices = np.concatenate([
        bootstrap_generator.choice(
            bootstrap_class_indices[
                class_index
            ],
            size=len(
                bootstrap_class_indices[
                    class_index
                ]
            ),
            replace=True
        )
        for class_index in FIXED_CLASS_INDICES
    ])

    bootstrap_generator.shuffle(
        sampled_indices
    )

    full_metrics = (
        calculate_multiclass_metrics(
            true_classes[
                sampled_indices
            ],
            full_predictions[
                sampled_indices
            ],
            full_probabilities[
                sampled_indices
            ]
        )
    )

    reduced_metrics = (
        calculate_multiclass_metrics(
            true_classes[
                sampled_indices
            ],
            reduced_predictions[
                sampled_indices
            ],
            reduced_probabilities[
                sampled_indices
            ]
        )
    )

    record = {
        "bootstrap_iteration":
            bootstrap_iteration
    }

    for metric_name in comparison_metric_names:
        record[
            f"{metric_name}_"
            "difference_reduced_minus_full"
        ] = (
            reduced_metrics[metric_name]
            - full_metrics[metric_name]
        )

    full_class_f1 = f1_score(
        true_classes[sampled_indices],
        full_predictions[sampled_indices],
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    reduced_class_f1 = f1_score(
        true_classes[sampled_indices],
        reduced_predictions[
            sampled_indices
        ],
        labels=FIXED_CLASS_INDICES,
        average=None,
        zero_division=0
    )

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        safe_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        record[
            f"{safe_name}_f1_"
            "difference_reduced_minus_full"
        ] = (
            reduced_class_f1[
                class_index
            ]
            - full_class_f1[
                class_index
            ]
        )

    redundancy_bootstrap_records.append(
        record
    )


redundancy_bootstrap_df = pd.DataFrame(
    redundancy_bootstrap_records
)


# ------------------------------------------------------------
# 10. Confidence intervals for paired differences
# ------------------------------------------------------------

redundancy_difference_ci_records = []

for metric_name in comparison_metric_names:
    column_name = (
        f"{metric_name}_"
        "difference_reduced_minus_full"
    )

    point_difference = (
        reduced_feature_results[
            "pooled_metrics"
        ][metric_name]
        - full_feature_results[
            "pooled_metrics"
        ][metric_name]
    )

    redundancy_difference_ci_records.append({
        "metric": metric_name,
        "point_difference_reduced_minus_full": (
            point_difference
        ),
        "bootstrap_mean_difference": (
            redundancy_bootstrap_df[
                column_name
            ].mean()
        ),
        "ci_2.5_percent": (
            redundancy_bootstrap_df[
                column_name
            ].quantile(0.025)
        ),
        "ci_97.5_percent": (
            redundancy_bootstrap_df[
                column_name
            ].quantile(0.975)
        )
    })

redundancy_difference_ci_df = pd.DataFrame(
    redundancy_difference_ci_records
)


redundancy_class_difference_ci_records = []

for class_name in ORDERED_CLUSTER_NAMES:
    safe_name = (
        class_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    bootstrap_column = (
        f"{safe_name}_f1_"
        "difference_reduced_minus_full"
    )

    point_difference = float(
        redundancy_class_comparison_df.loc[
            redundancy_class_comparison_df[
                "class_name"
            ] == class_name,
            "f1_difference_reduced_minus_full"
        ].iloc[0]
    )

    redundancy_class_difference_ci_records.append({
        "class_name": class_name,
        "metric": "f1",
        "point_difference_reduced_minus_full": (
            point_difference
        ),
        "bootstrap_mean_difference": (
            redundancy_bootstrap_df[
                bootstrap_column
            ].mean()
        ),
        "ci_2.5_percent": (
            redundancy_bootstrap_df[
                bootstrap_column
            ].quantile(0.025)
        ),
        "ci_97.5_percent": (
            redundancy_bootstrap_df[
                bootstrap_column
            ].quantile(0.975)
        )
    })

redundancy_class_difference_ci_df = (
    pd.DataFrame(
        redundancy_class_difference_ci_records
    )
)


# ------------------------------------------------------------
# 11. Display results
# ------------------------------------------------------------

print("=" * 80)
print("STEP 16 — CORRELATED-FEATURE SENSITIVITY")
print("=" * 80)

print("\nCorrelation groups with more than one feature:")

display(
    correlation_component_df.loc[
        correlation_component_df[
            "component_size"
        ] > 1
    ]
)

print("\nRetained reduced feature set:")

display(
    pd.DataFrame({
        "feature": (
            REDUCED_CORRELATION_FEATURES
        )
    })
)

print("\nRemoved only for sensitivity analysis:")

display(
    pd.DataFrame({
        "feature": sorted(
            features_to_remove,
            key=lambda feature:
                original_feature_rank[feature]
        )
    })
)

print("\nOverall performance comparison:")

display(
    redundancy_overall_comparison_df.round(4)
)

print("\nClass-specific comparison:")

display(
    redundancy_class_comparison_df.round(4)
)

print("\nPaired bootstrap confidence intervals:")

display(
    redundancy_difference_ci_df.round(4)
)

print("\nClass-specific F1 difference intervals:")

display(
    redundancy_class_difference_ci_df.round(4)
)

print("\nFull-feature confusion matrix:")

display(
    full_feature_results[
        "confusion_matrix"
    ]
)

print("\nReduced-feature confusion matrix:")

display(
    reduced_feature_results[
        "confusion_matrix"
    ]
)


# ------------------------------------------------------------
# 12. Save outputs
# ------------------------------------------------------------

high_correlation_pairs_step16_df.to_csv(
    OUTPUT_DIR
    / "step16_high_correlation_pairs.csv",
    index=False
)

correlation_component_df.to_csv(
    OUTPUT_DIR
    / "step16_correlation_components.csv",
    index=False
)

redundancy_overall_comparison_df.to_csv(
    OUTPUT_DIR
    / "correlation_reduction_overall_comparison.csv",
    index=False
)

redundancy_class_comparison_df.to_csv(
    OUTPUT_DIR
    / "correlation_reduction_class_comparison.csv",
    index=False
)

redundancy_difference_ci_df.to_csv(
    OUTPUT_DIR
    / "correlation_reduction_difference_ci.csv",
    index=False
)

redundancy_class_difference_ci_df.to_csv(
    OUTPUT_DIR
    / "correlation_reduction_class_difference_ci.csv",
    index=False
)

redundancy_bootstrap_df.to_csv(
    OUTPUT_DIR
    / "correlation_reduction_bootstrap.csv",
    index=False
)

full_feature_results[
    "fold_metrics"
].to_csv(
    OUTPUT_DIR
    / "correlation_sensitivity_full_fold_metrics.csv",
    index=False
)

reduced_feature_results[
    "fold_metrics"
].to_csv(
    OUTPUT_DIR
    / "correlation_sensitivity_reduced_fold_metrics.csv",
    index=False
)

correlation_reduction_configuration = {
    "correlation_method": "Spearman",
    "absolute_correlation_threshold": (
        CORRELATION_REDUCTION_THRESHOLD
    ),
    "target_used_for_grouping": False,
    "model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "model_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "full_feature_count": len(
        FINAL_WORKFLOW_FEATURES
    ),
    "reduced_feature_count": len(
        REDUCED_CORRELATION_FEATURES
    ),
    "full_features": (
        FINAL_WORKFLOW_FEATURES
    ),
    "reduced_features": (
        REDUCED_CORRELATION_FEATURES
    ),
    "removed_features": sorted(
        features_to_remove,
        key=lambda feature:
            original_feature_rank[feature]
    ),
    "same_cv_splits_used": True,
    "bootstrap_iterations": (
        REDUNDANCY_BOOTSTRAP_ITERATIONS
    ),
    "analysis_role": (
        "Secondary correlated-feature sensitivity"
    ),
    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation"
    ),
    "automatic_final_model_replacement": False
}

with open(
    OUTPUT_DIR
    / "correlation_reduction_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        correlation_reduction_configuration,
        configuration_file,
        indent=4
    )


print("\n" + "=" * 80)
print("STEP 16 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Full feature count:",
    len(FINAL_WORKFLOW_FEATURES)
)

print(
    "Reduced feature count:",
    len(REDUCED_CORRELATION_FEATURES)
)

print(
    "Full-feature macro-F1:",
    round(
        full_feature_results[
            "pooled_metrics"
        ]["macro_f1"],
        4
    )
)

print(
    "Reduced-feature macro-F1:",
    round(
        reduced_feature_results[
            "pooled_metrics"
        ]["macro_f1"],
        4
    )
)

print(
    "\nSaved Step 16 outputs to:",
    OUTPUT_DIR
)

In [ ]:

# ============================================================
# STEP 17 — FIT, SAVE, RELOAD, AND AUDIT THE FINAL RESEARCH PROTOTYPE
# ============================================================
# This step is run only after Steps 1–16.
#
# Performance must continue to be reported from the primary
# Step 4 nested out-of-fold evaluation. Fitting on all matched
# households creates a reusable artefact; it does not create a
# new unbiased performance estimate.
# ============================================================

import hashlib
import importlib.metadata
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone


required_step17_objects = [
    "dataset_all",
    "locked_labels",
    "FINAL_WORKFLOW_ID",
    "FINAL_WORKFLOW_FEATURE_SET",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_PARAMS",
    "final_workflow_pipeline_template",
    "CLASS_TO_INDEX",
    "INDEX_TO_CLASS",
    "FIXED_CLASS_INDICES",
    "ORDERED_CLUSTER_NAMES",
    "input_hashes",
    "pooled_performance_df",
    "pooled_class_metrics_df",
    "selected_stability_row",
    "OUTPUT_DIR"
]

missing_step17_objects = [
    name
    for name in required_step17_objects
    if name not in globals()
]

if missing_step17_objects:
    raise RuntimeError(
        "Run Steps 1–16 before Step 17. Missing objects: "
        f"{missing_step17_objects}"
    )


def to_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): to_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [to_json_safe(item) for item in value]

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if pd.isna(value):
        return None

    return value


def sha256_path(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(file_path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


final_training_df = (
    dataset_all
    .copy()
    .sort_values("household_ID")
    .reset_index(drop=True)
)

locked_household_count = int(
    len(locked_labels)
)

matched_household_count = int(
    len(final_training_df)
)

matched_class_counts = (
    final_training_df["cluster_name"]
    .astype(str)
    .value_counts()
)

daytime_matched_count = int(
    matched_class_counts.get(
        "Daytime-Active",
        0
    )
)

if final_training_df["household_ID"].duplicated().any():
    raise ValueError(
        "The final classifier dataset must contain one row per household."
    )

X_final_classifier = (
    final_training_df[
        FINAL_WORKFLOW_FEATURES
    ]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

y_final_classifier = (
    final_training_df["cluster_name"]
    .astype(str)
    .map(CLASS_TO_INDEX)
)

if y_final_classifier.isna().any():
    unknown_labels = sorted(
        final_training_df.loc[
            y_final_classifier.isna(),
            "cluster_name"
        ].astype(str).unique().tolist()
    )

    raise ValueError(
        f"Unknown class labels before final fitting: {unknown_labels}"
    )

y_final_classifier = y_final_classifier.astype(int)

if sorted(y_final_classifier.unique().tolist()) != FIXED_CLASS_INDICES:
    raise ValueError(
        "The final fitting target does not contain all four locked classes."
    )

if X_final_classifier.isna().all(axis=1).any():
    raise ValueError(
        "At least one household has all final predictors missing."
    )

final_research_prototype = clone(
    final_workflow_pipeline_template
)

final_research_prototype.fit(
    X_final_classifier,
    y_final_classifier
)

prototype_dir = (
    Path(OUTPUT_DIR)
    / "final_research_prototype"
)

prototype_dir.mkdir(
    parents=True,
    exist_ok=True
)

model_family_slug = (
    FINAL_WORKFLOW_MODEL_FAMILY
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
)

model_path = (
    prototype_dir
    / "archetype_classifier_bundle.joblib"
)

metadata_path = (
    prototype_dir
    / "archetype_transfer_model_metadata.json"
)

schema_path = (
    prototype_dir
    / "archetype_transfer_feature_schema.json"
)

model_card_path = (
    prototype_dir
    / "MODEL_CARD.md"
)

requirements_path = (
    prototype_dir
    / "requirements.txt"
)

audit_path = (
    prototype_dir
    / "PACKAGE_AUDIT.json"
)

manifest_path = (
    prototype_dir
    / "FILE_MANIFEST_SHA256.csv"
)

inference_helper_path = (
    prototype_dir
    / "predict_from_engineered_features.py"
)

usage_readme_path = (
    prototype_dir
    / "README_USAGE.md"
)

primary_performance = (
    pooled_performance_df
    .iloc[0]
    .to_dict()
)

class_performance = (
    pooled_class_metrics_df
    .to_dict(orient="records")
)

fixed_workflow_selection_performance = {
    "source": (
        "Step 9 repeated stratified cross-validation used "
        "for fixed-workflow selection"
    ),
    "macro_f1": float(
        selected_stability_row["mean_macro_f1"]
    ),
    "balanced_accuracy": float(
        selected_stability_row[
            "mean_balanced_accuracy"
        ]
    ),
    "accuracy": float(
        selected_stability_row["mean_accuracy"]
    ),
    "status": (
        "Secondary post-selection internal estimate; not an "
        "independent unbiased estimate for the fitted prototype."
    )
}

model_package = {
    "estimator": final_research_prototype,
    "feature_order": list(FINAL_WORKFLOW_FEATURES),
    "class_to_index": dict(CLASS_TO_INDEX),
    "index_to_class": dict(INDEX_TO_CLASS),
    "ordered_class_names": list(ORDERED_CLUSTER_NAMES),
    "workflow_id": FINAL_WORKFLOW_ID,
    "workflow_feature_set": FINAL_WORKFLOW_FEATURE_SET,
    "workflow_model_family": FINAL_WORKFLOW_MODEL_FAMILY,
    "workflow_parameters": dict(FINAL_WORKFLOW_PARAMS),
    "training_households": int(len(final_training_df)),
    "training_month_coverage": {
        "minimum_months": int(
            final_training_df["months_observed"].min()
        ),
        "median_months": float(
            final_training_df["months_observed"].median()
        ),
        "maximum_months": int(
            final_training_df["months_observed"].max()
        )
    },
    "raw_monthly_input_columns": [
        "household_ID",
        "month",
        "consumption"
    ],
    "feature_engineering_version": (
        "2.0_calendar_gap_aware"
    ),
    "scaling_policy": (
        "RobustScaler is fitted inside Logistic Regression pipelines. "
        "Tree pipelines use no scaler because scaling does not change "
        "tree split ordering."
    ),
    "probability_status": (
        "Raw uncalibrated relative model scores; not validated "
        "probabilities of correctness."
    ),
    "input_hashes": dict(input_hashes),
    "intended_use": (
        "Supplementary contemporaneous transfer of locked "
        "smart-meter-derived pseudo-labels from monthly billing features."
    ),
    "prohibited_interpretations": [
        "Not an archetype-discovery model.",
        "Not a prospective demand-forecasting model.",
        "Not a causal model.",
        "Not externally validated for operational deployment.",
        "Not a replacement for high-frequency smart-meter clustering."
    ],
    "primary_evaluation_target": (
        "Adaptive nested workflow-selection procedure"
    ),
    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation"
    ),
    "fixed_prototype_performance_status": (
        "No independent unbiased performance estimate exists "
        "for this exact full-data-fitted prototype."
    )
}

joblib.dump(
    model_package,
    model_path
)

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": importlib.metadata.version("joblib"),
    "workflow_id": FINAL_WORKFLOW_ID,
    "feature_set": FINAL_WORKFLOW_FEATURE_SET,
    "model_family": FINAL_WORKFLOW_MODEL_FAMILY,
    "parameters": to_json_safe(FINAL_WORKFLOW_PARAMS),
    "feature_count": int(len(FINAL_WORKFLOW_FEATURES)),
    "training_households": int(len(final_training_df)),
    "class_counts": to_json_safe(
        final_training_df["cluster_name"]
        .value_counts()
        .reindex(ORDERED_CLUSTER_NAMES)
        .fillna(0)
        .astype(int)
        .to_dict()
    ),
    "input_hashes": to_json_safe(input_hashes),
    "adaptive_nested_oof_performance": to_json_safe(
        primary_performance
    ),
    "adaptive_nested_oof_class_performance": to_json_safe(
        class_performance
    ),
    "fixed_workflow_selection_performance": to_json_safe(
        fixed_workflow_selection_performance
    ),
    "performance_attribution": {
        "step4_metrics_apply_to": (
            "Adaptive nested workflow-selection procedure"
        ),
        "step4_metrics_do_not_apply_directly_to": (
            "The exact fixed prototype fitted on all matched households"
        ),
        "exact_fixed_prototype_independent_performance_available": False
    },
    "limitations": [
        "The target consists of K-Means pseudo-labels rather than external ground truth.",
        f"Only {matched_household_count:,} of {locked_household_count:,} locked households matched the monthly billing data.",
        "Monthly billing match coverage differs significantly by archetype.",
        f"Daytime-Active has {daytime_matched_count:,} matched households.",
        "Monthly totals cannot directly observe time-of-use behaviour.",
        "The all-available billing predictors overlap the smart-meter label period and therefore represent contemporaneous transfer rather than prospective prediction.",
        "No independent external cohort or later untouched deployment dataset was available.",
        "Probability calibration did not provide a reliable four-class hard-decision improvement.",
        "Returned probabilities are raw uncalibrated model scores and must not be interpreted as validated confidence estimates.",
        "The training households contained 15 to 25 observed billing months. Predictions outside this coverage range are not validated."
    ]
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        metadata,
        handle,
        indent=4
    )

feature_schema = {
    "feature_order": list(FINAL_WORKFLOW_FEATURES),
    "feature_count": int(len(FINAL_WORKFLOW_FEATURES)),
    "input_unit": "one engineered feature row per household",
    "input_stage": "engineered_household_monthly_billing_features",
    "raw_monthly_csv_supported_directly": True,
    "raw_monthly_runtime_module": "archetype_runtime.py",
    "validated_month_coverage": {
        "minimum_months": 15,
        "maximum_months": 25,
        "deployment_policy": (
            "Require at least 15 valid monthly observations and "
            "retain at most the latest 25 observations."
        )
    },
    "target": "locked K-Means archetype pseudo-label",
    "required_identifier": "household_ID is not used as a predictor",
    "missing_value_handling": (
        "Handled by the fitted scikit-learn preprocessing pipeline."
    )
}

with open(
    schema_path,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        feature_schema,
        handle,
        indent=4
    )

inference_helper_code = r"""from pathlib import Path
import argparse
import joblib
import numpy as np
import pandas as pd


def predict_engineered_features(
    model_package_path,
    engineered_feature_csv,
    output_csv
):
    package = joblib.load(model_package_path)
    estimator = package["estimator"]
    feature_order = list(package["feature_order"])
    index_to_class = {
        int(key): value
        for key, value in package["index_to_class"].items()
    }

    data = pd.read_csv(engineered_feature_csv)

    missing_features = [
        feature
        for feature in feature_order
        if feature not in data.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing required engineered features: "
            f"{missing_features}"
        )

    X = (
        data[feature_order]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    if X.isna().all(axis=1).any():
        bad_rows = X.index[
            X.isna().all(axis=1)
        ].tolist()

        raise ValueError(
            "At least one row has all predictor values missing. "
            f"Row indices: {bad_rows[:20]}"
        )

    predicted_indices = estimator.predict(X).astype(int)
    probabilities = estimator.predict_proba(X)

    result = pd.DataFrame(index=data.index)

    if "household_ID" in data.columns:
        result["household_ID"] = data["household_ID"]

    result["predicted_class_index"] = predicted_indices
    result["predicted_archetype"] = [
        index_to_class[index]
        for index in predicted_indices
    ]
    result["highest_probability"] = probabilities.max(axis=1)

    estimator_classes = np.asarray(
        estimator.classes_,
        dtype=int
    )

    for probability_column, class_index in enumerate(
        estimator_classes
    ):
        safe_name = (
            index_to_class[int(class_index)]
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        result[
            f"probability_{safe_name}"
        ] = probabilities[:, probability_column]

    result.to_csv(output_csv, index=False)
    return result


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Predict locked smart-meter-derived archetype "
            "pseudo-labels from already engineered household "
            "monthly-billing features."
        )
    )

    parser.add_argument(
        "--model-package",
        required=True
    )

    parser.add_argument(
        "--features-csv",
        required=True
    )

    parser.add_argument(
        "--output-csv",
        default="archetype_predictions.csv"
    )

    arguments = parser.parse_args()

    predict_engineered_features(
        model_package_path=Path(arguments.model_package),
        engineered_feature_csv=Path(arguments.features_csv),
        output_csv=Path(arguments.output_csv)
    )


if __name__ == "__main__":
    main()
"""

inference_helper_path.write_text(
    inference_helper_code,
    encoding="utf-8"
)

usage_readme = f"""# Archetype-Transfer Prototype Usage

## Accepted input

The saved estimator accepts **one already engineered household-level row**
containing the exact {len(FINAL_WORKFLOW_FEATURES)} predictors listed in
`archetype_transfer_feature_schema.json`.

The accompanying Streamlit package accepts raw monthly billing transactions
and applies the same Step 2 feature-engineering definitions before prediction.
The fitted estimator itself still receives one engineered household-level row.

## Command-line prediction

```bash
python predict_from_engineered_features.py \
  --model-package {model_path.name} \
  --features-csv engineered_household_features.csv \
  --output-csv archetype_predictions.csv
```

## Interpretation

The output is a supplementary transfer of locked K-Means pseudo-labels.
It is not new archetype discovery, prospective forecasting, causal inference,
or an externally validated operational decision system.

## Performance reporting

Step 4 nested out-of-fold metrics estimate the complete adaptive
workflow-selection procedure. They must not be attributed as an independent
unbiased estimate of this exact full-data-fitted {FINAL_WORKFLOW_MODEL_FAMILY}
prototype.
"""

usage_readme_path.write_text(
    usage_readme,
    encoding="utf-8"
)


requirements = [
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={importlib.metadata.version('joblib')}"
]

requirements_path.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

primary_macro_f1 = float(
    primary_performance["macro_f1"]
)

primary_balanced_accuracy = float(
    primary_performance["balanced_accuracy"]
)

primary_accuracy = float(
    primary_performance["accuracy"]
)

model_card = f"""# Model Card — Monthly-Billing Archetype-Transfer Research Prototype

## Purpose

This model transfers four locked smart-meter-derived K-Means pseudo-labels
using household-level monthly billing features.

It is a supplementary contemporaneous label-transfer prototype, not an
archetype-discovery, forecasting, causal, or externally validated
operational model.

## Locked data endpoint

- Locked K-Means population: {locked_household_count:,} households
- Monthly-billing matched population used for fitting: {len(final_training_df):,}
- Predictor count: {len(FINAL_WORKFLOW_FEATURES)}
- Model family: {FINAL_WORKFLOW_MODEL_FAMILY}
- Feature set: {FINAL_WORKFLOW_FEATURE_SET}

## Primary evaluated procedure

Step 4 provides the primary approximately unbiased internal estimate for the
**adaptive nested workflow-selection procedure**, in which the selected feature
set, model family and parameters may differ across outer folds:

- Accuracy: {primary_accuracy:.4f}
- Balanced accuracy: {primary_balanced_accuracy:.4f}
- Macro-F1: {primary_macro_f1:.4f}

These values must not be presented as an independent unbiased estimate of the
exact fixed prototype saved in this package.

## Final fitted prototype

The saved {FINAL_WORKFLOW_MODEL_FAMILY} configuration was selected using
full-dataset repeated cross-validation and then fitted on all matched
households. Its repeated-CV values are secondary post-selection internal
estimates:

- Repeated-CV accuracy: {fixed_workflow_selection_performance['accuracy']:.4f}
- Repeated-CV balanced accuracy: {fixed_workflow_selection_performance['balanced_accuracy']:.4f}
- Repeated-CV macro-F1: {fixed_workflow_selection_performance['macro_f1']:.4f}

No independent unbiased performance estimate exists for this exact
full-data-fitted prototype.

## Input contract

The estimator requires the exact engineered feature columns listed in the
feature schema. The accompanying `archetype_runtime.py` and Streamlit app
transform validated raw monthly billing records into those features. At least
15 valid monthly observations are required, and at most the latest 25 are used.

## Intended use

- Methodological proof-of-concept
- Supplementary analysis of whether billing features contain information
  about smart-meter-derived archetypes
- Reproducible research inference on correctly engineered feature rows

## Not intended for

- Replacing 15-minute smart-meter clustering
- Discovering new archetypes
- Prospective household forecasting
- Causal, welfare, tariff, or policy eligibility decisions
- Operational deployment without external validation

## Principal limitations

- Pseudo-label rather than external ground truth
- Differential billing-match coverage across archetypes
- Only {daytime_matched_count:,} matched Daytime-Active households
- Monthly totals cannot directly recover time-of-use structure
- All-available billing predictors overlap the smart-meter label period
- Moderate balanced accuracy and macro-F1
- No independent external validation cohort
- Returned probabilities are raw uncalibrated model scores, not validated confidence estimates
- Training coverage was 15–25 observed months; inputs outside this range are not validated
"""

model_card_path.write_text(
    model_card,
    encoding="utf-8"
)

# Reload and deterministic smoke test.
reloaded_package = joblib.load(
    model_path
)

reloaded_estimator = reloaded_package[
    "estimator"
]

smoke_rows = min(
    10,
    len(X_final_classifier)
)

original_probabilities = (
    final_research_prototype
    .predict_proba(
        X_final_classifier.iloc[:smoke_rows]
    )
)

reloaded_probabilities = (
    reloaded_estimator
    .predict_proba(
        X_final_classifier.iloc[:smoke_rows]
    )
)

if not np.allclose(
    original_probabilities,
    reloaded_probabilities,
    atol=1e-12
):
    raise RuntimeError(
        "Reloaded estimator probabilities do not match the saved estimator."
    )

smoke_predictions = (
    reloaded_estimator
    .predict(
        X_final_classifier.iloc[:smoke_rows]
    )
)

if not set(
    np.asarray(smoke_predictions, dtype=int)
).issubset(
    set(FIXED_CLASS_INDICES)
):
    raise RuntimeError(
        "Reloaded estimator returned an unexpected class index."
    )

required_artifacts = [
    model_path,
    metadata_path,
    schema_path,
    model_card_path,
    requirements_path,
    inference_helper_path,
    usage_readme_path
]

missing_artifacts = [
    str(path)
    for path in required_artifacts
    if not path.exists()
]

if missing_artifacts:
    raise RuntimeError(
        f"Missing final prototype artifacts: {missing_artifacts}"
    )

manifest_records = []

for artifact in required_artifacts:
    manifest_records.append({
        "relative_path": artifact.name,
        "size_bytes": int(artifact.stat().st_size),
        "sha256": sha256_path(artifact)
    })

manifest_df = pd.DataFrame(
    manifest_records
)

manifest_df.to_csv(
    manifest_path,
    index=False
)

package_audit = {
    "package_complete": True,
    "model_reload_smoke_test_passed": True,
    "households_fitted": int(len(final_training_df)),
    "feature_count": int(len(FINAL_WORKFLOW_FEATURES)),
    "primary_evaluation_target": (
        "Adaptive nested workflow-selection procedure"
    ),
    "primary_performance_source": (
        "Step 4 nested out-of-fold evaluation"
    ),
    "exact_fixed_prototype_independent_performance_available": False,
    "external_validation_available": False,
    "deployment_ready": False,
    "files": [
        path.name
        for path in required_artifacts
    ] + [manifest_path.name]
}

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        package_audit,
        handle,
        indent=4
    )

print("=" * 80)
print("STEP 17 — FINAL RESEARCH PROTOTYPE PACKAGE")
print("=" * 80)
print("Model fitted households:", f"{len(final_training_df):,}")
print("Model family:", FINAL_WORKFLOW_MODEL_FAMILY)
print("Predictor count:", len(FINAL_WORKFLOW_FEATURES))
print("Reload smoke test: PASS")
print("Primary evaluation target: adaptive nested selection procedure")
print("Exact fixed prototype independently evaluated: NO")
print("\nSaved package:")
print(prototype_dir.resolve())


In [ ]:

# ============================================================
# STEP 18 — EXPORT FINAL THESIS TABLES AND FIGURES
# ============================================================
# This step does not retrain or reselect the model.
# Step 4 provides the primary adaptive-procedure estimate.
# Step 9 values are retained only as secondary post-selection
# estimates for the fixed prototype.
# ============================================================

from pathlib import Path
from IPython.display import display
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


required_objects = [
    "OUTPUT_DIR",
    "ORDERED_CLUSTER_NAMES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_FEATURE_SET",
    "FINAL_WORKFLOW_FEATURES",
    "pooled_performance_df",
    "pooled_class_metrics_df",
    "pooled_confusion_df",
    "baseline_comparison",
    "overall_bootstrap_ci_df",
    "class_bootstrap_ci_df",
    "probability_method_comparison_df",
    "imbalance_workflow_comparison",
    "step5_secondary_interpretation_df",
    "workflow_stability_decision_df",
    "selected_stability_row",
    "temporal_overall_comparison_df",
    "temporal_difference_ci_df",
    "permutation_summary_df",
    "discrimination_overall_comparison_df",
    "discrimination_class_comparison_df",
    "permutation_test_summary_df",
    "redundancy_overall_comparison_df",
    "redundancy_difference_ci_df",
    "oof_predictions_df",
    "dataset_all",
    "locked_labels"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Steps 1–17 first. Missing objects: "
        f"{missing_objects}"
    )


FINAL_REPORT_DIR = (
    Path(OUTPUT_DIR)
    / "final_thesis_reporting"
)

FINAL_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

locked_household_count = int(
    len(locked_labels)
)

matched_household_count = int(
    len(dataset_all)
)

matched_class_counts = (
    dataset_all["cluster_name"]
    .astype(str)
    .value_counts()
)

daytime_matched_count = int(
    matched_class_counts.get(
        "Daytime-Active",
        0
    )
)


# Verify the authoritative Step 4 OOF results.
if len(pooled_performance_df) != 1:
    raise ValueError(
        "Expected one pooled Step 4 performance row."
    )

primary_evaluation = str(
    pooled_performance_df.iloc[0]["evaluation"]
)

if "out-of-fold" not in primary_evaluation.lower():
    raise ValueError(
        "Primary performance is not labelled out-of-fold."
    )

expected_oof_households = matched_household_count

if len(oof_predictions_df) != expected_oof_households:
    raise ValueError(
        f"Expected {expected_oof_households:,} Step 4 OOF "
        f"household predictions, but found "
        f"{len(oof_predictions_df):,}."
    )

if oof_predictions_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households found in OOF predictions."
    )


# Separate the evaluated adaptive procedure from the fitted prototype.
primary_row = pooled_performance_df.iloc[0]

evaluation_and_prototype_decision_df = pd.DataFrame([
    {
        "record_type": (
            "Primary evaluated procedure"
        ),
        "evaluation_target": (
            "Adaptive nested workflow-selection procedure"
        ),
        "locked_cluster_households": locked_household_count,
        "matched_classifier_households": matched_household_count,
        "model_family": (
            "Adaptive selection across candidate workflows"
        ),
        "feature_set": (
            "Selected inside each outer-training partition"
        ),
        "predictor_count": np.nan,
        "performance_source": primary_evaluation,
        "accuracy": primary_row["accuracy"],
        "balanced_accuracy": primary_row[
            "balanced_accuracy"
        ],
        "macro_f1": primary_row["macro_f1"],
        "weighted_f1": primary_row["weighted_f1"],
        "top_2_accuracy": primary_row[
            "top_2_accuracy"
        ],
        "multiclass_log_loss": primary_row[
            "multiclass_log_loss"
        ],
        "multiclass_brier_sum": primary_row[
            "multiclass_brier_sum"
        ],
        "approximately_unbiased_internal_estimate_for_this_record": True,
        "interpretation": (
            "Use these metrics as the primary internal estimate "
            "of the complete adaptive development procedure."
        ),
        "operational_deployment_ready": False
    },
    {
        "record_type": (
            "Final fitted research prototype"
        ),
        "evaluation_target": (
            f"Fixed {FINAL_WORKFLOW_MODEL_FAMILY} prototype"
        ),
        "locked_cluster_households": locked_household_count,
        "matched_classifier_households": matched_household_count,
        "model_family": FINAL_WORKFLOW_MODEL_FAMILY,
        "feature_set": FINAL_WORKFLOW_FEATURE_SET,
        "predictor_count": len(FINAL_WORKFLOW_FEATURES),
        "performance_source": (
            "Step 9 repeated cross-validation used for "
            "post-selection fixed-workflow choice"
        ),
        "accuracy": float(
            selected_stability_row["mean_accuracy"]
        ),
        "balanced_accuracy": float(
            selected_stability_row[
                "mean_balanced_accuracy"
            ]
        ),
        "macro_f1": float(
            selected_stability_row["mean_macro_f1"]
        ),
        "weighted_f1": float(
            selected_stability_row["mean_weighted_f1"]
        ),
        "top_2_accuracy": np.nan,
        "multiclass_log_loss": np.nan,
        "multiclass_brier_sum": np.nan,
        "approximately_unbiased_internal_estimate_for_this_record": False,
        "interpretation": (
            "Secondary post-selection internal estimate only. "
            "No independent unbiased estimate exists for the "
            "exact full-data-fitted prototype."
        ),
        "operational_deployment_ready": False
    }
])


# Add readable confidence intervals.
overall_report_df = (
    overall_bootstrap_ci_df
    .copy()
)

overall_report_df[
    "confidence_interval"
] = overall_report_df.apply(
    lambda row: (
        f"[{row['ci_2.5_percent']:.4f}, "
        f"{row['ci_97.5_percent']:.4f}]"
    ),
    axis=1
)


# Merge class-level point estimates and confidence intervals.
class_ci_wide_df = (
    class_bootstrap_ci_df
    .pivot(
        index="class_name",
        columns="metric",
        values=[
            "point_estimate",
            "ci_2.5_percent",
            "ci_97.5_percent"
        ]
    )
)

class_ci_wide_df.columns = [
    f"{statistic}_{metric}"
    for statistic, metric
    in class_ci_wide_df.columns
]

class_ci_wide_df = (
    class_ci_wide_df
    .reset_index()
)

class_report_df = (
    pooled_class_metrics_df
    .merge(
        class_ci_wide_df,
        on="class_name",
        how="left",
        validate="one_to_one"
    )
)

class_order = {
    class_name: position
    for position, class_name
    in enumerate(ORDERED_CLUSTER_NAMES)
}

class_report_df[
    "_class_order"
] = class_report_df[
    "class_name"
].map(class_order)

class_report_df = (
    class_report_df
    .sort_values("_class_order")
    .drop(columns="_class_order")
    .reset_index(drop=True)
)


# Export final authoritative tables.
final_tables = {
    "01_evaluation_and_prototype_decision.csv":
        evaluation_and_prototype_decision_df,

    "02_primary_nested_oof_performance.csv":
        pooled_performance_df,

    "03_primary_overall_bootstrap_ci.csv":
        overall_report_df,

    "04_primary_class_metrics.csv":
        class_report_df,

    "05_primary_confusion_matrix.csv":
        pooled_confusion_df,

    "06_dummy_baseline_comparison.csv":
        baseline_comparison,

    "07_probability_method_comparison.csv":
        probability_method_comparison_df,

    "08_imbalance_sensitivity_comparison.csv":
        imbalance_workflow_comparison,

    "09_imbalance_sensitivity_interpretation.csv":
        step5_secondary_interpretation_df,

    "10_workflow_stability_decision.csv":
        workflow_stability_decision_df,

    "11_temporal_sensitivity_overall.csv":
        temporal_overall_comparison_df,

    "12_temporal_sensitivity_difference_ci.csv":
        temporal_difference_ci_df,

    "13_permutation_importance.csv":
        permutation_summary_df,

    "14_discrimination_overall.csv":
        discrimination_overall_comparison_df,

    "15_discrimination_by_class.csv":
        discrimination_class_comparison_df,

    "16_conditional_label_permutation_check.csv":
        permutation_test_summary_df,

    "17_feature_reduction_comparison.csv":
        redundancy_overall_comparison_df,

    "18_feature_reduction_difference_ci.csv":
        redundancy_difference_ci_df,

    "19_primary_oof_predictions.csv":
        oof_predictions_df
}

for filename, dataframe in final_tables.items():
    dataframe.to_csv(
        FINAL_REPORT_DIR / filename,
        index=(
            filename
            == "05_primary_confusion_matrix.csv"
        )
    )


# Export one combined Excel workbook when openpyxl is available.
excel_status = "PASS"

try:
    with pd.ExcelWriter(
        FINAL_REPORT_DIR
        / "archetype_classifier_final_results.xlsx",
        engine="openpyxl"
    ) as writer:

        for filename, dataframe in final_tables.items():
            sheet_name = (
                Path(filename).stem[:31]
            )

            dataframe.to_excel(
                writer,
                sheet_name=sheet_name,
                index=(
                    filename
                    == "05_primary_confusion_matrix.csv"
                )
            )

except ImportError:
    excel_status = (
        "SKIPPED — openpyxl is not installed"
    )


# Figure 1: raw Step 4 OOF confusion matrix.
confusion_values = (
    pooled_confusion_df
    .to_numpy(dtype=int)
)

fig, ax = plt.subplots(
    figsize=(8, 7)
)

image = ax.imshow(
    confusion_values
)

ax.set_xticks(
    np.arange(
        len(ORDERED_CLUSTER_NAMES)
    )
)

ax.set_yticks(
    np.arange(
        len(ORDERED_CLUSTER_NAMES)
    )
)

ax.set_xticklabels(
    ORDERED_CLUSTER_NAMES,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(
    ORDERED_CLUSTER_NAMES
)

ax.set_xlabel(
    "Predicted archetype"
)

ax.set_ylabel(
    "True archetype"
)

ax.set_title(
    "Primary Nested Out-of-Fold Confusion Matrix"
)

for row_index in range(
    confusion_values.shape[0]
):
    for column_index in range(
        confusion_values.shape[1]
    ):
        ax.text(
            column_index,
            row_index,
            str(
                confusion_values[
                    row_index,
                    column_index
                ]
            ),
            ha="center",
            va="center"
        )

fig.colorbar(
    image,
    ax=ax,
    label="Household count"
)

fig.tight_layout()

fig.savefig(
    FINAL_REPORT_DIR
    / "figure_01_primary_oof_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# Figure 2: class-specific metrics.
class_plot_df = (
    pooled_class_metrics_df
    .set_index("class_name")
    .reindex(ORDERED_CLUSTER_NAMES)
)

ax = class_plot_df[
    [
        "precision",
        "recall",
        "f1_score"
    ]
].plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_xlabel(
    "Archetype"
)

ax.set_ylabel(
    "Score"
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "Primary Nested Out-of-Fold Class Performance"
)

ax.legend(
    title="Metric"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FINAL_REPORT_DIR
    / "figure_02_primary_class_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


# Figure 3: classifier versus dummy baselines.
baseline_plot_df = (
    baseline_comparison
    .set_index("evaluation")[
        [
            "balanced_accuracy",
            "macro_f1"
        ]
    ]
)

ax = baseline_plot_df.plot(
    kind="bar",
    figsize=(9, 6)
)

ax.set_xlabel(
    "Evaluation"
)

ax.set_ylabel(
    "Score"
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "Classifier Compared with Dummy Baselines"
)

ax.legend(
    title="Metric"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FINAL_REPORT_DIR
    / "figure_03_baseline_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


# Save the final reporting rules.
reporting_rules = {
    "primary_evaluation_target":
        "Adaptive nested workflow-selection procedure",

    "primary_performance_source":
        "Step 4 nested out-of-fold predictions",

    "step4_metrics_directly_attributable_to_exact_fixed_prototype":
        False,

    "final_model_family":
        FINAL_WORKFLOW_MODEL_FAMILY,

    "final_predictor_count":
        len(FINAL_WORKFLOW_FEATURES),

    "predictor_decision":
        "Retain all 32 predictors",

    "matched_households":
        matched_household_count,

    "locked_cluster_households":
        locked_household_count,

    "fixed_prototype_performance_status":
        (
            "No independent unbiased estimate exists for the "
            "exact full-data-fitted prototype."
        ),

    "deployment_ready":
        False,

    "important_limitations": [
        "Target labels are K-Means pseudo-labels.",
        "Monthly totals do not directly observe time-of-use behaviour.",
        f"Only {daytime_matched_count:,} matched Daytime-Active households were available.",
        "No independent external validation cohort was available.",
        "The label-permutation result is conditional on the selected "
        "fixed workflow and is not a formal test of the complete "
        "model-selection procedure."
    ]
}

with open(
    FINAL_REPORT_DIR
    / "FINAL_REPORTING_RULES.json",
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        reporting_rules,
        handle,
        indent=4
    )


print("=" * 80)
print("STEP 18 — FINAL THESIS REPORTING EXPORT")
print("=" * 80)
print(
    "Primary evaluation target: adaptive nested "
    "workflow-selection procedure"
)
print("Primary performance source: Step 4 nested OOF")
print("Final prototype:", FINAL_WORKFLOW_MODEL_FAMILY)
print("Predictor count:", len(FINAL_WORKFLOW_FEATURES))
print("Matched households:", len(oof_predictions_df))
print("CSV tables exported:", len(final_tables))
print("Figures exported: 3")
print("Excel export:", excel_status)
print("Operational deployment ready: NO")
print("\nFinal reporting folder:")
print(FINAL_REPORT_DIR.resolve())

print("\nEvaluation and prototype reporting decision:")
display(
    evaluation_and_prototype_decision_df.round(4)
)

print("\nPrimary overall performance:")
display(
    overall_report_df.round(4)
)

print("\nPrimary class-specific performance:")
display(
    class_report_df.round(4)
)


In [ ]:
# ============================================================
# STEP 18 — EXPORT FINAL THESIS TABLES AND FIGURES
# ============================================================
# This step does not retrain or reselect the model.
# Step 4 provides the primary adaptive-procedure estimate.
# Step 9 values are retained only as secondary post-selection
# estimates for the fixed prototype.
#
# Figure styling in this step is designed for thesis/report use:
# - Sequential blue scale for count-based confusion matrix
# - Adaptive annotation colours for readability
# - Distinct, colourblind-friendly colours for metric bar charts
# - No analytical values or evaluation procedures are changed
# ============================================================

from pathlib import Path
from IPython.display import display
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# REQUIRED OBJECTS CHECK
# ============================================================

required_objects = [
    "OUTPUT_DIR",
    "ORDERED_CLUSTER_NAMES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_FEATURE_SET",
    "FINAL_WORKFLOW_FEATURES",
    "pooled_performance_df",
    "pooled_class_metrics_df",
    "pooled_confusion_df",
    "baseline_comparison",
    "overall_bootstrap_ci_df",
    "class_bootstrap_ci_df",
    "probability_method_comparison_df",
    "imbalance_workflow_comparison",
    "step5_secondary_interpretation_df",
    "workflow_stability_decision_df",
    "selected_stability_row",
    "temporal_overall_comparison_df",
    "temporal_difference_ci_df",
    "permutation_summary_df",
    "discrimination_overall_comparison_df",
    "discrimination_class_comparison_df",
    "permutation_test_summary_df",
    "redundancy_overall_comparison_df",
    "redundancy_difference_ci_df",
    "oof_predictions_df",
    "dataset_all",
    "locked_labels"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Steps 1–17 first. Missing objects: "
        f"{missing_objects}"
    )


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

FINAL_REPORT_DIR = (
    Path(OUTPUT_DIR)
    / "final_thesis_reporting"
)

FINAL_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# BASIC SAMPLE INFORMATION
# ============================================================

locked_household_count = int(
    len(locked_labels)
)

matched_household_count = int(
    len(dataset_all)
)

matched_class_counts = (
    dataset_all["cluster_name"]
    .astype(str)
    .value_counts()
)

daytime_matched_count = int(
    matched_class_counts.get(
        "Daytime-Active",
        0
    )
)


# ============================================================
# VERIFY AUTHORITATIVE STEP 4 OOF RESULTS
# ============================================================

if len(pooled_performance_df) != 1:
    raise ValueError(
        "Expected one pooled Step 4 performance row."
    )

primary_evaluation = str(
    pooled_performance_df.iloc[0]["evaluation"]
)

if "out-of-fold" not in primary_evaluation.lower():
    raise ValueError(
        "Primary performance is not labelled out-of-fold."
    )

expected_oof_households = matched_household_count

if len(oof_predictions_df) != expected_oof_households:
    raise ValueError(
        f"Expected {expected_oof_households:,} Step 4 OOF "
        f"household predictions, but found "
        f"{len(oof_predictions_df):,}."
    )

if oof_predictions_df["household_ID"].duplicated().any():
    raise ValueError(
        "Duplicate households found in OOF predictions."
    )


# ============================================================
# SEPARATE EVALUATED PROCEDURE FROM FINAL FITTED PROTOTYPE
# ============================================================

primary_row = pooled_performance_df.iloc[0]

evaluation_and_prototype_decision_df = pd.DataFrame([
    {
        "record_type": (
            "Primary evaluated procedure"
        ),
        "evaluation_target": (
            "Adaptive nested workflow-selection procedure"
        ),
        "locked_cluster_households": locked_household_count,
        "matched_classifier_households": matched_household_count,
        "model_family": (
            "Adaptive selection across candidate workflows"
        ),
        "feature_set": (
            "Selected inside each outer-training partition"
        ),
        "predictor_count": np.nan,
        "performance_source": primary_evaluation,
        "accuracy": primary_row["accuracy"],
        "balanced_accuracy": primary_row[
            "balanced_accuracy"
        ],
        "macro_f1": primary_row["macro_f1"],
        "weighted_f1": primary_row["weighted_f1"],
        "top_2_accuracy": primary_row[
            "top_2_accuracy"
        ],
        "multiclass_log_loss": primary_row[
            "multiclass_log_loss"
        ],
        "multiclass_brier_sum": primary_row[
            "multiclass_brier_sum"
        ],
        "approximately_unbiased_internal_estimate_for_this_record": True,
        "interpretation": (
            "Use these metrics as the primary internal estimate "
            "of the complete adaptive development procedure."
        ),
        "operational_deployment_ready": False
    },
    {
        "record_type": (
            "Final fitted research prototype"
        ),
        "evaluation_target": (
            f"Fixed {FINAL_WORKFLOW_MODEL_FAMILY} prototype"
        ),
        "locked_cluster_households": locked_household_count,
        "matched_classifier_households": matched_household_count,
        "model_family": FINAL_WORKFLOW_MODEL_FAMILY,
        "feature_set": FINAL_WORKFLOW_FEATURE_SET,
        "predictor_count": len(FINAL_WORKFLOW_FEATURES),
        "performance_source": (
            "Step 9 repeated cross-validation used for "
            "post-selection fixed-workflow choice"
        ),
        "accuracy": float(
            selected_stability_row["mean_accuracy"]
        ),
        "balanced_accuracy": float(
            selected_stability_row[
                "mean_balanced_accuracy"
            ]
        ),
        "macro_f1": float(
            selected_stability_row["mean_macro_f1"]
        ),
        "weighted_f1": float(
            selected_stability_row["mean_weighted_f1"]
        ),
        "top_2_accuracy": np.nan,
        "multiclass_log_loss": np.nan,
        "multiclass_brier_sum": np.nan,
        "approximately_unbiased_internal_estimate_for_this_record": False,
        "interpretation": (
            "Secondary post-selection internal estimate only. "
            "No independent unbiased estimate exists for the "
            "exact full-data-fitted prototype."
        ),
        "operational_deployment_ready": False
    }
])


# ============================================================
# READABLE OVERALL CONFIDENCE INTERVALS
# ============================================================

overall_report_df = (
    overall_bootstrap_ci_df
    .copy()
)

overall_report_df[
    "confidence_interval"
] = overall_report_df.apply(
    lambda row: (
        f"[{row['ci_2.5_percent']:.4f}, "
        f"{row['ci_97.5_percent']:.4f}]"
    ),
    axis=1
)


# ============================================================
# CLASS-LEVEL METRICS + CONFIDENCE INTERVALS
# ============================================================

class_ci_wide_df = (
    class_bootstrap_ci_df
    .pivot(
        index="class_name",
        columns="metric",
        values=[
            "point_estimate",
            "ci_2.5_percent",
            "ci_97.5_percent"
        ]
    )
)

class_ci_wide_df.columns = [
    f"{statistic}_{metric}"
    for statistic, metric
    in class_ci_wide_df.columns
]

class_ci_wide_df = (
    class_ci_wide_df
    .reset_index()
)

class_report_df = (
    pooled_class_metrics_df
    .merge(
        class_ci_wide_df,
        on="class_name",
        how="left",
        validate="one_to_one"
    )
)

class_order = {
    class_name: position
    for position, class_name
    in enumerate(ORDERED_CLUSTER_NAMES)
}

class_report_df[
    "_class_order"
] = class_report_df[
    "class_name"
].map(class_order)

class_report_df = (
    class_report_df
    .sort_values("_class_order")
    .drop(columns="_class_order")
    .reset_index(drop=True)
)


# ============================================================
# EXPORT FINAL AUTHORITATIVE TABLES
# ============================================================

final_tables = {
    "01_evaluation_and_prototype_decision.csv":
        evaluation_and_prototype_decision_df,

    "02_primary_nested_oof_performance.csv":
        pooled_performance_df,

    "03_primary_overall_bootstrap_ci.csv":
        overall_report_df,

    "04_primary_class_metrics.csv":
        class_report_df,

    "05_primary_confusion_matrix.csv":
        pooled_confusion_df,

    "06_dummy_baseline_comparison.csv":
        baseline_comparison,

    "07_probability_method_comparison.csv":
        probability_method_comparison_df,

    "08_imbalance_sensitivity_comparison.csv":
        imbalance_workflow_comparison,

    "09_imbalance_sensitivity_interpretation.csv":
        step5_secondary_interpretation_df,

    "10_workflow_stability_decision.csv":
        workflow_stability_decision_df,

    "11_temporal_sensitivity_overall.csv":
        temporal_overall_comparison_df,

    "12_temporal_sensitivity_difference_ci.csv":
        temporal_difference_ci_df,

    "13_permutation_importance.csv":
        permutation_summary_df,

    "14_discrimination_overall.csv":
        discrimination_overall_comparison_df,

    "15_discrimination_by_class.csv":
        discrimination_class_comparison_df,

    "16_conditional_label_permutation_check.csv":
        permutation_test_summary_df,

    "17_feature_reduction_comparison.csv":
        redundancy_overall_comparison_df,

    "18_feature_reduction_difference_ci.csv":
        redundancy_difference_ci_df,

    "19_primary_oof_predictions.csv":
        oof_predictions_df
}

for filename, dataframe in final_tables.items():
    dataframe.to_csv(
        FINAL_REPORT_DIR / filename,
        index=(
            filename
            == "05_primary_confusion_matrix.csv"
        )
    )


# ============================================================
# EXPORT COMBINED EXCEL WORKBOOK
# ============================================================

excel_status = "PASS"

try:
    with pd.ExcelWriter(
        FINAL_REPORT_DIR
        / "archetype_classifier_final_results.xlsx",
        engine="openpyxl"
    ) as writer:

        for filename, dataframe in final_tables.items():
            sheet_name = (
                Path(filename).stem[:31]
            )

            dataframe.to_excel(
                writer,
                sheet_name=sheet_name,
                index=(
                    filename
                    == "05_primary_confusion_matrix.csv"
                )
            )

except ImportError:
    excel_status = (
        "SKIPPED — openpyxl is not installed"
    )


# ============================================================
# THESIS FIGURE COLOURS
# ============================================================

# Sequential blue is appropriate for count magnitude:
# lighter = fewer households, darker = more households.
CONFUSION_CMAP = "Blues"

# Distinct colourblind-friendly colours for categorical metrics.
PRECISION_COLOR = "#4C78A8"       # blue
RECALL_COLOR = "#F58518"          # orange
F1_COLOR = "#54A24B"              # green

BALANCED_ACC_COLOR = "#4C78A8"    # blue
MACRO_F1_COLOR = "#E45756"        # muted red


# ============================================================
# FIGURE 1 — PRIMARY STEP 4 OOF CONFUSION MATRIX
# ============================================================

confusion_values = (
    pooled_confusion_df
    .to_numpy(dtype=int)
)

fig, ax = plt.subplots(
    figsize=(8.5, 7)
)

image = ax.imshow(
    confusion_values,
    cmap=CONFUSION_CMAP,
    interpolation="nearest",
    vmin=0,
    vmax=confusion_values.max()
)

ax.set_xticks(
    np.arange(
        len(ORDERED_CLUSTER_NAMES)
    )
)

ax.set_yticks(
    np.arange(
        len(ORDERED_CLUSTER_NAMES)
    )
)

ax.set_xticklabels(
    ORDERED_CLUSTER_NAMES,
    rotation=35,
    ha="right",
    fontsize=10
)

ax.set_yticklabels(
    ORDERED_CLUSTER_NAMES,
    fontsize=10
)

ax.set_xlabel(
    "Predicted archetype",
    fontsize=11
)

ax.set_ylabel(
    "True archetype",
    fontsize=11
)

ax.set_title(
    "Primary Nested Out-of-Fold Confusion Matrix",
    fontsize=13,
    pad=12
)


# ------------------------------------------------------------
# Adaptive annotation colour:
# - black text on light cells
# - white text on dark cells
# ------------------------------------------------------------

max_count = confusion_values.max()

text_threshold = (
    0.50 * max_count
)

for row_index in range(
    confusion_values.shape[0]
):
    for column_index in range(
        confusion_values.shape[1]
    ):

        cell_value = confusion_values[
            row_index,
            column_index
        ]

        annotation_colour = (
            "white"
            if cell_value > text_threshold
            else "black"
        )

        ax.text(
            column_index,
            row_index,
            f"{cell_value:,}",
            ha="center",
            va="center",
            color=annotation_colour,
            fontsize=10,
            fontweight="medium"
        )


# Thin white borders improve cell separation in print.
ax.set_xticks(
    np.arange(
        confusion_values.shape[1] + 1
    ) - 0.5,
    minor=True
)

ax.set_yticks(
    np.arange(
        confusion_values.shape[0] + 1
    ) - 0.5,
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=1.5
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)


colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

colorbar.set_label(
    "Household count",
    fontsize=10
)

fig.tight_layout()

fig.savefig(
    FINAL_REPORT_DIR
    / "figure_01_primary_oof_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


# ============================================================
# FIGURE 2 — CLASS-SPECIFIC PERFORMANCE
# ============================================================

class_plot_df = (
    pooled_class_metrics_df
    .set_index("class_name")
    .reindex(ORDERED_CLUSTER_NAMES)
)

ax = class_plot_df[
    [
        "precision",
        "recall",
        "f1_score"
    ]
].plot(
    kind="bar",
    figsize=(10, 6),
    color=[
        PRECISION_COLOR,
        RECALL_COLOR,
        F1_COLOR
    ],
    edgecolor="black",
    linewidth=0.5,
    width=0.72
)

ax.set_xlabel(
    "Archetype",
    fontsize=11
)

ax.set_ylabel(
    "Score",
    fontsize=11
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "Primary Nested Out-of-Fold Class Performance",
    fontsize=13,
    pad=12
)

ax.legend(
    labels=[
        "Precision",
        "Recall",
        "F1-score"
    ],
    title="Metric",
    frameon=True
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.30
)

ax.set_axisbelow(True)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FINAL_REPORT_DIR
    / "figure_02_primary_class_metrics.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()


# ============================================================
# FIGURE 3 — CLASSIFIER VERSUS DUMMY BASELINES
# ============================================================

baseline_plot_df = (
    baseline_comparison
    .set_index("evaluation")[
        [
            "balanced_accuracy",
            "macro_f1"
        ]
    ]
)

ax = baseline_plot_df.plot(
    kind="bar",
    figsize=(9, 6),
    color=[
        BALANCED_ACC_COLOR,
        MACRO_F1_COLOR
    ],
    edgecolor="black",
    linewidth=0.5,
    width=0.70
)

ax.set_xlabel(
    "Evaluation",
    fontsize=11
)

ax.set_ylabel(
    "Score",
    fontsize=11
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "Classifier Compared with Dummy Baselines",
    fontsize=13,
    pad=12
)

ax.legend(
    labels=[
        "Balanced accuracy",
        "Macro-F1"
    ],
    title="Metric",
    frameon=True
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.30
)

ax.set_axisbelow(True)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FINAL_REPORT_DIR
    / "figure_03_baseline_comparison.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()


# ============================================================
# SAVE FINAL REPORTING RULES
# ============================================================

reporting_rules = {
    "primary_evaluation_target":
        "Adaptive nested workflow-selection procedure",

    "primary_performance_source":
        "Step 4 nested out-of-fold predictions",

    "step4_metrics_directly_attributable_to_exact_fixed_prototype":
        False,

    "final_model_family":
        FINAL_WORKFLOW_MODEL_FAMILY,

    "final_predictor_count":
        len(FINAL_WORKFLOW_FEATURES),

    "predictor_decision":
        "Retain all 32 predictors",

    "matched_households":
        matched_household_count,

    "locked_cluster_households":
        locked_household_count,

    "fixed_prototype_performance_status":
        (
            "No independent unbiased estimate exists for the "
            "exact full-data-fitted prototype."
        ),

    "deployment_ready":
        False,

    "important_limitations": [
        "Target labels are K-Means pseudo-labels.",
        "Monthly totals do not directly observe time-of-use behaviour.",
        f"Only {daytime_matched_count:,} matched Daytime-Active households were available.",
        "No independent external validation cohort was available.",
        "The label-permutation result is conditional on the selected "
        "fixed workflow and is not a formal test of the complete "
        "model-selection procedure."
    ]
}

with open(
    FINAL_REPORT_DIR
    / "FINAL_REPORTING_RULES.json",
    "w",
    encoding="utf-8"
) as handle:
    json.dump(
        reporting_rules,
        handle,
        indent=4
    )


# ============================================================
# FINAL REPORTING SUMMARY
# ============================================================

print("=" * 80)
print("STEP 18 — FINAL THESIS REPORTING EXPORT")
print("=" * 80)

print(
    "Primary evaluation target: adaptive nested "
    "workflow-selection procedure"
)

print(
    "Primary performance source: Step 4 nested OOF"
)

print(
    "Final prototype:",
    FINAL_WORKFLOW_MODEL_FAMILY
)

print(
    "Predictor count:",
    len(FINAL_WORKFLOW_FEATURES)
)

print(
    "Matched households:",
    len(oof_predictions_df)
)

print(
    "CSV tables exported:",
    len(final_tables)
)

print(
    "Figures exported: 3"
)

print(
    "Excel export:",
    excel_status
)

print(
    "Operational deployment ready: NO"
)

print(
    "\nFinal reporting folder:"
)

print(
    FINAL_REPORT_DIR.resolve()
)


print(
    "\nEvaluation and prototype reporting decision:"
)

display(
    evaluation_and_prototype_decision_df.round(4)
)


print(
    "\nPrimary overall performance:"
)

display(
    overall_report_df.round(4)
)


print(
    "\nPrimary class-specific performance:"
)

display(
    class_report_df.round(4)
)

In [ ]:
# ============================================================
# RECREATE SAVED CONFUSION MATRIX WITH LARGER FONTS
# NO MODEL RETRAINING / NO PREVIOUS STEPS REQUIRED
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# STEP 1 — LOCATE SAVED CONFUSION MATRIX
# ============================================================

# Public-draft path: an authorised, private local analysis workspace.
# The confusion-matrix CSV is intentionally not bundled with the public repository.
# This cell must not search a person's Downloads folder or open a file picker.
PROJECT_DIR = Path.cwd()
csv_path = (
    PROJECT_DIR
    / "archetype_classifier_final_outputs"
    / "final_thesis_reporting"
    / "05_primary_confusion_matrix.csv"
)

if not csv_path.is_file():
    raise FileNotFoundError(
        "Required saved aggregate confusion matrix is not included in the "
        "public research draft. In an authorised private working copy, "
        "place 05_primary_confusion_matrix.csv under "
        "archetype_classifier_final_outputs/final_thesis_reporting/ "
        "relative to the working directory."
    )

FINAL_REPORT_DIR = csv_path.parent

print("Loaded confusion matrix from:")
print(csv_path)


# ============================================================
# STEP 2 — LOAD ORIGINAL CONFUSION MATRIX VALUES
# ============================================================

confusion_df = pd.read_csv(
    csv_path,
    index_col=0
)

confusion_values = confusion_df.to_numpy(dtype=int)

true_class_names = confusion_df.index.astype(str).tolist()
predicted_class_names = confusion_df.columns.astype(str).tolist()

if confusion_values.shape[0] != confusion_values.shape[1]:
    raise ValueError("Confusion matrix must be square.")

print("\nConfusion matrix:")
display(confusion_df)


# ============================================================
# STEP 3 — CREATE FIGURE WITH LARGER FONTS
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 8)
)

image = ax.imshow(
    confusion_values,
    cmap="Blues",
    interpolation="nearest",
    vmin=0,
    vmax=confusion_values.max()
)

# X-axis labels
ax.set_xticks(
    np.arange(len(predicted_class_names))
)

ax.set_xticklabels(
    predicted_class_names,
    rotation=30,
    ha="right",
    fontsize=15
)

# Y-axis labels
ax.set_yticks(
    np.arange(len(true_class_names))
)

ax.set_yticklabels(
    true_class_names,
    fontsize=15
)

# Axis titles
ax.set_xlabel(
    "Predicted archetype",
    fontsize=16,
    labelpad=12
)

ax.set_ylabel(
    "True archetype",
    fontsize=16,
    labelpad=12
)

# Figure title
ax.set_title(
    "Primary Nested Out-of-Fold Confusion Matrix",
    fontsize=17,
    pad=18
)


# ============================================================
# STEP 4 — LARGER HOUSEHOLD COUNT ANNOTATIONS
# ============================================================

max_count = confusion_values.max()

for i in range(confusion_values.shape[0]):
    for j in range(confusion_values.shape[1]):

        value = confusion_values[i, j]

        text_color = (
            "white"
            if value > 0.50 * max_count
            else "black"
        )

        ax.text(
            j,
            i,
            f"{value:,}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=17,
            fontweight="medium"
        )


# ============================================================
# STEP 5 — CELL BORDERS
# ============================================================

ax.set_xticks(
    np.arange(confusion_values.shape[1] + 1) - 0.5,
    minor=True
)

ax.set_yticks(
    np.arange(confusion_values.shape[0] + 1) - 0.5,
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=1.5
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)


# ============================================================
# STEP 6 — LARGER COLORBAR TEXT
# ============================================================

colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

colorbar.set_label(
    "Household count",
    fontsize=15
)

colorbar.ax.tick_params(
    labelsize=13
)


# ============================================================
# STEP 7 — SAVE NEW HIGH-RESOLUTION IMAGE
# ============================================================

fig.tight_layout()

new_figure_path = (
    FINAL_REPORT_DIR
    / "figure_01_primary_oof_confusion_matrix_large_font.png"
)

fig.savefig(
    new_figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


# ============================================================
# STEP 8 — FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("CONFUSION MATRIX FONT ENLARGEMENT COMPLETE")
print("=" * 65)

print("Original confusion matrix values: UNCHANGED")
print("Model retraining: NOT REQUIRED")
print("Previous notebook steps: NOT REQUIRED")
print("Figure resolution: 300 DPI")
print("Original PNG: PRESERVED")

print("\nNew figure saved at:")
print(new_figure_path.resolve())

print("\nFile exists:", new_figure_path.exists())

In [ ]:
# ============================================================
# STEP 19 — BUILD THE STREAMLIT RESEARCH-DEMO PACKAGE
# ============================================================
# This step packages the fitted pipeline together with the exact
# raw-monthly feature engineering used by the model.
#
# The package is suitable for a controlled thesis demonstration.
# It is not an externally validated operational deployment.
# ============================================================

from pathlib import Path
import importlib.metadata
import importlib.util
import shutil
import zipfile

import numpy as np
import pandas as pd
import sklearn


required_step19_objects = [
    "model_path",
    "prototype_dir",
    "dataset_all",
    "valid_monthly",
    "OUTPUT_DIR"
]

missing_step19_objects = [
    name
    for name in required_step19_objects
    if name not in globals()
]

if missing_step19_objects:
    raise RuntimeError(
        "Run Steps 1–18 before Step 19. Missing objects: "
        f"{missing_step19_objects}"
    )


RUNTIME_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\n\nMIN_VALID_MONTHS = 15\nMAX_VALID_MONTHS = 25\n\nCORE_FEATURES = [\n    "billing_median_kwh",\n    "billing_mean_kwh",\n    "billing_std_kwh",\n    "billing_iqr_kwh",\n    "billing_cv",\n    "billing_p10_kwh",\n    "billing_p90_kwh",\n    "billing_robust_range_kwh",\n    "recent_3_month_median_kwh",\n    "recent_6_month_median_kwh",\n    "recent_vs_overall_log_ratio",\n    "early_6_month_median_kwh",\n    "late_6_month_median_kwh",\n    "late_vs_early_log_ratio",\n    "monthly_trend_slope_log1p",\n    "median_abs_log_change",\n    "near_zero_month_rate",\n    "above_120_month_rate",\n]\n\nEXTENDED_ONLY_FEATURES = [\n    "billing_min_kwh",\n    "billing_max_kwh",\n    "billing_range_kwh",\n    "billing_log_std",\n    "mean_abs_monthly_change_kwh",\n    "maximum_monthly_increase_kwh",\n    "maximum_monthly_decrease_kwh",\n    "low_30_month_rate",\n    "above_90_month_rate",\n    "above_180_month_rate",\n    "above_240_month_rate",\n    "median_kwh_calendar_q1",\n    "median_kwh_calendar_q2",\n    "median_kwh_calendar_q3",\n    "median_kwh_calendar_q4",\n]\n\nAUDIT_COLUMNS = [\n    "months_observed",\n    "first_month",\n    "last_month",\n    "observation_span_months",\n]\n\n\ndef _normalise_household_id(series: pd.Series) -> pd.Series:\n    return series.astype(str).str.strip()\n\n\ndef _calculate_month_span(\n    start_series: pd.Series,\n    end_series: pd.Series,\n) -> pd.Series:\n    return (\n        (end_series.dt.year - start_series.dt.year) * 12\n        + (end_series.dt.month - start_series.dt.month)\n        + 1\n    )\n\n\ndef _calculate_household_trend(group: pd.DataFrame) -> float:\n    group = group.sort_values("month")\n\n    if len(group) < 3:\n        return np.nan\n\n    month_index = (\n        group["month"].dt.year.to_numpy() * 12\n        + group["month"].dt.month.to_numpy()\n    ).astype(float)\n\n    month_index = month_index - month_index.min()\n\n    if np.unique(month_index).size < 3:\n        return np.nan\n\n    log_consumption = np.log1p(\n        group["consumption"].to_numpy(dtype=float)\n    )\n\n    return float(\n        np.polyfit(month_index, log_consumption, 1)[0]\n    )\n\n\ndef prepare_monthly_records(\n    raw_monthly: pd.DataFrame,\n    minimum_months: int = MIN_VALID_MONTHS,\n    maximum_months: int = MAX_VALID_MONTHS,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """\n    Validate raw monthly billing rows and enforce the deployment\n    coverage contract used by the research prototype.\n\n    Required columns:\n    - household_ID\n    - month\n    - consumption\n\n    Households need at least 15 valid monthly observations. When more\n    than 25 are supplied, only the most recent 25 observations are used\n    to stay close to the training coverage range.\n    """\n\n    required = {"household_ID", "month", "consumption"}\n    missing = sorted(required - set(raw_monthly.columns))\n\n    if missing:\n        raise ValueError(\n            f"Missing required columns: {missing}. "\n            "Expected household_ID, month and consumption."\n        )\n\n    data = raw_monthly[\n        ["household_ID", "month", "consumption"]\n    ].copy()\n\n    data["household_ID"] = _normalise_household_id(\n        data["household_ID"]\n    )\n\n    invalid_ids = data["household_ID"].isin(\n        ["", "nan", "None", "<NA>"]\n    )\n\n    if invalid_ids.any():\n        raise ValueError(\n            "Blank or invalid household_ID values were found."\n        )\n\n    original_month = data["month"].copy()\n\n    data["month"] = pd.to_datetime(\n        data["month"],\n        errors="coerce",\n    )\n\n    invalid_months = data["month"].isna()\n\n    if invalid_months.any():\n        examples = (\n            original_month[invalid_months]\n            .head(10)\n            .astype(str)\n            .tolist()\n        )\n\n        raise ValueError(\n            "Invalid month values were found. "\n            f"Examples: {examples}"\n        )\n\n    data["month"] = (\n        data["month"]\n        .dt.to_period("M")\n        .dt.to_timestamp()\n    )\n\n    data["consumption"] = pd.to_numeric(\n        data["consumption"]\n        .astype(str)\n        .str.replace(",", "", regex=False),\n        errors="coerce",\n    )\n\n    missing_consumption_count = int(\n        data["consumption"].isna().sum()\n    )\n\n    data = data.dropna(\n        subset=["consumption"]\n    ).copy()\n\n    if (data["consumption"] < 0).any():\n        examples = (\n            data.loc[\n                data["consumption"] < 0,\n                ["household_ID", "month", "consumption"],\n            ]\n            .head(10)\n            .to_dict("records")\n        )\n\n        raise ValueError(\n            "Negative monthly consumption is not supported. "\n            f"Examples: {examples}"\n        )\n\n    duplicate_mask = data.duplicated(\n        subset=["household_ID", "month"],\n        keep=False,\n    )\n\n    if duplicate_mask.any():\n        examples = (\n            data.loc[\n                duplicate_mask,\n                ["household_ID", "month", "consumption"],\n            ]\n            .head(20)\n            .to_dict("records")\n        )\n\n        raise ValueError(\n            "Duplicate household-month rows were found. "\n            "Resolve them before prediction. "\n            f"Examples: {examples}"\n        )\n\n    data = data.sort_values(\n        ["household_ID", "month"]\n    ).reset_index(drop=True)\n\n    original_counts = (\n        data.groupby("household_ID")["month"]\n        .nunique()\n        .rename("months_supplied")\n    )\n\n    too_short = original_counts[\n        original_counts < minimum_months\n    ]\n\n    if not too_short.empty:\n        raise ValueError(\n            f"At least {minimum_months} valid monthly observations "\n            "are required per household. Insufficient households: "\n            f"{too_short.to_dict()}"\n        )\n\n    # Preserve the most recent months when a longer history is supplied.\n    data = (\n        data.groupby(\n            "household_ID",\n            group_keys=False,\n        )\n        .tail(maximum_months)\n        .reset_index(drop=True)\n    )\n\n    retained_counts = (\n        data.groupby("household_ID")["month"]\n        .nunique()\n        .rename("months_used")\n    )\n\n    audit = (\n        pd.concat(\n            [original_counts, retained_counts],\n            axis=1,\n        )\n        .reset_index()\n    )\n\n    audit["rows_dropped_for_missing_consumption_total"] = (\n        missing_consumption_count\n    )\n\n    audit["history_truncated_to_latest_25"] = (\n        audit["months_supplied"] > maximum_months\n    )\n\n    return data, audit\n\n\ndef build_household_billing_features(\n    validated_monthly: pd.DataFrame,\n) -> pd.DataFrame:\n    """Create one engineered feature row per household."""\n\n    df = validated_monthly.copy()\n\n    df = df.sort_values(\n        ["household_ID", "month"]\n    ).reset_index(drop=True)\n\n    df["log_consumption"] = np.log1p(\n        df["consumption"]\n    )\n\n    grouped = df.groupby(\n        "household_ID",\n        sort=False,\n    )\n\n    features = grouped.agg(\n        billing_median_kwh=("consumption", "median"),\n        billing_mean_kwh=("consumption", "mean"),\n        billing_std_kwh=("consumption", "std"),\n        billing_min_kwh=("consumption", "min"),\n        billing_max_kwh=("consumption", "max"),\n        billing_p10_kwh=(\n            "consumption",\n            lambda values: values.quantile(0.10),\n        ),\n        billing_p25_kwh=(\n            "consumption",\n            lambda values: values.quantile(0.25),\n        ),\n        billing_p75_kwh=(\n            "consumption",\n            lambda values: values.quantile(0.75),\n        ),\n        billing_p90_kwh=(\n            "consumption",\n            lambda values: values.quantile(0.90),\n        ),\n        billing_log_std=("log_consumption", "std"),\n        months_observed=("month", "nunique"),\n        first_month=("month", "min"),\n        last_month=("month", "max"),\n    ).reset_index()\n\n    features["billing_iqr_kwh"] = (\n        features["billing_p75_kwh"]\n        - features["billing_p25_kwh"]\n    )\n\n    features["billing_robust_range_kwh"] = (\n        features["billing_p90_kwh"]\n        - features["billing_p10_kwh"]\n    )\n\n    features["billing_range_kwh"] = (\n        features["billing_max_kwh"]\n        - features["billing_min_kwh"]\n    )\n\n    features["billing_cv"] = np.where(\n        features["billing_mean_kwh"] > 0,\n        features["billing_std_kwh"]\n        / features["billing_mean_kwh"],\n        np.nan,\n    )\n\n    features["observation_span_months"] = (\n        _calculate_month_span(\n            features["first_month"],\n            features["last_month"],\n        )\n    )\n\n    rate_source = df.assign(\n        near_zero_month=(df["consumption"] < 10).astype(float),\n        low_30_month=(df["consumption"] <= 30).astype(float),\n        above_90_month=(df["consumption"] > 90).astype(float),\n        above_120_month=(df["consumption"] > 120).astype(float),\n        above_180_month=(df["consumption"] > 180).astype(float),\n        above_240_month=(df["consumption"] > 240).astype(float),\n    )\n\n    rate_features = (\n        rate_source.groupby("household_ID")\n        .agg(\n            near_zero_month_rate=("near_zero_month", "mean"),\n            low_30_month_rate=("low_30_month", "mean"),\n            above_90_month_rate=("above_90_month", "mean"),\n            above_120_month_rate=("above_120_month", "mean"),\n            above_180_month_rate=("above_180_month", "mean"),\n            above_240_month_rate=("above_240_month", "mean"),\n        )\n        .reset_index()\n    )\n\n    features = features.merge(\n        rate_features,\n        on="household_ID",\n        how="left",\n        validate="one_to_one",\n    )\n\n    def window_median(\n        frame: pd.DataFrame,\n        count: int,\n        take_first: bool,\n        output_name: str,\n    ) -> pd.DataFrame:\n        grouped_rows = frame.groupby(\n            "household_ID",\n            group_keys=False,\n        )\n\n        selected = (\n            grouped_rows.head(count)\n            if take_first\n            else grouped_rows.tail(count)\n        )\n\n        return (\n            selected.groupby("household_ID")["consumption"]\n            .median()\n            .rename(output_name)\n            .reset_index()\n        )\n\n    early_6 = window_median(\n        df,\n        6,\n        True,\n        "early_6_month_median_kwh",\n    )\n\n    late_6 = window_median(\n        df,\n        6,\n        False,\n        "late_6_month_median_kwh",\n    )\n\n    recent_3 = window_median(\n        df,\n        3,\n        False,\n        "recent_3_month_median_kwh",\n    )\n\n    recent_6 = late_6.rename(\n        columns={\n            "late_6_month_median_kwh":\n            "recent_6_month_median_kwh"\n        }\n    )\n\n    for window_table in [\n        early_6,\n        late_6,\n        recent_3,\n        recent_6,\n    ]:\n        features = features.merge(\n            window_table,\n            on="household_ID",\n            how="left",\n            validate="one_to_one",\n        )\n\n    features["recent_vs_overall_log_ratio"] = (\n        np.log1p(\n            features["recent_3_month_median_kwh"]\n        )\n        - np.log1p(features["billing_median_kwh"])\n    )\n\n    features["late_vs_early_log_ratio"] = (\n        np.log1p(\n            features["late_6_month_median_kwh"]\n        )\n        - np.log1p(\n            features["early_6_month_median_kwh"]\n        )\n    )\n\n    df["calendar_month_number"] = (\n        df["month"].dt.year * 12\n        + df["month"].dt.month\n    )\n\n    df["month_gap"] = (\n        df.groupby("household_ID")[\n            "calendar_month_number"\n        ]\n        .diff()\n    )\n\n    raw_change = (\n        df.groupby("household_ID")["consumption"]\n        .diff()\n    )\n\n    raw_log_change = (\n        df.groupby("household_ID")["log_consumption"]\n        .diff()\n    )\n\n    df["monthly_change_kwh"] = raw_change.where(\n        df["month_gap"].eq(1)\n    )\n\n    df["log_change"] = raw_log_change.where(\n        df["month_gap"].eq(1)\n    )\n\n    dynamic_features = (\n        df.groupby("household_ID")\n        .agg(\n            mean_abs_monthly_change_kwh=(\n                "monthly_change_kwh",\n                lambda values: values.abs().mean(),\n            ),\n            maximum_monthly_increase_kwh=(\n                "monthly_change_kwh",\n                lambda values: (\n                    max(float(values.max(skipna=True)), 0.0)\n                    if values.notna().any()\n                    else np.nan\n                ),\n            ),\n            maximum_monthly_decrease_kwh=(\n                "monthly_change_kwh",\n                lambda values: (\n                    max(-float(values.min(skipna=True)), 0.0)\n                    if values.notna().any()\n                    else np.nan\n                ),\n            ),\n            median_abs_log_change=(\n                "log_change",\n                lambda values: values.abs().median(),\n            ),\n        )\n        .reset_index()\n    )\n\n    features = features.merge(\n        dynamic_features,\n        on="household_ID",\n        how="left",\n        validate="one_to_one",\n    )\n\n    df["calendar_quarter"] = (\n        (df["month"].dt.month - 1) // 3 + 1\n    )\n\n    quarter_features = (\n        df.pivot_table(\n            index="household_ID",\n            columns="calendar_quarter",\n            values="consumption",\n            aggfunc="median",\n        )\n        .rename(\n            columns={\n                1: "median_kwh_calendar_q1",\n                2: "median_kwh_calendar_q2",\n                3: "median_kwh_calendar_q3",\n                4: "median_kwh_calendar_q4",\n            }\n        )\n        .reset_index()\n    )\n\n    quarter_columns = [\n        "median_kwh_calendar_q1",\n        "median_kwh_calendar_q2",\n        "median_kwh_calendar_q3",\n        "median_kwh_calendar_q4",\n    ]\n\n    for column in quarter_columns:\n        if column not in quarter_features.columns:\n            quarter_features[column] = np.nan\n\n    features = features.merge(\n        quarter_features[\n            ["household_ID"] + quarter_columns\n        ],\n        on="household_ID",\n        how="left",\n        validate="one_to_one",\n    )\n\n    trend_features = pd.DataFrame(\n        [\n            {\n                "household_ID": household_id,\n                "monthly_trend_slope_log1p":\n                    _calculate_household_trend(group),\n            }\n            for household_id, group\n            in df.groupby("household_ID")\n        ]\n    )\n\n    features = features.merge(\n        trend_features,\n        on="household_ID",\n        how="left",\n        validate="one_to_one",\n    )\n\n    features = features.replace(\n        [np.inf, -np.inf],\n        np.nan,\n    )\n\n    return features\n\n\ndef load_model_package(\n    package_path: str | Path,\n) -> dict:\n    package = joblib.load(package_path)\n\n    required = {\n        "estimator",\n        "feature_order",\n        "index_to_class",\n        "workflow_model_family",\n    }\n\n    missing = sorted(required - set(package))\n\n    if missing:\n        raise ValueError(\n            f"Invalid model package. Missing keys: {missing}"\n        )\n\n    return package\n\n\ndef predict_from_monthly_records(\n    raw_monthly: pd.DataFrame,\n    package_path: str | Path,\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """\n    Validate raw monthly rows, engineer household features and\n    predict the locked archetype pseudo-label.\n    """\n\n    package = load_model_package(package_path)\n\n    validated, coverage_audit = prepare_monthly_records(\n        raw_monthly\n    )\n\n    features = build_household_billing_features(\n        validated\n    )\n\n    feature_order = list(package["feature_order"])\n\n    missing_features = [\n        feature\n        for feature in feature_order\n        if feature not in features.columns\n    ]\n\n    if missing_features:\n        raise ValueError(\n            "Feature-engineering output is missing model "\n            f"predictors: {missing_features}"\n        )\n\n    X = (\n        features[feature_order]\n        .apply(pd.to_numeric, errors="coerce")\n        .replace([np.inf, -np.inf], np.nan)\n    )\n\n    if X.isna().all(axis=1).any():\n        bad_households = features.loc[\n            X.isna().all(axis=1),\n            "household_ID",\n        ].tolist()\n\n        raise ValueError(\n            "All model predictors are missing for households: "\n            f"{bad_households}"\n        )\n\n    estimator = package["estimator"]\n    predicted_indices = estimator.predict(X).astype(int)\n    probabilities = estimator.predict_proba(X)\n\n    index_to_class = {\n        int(key): value\n        for key, value in package["index_to_class"].items()\n    }\n\n    estimator_classes = np.asarray(\n        estimator.classes_,\n        dtype=int,\n    )\n\n    if probabilities.shape[1] != len(estimator_classes):\n        raise RuntimeError(\n            "Probability columns do not match estimator classes."\n        )\n\n    order = np.argsort(\n        probabilities,\n        axis=1,\n    )[:, ::-1]\n\n    top_1_columns = order[:, 0]\n    top_2_columns = order[:, 1]\n\n    results = pd.DataFrame({\n        "household_ID": features["household_ID"],\n        "predicted_class_index": predicted_indices,\n        "predicted_archetype": [\n            index_to_class[index]\n            for index in predicted_indices\n        ],\n        "top_model_score": probabilities[\n            np.arange(len(probabilities)),\n            top_1_columns,\n        ],\n        "second_archetype": [\n            index_to_class[\n                int(estimator_classes[column])\n            ]\n            for column in top_2_columns\n        ],\n        "second_model_score": probabilities[\n            np.arange(len(probabilities)),\n            top_2_columns,\n        ],\n    })\n\n    for probability_column, class_index in enumerate(\n        estimator_classes\n    ):\n        safe_name = (\n            index_to_class[int(class_index)]\n            .lower()\n            .replace("-", "_")\n            .replace(" ", "_")\n            .replace("/", "_")\n        )\n\n        results[\n            f"model_score_{safe_name}"\n        ] = probabilities[:, probability_column]\n\n    results["score_interpretation"] = (\n        "Raw uncalibrated relative model score; "\n        "not a validated probability of correctness."\n    )\n\n    results = results.merge(\n        coverage_audit,\n        on="household_ID",\n        how="left",\n        validate="one_to_one",\n    )\n\n    return results, features, coverage_audit\n'
STREAMLIT_APP_SOURCE = 'from pathlib import Path\n\nimport pandas as pd\nimport streamlit as st\n\nfrom archetype_runtime import (\n    MIN_VALID_MONTHS,\n    MAX_VALID_MONTHS,\n    predict_from_monthly_records,\n)\n\n\nAPP_DIR = Path(__file__).resolve().parent\nMODEL_PATH = APP_DIR / "archetype_classifier_bundle.joblib"\n\n\nst.set_page_config(\n    page_title="Household Electricity Archetype Transfer",\n    page_icon="⚡",\n    layout="wide",\n)\n\nst.title("Household Electricity Archetype Transfer")\nst.caption(\n    "Research demonstration using monthly billing data to approximate "\n    "smart-meter-derived archetype pseudo-labels."\n)\n\nst.warning(\n    "This is a supplementary research prototype. It does not discover "\n    "new archetypes, forecast future demand, or replace the 15-minute "\n    "smart-meter clustering pipeline."\n)\n\nwith st.expander("Required CSV format", expanded=True):\n    st.markdown(\n        f"""\nUpload a CSV with exactly these columns:\n\n- `household_ID`\n- `month`\n- `consumption`\n\nEach household needs at least **{MIN_VALID_MONTHS} valid months**.\nWhen more than **{MAX_VALID_MONTHS} months** are supplied, only the\nlatest {MAX_VALID_MONTHS} are used.\n\nDuplicate household-month rows, invalid dates, negative consumption,\nor insufficient history are rejected rather than silently corrected.\n"""\n    )\n\n    template = pd.DataFrame({\n        "household_ID": ["DEMO_001"] * MIN_VALID_MONTHS,\n        "month": pd.date_range(\n            "2023-08-01",\n            periods=MIN_VALID_MONTHS,\n            freq="MS",\n        ).strftime("%Y-%m-%d"),\n        "consumption": [\n            92, 95, 101, 98, 106,\n            111, 109, 115, 118, 121,\n            117, 123, 128, 126, 131,\n        ],\n    })\n\n    st.download_button(\n        "Download input template",\n        template.to_csv(index=False).encode("utf-8"),\n        file_name="monthly_billing_input_template.csv",\n        mime="text/csv",\n    )\n\nif not MODEL_PATH.exists():\n    st.error(\n        "Model bundle not found. Run the final notebook through "\n        "Step 19 and place archetype_classifier_bundle.joblib "\n        "beside this app."\n    )\n    st.stop()\n\nuploaded_file = st.file_uploader(\n    "Upload monthly billing CSV",\n    type=["csv"],\n)\n\nif uploaded_file is not None:\n    try:\n        raw_monthly = pd.read_csv(uploaded_file)\n\n        predictions, engineered_features, coverage_audit = (\n            predict_from_monthly_records(\n                raw_monthly=raw_monthly,\n                package_path=MODEL_PATH,\n            )\n        )\n\n    except Exception as error:\n        st.error(str(error))\n        st.stop()\n\n    st.success(\n        f"Predictions generated for {len(predictions):,} household(s)."\n    )\n\n    st.subheader("Coverage audit")\n    st.dataframe(\n        coverage_audit,\n        use_container_width=True,\n    )\n\n    st.subheader("Predicted archetypes")\n    st.dataframe(\n        predictions,\n        use_container_width=True,\n    )\n\n    st.info(\n        "The model scores are raw, uncalibrated relative scores. "\n        "They must not be interpreted as validated confidence "\n        "probabilities."\n    )\n\n    st.download_button(\n        "Download predictions",\n        predictions.to_csv(index=False).encode("utf-8"),\n        file_name="archetype_predictions.csv",\n        mime="text/csv",\n    )\n\n    with st.expander("View engineered household features"):\n        st.dataframe(\n            engineered_features,\n            use_container_width=True,\n        )\n\n        st.download_button(\n            "Download engineered features",\n            engineered_features.to_csv(index=False).encode("utf-8"),\n            file_name="engineered_household_features.csv",\n            mime="text/csv",\n        )\n\nst.divider()\nst.caption(\n    "Interpretation: Peak-Exposed, Typical Household, Heavy Continuous "\n    "and Daytime-Active are locked K-Means pseudo-labels derived from "\n    "the smart-meter study. Monthly billing totals only approximate "\n    "these time-of-use patterns."\n)\n'
STREAMLIT_README_SOURCE = '# Streamlit Archetype-Transfer Research Demo\n\n## 1. Train and package the model\n\nPlace these files beside the notebook:\n\n- `clean_fm_clusters.pkl`\n- `monthly_consumption.csv`\n\nRun the notebook from Step 1 through Step 19. Step 19 creates a\n`streamlit_archetype_app` folder and a ZIP archive containing the\ntrained `archetype_classifier_bundle.joblib`.\n\n## 2. Install dependencies\n\n```bash\npython -m venv venv\nvenv\\Scripts\\activate\npip install -r requirements.txt\n```\n\nOn macOS/Linux:\n\n```bash\npython -m venv venv\nsource venv/bin/activate\npip install -r requirements.txt\n```\n\n## 3. Run the app\n\n```bash\nstreamlit run app.py\n```\n\n## Input\n\nUpload a CSV with:\n\n- `household_ID`\n- `month`\n- `consumption`\n\nAt least 15 valid months are required per household. At most the latest\n25 months are used, matching the coverage range observed during model\ndevelopment.\n\n## Methodological contract\n\n- Logistic Regression is scaled with `RobustScaler` inside its pipeline.\n- Random Forest and Extra Trees are not scaled because scaling does not\n  affect tree split ordering.\n- Median imputation is fitted inside every training fold and is included\n  in the saved final pipeline.\n- Primary performance comes from 5x4 nested stratified cross-validation.\n- The final prototype is fitted on all 1,146 matched households only after\n  evaluation.\n- There is no independent external test cohort for the exact saved model.\n- Output scores are uncalibrated relative model scores, not validated\n  probabilities of correctness.\n\n## Intended use\n\nThis package is suitable for a thesis demonstration and controlled research\nprototype. It is not validated for operational tariff, welfare, credit, or\npolicy decisions.\n'
STREAMLIT_REQUIREMENTS_SOURCE = "\n".join([
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={importlib.metadata.version('joblib')}",
    "streamlit>=1.35,<2"
]) + "\n"


deployment_dir = (
    Path(OUTPUT_DIR)
    / "streamlit_archetype_app"
)

deployment_dir.mkdir(
    parents=True,
    exist_ok=True
)

deployment_model_path = (
    deployment_dir
    / "archetype_classifier_bundle.joblib"
)

shutil.copy2(
    model_path,
    deployment_model_path
)

runtime_path = (
    deployment_dir
    / "archetype_runtime.py"
)

app_path = (
    deployment_dir
    / "app.py"
)

readme_path = (
    deployment_dir
    / "README.md"
)

deployment_requirements_path = (
    deployment_dir
    / "requirements.txt"
)

runtime_path.write_text(
    RUNTIME_SOURCE,
    encoding="utf-8"
)

app_path.write_text(
    STREAMLIT_APP_SOURCE,
    encoding="utf-8"
)

readme_path.write_text(
    STREAMLIT_README_SOURCE,
    encoding="utf-8"
)

deployment_requirements_path.write_text(
    STREAMLIT_REQUIREMENTS_SOURCE,
    encoding="utf-8"
)


# Create a synthetic input template with the validated minimum
# monthly history. It contains no research-household data.
template_months = pd.date_range(
    "2023-08-01",
    periods=15,
    freq="MS"
)

template_df = pd.DataFrame({
    "household_ID": ["DEMO_001"] * 15,
    "month": template_months.strftime("%Y-%m-%d"),
    "consumption": [
        92, 95, 101, 98, 106,
        111, 109, 115, 118, 121,
        117, 123, 128, 126, 131
    ]
})

template_path = (
    deployment_dir
    / "monthly_billing_input_template.csv"
)

template_df.to_csv(
    template_path,
    index=False
)


# Import the written runtime module and run an end-to-end smoke test
# using two training households. This checks raw input -> engineered
# features -> saved pipeline -> predictions.
module_spec = importlib.util.spec_from_file_location(
    "archetype_runtime",
    runtime_path
)

runtime_module = importlib.util.module_from_spec(
    module_spec
)

module_spec.loader.exec_module(
    runtime_module
)

smoke_households = (
    dataset_all["household_ID"]
    .astype(str)
    .head(2)
    .tolist()
)

smoke_monthly = valid_monthly[
    valid_monthly["household_ID"]
    .astype(str)
    .isin(smoke_households)
].copy()

(
    smoke_predictions,
    smoke_features,
    smoke_coverage
) = runtime_module.predict_from_monthly_records(
    raw_monthly=smoke_monthly,
    package_path=deployment_model_path
)

if len(smoke_predictions) != len(smoke_households):
    raise RuntimeError(
        "Streamlit runtime smoke test returned an unexpected "
        "number of household predictions."
    )

smoke_predictions.to_csv(
    deployment_dir
    / "STREAMLIT_SMOKE_TEST_PREDICTIONS.csv",
    index=False
)

smoke_coverage.to_csv(
    deployment_dir
    / "STREAMLIT_SMOKE_TEST_COVERAGE.csv",
    index=False
)


# Copy the research documentation produced in Step 17.
for optional_name in [
    "MODEL_CARD.md",
    "archetype_transfer_model_metadata.json",
    "archetype_transfer_feature_schema.json",
    "PACKAGE_AUDIT.json",
    "FILE_MANIFEST_SHA256.csv"
]:
    source_file = (
        prototype_dir
        / optional_name
    )

    if source_file.exists():
        shutil.copy2(
            source_file,
            deployment_dir
            / optional_name
        )


# Create one portable ZIP archive.
deployment_zip_path = (
    Path(OUTPUT_DIR)
    / "streamlit_archetype_app.zip"
)

with zipfile.ZipFile(
    deployment_zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for file_path in sorted(
        deployment_dir.rglob("*")
    ):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=file_path.relative_to(
                    deployment_dir
                )
            )


print("=" * 80)
print("STEP 19 — STREAMLIT PACKAGE")
print("=" * 80)
print("Model bundle:", deployment_model_path)
print("Runtime module:", runtime_path)
print("Streamlit app:", app_path)
print("Input template:", template_path)
print("Smoke test: PASS")
print("Portable ZIP:", deployment_zip_path)
print(
    "Research-demo readiness: YES | "
    "Operational deployment readiness: NO"
)


# STEP 20 — SUPPLEMENTARY LEAKAGE-SAFE ADVANCED MODEL COMPARISON

This section is appended without changing any code in Steps 1–19.

It tests whether additional, methodologically valid tabular-learning techniques improve four-archetype transfer from monthly billing features:

1. **Original balanced Random Forest** — direct reference using the existing Step 4 search space.
2. **Balanced Random Forest** — balances each tree's training sample internally.
3. **Balanced Histogram Gradient Boosting** — uses class-balanced loss weighting and regularised boosting.
4. **Balanced XGBoost** — uses class-balanced training weights and regularised gradient boosting when `xgboost` is available.

Methodological safeguards:

- The same household-level 5-fold outer and 4-fold inner stratified cross-validation design is retained.
- Feature-set and hyperparameter selection occurs only within each outer-training partition.
- Median imputation is fitted only inside training folds.
- No synthetic or duplicated observations enter validation or outer-test folds.
- Macro F1 is the primary metric, followed by balanced accuracy.
- Accuracy remains supplementary because the largest class dominates the dataset.
- Step 4 remains the primary thesis result. Step 20 is a supplementary extension and cannot replace the locked workflow solely because one added model scores higher on these same data.

In [ ]:
# ============================================================
# STEP 20 — SUPPLEMENTARY LEAKAGE-SAFE ADVANCED MODEL COMPARISON
# ============================================================
# This section was appended without changing Steps 1–19.
#
# Purpose:
# 1. Test additional valid classifiers for imbalanced tabular data.
# 2. Tune every model and feature set inside the inner CV only.
# 3. Evaluate each model family on untouched outer folds.
# 4. Compare each new model with the original Step 4 OOF result.
#
# Important:
# - Macro-F1 is primary.
# - Balanced accuracy is the first supporting metric.
# - Accuracy is supplementary only.
# - Step 4 remains the primary thesis result.
# - Step 20 is an exploratory extension, not post-hoc replacement.
# ============================================================

import importlib.util
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight


# ------------------------------------------------------------
# 1. Confirm required objects from the existing notebook
# ------------------------------------------------------------

REQUIRED_STEP20_OBJECTS = [
    "dataset_all",
    "nested_df",
    "X_nested_source",
    "y_nested",
    "MODEL_FEATURE_SETS",
    "ORDERED_CLUSTER_NAMES",
    "CLASS_TO_INDEX",
    "INDEX_TO_CLASS",
    "FIXED_CLASS_INDICES",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "RANDOM_STATE",
    "OUTPUT_DIR",
    "SCORING",
    "build_candidate_pipeline",
    "align_class_probabilities",
    "calculate_multiclass_metrics",
    "serialise_parameters",
    "pooled_performance_df",
    "oof_predictions_df"
]

missing_step20_objects = [
    object_name
    for object_name in REQUIRED_STEP20_OBJECTS
    if object_name not in globals()
]

if missing_step20_objects:
    raise RuntimeError(
        "Run Steps 1–19 before Step 20. Missing objects: "
        f"{missing_step20_objects}"
    )

if len(dataset_all) != len(y_nested):
    raise RuntimeError(
        "Step 20 detected a mismatch between predictors and labels."
    )

if nested_df["household_ID"].duplicated().any():
    raise RuntimeError(
        "Step 20 requires one row per household."
    )

minimum_class_count_step20 = int(
    pd.Series(y_nested).value_counts().min()
)

if minimum_class_count_step20 < OUTER_FOLDS:
    raise RuntimeError(
        "The smallest class is insufficient for the requested "
        "outer-fold count."
    )


# ------------------------------------------------------------
# 2. Optional balanced XGBoost wrapper
# ------------------------------------------------------------

XGBOOST_AVAILABLE_STEP20 = (
    importlib.util.find_spec("xgboost") is not None
)


class BalancedXGBClassifier(
    ClassifierMixin,
    BaseEstimator
):
    """
    Multiclass XGBoost wrapper that calculates balanced sample
    weights inside every fit call.

    Because GridSearchCV calls fit separately for each inner
    training fold, the class weights are recalculated from that
    fold only. No validation labels influence weighting.
    """

    def __init__(
        self,
        n_estimators=250,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=8.0,
        subsample=0.80,
        colsample_bytree=0.80,
        gamma=0.0,
        reg_alpha=0.10,
        reg_lambda=8.0,
        random_state=42,
        n_jobs=1
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.min_child_weight = min_child_weight
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.gamma = gamma
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y):
        if not XGBOOST_AVAILABLE_STEP20:
            raise ImportError(
                "xgboost is not installed. Install xgboost or "
                "allow Step 20 to run the other model families."
            )

        from xgboost import XGBClassifier

        y_array = np.asarray(y, dtype=int)
        observed_classes = np.unique(y_array)

        expected_classes = np.arange(
            observed_classes.size,
            dtype=int
        )

        if not np.array_equal(
            observed_classes,
            expected_classes
        ):
            raise ValueError(
                "BalancedXGBClassifier requires consecutive "
                "class indices starting from zero."
            )

        balanced_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_array
        )

        self.model_ = XGBClassifier(
            objective="multi:softprob",
            num_class=observed_classes.size,
            eval_metric="mlogloss",
            tree_method="hist",
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            min_child_weight=self.min_child_weight,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            gamma=self.gamma,
            reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
            verbosity=0
        )

        self.model_.fit(
            X,
            y_array,
            sample_weight=balanced_weights,
            verbose=False
        )

        self.classes_ = observed_classes
        self.n_features_in_ = int(
            np.asarray(X).shape[1]
        )

        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        probabilities = np.asarray(
            self.model_.predict_proba(X),
            dtype=float
        )

        row_sums = probabilities.sum(
            axis=1,
            keepdims=True
        )

        if (
            np.any(~np.isfinite(probabilities))
            or np.any(row_sums <= 0)
        ):
            raise RuntimeError(
                "XGBoost returned invalid class probabilities."
            )

        return probabilities / row_sums


# ------------------------------------------------------------
# 3. Advanced candidate builders
# ------------------------------------------------------------

def build_step20_balanced_random_forest():
    try:
        from imblearn.ensemble import (
            BalancedRandomForestClassifier
        )
    except ImportError as error:
        raise ImportError(
            "Step 20 requires imbalanced-learn, which was already "
            "used by Step 5 for oversampling and SMOTE."
        ) from error

    estimator = BalancedRandomForestClassifier(
        n_estimators=400,
        sampling_strategy="all",
        replacement=True,
        bootstrap=False,
        class_weight=None,
        random_state=RANDOM_STATE,
        n_jobs=1
    )

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            estimator
        )
    ])

    parameter_grid = [
        {
            "model__max_depth": [3],
            "model__min_samples_leaf": [10],
            "model__min_samples_split": [20],
            "model__max_features": ["sqrt"]
        },
        {
            "model__max_depth": [4],
            "model__min_samples_leaf": [8],
            "model__min_samples_split": [16],
            "model__max_features": ["sqrt"]
        },
        {
            "model__max_depth": [5],
            "model__min_samples_leaf": [6],
            "model__min_samples_split": [12],
            "model__max_features": [0.50]
        },
        {
            "model__max_depth": [6],
            "model__min_samples_leaf": [5],
            "model__min_samples_split": [10],
            "model__max_features": [0.50]
        },
        {
            "model__max_depth": [8],
            "model__min_samples_leaf": [8],
            "model__min_samples_split": [16],
            "model__max_features": [0.30]
        }
    ]

    return pipeline, parameter_grid


def build_step20_hist_gradient_boosting():
    estimator = HistGradientBoostingClassifier(
        loss="log_loss",
        class_weight="balanced",
        early_stopping=False,
        random_state=RANDOM_STATE
    )

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            estimator
        )
    ])

    parameter_grid = [
        {
            "model__learning_rate": [0.03],
            "model__max_iter": [200],
            "model__max_leaf_nodes": [7],
            "model__max_depth": [3],
            "model__min_samples_leaf": [20],
            "model__l2_regularization": [5.0]
        },
        {
            "model__learning_rate": [0.05],
            "model__max_iter": [150],
            "model__max_leaf_nodes": [7],
            "model__max_depth": [3],
            "model__min_samples_leaf": [20],
            "model__l2_regularization": [2.0]
        },
        {
            "model__learning_rate": [0.03],
            "model__max_iter": [250],
            "model__max_leaf_nodes": [15],
            "model__max_depth": [4],
            "model__min_samples_leaf": [25],
            "model__l2_regularization": [5.0]
        },
        {
            "model__learning_rate": [0.05],
            "model__max_iter": [200],
            "model__max_leaf_nodes": [15],
            "model__max_depth": [4],
            "model__min_samples_leaf": [30],
            "model__l2_regularization": [8.0]
        },
        {
            "model__learning_rate": [0.02],
            "model__max_iter": [300],
            "model__max_leaf_nodes": [7],
            "model__max_depth": [3],
            "model__min_samples_leaf": [30],
            "model__l2_regularization": [10.0]
        }
    ]

    return pipeline, parameter_grid


def build_step20_xgboost():
    estimator = BalancedXGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1
    )

    pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            estimator
        )
    ])

    parameter_grid = [
        {
            "model__n_estimators": [200],
            "model__max_depth": [2],
            "model__learning_rate": [0.03],
            "model__min_child_weight": [8.0],
            "model__subsample": [0.80],
            "model__colsample_bytree": [0.80],
            "model__gamma": [0.0],
            "model__reg_alpha": [0.10],
            "model__reg_lambda": [8.0]
        },
        {
            "model__n_estimators": [300],
            "model__max_depth": [3],
            "model__learning_rate": [0.03],
            "model__min_child_weight": [10.0],
            "model__subsample": [0.80],
            "model__colsample_bytree": [0.80],
            "model__gamma": [0.10],
            "model__reg_alpha": [0.20],
            "model__reg_lambda": [10.0]
        },
        {
            "model__n_estimators": [200],
            "model__max_depth": [3],
            "model__learning_rate": [0.05],
            "model__min_child_weight": [12.0],
            "model__subsample": [0.75],
            "model__colsample_bytree": [0.75],
            "model__gamma": [0.20],
            "model__reg_alpha": [0.25],
            "model__reg_lambda": [12.0]
        },
        {
            "model__n_estimators": [400],
            "model__max_depth": [2],
            "model__learning_rate": [0.02],
            "model__min_child_weight": [10.0],
            "model__subsample": [0.85],
            "model__colsample_bytree": [0.85],
            "model__gamma": [0.10],
            "model__reg_alpha": [0.10],
            "model__reg_lambda": [10.0]
        }
    ]

    return pipeline, parameter_grid


def build_step20_reference_random_forest():
    """
    Reuse the original balanced Random Forest implementation and
    search space from Step 4 as the direct nested-CV reference.
    """

    return build_candidate_pipeline(
        model_family="Random Forest",
        class_weight="balanced"
    )


STEP20_MODEL_BUILDERS = {
    "Original balanced Random Forest": (
        build_step20_reference_random_forest
    ),
    "Balanced Random Forest": (
        build_step20_balanced_random_forest
    ),
    "Balanced HistGradientBoosting": (
        build_step20_hist_gradient_boosting
    )
}

if XGBOOST_AVAILABLE_STEP20:
    STEP20_MODEL_BUILDERS[
        "Balanced XGBoost"
    ] = build_step20_xgboost
else:
    warnings.warn(
        "xgboost is unavailable. Step 20 will continue with "
        "the other valid candidate models."
    )


# ------------------------------------------------------------
# 4. Inner-CV selection rule
# ------------------------------------------------------------

def select_step20_inner_candidate(
    cv_results
):
    """
    Select using inner validation only.

    Ordering:
    1. highest mean validation macro-F1;
    2. highest mean validation balanced accuracy;
    3. highest mean validation weighted-F1;
    4. highest mean validation accuracy;
    5. smallest train-validation macro-F1 gap;
    6. lowest macro-F1 standard deviation.
    """

    results = cv_results.copy()

    results[
        "macro_f1_train_validation_gap"
    ] = (
        results["mean_train_macro_f1"]
        - results["mean_test_macro_f1"]
    )

    return (
        results
        .sort_values(
            [
                "mean_test_macro_f1",
                "mean_test_balanced_accuracy",
                "mean_test_weighted_f1",
                "mean_test_accuracy",
                "macro_f1_train_validation_gap",
                "std_test_macro_f1"
            ],
            ascending=[
                False,
                False,
                False,
                False,
                True,
                True
            ]
        )
        .iloc[0]
    )


# ------------------------------------------------------------
# 5. Nested evaluation for every model family
# ------------------------------------------------------------

step20_outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

step20_inner_records = []
step20_outer_selection_records = []
step20_outer_metric_records = []
step20_oof_records = []

step20_start_time = time.time()

for model_position, (
    model_family,
    model_builder
) in enumerate(
    STEP20_MODEL_BUILDERS.items(),
    start=1
):
    print("\n" + "=" * 80)
    print(
        f"STEP 20 MODEL {model_position}/"
        f"{len(STEP20_MODEL_BUILDERS)}: "
        f"{model_family}"
    )
    print("=" * 80)

    for outer_fold, (
        outer_train_indices,
        outer_test_indices
    ) in enumerate(
        step20_outer_cv.split(
            X_nested_source,
            y_nested
        ),
        start=1
    ):
        outer_train_source = (
            X_nested_source.iloc[
                outer_train_indices
            ]
            .reset_index(drop=True)
        )

        outer_test_source = (
            X_nested_source.iloc[
                outer_test_indices
            ]
            .reset_index(drop=True)
        )

        y_outer_train = (
            y_nested.iloc[
                outer_train_indices
            ]
            .reset_index(drop=True)
        )

        y_outer_test = (
            y_nested.iloc[
                outer_test_indices
            ]
            .reset_index(drop=True)
        )

        outer_test_metadata = (
            nested_df.iloc[
                outer_test_indices
            ][
                [
                    "household_ID",
                    "cluster_name"
                ]
            ]
            .reset_index(drop=True)
        )

        inner_cv = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=(
                RANDOM_STATE + outer_fold
            )
        )

        fold_candidates = []

        for feature_set_name, feature_columns in (
            MODEL_FEATURE_SETS.items()
        ):
            X_outer_train = (
                outer_train_source[
                    feature_columns
                ]
            )

            pipeline, parameter_grid = model_builder()

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=parameter_grid,
                scoring=SCORING,
                refit=False,
                cv=inner_cv,
                n_jobs=-1,
                pre_dispatch="2*n_jobs",
                return_train_score=True,
                error_score="raise"
            )

            search_start_time = time.time()

            search.fit(
                X_outer_train,
                y_outer_train
            )

            search_runtime_seconds = (
                time.time()
                - search_start_time
            )

            cv_results = pd.DataFrame(
                search.cv_results_
            )

            selected_result = (
                select_step20_inner_candidate(
                    cv_results
                )
            )

            selected_parameters = dict(
                selected_result["params"]
            )

            candidate_record = {
                "model_family": model_family,
                "outer_fold": outer_fold,
                "feature_set": feature_set_name,
                "feature_count": len(feature_columns),
                "selected_parameters": (
                    serialise_parameters(
                        selected_parameters
                    )
                ),
                "mean_inner_macro_f1": float(
                    selected_result[
                        "mean_test_macro_f1"
                    ]
                ),
                "std_inner_macro_f1": float(
                    selected_result[
                        "std_test_macro_f1"
                    ]
                ),
                "mean_inner_balanced_accuracy": float(
                    selected_result[
                        "mean_test_balanced_accuracy"
                    ]
                ),
                "mean_inner_weighted_f1": float(
                    selected_result[
                        "mean_test_weighted_f1"
                    ]
                ),
                "mean_inner_accuracy": float(
                    selected_result[
                        "mean_test_accuracy"
                    ]
                ),
                "mean_inner_train_macro_f1": float(
                    selected_result[
                        "mean_train_macro_f1"
                    ]
                ),
                "inner_macro_f1_gap": float(
                    selected_result[
                        "macro_f1_train_validation_gap"
                    ]
                ),
                "search_runtime_seconds": float(
                    search_runtime_seconds
                )
            }

            step20_inner_records.append(
                candidate_record
            )

            fold_candidates.append({
                **candidate_record,
                "feature_columns": list(
                    feature_columns
                ),
                "pipeline_template": pipeline,
                "best_parameters_dict": (
                    selected_parameters
                )
            })

            print(
                f"Outer fold {outer_fold} | "
                f"{feature_set_name}: "
                f"inner macro-F1="
                f"{candidate_record['mean_inner_macro_f1']:.4f}, "
                f"balanced accuracy="
                f"{candidate_record['mean_inner_balanced_accuracy']:.4f}"
            )

        selected_candidate = (
            pd.DataFrame([
                {
                    "candidate_index": candidate_index,
                    "mean_inner_macro_f1": candidate[
                        "mean_inner_macro_f1"
                    ],
                    "mean_inner_balanced_accuracy": candidate[
                        "mean_inner_balanced_accuracy"
                    ],
                    "mean_inner_weighted_f1": candidate[
                        "mean_inner_weighted_f1"
                    ],
                    "mean_inner_accuracy": candidate[
                        "mean_inner_accuracy"
                    ],
                    "inner_macro_f1_gap": candidate[
                        "inner_macro_f1_gap"
                    ],
                    "std_inner_macro_f1": candidate[
                        "std_inner_macro_f1"
                    ]
                }
                for candidate_index, candidate
                in enumerate(fold_candidates)
            ])
            .sort_values(
                [
                    "mean_inner_macro_f1",
                    "mean_inner_balanced_accuracy",
                    "mean_inner_weighted_f1",
                    "mean_inner_accuracy",
                    "inner_macro_f1_gap",
                    "std_inner_macro_f1"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    False,
                    True,
                    True
                ]
            )
            .iloc[0]
        )

        selected_candidate = fold_candidates[
            int(selected_candidate["candidate_index"])
        ]

        selected_pipeline = clone(
            selected_candidate[
                "pipeline_template"
            ]
        )

        selected_pipeline.set_params(
            **selected_candidate[
                "best_parameters_dict"
            ]
        )

        selected_features = selected_candidate[
            "feature_columns"
        ]

        selected_pipeline.fit(
            outer_train_source[
                selected_features
            ],
            y_outer_train
        )

        outer_predictions = (
            selected_pipeline.predict(
                outer_test_source[
                    selected_features
                ]
            )
            .astype(int)
        )

        raw_outer_probabilities = (
            selected_pipeline.predict_proba(
                outer_test_source[
                    selected_features
                ]
            )
        )

        outer_probabilities = (
            align_class_probabilities(
                raw_outer_probabilities,
                selected_pipeline.classes_
            )
        )

        outer_metrics = (
            calculate_multiclass_metrics(
                y_outer_test.to_numpy(dtype=int),
                outer_predictions,
                outer_probabilities
            )
        )

        step20_outer_selection_records.append({
            "model_family": model_family,
            "outer_fold": outer_fold,
            "selected_feature_set": (
                selected_candidate[
                    "feature_set"
                ]
            ),
            "selected_feature_count": len(
                selected_features
            ),
            "selected_parameters": (
                selected_candidate[
                    "selected_parameters"
                ]
            ),
            "inner_macro_f1": (
                selected_candidate[
                    "mean_inner_macro_f1"
                ]
            ),
            "inner_balanced_accuracy": (
                selected_candidate[
                    "mean_inner_balanced_accuracy"
                ]
            ),
            "outer_macro_f1": (
                outer_metrics["macro_f1"]
            ),
            "outer_balanced_accuracy": (
                outer_metrics[
                    "balanced_accuracy"
                ]
            ),
            "outer_accuracy": (
                outer_metrics["accuracy"]
            ),
            "outer_weighted_f1": (
                outer_metrics[
                    "weighted_f1"
                ]
            )
        })

        step20_outer_metric_records.append({
            "model_family": model_family,
            "outer_fold": outer_fold,
            **outer_metrics
        })

        for row_position in range(
            len(outer_test_metadata)
        ):
            prediction_record = {
                "model_family": model_family,
                "household_ID": (
                    outer_test_metadata.iloc[
                        row_position
                    ]["household_ID"]
                ),
                "outer_fold": outer_fold,
                "true_class_index": int(
                    y_outer_test.iloc[
                        row_position
                    ]
                ),
                "true_class_name": (
                    outer_test_metadata.iloc[
                        row_position
                    ]["cluster_name"]
                ),
                "predicted_class_index": int(
                    outer_predictions[
                        row_position
                    ]
                ),
                "predicted_class_name": (
                    INDEX_TO_CLASS[
                        int(
                            outer_predictions[
                                row_position
                            ]
                        )
                    ]
                )
            }

            for class_index, class_name in enumerate(
                ORDERED_CLUSTER_NAMES
            ):
                safe_class_name = (
                    class_name
                    .lower()
                    .replace("-", "_")
                    .replace(" ", "_")
                    .replace("/", "_")
                )

                prediction_record[
                    f"probability_{safe_class_name}"
                ] = float(
                    outer_probabilities[
                        row_position,
                        class_index
                    ]
                )

            step20_oof_records.append(
                prediction_record
            )

        print(
            f"Selected {selected_candidate['feature_set']} | "
            f"outer macro-F1={outer_metrics['macro_f1']:.4f} | "
            f"outer balanced accuracy="
            f"{outer_metrics['balanced_accuracy']:.4f}"
        )


# ------------------------------------------------------------
# 6. Compile and audit Step 20 predictions
# ------------------------------------------------------------

step20_inner_results_df = pd.DataFrame(
    step20_inner_records
)

step20_outer_selection_df = pd.DataFrame(
    step20_outer_selection_records
)

step20_outer_metrics_df = pd.DataFrame(
    step20_outer_metric_records
)

step20_oof_predictions_df = pd.DataFrame(
    step20_oof_records
)

expected_step20_predictions = (
    len(nested_df)
    * len(STEP20_MODEL_BUILDERS)
)

if len(step20_oof_predictions_df) != (
    expected_step20_predictions
):
    raise RuntimeError(
        "Step 20 did not create exactly one OOF prediction "
        "per household and model family."
    )

duplicate_step20_predictions = int(
    step20_oof_predictions_df.duplicated(
        subset=[
            "model_family",
            "household_ID"
        ]
    ).sum()
)

if duplicate_step20_predictions > 0:
    raise RuntimeError(
        "Step 20 created duplicate household predictions."
    )

step20_counts_by_model = (
    step20_oof_predictions_df
    .groupby("model_family")
    .size()
)

if not (
    step20_counts_by_model
    == len(nested_df)
).all():
    raise RuntimeError(
        "At least one Step 20 model is missing household OOF "
        "predictions."
    )


# ------------------------------------------------------------
# 7. Pooled performance and class-specific metrics
# ------------------------------------------------------------

step20_performance_records = []
step20_class_metric_records = []
step20_confusion_tables = {}

for model_family in STEP20_MODEL_BUILDERS:
    model_oof = (
        step20_oof_predictions_df[
            step20_oof_predictions_df[
                "model_family"
            ] == model_family
        ]
        .sort_values("household_ID")
        .reset_index(drop=True)
    )

    y_true_model = (
        model_oof[
            "true_class_index"
        ]
        .to_numpy(dtype=int)
    )

    y_pred_model = (
        model_oof[
            "predicted_class_index"
        ]
        .to_numpy(dtype=int)
    )

    model_probability_columns = []

    for class_name in ORDERED_CLUSTER_NAMES:
        safe_class_name = (
            class_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        model_probability_columns.append(
            f"probability_{safe_class_name}"
        )

    model_probabilities = (
        model_oof[
            model_probability_columns
        ]
        .to_numpy(dtype=float)
    )

    model_metrics = (
        calculate_multiclass_metrics(
            y_true_model,
            y_pred_model,
            model_probabilities
        )
    )

    step20_performance_records.append({
        "evaluation": model_family,
        **model_metrics
    })

    report = classification_report(
        y_true_model,
        y_pred_model,
        labels=FIXED_CLASS_INDICES,
        target_names=ORDERED_CLUSTER_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in ORDERED_CLUSTER_NAMES:
        step20_class_metric_records.append({
            "model_family": model_family,
            "class_name": class_name,
            "precision": report[
                class_name
            ]["precision"],
            "recall": report[
                class_name
            ]["recall"],
            "f1_score": report[
                class_name
            ]["f1-score"],
            "support": int(
                report[
                    class_name
                ]["support"]
            )
        })

    confusion = confusion_matrix(
        y_true_model,
        y_pred_model,
        labels=FIXED_CLASS_INDICES
    )

    step20_confusion_tables[
        model_family
    ] = pd.DataFrame(
        confusion,
        index=ORDERED_CLUSTER_NAMES,
        columns=ORDERED_CLUSTER_NAMES
    )

step20_performance_df = pd.DataFrame(
    step20_performance_records
)

step20_class_metrics_df = pd.DataFrame(
    step20_class_metric_records
)


# ------------------------------------------------------------
# 8. Add the original Step 4 result for descriptive comparison
# ------------------------------------------------------------

step4_reference_row = (
    pooled_performance_df
    .iloc[[0]]
    .copy()
)

step4_reference_row[
    "evaluation"
] = "Existing Step 4 adaptive nested procedure"

step20_performance_comparison_df = pd.concat(
    [
        step4_reference_row[
            step20_performance_df.columns
        ],
        step20_performance_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 9. Paired stratified bootstrap differences versus Step 4
# ------------------------------------------------------------

STEP20_BOOTSTRAP_REPETITIONS = 2000
step20_bootstrap_rng = np.random.default_rng(
    RANDOM_STATE + 2000
)

step4_reference_predictions = (
    oof_predictions_df[
        [
            "household_ID",
            "true_class_index",
            "predicted_class_index"
        ]
    ]
    .rename(
        columns={
            "predicted_class_index": (
                "step4_predicted_class_index"
            )
        }
    )
)

step20_difference_records = []

for model_family in STEP20_MODEL_BUILDERS:
    candidate_predictions = (
        step20_oof_predictions_df[
            step20_oof_predictions_df[
                "model_family"
            ] == model_family
        ][
            [
                "household_ID",
                "true_class_index",
                "predicted_class_index"
            ]
        ]
        .rename(
            columns={
                "true_class_index": (
                    "step20_true_class_index"
                ),
                "predicted_class_index": (
                    "step20_predicted_class_index"
                )
            }
        )
    )

    paired = step4_reference_predictions.merge(
        candidate_predictions,
        on="household_ID",
        how="inner",
        validate="one_to_one"
    )

    if len(paired) != len(nested_df):
        raise RuntimeError(
            f"{model_family}: paired comparison lost households."
        )

    if not np.array_equal(
        paired[
            "true_class_index"
        ].to_numpy(dtype=int),
        paired[
            "step20_true_class_index"
        ].to_numpy(dtype=int)
    ):
        raise RuntimeError(
            f"{model_family}: true labels do not align with "
            "the Step 4 OOF table."
        )

    y_true_paired = (
        paired[
            "true_class_index"
        ]
        .to_numpy(dtype=int)
    )

    step4_pred_paired = (
        paired[
            "step4_predicted_class_index"
        ]
        .to_numpy(dtype=int)
    )

    step20_pred_paired = (
        paired[
            "step20_predicted_class_index"
        ]
        .to_numpy(dtype=int)
    )

    class_positions = {
        class_index: np.flatnonzero(
            y_true_paired == class_index
        )
        for class_index in FIXED_CLASS_INDICES
    }

    observed_differences = {
        "accuracy": (
            accuracy_score(
                y_true_paired,
                step20_pred_paired
            )
            - accuracy_score(
                y_true_paired,
                step4_pred_paired
            )
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true_paired,
                step20_pred_paired
            )
            - balanced_accuracy_score(
                y_true_paired,
                step4_pred_paired
            )
        ),
        "macro_f1": (
            f1_score(
                y_true_paired,
                step20_pred_paired,
                average="macro",
                zero_division=0
            )
            - f1_score(
                y_true_paired,
                step4_pred_paired,
                average="macro",
                zero_division=0
            )
        ),
        "weighted_f1": (
            f1_score(
                y_true_paired,
                step20_pred_paired,
                average="weighted",
                zero_division=0
            )
            - f1_score(
                y_true_paired,
                step4_pred_paired,
                average="weighted",
                zero_division=0
            )
        )
    }

    bootstrap_differences = {
        metric_name: np.empty(
            STEP20_BOOTSTRAP_REPETITIONS,
            dtype=float
        )
        for metric_name in observed_differences
    }

    for repetition in range(
        STEP20_BOOTSTRAP_REPETITIONS
    ):
        sampled_positions = np.concatenate([
            step20_bootstrap_rng.choice(
                positions,
                size=len(positions),
                replace=True
            )
            for positions in class_positions.values()
        ])

        bootstrap_true = (
            y_true_paired[
                sampled_positions
            ]
        )

        bootstrap_step4 = (
            step4_pred_paired[
                sampled_positions
            ]
        )

        bootstrap_step20 = (
            step20_pred_paired[
                sampled_positions
            ]
        )

        bootstrap_differences[
            "accuracy"
        ][repetition] = (
            accuracy_score(
                bootstrap_true,
                bootstrap_step20
            )
            - accuracy_score(
                bootstrap_true,
                bootstrap_step4
            )
        )

        bootstrap_differences[
            "balanced_accuracy"
        ][repetition] = (
            balanced_accuracy_score(
                bootstrap_true,
                bootstrap_step20
            )
            - balanced_accuracy_score(
                bootstrap_true,
                bootstrap_step4
            )
        )

        bootstrap_differences[
            "macro_f1"
        ][repetition] = (
            f1_score(
                bootstrap_true,
                bootstrap_step20,
                average="macro",
                zero_division=0
            )
            - f1_score(
                bootstrap_true,
                bootstrap_step4,
                average="macro",
                zero_division=0
            )
        )

        bootstrap_differences[
            "weighted_f1"
        ][repetition] = (
            f1_score(
                bootstrap_true,
                bootstrap_step20,
                average="weighted",
                zero_division=0
            )
            - f1_score(
                bootstrap_true,
                bootstrap_step4,
                average="weighted",
                zero_division=0
            )
        )

    for metric_name, observed_difference in (
        observed_differences.items()
    ):
        difference_distribution = (
            bootstrap_differences[
                metric_name
            ]
        )

        lower_bound, upper_bound = np.quantile(
            difference_distribution,
            [0.025, 0.975]
        )

        step20_difference_records.append({
            "model_family": model_family,
            "metric": metric_name,
            "observed_difference_vs_step4": float(
                observed_difference
            ),
            "bootstrap_ci_lower": float(
                lower_bound
            ),
            "bootstrap_ci_upper": float(
                upper_bound
            ),
            "bootstrap_repetitions": (
                STEP20_BOOTSTRAP_REPETITIONS
            )
        })

step20_paired_difference_ci_df = pd.DataFrame(
    step20_difference_records
)


# ------------------------------------------------------------
# 10. Conservative interpretation screen
# ------------------------------------------------------------

step20_interpretation_records = []

for model_family in STEP20_MODEL_BUILDERS:
    model_rows = (
        step20_paired_difference_ci_df[
            step20_paired_difference_ci_df[
                "model_family"
            ] == model_family
        ]
        .set_index("metric")
    )

    macro_row = model_rows.loc["macro_f1"]
    balanced_row = model_rows.loc[
        "balanced_accuracy"
    ]

    if (
        macro_row["bootstrap_ci_lower"] > 0
        and balanced_row[
            "bootstrap_ci_lower"
        ] > 0
    ):
        interpretation = (
            "Promising improvement in both primary imbalance-aware "
            "metrics; requires independent confirmation before "
            "replacing Step 4."
        )
    elif (
        macro_row[
            "observed_difference_vs_step4"
        ] > 0
        and balanced_row[
            "observed_difference_vs_step4"
        ] > 0
    ):
        interpretation = (
            "Observed improvement, but uncertainty overlaps no "
            "difference; retain as exploratory."
        )
    else:
        interpretation = (
            "No consistent improvement over Step 4 in both "
            "imbalance-aware metrics."
        )

    step20_interpretation_records.append({
        "model_family": model_family,
        "macro_f1_difference": float(
            macro_row[
                "observed_difference_vs_step4"
            ]
        ),
        "macro_f1_ci_lower": float(
            macro_row[
                "bootstrap_ci_lower"
            ]
        ),
        "macro_f1_ci_upper": float(
            macro_row[
                "bootstrap_ci_upper"
            ]
        ),
        "balanced_accuracy_difference": float(
            balanced_row[
                "observed_difference_vs_step4"
            ]
        ),
        "balanced_accuracy_ci_lower": float(
            balanced_row[
                "bootstrap_ci_lower"
            ]
        ),
        "balanced_accuracy_ci_upper": float(
            balanced_row[
                "bootstrap_ci_upper"
            ]
        ),
        "interpretation": interpretation
    })

step20_interpretation_df = pd.DataFrame(
    step20_interpretation_records
)


# ------------------------------------------------------------
# 11. Display results
# ------------------------------------------------------------

step20_runtime_minutes = (
    time.time()
    - step20_start_time
) / 60.0

print("\n" + "=" * 80)
print("STEP 20 POOLED OOF PERFORMANCE")
print("=" * 80)

display(
    step20_performance_comparison_df.round(4)
)

print("\nClass-specific performance:")

display(
    step20_class_metrics_df.round(4)
)

print(
    "\nPaired bootstrap differences versus the existing "
    "Step 4 nested OOF predictions:"
)

display(
    step20_paired_difference_ci_df.round(4)
)

print("\nConservative interpretation:")

display(
    step20_interpretation_df.round(4)
)

print("\nSelected feature set and parameters by outer fold:")

display(
    step20_outer_selection_df
)

print(
    "\nStep 20 runtime in minutes:",
    round(step20_runtime_minutes, 2)
)

print(
    "\nInterpretation warning: comparing several additional "
    "models increases the chance of finding an apparently better "
    "result by chance. Do not replace the locked Step 4 workflow "
    "without independent confirmation."
)


# ------------------------------------------------------------
# 12. Save Step 20 outputs
# ------------------------------------------------------------

step20_inner_results_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_inner_results.csv",
    index=False
)

step20_outer_selection_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_outer_selections.csv",
    index=False
)

step20_outer_metrics_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_outer_metrics.csv",
    index=False
)

step20_oof_predictions_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_oof_predictions.csv",
    index=False
)

step20_performance_comparison_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_performance_comparison.csv",
    index=False
)

step20_class_metrics_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_class_metrics.csv",
    index=False
)

step20_paired_difference_ci_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_paired_difference_ci.csv",
    index=False
)

step20_interpretation_df.to_csv(
    OUTPUT_DIR
    / "step20_advanced_models_interpretation.csv",
    index=False
)

for model_family, confusion_table in (
    step20_confusion_tables.items()
):
    safe_model_name = (
        model_family
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    confusion_table.to_csv(
        OUTPUT_DIR
        / (
            "step20_confusion_matrix_"
            f"{safe_model_name}.csv"
        )
    )

step20_methodology = {
    "role": "supplementary exploratory extension",
    "primary_existing_result_retained": True,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "outer_random_state": RANDOM_STATE,
    "primary_selection_metric": "macro_f1",
    "first_supporting_metric": "balanced_accuracy",
    "accuracy_primary": False,
    "model_families": list(
        STEP20_MODEL_BUILDERS.keys()
    ),
    "feature_sets": {
        feature_set_name: list(feature_columns)
        for feature_set_name, feature_columns
        in MODEL_FEATURE_SETS.items()
    },
    "xgboost_available": (
        XGBOOST_AVAILABLE_STEP20
    ),
    "bootstrap_repetitions": (
        STEP20_BOOTSTRAP_REPETITIONS
    ),
    "promotion_rule": (
        "No automatic promotion. Independent confirmation is "
        "required before replacing the locked Step 4 workflow."
    )
}

with open(
    OUTPUT_DIR
    / "step20_advanced_models_methodology.json",
    "w",
    encoding="utf-8"
) as methodology_file:
    json.dump(
        step20_methodology,
        methodology_file,
        indent=4
    )

print("\n" + "=" * 80)
print("STEP 20 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "Advanced model families evaluated:",
    list(STEP20_MODEL_BUILDERS.keys())
)

print(
    "Saved Step 20 outputs to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# STEP 21 — REPEATED NESTED-CV SPLIT-STABILITY AUDIT
# ============================================================
# Purpose:
# 1. Re-run the already frozen Step 4 adaptive nested workflow
#    under several different stratified outer-fold allocations.
# 2. Measure sensitivity of the COMPLETE adaptive procedure to
#    fold randomisation.
# 3. Provide supplementary split-stability evidence only.
#
# IMPORTANT:
# - This DOES NOT replace the original Step 4 nested OOF result.
# - This DOES NOT select a new final model.
# - This DOES NOT modify the Step 9 fixed prototype.
# - The same households are reused across repeats, so repeated
#   results are NOT independent external validation samples.
# - Report this only as a robustness / split-stability audit.
# ============================================================

from pathlib import Path
from collections import Counter
import time

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)


# ------------------------------------------------------------
# 1. Confirm required frozen Step-4 objects
# ------------------------------------------------------------

REQUIRED_STEP21_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "y_nested",
    "MODEL_FEATURE_SETS",
    "PRIMARY_CANDIDATE_SPECIFICATIONS",
    "build_candidate_pipeline",
    "select_balanced_cv_candidate",
    "SCORING",
    "ORDERED_CLUSTER_NAMES",
    "RANDOM_STATE",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "pooled_performance_df",
    "OUTPUT_DIR",
]

missing_step21_objects = [
    name
    for name in REQUIRED_STEP21_OBJECTS
    if name not in globals()
]

if missing_step21_objects:
    raise RuntimeError(
        "Run the existing notebook through the primary modelling "
        f"steps first. Missing objects: {missing_step21_objects}"
    )


# ------------------------------------------------------------
# 2. Freeze the audit configuration
# ------------------------------------------------------------

STABILITY_REPEATS = 3

# Use deterministic but different seeds from the original Step 4.
STABILITY_SEEDS = [
    int(RANDOM_STATE + 10_000),
    int(RANDOM_STATE + 20_000),
    int(RANDOM_STATE + 30_000),
]

if len(STABILITY_SEEDS) != STABILITY_REPEATS:
    raise RuntimeError(
        "STABILITY_SEEDS length must equal STABILITY_REPEATS."
    )

STEP21_OUTPUT_DIR = (
    Path(OUTPUT_DIR)
    / "repeated_nested_cv_stability_audit"
)

STEP21_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 3. Utility for safe positional indexing
# ------------------------------------------------------------

def _take_rows(obj, indices):
    """
    Positionally subset pandas or NumPy objects.
    """
    if hasattr(obj, "iloc"):
        return obj.iloc[indices]
    return obj[indices]


# ------------------------------------------------------------
# 4. Containers
# ------------------------------------------------------------

repeat_summary_records = []
fold_metric_records = []
selection_records = []
prediction_records = []
class_metric_records = []

step21_start_time = time.time()


# ------------------------------------------------------------
# 5. Repeated nested-CV audit
# ------------------------------------------------------------

for repeat_number, repeat_seed in enumerate(
    STABILITY_SEEDS,
    start=1,
):

    print("\n" + "=" * 88)
    print(
        f"STEP 21 — REPEAT {repeat_number}/{STABILITY_REPEATS} "
        f"| outer seed = {repeat_seed}"
    )
    print("=" * 88)

    outer_cv = StratifiedKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=repeat_seed,
    )

    repeat_predictions = np.full(
        shape=len(nested_df),
        fill_value=-1,
        dtype=int,
    )

    repeat_fold_assignments = np.full(
        shape=len(nested_df),
        fill_value=-1,
        dtype=int,
    )

    for outer_fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(
            X_nested_source,
            y_nested,
        ),
        start=1,
    ):

        print(
            f"\nRepeat {repeat_number} | "
            f"Outer fold {outer_fold}/{OUTER_FOLDS}"
        )

        outer_train_source = _take_rows(
            X_nested_source,
            train_idx,
        ).reset_index(drop=True)

        outer_test_source = _take_rows(
            X_nested_source,
            test_idx,
        ).reset_index(drop=True)

        y_outer_train = np.asarray(
            _take_rows(
                y_nested,
                train_idx,
            ),
            dtype=int,
        )

        y_outer_test = np.asarray(
            _take_rows(
                y_nested,
                test_idx,
            ),
            dtype=int,
        )

        inner_cv = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=(
                repeat_seed
                + outer_fold
            ),
        )

        fold_best_candidates = []

        # ----------------------------------------------------
        # Inner selection: EXACT SAME candidate space already
        # frozen in the primary Step 4 notebook
        # ----------------------------------------------------

        for feature_set_name, feature_columns in (
            MODEL_FEATURE_SETS.items()
        ):

            X_outer_train = (
                outer_train_source[
                    feature_columns
                ]
            )

            for candidate_specification in (
                PRIMARY_CANDIDATE_SPECIFICATIONS
            ):

                model_family = (
                    candidate_specification[
                        "model_family"
                    ]
                )

                imbalance_strategy = (
                    candidate_specification[
                        "imbalance_strategy"
                    ]
                )

                class_weight = (
                    candidate_specification[
                        "class_weight"
                    ]
                )

                pipeline, parameter_grid = (
                    build_candidate_pipeline(
                        model_family,
                        class_weight,
                    )
                )

                search = GridSearchCV(
                    estimator=pipeline,
                    param_grid=parameter_grid,
                    scoring=SCORING,
                    refit=False,
                    cv=inner_cv,
                    n_jobs=-1,
                    pre_dispatch="2*n_jobs",
                    return_train_score=True,
                    error_score="raise",
                )

                search.fit(
                    X_outer_train,
                    y_outer_train,
                )

                candidate_results = pd.DataFrame(
                    search.cv_results_
                )

                # Preserve the Step 4 ordering.
                candidate_results = (
                    candidate_results
                    .sort_values(
                        [
                            "mean_test_macro_f1",
                            "mean_test_balanced_accuracy",
                            "mean_test_weighted_f1",
                            "mean_test_accuracy",
                        ],
                        ascending=[
                            False,
                            False,
                            False,
                            False,
                        ],
                    )
                    .reset_index(drop=True)
                )

                # Preserve the regularisation-aware selection
                # rule already defined in BALANCED_OVERFIT_FINAL.
                best_result = (
                    select_balanced_cv_candidate(
                        candidate_results,
                        model_family,
                    )
                )

                best_parameters = dict(
                    best_result["params"]
                )

                candidate_record = {
                    "repeat": repeat_number,
                    "outer_fold": outer_fold,
                    "feature_set": feature_set_name,
                    "feature_count": len(
                        feature_columns
                    ),
                    "model_family": model_family,
                    "imbalance_strategy": (
                        imbalance_strategy
                    ),
                    "class_weight": str(
                        class_weight
                    ),
                    "mean_inner_macro_f1": float(
                        best_result[
                            "mean_test_macro_f1"
                        ]
                    ),
                    "mean_inner_balanced_accuracy": float(
                        best_result[
                            "mean_test_balanced_accuracy"
                        ]
                    ),
                    "mean_inner_weighted_f1": float(
                        best_result[
                            "mean_test_weighted_f1"
                        ]
                    ),
                    "mean_inner_accuracy": float(
                        best_result[
                            "mean_test_accuracy"
                        ]
                    ),
                    "feature_columns": list(
                        feature_columns
                    ),
                    "pipeline_template": pipeline,
                    "best_parameters_dict": (
                        best_parameters
                    ),
                }

                fold_best_candidates.append(
                    candidate_record
                )

        # ----------------------------------------------------
        # Select full workflow using ONLY inner-validation data
        # ----------------------------------------------------

        fold_best_candidates = sorted(
            fold_best_candidates,
            key=lambda record: (
                record[
                    "mean_inner_macro_f1"
                ],
                record[
                    "mean_inner_balanced_accuracy"
                ],
                record[
                    "mean_inner_weighted_f1"
                ],
                record[
                    "mean_inner_accuracy"
                ],
            ),
            reverse=True,
        )

        selected_candidate = (
            fold_best_candidates[0]
        )

        selected_model = clone(
            selected_candidate[
                "pipeline_template"
            ]
        )

        selected_model.set_params(
            **selected_candidate[
                "best_parameters_dict"
            ]
        )

        selected_features = (
            selected_candidate[
                "feature_columns"
            ]
        )

        selected_model.fit(
            outer_train_source[
                selected_features
            ],
            y_outer_train,
        )

        outer_predictions = (
            selected_model.predict(
                outer_test_source[
                    selected_features
                ]
            )
        )

        outer_predictions = np.asarray(
            outer_predictions,
            dtype=int,
        )

        repeat_predictions[
            test_idx
        ] = outer_predictions

        repeat_fold_assignments[
            test_idx
        ] = outer_fold

        # ----------------------------------------------------
        # Outer-fold metrics
        # ----------------------------------------------------

        outer_macro_f1 = f1_score(
            y_outer_test,
            outer_predictions,
            average="macro",
            zero_division=0,
        )

        outer_balanced_accuracy = (
            balanced_accuracy_score(
                y_outer_test,
                outer_predictions,
            )
        )

        outer_accuracy = accuracy_score(
            y_outer_test,
            outer_predictions,
        )

        fold_metric_records.append({
            "repeat": repeat_number,
            "outer_seed": repeat_seed,
            "outer_fold": outer_fold,
            "n_test": len(test_idx),
            "macro_f1": float(
                outer_macro_f1
            ),
            "balanced_accuracy": float(
                outer_balanced_accuracy
            ),
            "accuracy": float(
                outer_accuracy
            ),
        })

        selection_records.append({
            "repeat": repeat_number,
            "outer_seed": repeat_seed,
            "outer_fold": outer_fold,
            "feature_set": (
                selected_candidate[
                    "feature_set"
                ]
            ),
            "model_family": (
                selected_candidate[
                    "model_family"
                ]
            ),
            "imbalance_strategy": (
                selected_candidate[
                    "imbalance_strategy"
                ]
            ),
            "feature_count": len(
                selected_features
            ),
            "inner_macro_f1": (
                selected_candidate[
                    "mean_inner_macro_f1"
                ]
            ),
            "inner_balanced_accuracy": (
                selected_candidate[
                    "mean_inner_balanced_accuracy"
                ]
            ),
            "selected_parameters": str(
                selected_candidate[
                    "best_parameters_dict"
                ]
            ),
        })

        print(
            "Selected:",
            selected_candidate[
                "feature_set"
            ],
            "|",
            selected_candidate[
                "model_family"
            ],
            "|",
            selected_candidate[
                "imbalance_strategy"
            ],
        )

        print(
            f"Outer macro-F1 = "
            f"{outer_macro_f1:.4f}"
        )

        print(
            f"Outer balanced accuracy = "
            f"{outer_balanced_accuracy:.4f}"
        )


    # --------------------------------------------------------
    # 6. Verify exactly one OOF prediction per household
    #    WITHIN this repeat
    # --------------------------------------------------------

    if np.any(repeat_predictions < 0):
        raise RuntimeError(
            f"Repeat {repeat_number} did not generate "
            "one OOF prediction for every household."
        )

    if np.any(repeat_fold_assignments < 1):
        raise RuntimeError(
            f"Repeat {repeat_number} contains missing "
            "outer-fold assignments."
        )

    y_full = np.asarray(
        y_nested,
        dtype=int,
    )

    # --------------------------------------------------------
    # 7. Pooled OOF metrics for this repeat
    # --------------------------------------------------------

    repeat_macro_f1 = f1_score(
        y_full,
        repeat_predictions,
        average="macro",
        zero_division=0,
    )

    repeat_balanced_accuracy = (
        balanced_accuracy_score(
            y_full,
            repeat_predictions,
        )
    )

    repeat_accuracy = accuracy_score(
        y_full,
        repeat_predictions,
    )

    repeat_summary_records.append({
        "repeat": repeat_number,
        "outer_seed": repeat_seed,
        "households": len(y_full),
        "macro_f1": float(
            repeat_macro_f1
        ),
        "balanced_accuracy": float(
            repeat_balanced_accuracy
        ),
        "accuracy": float(
            repeat_accuracy
        ),
    })

    # --------------------------------------------------------
    # 8. Class-specific recall and F1 for this repeat
    # --------------------------------------------------------

    precision_values, recall_values, f1_values, supports = (
        precision_recall_fscore_support(
            y_full,
            repeat_predictions,
            labels=np.arange(
                len(ORDERED_CLUSTER_NAMES)
            ),
            zero_division=0,
        )
    )

    for class_index, class_name in enumerate(
        ORDERED_CLUSTER_NAMES
    ):
        class_metric_records.append({
            "repeat": repeat_number,
            "outer_seed": repeat_seed,
            "class_index": class_index,
            "class_name": class_name,
            "support": int(
                supports[class_index]
            ),
            "precision": float(
                precision_values[
                    class_index
                ]
            ),
            "recall": float(
                recall_values[
                    class_index
                ]
            ),
            "f1": float(
                f1_values[
                    class_index
                ]
            ),
        })

    # --------------------------------------------------------
    # 9. Save household-level prediction audit
    # --------------------------------------------------------

    household_ids = (
        nested_df[
            "household_ID"
        ]
        .astype(str)
        .to_numpy()
    )

    for row_position in range(
        len(nested_df)
    ):
        prediction_records.append({
            "repeat": repeat_number,
            "outer_seed": repeat_seed,
            "outer_fold": int(
                repeat_fold_assignments[
                    row_position
                ]
            ),
            "household_ID": (
                household_ids[
                    row_position
                ]
            ),
            "true_class_index": int(
                y_full[
                    row_position
                ]
            ),
            "predicted_class_index": int(
                repeat_predictions[
                    row_position
                ]
            ),
            "true_class_name": (
                ORDERED_CLUSTER_NAMES[
                    int(
                        y_full[
                            row_position
                        ]
                    )
                ]
            ),
            "predicted_class_name": (
                ORDERED_CLUSTER_NAMES[
                    int(
                        repeat_predictions[
                            row_position
                        ]
                    )
                ]
            ),
        })

    print(
        f"\nRepeat {repeat_number} pooled OOF "
        f"macro-F1 = {repeat_macro_f1:.4f}"
    )

    print(
        f"Repeat {repeat_number} pooled OOF "
        f"balanced accuracy = "
        f"{repeat_balanced_accuracy:.4f}"
    )


# ------------------------------------------------------------
# 10. Compile results
# ------------------------------------------------------------

repeated_nested_summary_df = pd.DataFrame(
    repeat_summary_records
)

repeated_nested_fold_metrics_df = pd.DataFrame(
    fold_metric_records
)

repeated_nested_selection_df = pd.DataFrame(
    selection_records
)

repeated_nested_predictions_df = pd.DataFrame(
    prediction_records
)

repeated_nested_class_metrics_df = pd.DataFrame(
    class_metric_records
)


# ------------------------------------------------------------
# 11. Summarise repeat-to-repeat stability
# ------------------------------------------------------------

metric_summary_records = []

for metric in [
    "macro_f1",
    "balanced_accuracy",
    "accuracy",
]:
    values = (
        repeated_nested_summary_df[
            metric
        ]
        .astype(float)
        .to_numpy()
    )

    metric_summary_records.append({
        "metric": metric,
        "n_repeats": len(values),
        "mean": float(
            np.mean(values)
        ),
        "std": float(
            np.std(
                values,
                ddof=1
            )
        ) if len(values) > 1 else np.nan,
        "minimum": float(
            np.min(values)
        ),
        "maximum": float(
            np.max(values)
        ),
    })

repeated_nested_metric_summary_df = pd.DataFrame(
    metric_summary_records
)


# ------------------------------------------------------------
# 12. Class-level stability summary
# ------------------------------------------------------------

repeated_nested_class_summary_df = (
    repeated_nested_class_metrics_df
    .groupby(
        "class_name",
        as_index=False,
    )
    .agg(
        support=(
            "support",
            "first",
        ),
        mean_precision=(
            "precision",
            "mean",
        ),
        sd_precision=(
            "precision",
            "std",
        ),
        mean_recall=(
            "recall",
            "mean",
        ),
        sd_recall=(
            "recall",
            "std",
        ),
        mean_f1=(
            "f1",
            "mean",
        ),
        sd_f1=(
            "f1",
            "std",
        ),
    )
)

class_order_map = {
    class_name: idx
    for idx, class_name
    in enumerate(
        ORDERED_CLUSTER_NAMES
    )
}

repeated_nested_class_summary_df[
    "_order"
] = (
    repeated_nested_class_summary_df[
        "class_name"
    ]
    .map(
        class_order_map
    )
)

repeated_nested_class_summary_df = (
    repeated_nested_class_summary_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. Workflow-selection frequency
# ------------------------------------------------------------

workflow_selection_frequency_df = (
    repeated_nested_selection_df
    .groupby(
        [
            "feature_set",
            "model_family",
            "imbalance_strategy",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "selected_outer_folds"
        }
    )
    .sort_values(
        "selected_outer_folds",
        ascending=False,
    )
    .reset_index(drop=True)
)

workflow_selection_frequency_df[
    "selection_percentage"
] = (
    100
    * workflow_selection_frequency_df[
        "selected_outer_folds"
    ]
    / len(
        repeated_nested_selection_df
    )
)


# ------------------------------------------------------------
# 14. Compare ONLY descriptively with original Step 4
# ------------------------------------------------------------

primary_step4_row = (
    pooled_performance_df
    .iloc[0]
)

comparison_to_primary_df = pd.DataFrame([
    {
        "evaluation": (
            "Primary Step 4 nested OOF"
        ),
        "macro_f1": float(
            primary_step4_row[
                "macro_f1"
            ]
        ),
        "balanced_accuracy": float(
            primary_step4_row[
                "balanced_accuracy"
            ]
        ),
        "accuracy": float(
            primary_step4_row[
                "accuracy"
            ]
        ),
        "status": (
            "PRIMARY THESIS ESTIMATE"
        ),
    },
    {
        "evaluation": (
            "Step 21 repeated nested-CV "
            "split-stability mean"
        ),
        "macro_f1": float(
            repeated_nested_summary_df[
                "macro_f1"
            ].mean()
        ),
        "balanced_accuracy": float(
            repeated_nested_summary_df[
                "balanced_accuracy"
            ].mean()
        ),
        "accuracy": float(
            repeated_nested_summary_df[
                "accuracy"
            ].mean()
        ),
        "status": (
            "SUPPLEMENTARY STABILITY AUDIT ONLY"
        ),
    },
])


# ------------------------------------------------------------
# 15. Save audit outputs
# ------------------------------------------------------------

repeated_nested_summary_df.to_csv(
    STEP21_OUTPUT_DIR
    / "01_repeat_level_metrics.csv",
    index=False,
)

repeated_nested_metric_summary_df.to_csv(
    STEP21_OUTPUT_DIR
    / "02_metric_stability_summary.csv",
    index=False,
)

repeated_nested_fold_metrics_df.to_csv(
    STEP21_OUTPUT_DIR
    / "03_outer_fold_metrics.csv",
    index=False,
)

repeated_nested_class_metrics_df.to_csv(
    STEP21_OUTPUT_DIR
    / "04_repeat_class_metrics.csv",
    index=False,
)

repeated_nested_class_summary_df.to_csv(
    STEP21_OUTPUT_DIR
    / "05_class_stability_summary.csv",
    index=False,
)

repeated_nested_selection_df.to_csv(
    STEP21_OUTPUT_DIR
    / "06_selected_workflows_by_fold.csv",
    index=False,
)

workflow_selection_frequency_df.to_csv(
    STEP21_OUTPUT_DIR
    / "07_workflow_selection_frequency.csv",
    index=False,
)

repeated_nested_predictions_df.to_csv(
    STEP21_OUTPUT_DIR
    / "08_repeated_nested_predictions.csv",
    index=False,
)

comparison_to_primary_df.to_csv(
    STEP21_OUTPUT_DIR
    / "09_primary_vs_stability_audit.csv",
    index=False,
)


# ------------------------------------------------------------
# 16. Final display
# ------------------------------------------------------------

step21_runtime_minutes = (
    time.time()
    - step21_start_time
) / 60.0

print("\n" + "=" * 88)
print("STEP 21 — REPEATED NESTED-CV STABILITY AUDIT COMPLETE")
print("=" * 88)

print(
    f"Runtime: {step21_runtime_minutes:.2f} minutes"
)

print(
    f"Repeats: {STABILITY_REPEATS}"
)

print(
    f"Outer folds per repeat: {OUTER_FOLDS}"
)

print(
    f"Inner folds: {INNER_FOLDS}"
)

print(
    "\nIMPORTANT: Step 4 remains the primary thesis estimate."
)

print(
    "Step 21 is split-stability evidence only and must not "
    "be used for model reselection."
)

print(
    "\nRepeat-level pooled metrics:"
)

display(
    repeated_nested_summary_df.round(4)
)

print(
    "\nAcross-repeat stability summary:"
)

display(
    repeated_nested_metric_summary_df.round(4)
)

print(
    "\nClass-specific stability:"
)

display(
    repeated_nested_class_summary_df.round(4)
)

print(
    "\nWorkflow-selection frequency:"
)

display(
    workflow_selection_frequency_df.round(2)
)

print(
    "\nPrimary Step 4 versus supplementary stability mean:"
)

display(
    comparison_to_primary_df.round(4)
)

print(
    "\nSaved Step 21 outputs to:"
)

print(
    STEP21_OUTPUT_DIR.resolve()
)

In [ ]:
# ============================================================
# STEP 22 — FINAL FROZEN-PIPELINE QA AND REPRODUIBILITY AUDIT
# ============================================================
# Purpose:
# 1. Verify the final modelling population and pseudo-labels.
# 2. Verify the exact frozen 32-predictor feature schema.
# 3. Confirm the known duplicate feature is absent.
# 4. Confirm no target/leakage columns entered the model.
# 5. Confirm primary Step 4 OOF predictions remain complete.
# 6. Verify the saved research-prototype package against the
#    current in-memory final model.
# 7. Save exact environment, workflow, hashes, and audit results.
#
# IMPORTANT:
# - This cell performs NO training.
# - This cell performs NO hyperparameter selection.
# - This cell performs NO model reselection.
# - Any failed assertion should be investigated rather than
#   silently bypassed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

REQUIRED_STEP22_OBJECTS = [
    "nested_df",
    "X_nested_source",
    "y_nested",
    "FINAL_WORKFLOW_FEATURES",
    "FINAL_WORKFLOW_MODEL_FAMILY",
    "FINAL_WORKFLOW_FEATURE_SET",
    "FINAL_WORKFLOW_PARAMS",
    "ORDERED_CLUSTER_NAMES",
    "CLASS_TO_INDEX",
    "INDEX_TO_CLASS",
    "pooled_performance_df",
    "oof_predictions_df",
    "final_research_prototype",
    "model_path",
    "OUTPUT_DIR",
]

missing_step22_objects = [
    name
    for name in REQUIRED_STEP22_OBJECTS
    if name not in globals()
]

if missing_step22_objects:
    raise RuntimeError(
        "Run the notebook through the final prototype-fitting "
        f"step first. Missing objects: {missing_step22_objects}"
    )


# ------------------------------------------------------------
# 2. Frozen expected study endpoint
# ------------------------------------------------------------

EXPECTED_MATCHED_HOUSEHOLDS = 1146

EXPECTED_CLASS_COUNTS = {
    "Peak-Exposed": 73,
    "Typical Household": 809,
    "Heavy Continuous": 243,
    "Daytime-Active": 21,
}

EXPECTED_FINAL_PREDICTOR_COUNT = 32

KNOWN_DUPLICATE_FEATURE = (
    "late_6_month_median_kwh"
)

REQUIRED_RETAINED_FEATURE = (
    "recent_6_month_median_kwh"
)


# ------------------------------------------------------------
# 3. Output folder
# ------------------------------------------------------------

STEP22_OUTPUT_DIR = (
    Path(OUTPUT_DIR)
    / "final_frozen_pipeline_qa"
)

STEP22_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. Hash utilities
# ------------------------------------------------------------

def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def sha256_file(path):
    path = Path(path)

    hasher = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def canonical_json_hash(obj):
    canonical_text = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )

    return sha256_text(
        canonical_text
    )


# ------------------------------------------------------------
# 5. QA record helper
# ------------------------------------------------------------

qa_records = []


def record_check(
    check_name,
    condition,
    observed,
    expected,
):
    condition = bool(condition)

    qa_records.append({
        "check": check_name,
        "status": (
            "PASS"
            if condition
            else "FAIL"
        ),
        "observed": str(observed),
        "expected": str(expected),
    })

    if not condition:
        raise AssertionError(
            f"{check_name} FAILED | "
            f"observed={observed} | "
            f"expected={expected}"
        )


# ------------------------------------------------------------
# 6. Household-level dataset integrity
# ------------------------------------------------------------

record_check(
    "Matched modelling population",
    len(nested_df)
    == EXPECTED_MATCHED_HOUSEHOLDS,
    len(nested_df),
    EXPECTED_MATCHED_HOUSEHOLDS,
)

record_check(
    "Unique household_ID values",
    not nested_df[
        "household_ID"
    ].duplicated().any(),
    nested_df[
        "household_ID"
    ].nunique(),
    EXPECTED_MATCHED_HOUSEHOLDS,
)

record_check(
    "X source row count",
    len(X_nested_source)
    == EXPECTED_MATCHED_HOUSEHOLDS,
    len(X_nested_source),
    EXPECTED_MATCHED_HOUSEHOLDS,
)

record_check(
    "Target row count",
    len(y_nested)
    == EXPECTED_MATCHED_HOUSEHOLDS,
    len(y_nested),
    EXPECTED_MATCHED_HOUSEHOLDS,
)


# ------------------------------------------------------------
# 7. Locked class-distribution integrity
# ------------------------------------------------------------

observed_class_counts = (
    nested_df[
        "cluster_name"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)

for class_name, expected_count in (
    EXPECTED_CLASS_COUNTS.items()
):
    observed_count = int(
        observed_class_counts.get(
            class_name,
            0,
        )
    )

    record_check(
        f"Class count — {class_name}",
        observed_count
        == expected_count,
        observed_count,
        expected_count,
    )


# ------------------------------------------------------------
# 8. Target encoding consistency
# ------------------------------------------------------------

y_array = np.asarray(
    y_nested,
    dtype=int,
)

record_check(
    "Target class indices valid",
    set(
        np.unique(
            y_array
        )
    )
    == set(
        range(
            len(
                ORDERED_CLUSTER_NAMES
            )
        )
    ),
    sorted(
        np.unique(
            y_array
        ).tolist()
    ),
    list(
        range(
            len(
                ORDERED_CLUSTER_NAMES
            )
        )
    ),
)

for class_index, class_name in enumerate(
    ORDERED_CLUSTER_NAMES
):

    mapped_name = (
        INDEX_TO_CLASS[
            class_index
        ]
    )

    record_check(
        f"Index-to-class mapping {class_index}",
        mapped_name
        == class_name,
        mapped_name,
        class_name,
    )

    mapped_index = (
        CLASS_TO_INDEX[
            class_name
        ]
    )

    record_check(
        f"Class-to-index mapping {class_name}",
        int(mapped_index)
        == class_index,
        mapped_index,
        class_index,
    )


# ------------------------------------------------------------
# 9. Exact final feature schema
# ------------------------------------------------------------

final_features = list(
    FINAL_WORKFLOW_FEATURES
)

record_check(
    "Final predictor count",
    len(final_features)
    == EXPECTED_FINAL_PREDICTOR_COUNT,
    len(final_features),
    EXPECTED_FINAL_PREDICTOR_COUNT,
)

record_check(
    "Final predictor names unique",
    len(final_features)
    == len(
        set(
            final_features
        )
    ),
    len(
        set(
            final_features
        )
    ),
    len(final_features),
)

missing_final_features = [
    feature
    for feature in final_features
    if feature
    not in X_nested_source.columns
]

record_check(
    "All final predictors exist in modelling matrix",
    len(
        missing_final_features
    )
    == 0,
    missing_final_features,
    [],
)

record_check(
    "Known duplicate predictor removed",
    KNOWN_DUPLICATE_FEATURE
    not in final_features,
    (
        KNOWN_DUPLICATE_FEATURE
        in final_features
    ),
    False,
)

record_check(
    "Canonical recent-six-month predictor retained",
    REQUIRED_RETAINED_FEATURE
    in final_features,
    (
        REQUIRED_RETAINED_FEATURE
        in final_features
    ),
    True,
)


# ------------------------------------------------------------
# 10. Direct leakage-name audit
# ------------------------------------------------------------

FORBIDDEN_EXACT_COLUMNS = {
    "cluster",
    "cluster_name",
    "cluster_label",
    "label",
    "target",
    "target_label",
    "archetype",
    "archetype_label",
    "predicted_cluster",
    "true_cluster",
}

direct_leakage_features = [
    feature
    for feature in final_features
    if feature.lower()
    in FORBIDDEN_EXACT_COLUMNS
]

record_check(
    "No direct target/leakage columns in final predictors",
    len(
        direct_leakage_features
    )
    == 0,
    direct_leakage_features,
    [],
)


# ------------------------------------------------------------
# 11. Numeric integrity of final feature matrix
# ------------------------------------------------------------

final_X = (
    X_nested_source[
        final_features
    ]
    .copy()
)

all_null_features = [
    column
    for column in final_features
    if final_X[
        column
    ].isna().all()
]

record_check(
    "No completely missing final predictor",
    len(
        all_null_features
    )
    == 0,
    all_null_features,
    [],
)


# ------------------------------------------------------------
# 12. Primary Step-4 OOF integrity
# ------------------------------------------------------------

record_check(
    "Primary OOF prediction count",
    len(
        oof_predictions_df
    )
    == EXPECTED_MATCHED_HOUSEHOLDS,
    len(
        oof_predictions_df
    ),
    EXPECTED_MATCHED_HOUSEHOLDS,
)

record_check(
    "Primary OOF household IDs unique",
    not oof_predictions_df[
        "household_ID"
    ].duplicated().any(),
    oof_predictions_df[
        "household_ID"
    ].nunique(),
    EXPECTED_MATCHED_HOUSEHOLDS,
)

primary_evaluation_name = str(
    pooled_performance_df
    .iloc[0][
        "evaluation"
    ]
)

record_check(
    "Primary result labelled out-of-fold",
    "out-of-fold"
    in primary_evaluation_name.lower(),
    primary_evaluation_name,
    "Contains 'out-of-fold'",
)


# ------------------------------------------------------------
# 13. Final fitted prototype audit
# ------------------------------------------------------------

record_check(
    "Final model is sklearn Pipeline",
    isinstance(
        final_research_prototype,
        Pipeline,
    ),
    type(
        final_research_prototype
    ).__name__,
    "Pipeline",
)

pipeline_step_names = [
    step_name
    for step_name, _
    in final_research_prototype.steps
]

record_check(
    "Final pipeline contains estimator step",
    len(
        pipeline_step_names
    )
    >= 1,
    pipeline_step_names,
    "At least one fitted pipeline step",
)


# ------------------------------------------------------------
# 14. Saved model-package existence and hash
# ------------------------------------------------------------

model_path = Path(
    model_path
)

record_check(
    "Saved model package exists",
    model_path.exists(),
    model_path.exists(),
    True,
)

model_sha256 = sha256_file(
    model_path
)


# ------------------------------------------------------------
# 15. Reload saved package
# ------------------------------------------------------------

reloaded_package = joblib.load(
    model_path
)

record_check(
    "Reloaded object is package dictionary",
    isinstance(
        reloaded_package,
        dict,
    ),
    type(
        reloaded_package
    ).__name__,
    "dict",
)

REQUIRED_PACKAGE_KEYS = {
    "estimator",
    "feature_order",
    "class_to_index",
    "index_to_class",
    "ordered_class_names",
    "workflow_feature_set",
    "workflow_model_family",
    "workflow_parameters",
    "training_households",
    "primary_performance_source",
}

missing_package_keys = sorted(
    REQUIRED_PACKAGE_KEYS
    - set(
        reloaded_package.keys()
    )
)

record_check(
    "Saved package contains required metadata",
    len(
        missing_package_keys
    )
    == 0,
    missing_package_keys,
    [],
)


# ------------------------------------------------------------
# 16. Reloaded feature-order audit
# ------------------------------------------------------------

reloaded_feature_order = list(
    reloaded_package[
        "feature_order"
    ]
)

record_check(
    "Reloaded feature order exactly matches frozen order",
    reloaded_feature_order
    == final_features,
    reloaded_feature_order,
    final_features,
)

record_check(
    "Reloaded model family matches frozen model family",
    str(
        reloaded_package[
            "workflow_model_family"
        ]
    )
    == str(
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    reloaded_package[
        "workflow_model_family"
    ],
    FINAL_WORKFLOW_MODEL_FAMILY,
)

record_check(
    "Reloaded feature-set name matches frozen feature set",
    str(
        reloaded_package[
            "workflow_feature_set"
        ]
    )
    == str(
        FINAL_WORKFLOW_FEATURE_SET
    ),
    reloaded_package[
        "workflow_feature_set"
    ],
    FINAL_WORKFLOW_FEATURE_SET,
)

record_check(
    "Reloaded training-household count",
    int(
        reloaded_package[
            "training_households"
        ]
    )
    == EXPECTED_MATCHED_HOUSEHOLDS,
    reloaded_package[
        "training_households"
    ],
    EXPECTED_MATCHED_HOUSEHOLDS,
)


# ------------------------------------------------------------
# 17. Saved versus in-memory prediction smoke test
# ------------------------------------------------------------

reloaded_estimator = (
    reloaded_package[
        "estimator"
    ]
)

SMOKE_TEST_ROWS = min(
    50,
    len(final_X),
)

smoke_X = (
    final_X
    .iloc[
        :SMOKE_TEST_ROWS
    ]
    .copy()
)

in_memory_predictions = (
    final_research_prototype.predict(
        smoke_X
    )
)

reloaded_predictions = (
    reloaded_estimator.predict(
        smoke_X
    )
)

prediction_match = np.array_equal(
    np.asarray(
        in_memory_predictions
    ),
    np.asarray(
        reloaded_predictions
    ),
)

record_check(
    "Reload smoke-test predictions identical",
    prediction_match,
    prediction_match,
    True,
)


# ------------------------------------------------------------
# 18. Package mapping consistency
# ------------------------------------------------------------

record_check(
    "Saved class order matches current class order",
    list(
        reloaded_package[
            "ordered_class_names"
        ]
    )
    == list(
        ORDERED_CLUSTER_NAMES
    ),
    reloaded_package[
        "ordered_class_names"
    ],
    ORDERED_CLUSTER_NAMES,
)

record_check(
    "Saved primary performance source remains Step 4",
    "step 4"
    in str(
        reloaded_package[
            "primary_performance_source"
        ]
    ).lower(),
    reloaded_package[
        "primary_performance_source"
    ],
    "Step 4 nested OOF evaluation",
)


# ------------------------------------------------------------
# 19. Reproducibility hashes
# ------------------------------------------------------------

sorted_household_ids = sorted(
    nested_df[
        "household_ID"
    ]
    .astype(str)
    .tolist()
)

feature_order_sha256 = (
    canonical_json_hash(
        final_features
    )
)

household_population_sha256 = (
    canonical_json_hash(
        sorted_household_ids
    )
)

class_mapping_sha256 = (
    canonical_json_hash({
        "class_to_index": (
            CLASS_TO_INDEX
        ),
        "index_to_class": (
            INDEX_TO_CLASS
        ),
        "ordered_names": list(
            ORDERED_CLUSTER_NAMES
        ),
    })
)

workflow_sha256 = (
    canonical_json_hash({
        "model_family": (
            FINAL_WORKFLOW_MODEL_FAMILY
        ),
        "feature_set": (
            FINAL_WORKFLOW_FEATURE_SET
        ),
        "parameters": (
            FINAL_WORKFLOW_PARAMS
        ),
        "features": (
            final_features
        ),
    })
)


# ------------------------------------------------------------
# 20. Environment versions
# ------------------------------------------------------------

def package_version_or_na(
    package_name
):
    try:
        return (
            importlib.metadata.version(
                package_name
            )
        )
    except Exception:
        return "not available"


environment_record = {
    "audit_created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "python_version": (
        platform.python_version()
    ),
    "platform": (
        platform.platform()
    ),
    "numpy_version": (
        np.__version__
    ),
    "pandas_version": (
        pd.__version__
    ),
    "scikit_learn_version": (
        sklearn.__version__
    ),
    "joblib_version": (
        package_version_or_na(
            "joblib"
        )
    ),
}


# ------------------------------------------------------------
# 21. Frozen configuration record
# ------------------------------------------------------------

frozen_configuration = {
    "study_component": (
        "Billing-only archetype-transfer "
        "research prototype"
    ),
    "matched_households": (
        EXPECTED_MATCHED_HOUSEHOLDS
    ),
    "target_type": (
        "K-Means-derived pseudo-labels"
    ),
    "ordered_cluster_names": list(
        ORDERED_CLUSTER_NAMES
    ),
    "class_counts": (
        EXPECTED_CLASS_COUNTS
    ),
    "primary_evaluation": (
        "Step 4 nested out-of-fold "
        "adaptive workflow-selection procedure"
    ),
    "final_model_family": (
        FINAL_WORKFLOW_MODEL_FAMILY
    ),
    "final_feature_set": (
        FINAL_WORKFLOW_FEATURE_SET
    ),
    "final_predictor_count": (
        len(
            final_features
        )
    ),
    "final_predictors": (
        final_features
    ),
    "final_parameters": (
        FINAL_WORKFLOW_PARAMS
    ),
    "external_validation_available": False,
    "operational_deployment_ready": False,
    "important_interpretation": (
        "The fixed full-data research prototype "
        "does not have an independent unbiased "
        "performance estimate. Step 4 remains "
        "the primary internal performance estimate."
    ),
    "hashes": {
        "feature_order_sha256": (
            feature_order_sha256
        ),
        "household_population_sha256": (
            household_population_sha256
        ),
        "class_mapping_sha256": (
            class_mapping_sha256
        ),
        "workflow_sha256": (
            workflow_sha256
        ),
        "saved_model_sha256": (
            model_sha256
        ),
    },
    "environment": (
        environment_record
    ),
}


# ------------------------------------------------------------
# 22. Compile QA table
# ------------------------------------------------------------

final_frozen_qa_df = (
    pd.DataFrame(
        qa_records
    )
)

failed_checks = (
    final_frozen_qa_df[
        final_frozen_qa_df[
            "status"
        ]
        != "PASS"
    ]
)


# ------------------------------------------------------------
# 23. Save all audit outputs
# ------------------------------------------------------------

final_frozen_qa_df.to_csv(
    STEP22_OUTPUT_DIR
    / "FINAL_FROZEN_PIPELINE_QA.csv",
    index=False,
)

with open(
    STEP22_OUTPUT_DIR
    / "FINAL_FROZEN_CONFIGURATION.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        frozen_configuration,
        handle,
        indent=4,
        default=str,
    )

with open(
    STEP22_OUTPUT_DIR
    / "ENVIRONMENT_VERSIONS.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        environment_record,
        handle,
        indent=4,
    )

hash_manifest_df = pd.DataFrame([
    {
        "item": (
            "Final predictor order"
        ),
        "sha256": (
            feature_order_sha256
        ),
    },
    {
        "item": (
            "Matched household population"
        ),
        "sha256": (
            household_population_sha256
        ),
    },
    {
        "item": (
            "Class mapping"
        ),
        "sha256": (
            class_mapping_sha256
        ),
    },
    {
        "item": (
            "Frozen workflow configuration"
        ),
        "sha256": (
            workflow_sha256
        ),
    },
    {
        "item": (
            "Saved model package"
        ),
        "sha256": (
            model_sha256
        ),
    },
])

hash_manifest_df.to_csv(
    STEP22_OUTPUT_DIR
    / "REPRODUCIBILITY_HASHES.csv",
    index=False,
)


# ------------------------------------------------------------
# 24. Final result
# ------------------------------------------------------------

if not failed_checks.empty:
    raise RuntimeError(
        "FINAL FROZEN QA FAILED. "
        "Do not ignore failed checks."
    )

print("=" * 88)
print("STEP 22 — FINAL FROZEN-PIPELINE QA")
print("=" * 88)

print(
    f"Checks passed: "
    f"{len(final_frozen_qa_df)}/"
    f"{len(final_frozen_qa_df)}"
)

print(
    "Matched households:",
    EXPECTED_MATCHED_HOUSEHOLDS,
)

print(
    "Final model family:",
    FINAL_WORKFLOW_MODEL_FAMILY,
)

print(
    "Final feature set:",
    FINAL_WORKFLOW_FEATURE_SET,
)

print(
    "Final predictor count:",
    len(
        final_features
    ),
)

print(
    "Saved model SHA-256:",
    model_sha256,
)

print(
    "\nPASS: final package reload produced identical "
    "smoke-test predictions."
)

print(
    "\nPASS: Step 4 remains the primary internal "
    "performance estimate."
)

print(
    "\nPASS: no new training, tuning, or model "
    "selection was performed in Step 22."
)

print(
    "\nFinal QA table:"
)

display(
    final_frozen_qa_df
)

print(
    "\nReproducibility hashes:"
)

display(
    hash_manifest_df
)

print(
    "\nSaved Step 22 outputs to:"
)

print(
    STEP22_OUTPUT_DIR.resolve()
)